# Entorno de trabajo

In [1]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb:///

In [2]:
%pip install lightgbm optuna

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
import lightgbm as lgb
import optuna


/Users/manuelmarasco/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import numpy as np
import pandas as pd
from datetime import date

from sklearn.metrics import roc_auc_score
pd.set_option('display.max_rows', 200)

In [5]:
semillas = [
    100129,
    100183,
    100151,
    500041,
    900007
]

In [6]:
# Habilitar SQL dentro del notebook
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [7]:
dataset_path = (
    "/Users/manuelmarasco/Library/CloudStorage/"
    "GoogleDrive-hola@manuelmarasco.com/"
    "Mi unidad/DMEyF/2026/notebooks/data/"
)

dataset_filename = "competencia_01_crudo.csv"

# Carga de archivo

In [8]:
%%sql

CREATE OR REPLACE TABLE competencia_01_crudo AS

SELECT *
FROM read_csv_auto(
    '{{dataset_path + dataset_filename}}',
    sample_size = -1
);

,Success


In [9]:
%%sql

SELECT *
FROM competencia_01_crudo
LIMIT 5;

,numero_de_cliente,foto_mes,active_quarter,cliente_vip,internet,cliente_edad,cliente_antiguedad,mrentabilidad,mrentabilidad_annual,mcomisiones,...,Visa_madelantodolares,Visa_fultimo_cierre,Visa_mpagado,Visa_mpagospesos,Visa_mpagosdolares,Visa_fechaalta,Visa_mconsumototal,Visa_cconsumos,Visa_cadelantosefectivo,Visa_mpagominimo
0,12159854.0,202103,1,0,0,56,134,688.41,26701.84,98.82,...,0.0,1,0.0,-16247.77,0.0,4056,15732.34,1,0,1137.81
1,12159858.0,202103,1,0,0,48,102,78.43,24418.75,-73.62,...,NaN,<NA>,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>,NaN
2,12160484.0,202103,1,0,0,60,55,8101.55,3162.23,13399.50,...,0.0,1,0.0,-31103.23,0.0,1632,2860.54,2,0,19858.89
3,12160591.0,202103,1,0,0,46,275,14825.78,138050.05,1146.27,...,0.0,1,0.0,-13733.44,0.0,2122,1419.36,3,0,1231.65
4,12160747.0,202103,1,0,0,47,194,2015.61,31240.49,1791.25,...,0.0,1,0.0,0.00,0.0,5901,1286.93,1,0,82.11


## Verificación de Granularidad 
Granularidad = Que no se repita la clave. Unicidad de la clave — y el par (numero_de_cliente, foto_mes) se llama clave primaria compuesta (compuesta porque son dos columnas, no una). Cuando se cumple, se dice que la tabla respeta su grano
¿Qué representa exactamente una fila de la tabla?
Verificar la granularidad = comprobar que el par (numero_de_cliente, foto_mes) no se repite nunca. Si un cliente apareciera dos veces en el mismo mes, todo lo que viene después se rompe: por ejemplo lead() y lag() asumen una fila por cliente por mes. Con duplicados, lead(mes_0, 1) no te devuelve "el mes siguiente", te devuelve "la otra fila del mismo mes", y el target sale mal sin tirar ningún error.

In [10]:
%%sql
select foto_mes, count(*) as cantidad
from competencia_01_crudo
group by foto_mes

,foto_mes,cantidad
0,202103,162900
1,202104,163284
2,202105,163768
3,202106,164114
4,202107,164348
5,202108,164647


In [11]:
suma_meses = 162900+163284+163768+164114+164348+164647
print(suma_meses)

983061


In [12]:
%%sql
select count(*) from competencia_01_crudo

,count_star()
0,983061


In [13]:
%%sql
select numero_de_cliente, foto_mes, count(*) as n
from competencia_01_crudo
group by numero_de_cliente, foto_mes
having count(*) > 1

,numero_de_cliente,foto_mes,n


Cómo se lee: se agrupa por la clave y pido solo los grupos que tienen más de una fila. Si no hay repetidos esto devuelve cero filas. Si devuelve algo, cada fila es un duplicado y hay un problema antes de empezar.

# Construcción del target 

In [14]:
%%sql

CREATE OR REPLACE TABLE competencia_01 AS

WITH periodos AS (
    SELECT DISTINCT foto_mes FROM competencia_01_crudo
),

clientes AS (
    SELECT DISTINCT numero_de_cliente FROM competencia_01_crudo
),

todo AS (
    SELECT numero_de_cliente, foto_mes
    FROM clientes CROSS JOIN periodos
),

presencia AS (
    SELECT
        c.* EXCLUDE (numero_de_cliente, foto_mes),
        t.numero_de_cliente,
        t.foto_mes,
        CASE WHEN c.numero_de_cliente IS NULL THEN 0 ELSE 1 END AS mes_0
    FROM todo AS t
    LEFT JOIN competencia_01_crudo AS c
        ON  c.numero_de_cliente = t.numero_de_cliente
        AND c.foto_mes          = t.foto_mes
),

etiquetado AS (
    SELECT
        *,
        LEAD(mes_0, 1) OVER w AS m1,
        LEAD(mes_0, 2) OVER w AS m2
    FROM presencia
    WINDOW w AS (PARTITION BY numero_de_cliente ORDER BY foto_mes)
)

SELECT
    * EXCLUDE (mes_0, m1, m2),
    CASE
        WHEN m1 = 0            THEN 'BAJA+1'
        WHEN m1 = 1 AND m2 = 0 THEN 'BAJA+2'
        WHEN m1 = 1 AND m2 = 1 THEN 'CONTINUA'
    END AS clase_ternaria
FROM etiquetado
WHERE mes_0 = 1
ORDER BY foto_mes, numero_de_cliente;

,Success


In [15]:
%%sql
PIVOT competencia_01
on clase_ternaria
USING count(numero_de_cliente)
GROUP BY foto_mes

,foto_mes,BAJA+1,BAJA+2,CONTINUA
0,202103,1019,960,160921
1,202104,964,1139,161181
2,202105,1143,870,161755
3,202106,874,1098,162142
4,202107,1103,0,0
5,202108,0,0,0


In [16]:
%%sql
SELECT foto_mes,
       COUNT(*) AS n,
       COUNT(*) FILTER (clase_ternaria IS NULL)       AS sin_clase,
       COUNT(*) FILTER (clase_ternaria = 'BAJA+1')    AS baja1,
       COUNT(*) FILTER (clase_ternaria = 'BAJA+2')    AS baja2,
       COUNT(*) FILTER (clase_ternaria = 'CONTINUA')  AS continua
FROM competencia_01
GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,n,sin_clase,baja1,baja2,continua
0,202103,162900,0,1019,960,160921
1,202104,163284,0,964,1139,161181
2,202105,163768,0,1143,870,161755
3,202106,164114,0,874,1098,162142
4,202107,164348,163245,1103,0,0
5,202108,164647,164647,0,0,0


Julio de 2021 tiene etiquetas incompletas: sólo puede identificarse `BAJA+1` porque existe agosto, pero no puede determinarse `BAJA+2` ni `CONTINUA` sin observar septiembre. Agosto queda completamente sin etiqueta. Por eso ninguno de los dos meses debe utilizarse como conjunto supervisado de entrenamiento o validación. Julio sí puede aportar historia para construir variables de agosto, y agosto es el mes que se deberá predecir.

# Función de ganancia

In [17]:
GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   =    27_500
BREAK_EVEN       = COSTO_ESTIMULO / (GANANCIA_ACIERTO + COSTO_ESTIMULO)  # 0.025


def ganancia_por_fila(clase_ternaria):
    """Ganancia de estimular cada cliente. Se construye SIEMPRE desde la
    ternaria, nunca desde la binaria con la que se entrena."""
    return np.where(
        np.asarray(clase_ternaria) == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO
    )


def _validar_alineacion(y_prob, gan_fila):
    y_prob = np.asarray(y_prob)
    gan_fila = np.asarray(gan_fila)
    if len(y_prob) != len(gan_fila):
        raise ValueError(
            f"Vectores desalineados: probabilidades={len(y_prob)}, "
            f"ganancias={len(gan_fila)}"
        )
    return y_prob, gan_fila


def ganancia(y_prob, gan_fila, threshold=BREAK_EVEN):
    """Ganancia con umbral fijo sobre un conjunto completo y sin submuestreo."""
    y_prob, gan_fila = _validar_alineacion(y_prob, gan_fila)
    return gan_fila[y_prob >= threshold].sum()


def ganancia_maxima(y_prob, gan_fila):
    """Busca el corte óptimo dentro de validación, incluyendo la opción
    de no estimular a nadie (ganancia 0, k=0). No debe reportarse como
    estimación insesgada sobre el mismo conjunto donde se eligió el corte."""
    y_prob, gan_fila = _validar_alineacion(y_prob, gan_fila)
    orden = np.argsort(-y_prob)
    acum = np.concatenate(([0], np.cumsum(gan_fila[orden])))
    k = int(np.argmax(acum))
    return acum[k], k


def feval_ganancia(gan_valid):
    """Fábrica de feval para un valid_set concreto de lgb.train.

    `gan_valid` debe estar alineado fila a fila con ese valid_set. Para CV se
    debe construir un feval distinto dentro de cada fold.
    """
    gan_valid = np.asarray(gan_valid)

    def _feval(y_prob, data_lgb):
        if data_lgb.num_data() != len(gan_valid):
            raise ValueError(
                f"Validación desalineada: dataset={data_lgb.num_data()}, "
                f"ganancias={len(gan_valid)}"
            )
        return "ganancia", ganancia(y_prob, gan_valid), True

    return _feval

# Explicación de la fc Ganancia

### Vamos por partes

Sos el banco. Cada mes se te van clientes. Tenés un modelo que trata de adivinar quiénes se van a ir.

Podés mandarle un **estímulo** a quien quieras (una oferta, un llamado, lo que sea para que se queden). 

- Según la función de evaluación, estimular correctamente a un `BAJA+2` aporta **+$1.072.500**.
- Estimular a cualquier otro cliente aporta **−$27.500**.
- Esos importes son directamente los valores netos que usa la competencia; no hay que volver a restar el costo al premio positivo.

La función de ganancia hace exactamente eso: mira a quiénes decidiste estimular, suma $1.072.500 por cada `BAJA+2` y resta $27.500 por cada caso restante. Un solo número al final. Ese número es tu puntaje en la competencia.

**Ahora el dato que explica todo lo demás.** Dividí los dos premios:

```
1.072.500 / 27.500 = 39
```

Un acierto paga 39 errores. O dicho al revés: de cada **40** personas que estimulás, con que **1** sea un `BAJA+2` real ya empatás. Y 1 de cada 40 es **0,025** — ese es el famoso umbral.

Esto es contraintuitivo y es la clave de la competencia: solo ~0,6% de tus clientes son `BAJA+2`, pero igual te conviene estimular a mucha más gente que eso, porque errarle es barato y pegarle es carísimo. No estás tratando de adivinar con precisión. Estás tratando de pescar con una red ancha porque el pescado vale 39 veces la red.

## 2. El "blindaje" 

El modelo no entiende las palabras `CONTINUA`, `BAJA+1`, `BAJA+2`. Necesita una columna de 0 y 1. Así que tenés que elegir cómo traducir, y hay dos formas razonables:

| | `BAJA+2` | `BAJA+1` | `CONTINUA` |
|---|---|---|---|
| **Opción A** | 1 | 0 | 0 |
| **Opción B** | 1 | 1 | 0 |

La B tienta porque te da casi el doble de unos para aprender, y los `BAJA+1` se parecen mucho a los `BAJA+2`, que al fin y al cabo también se están yendo. 

**Y acá está la bomba.** La función de clase  dice `y_true == 1` para decidir a quién le paga el $1.072.500. Si le pasás la columna de la opción B, la función le paga el premio **a todos los `BAJA+1` también**. Y los `BAJA+1` no valen nada: ese cliente ya decidió irse, el mes que viene no está, no hay nada que retener. Vale –$27.500 igual que un `CONTINUA`.

¿Qué vería? Tu ganancia casi duplicada. Ningún error, ninguna advertencia, ningún número raro. Pensás que encontraste oro y te estrellás sin entender por qué. Es el peor tipo de bug que existe: **el que no falla, sino que miente y encima te felicita.**

El blindaje es una regla de una línea para que no pueda pasar nunca:

> La ganancia se calcula **siempre desde la columna ternaria** (la de las palabras). La binaria es solo comida del modelo.

Construís el vector de plata una vez, al principio, desde la ternaria, y lo dejás ahí al lado del dataset. Después podés entrenar con la opción A, con la B, o con quince variantes más, y la ganancia no se entera ni le importa. Los dos caminos nunca se cruzan. Por eso en el código la función se llama `ganancia_por_fila(clase_ternaria)` y recibe la ternaria explícitamente: para que sea imposible pasarle la otra por accidente.

## 3. El "arreglo 2"

Tu modelo no te dice "este se va". Te dice un número entre 0 y 1: `0.08`, que querría decir "8% de probabilidad".

Pero vos no podés mandarle un 8% de estímulo a nadie. Tenés que decidir: **le mando o no le mando**. Hay dos formas de pasar del número a la decisión.

### Forma 1: umbral

Le mandás a todos los que están por encima de 0,025, que como vimos es el punto de empate.

**El problema:** esto funciona solo si el `0.08` del modelo significa de verdad 8 de cada 100. Y casi nunca significa eso. Como los positivos son rarísimos (0,6%), todo el mundo entrena con pesos o rebalanceo para que el modelo les preste atención — y eso le deforma las probabilidades de manera sistemática. El modelo queda bueno para *ordenar* pero mentiroso para *cuantificar*. Entonces 0,025 deja de ser el punto de empate **de tu modelo en particular**, y lo peor es que no sabés para qué lado se te corrió.

### Forma 2: la curva de ganancia acumulada

No le creés los números absolutos. Le creés **solamente el orden**.

Ordenás a todos de más probable a menos probable y vas preguntando: ¿y si le mando solo al primero? ¿a los dos primeros? ¿a los mil primeros? Calculás la ganancia de cada escenario. Con 6 clientes se ve perfecto:

| # | Realidad | Plata de esta fila | Ganancia acumulada |
|---|---|---|---|
| 1 | `BAJA+2` | +1.072.500 | **1.072.500** |
| 2 | `CONTINUA` | –27.500 | 1.045.000 |
| 3 | `BAJA+2` | +1.072.500 | **2.117.500** ← máximo |
| 4 | `CONTINUA` | –27.500 | 2.090.000 |
| 5 | `CONTINUA` | –27.500 | 2.062.500 |
| 6 | `CONTINUA` | –27.500 | 2.035.000 |

La curva sube mientras aparecen aciertos, llega a un pico, y después baja porque seguís gastando sin pegarle a nadie. **El pico te da las dos cosas que necesitás: cuánta plata podés hacer, y a cuántos mandarles.** Acá: $2.117.500, mandando a 3.

Por qué es más robusto: solo le pedís al modelo que **ordene bien**, no que sea honesto con las magnitudes. Ordenar es un pedido mucho más fácil de cumplir. Y además la prevalencia de `BAJA+2` se mueve ~25% entre meses según la paridad, y que 202108 cae en la paridad alta: un umbral fijo en probabilidad te agarra una cantidad distinta de gente según el mes, mientras que "los mejores N" no se te mueve.

### ¿Y entonces para qué queda el umbral?

El 0,025 queda como **referencia económica** si las probabilidades están bien calibradas. No conviene usarlo automáticamente para comparar modelos cuando cambian los pesos, el rebalanceo o el algoritmo, porque esas decisiones pueden cambiar la escala de las probabilidades.

Para comparar modelos conviene usar el ranking y la ganancia por cantidad de envíos con validación temporal. La cantidad de envíos se elige en un mes anterior y se aplica, sin volver a optimizarla, en un mes posterior.

**Resumen:** 0,025 es una referencia económica para probabilidades calibradas; el ranking y la validación temporal sirven para elegir el modelo y el volumen de envíos.

# Data Quality

## Verifico ceros y nulos 

In [18]:
data = %sql select * from competencia_01

In [19]:
COLUMNAS_EXCLUIDAS = {"numero_de_cliente", "foto_mes", "clase_ternaria"}
features = [c for c in data.columns if c not in COLUMNAS_EXCLUIDAS]

es_nulo = data[features].isna()
es_cero = data[features].eq(0)

# Nulos y ceros sobre el total de filas de cada mes.
pct_nulos = (
    es_nulo.groupby(data["foto_mes"]).mean().mul(100).T.astype(float).round(2)
)
pct_ceros = (
    es_cero.fillna(False).astype(bool)
    .groupby(data["foto_mes"]).mean().mul(100).T.astype(float).round(2)
)

# Ceros entre valores observados: numerador explícito / cantidad no nula.
# Así el cálculo no depende de cómo cada dtype representa los NaN.
cantidad_ceros = (
    es_cero.fillna(False).astype(bool)
    .groupby(data["foto_mes"]).sum().T
)
cantidad_observados = (
    (~es_nulo).groupby(data["foto_mes"]).sum().T
)
pct_ceros_observados = (
    cantidad_ceros
    .div(cantidad_observados.replace(0, np.nan))
    .mul(100).astype(float).round(2)
)

assert (pct_nulos + pct_ceros <= 100.0001).all().all()
assert (
    pct_ceros_observados.isna()
    | ((pct_ceros_observados >= 0) & (pct_ceros_observados <= 100))
).all().all()

In [20]:
def se_mueve(tabla, minimo_pp=0.10):
    """Muestra variables cuyo rango temporal supera `minimo_pp`.

    Una variación pequeña no implica una rotura: puede ser variación muestral.
    """
    rango = tabla.max(axis=1) - tabla.min(axis=1)
    return tabla.loc[rango > minimo_pp]

display(se_mueve(pct_nulos))


foto_mes,202103,202104,202105,202106,202107,202108
mtarjeta_visa_descuentos,2.26,1.71,2.24,1.87,2.68,1.88
mtarjeta_master_descuentos,2.94,3.24,2.96,2.60,2.63,3.43
Master_delinquency,10.34,10.30,10.23,10.11,9.69,10.15
Master_status,10.34,10.30,10.23,10.11,9.69,10.15
Master_mfinanciacion_limite,10.34,10.30,10.23,10.11,9.69,10.15
Master_Fvencimiento,10.34,10.30,10.23,10.11,9.69,10.15
Master_Finiciomora,99.31,99.40,99.50,99.60,99.58,99.66
Master_msaldototal,10.34,10.30,10.23,10.11,9.69,10.15
Master_msaldopesos,10.34,10.30,10.23,10.11,9.69,10.15
Master_msaldodolares,10.34,10.30,10.23,10.11,9.69,10.15


In [21]:
display(se_mueve(pct_ceros))

foto_mes,202103,202104,202105,202106,202107,202108
active_quarter,1.43,1.31,1.27,1.23,1.25,1.34
cliente_vip,99.74,99.74,99.73,99.44,99.73,99.72
internet,95.96,95.43,95.57,96.31,97.41,96.82
mcomisiones,2.36,2.57,2.69,2.57,2.91,2.94
mactivos_margen,4.37,3.99,3.96,4.35,4.32,3.91
mcuenta_corriente,48.73,47.12,48.21,49.64,52.51,49.16
mcaja_ahorro,6.56,7.29,7.15,6.70,6.23,6.13
mcaja_ahorro_dolares,44.24,44.63,44.94,44.95,44.47,44.33
cdescubierto_preacordado,3.58,3.55,3.59,3.58,4.31,4.46
ctarjeta_debito_transacciones,29.02,29.70,30.07,29.14,28.01,28.05


In [22]:
print(pct_nulos.shape, se_mueve(pct_nulos).shape)
print(pct_ceros.shape, se_mueve(pct_ceros).shape)

(152, 6) (46, 6)
(152, 6) (115, 6)


In [23]:
display(pct_nulos[pct_nulos.min(axis=1) > 99])
display(pct_ceros[pct_ceros.min(axis=1) > 99])

foto_mes,202103,202104,202105,202106,202107,202108
Master_Finiciomora,99.31,99.4,99.5,99.6,99.58,99.66


foto_mes,202103,202104,202105,202106,202107,202108
cliente_vip,99.74,99.74,99.73,99.44,99.73,99.72
mcuenta_corriente_adicional,99.95,99.95,99.95,99.95,99.95,99.95
cprestamos_prendarios,99.10,99.05,99.06,99.08,99.08,99.10
mprestamos_prendarios,99.11,99.16,99.07,99.09,99.09,99.10
mplazo_fijo_pesos,99.58,99.58,99.57,99.56,99.55,99.53
minversion1_dolares,99.66,99.66,99.67,99.67,99.68,99.68
mpayroll2,99.90,99.91,99.91,99.88,99.95,99.94
cpayroll2_trx,99.90,99.91,99.91,99.88,99.95,99.94
cforex_buy,99.45,99.60,99.66,99.71,99.76,99.73
mforex_buy,99.45,99.60,99.66,99.71,99.76,99.73


In [24]:
(pct_ceros.max(axis=1) - pct_ceros.min(axis=1)).sort_values(ascending=False).head(30)

Master_fultimo_cierre                 69.09
Visa_fultimo_cierre                   69.04
Visa_mpagado                          17.73
mttarjeta_visa_debitos_automaticos     7.98
ctarjeta_visa_debitos_automaticos      7.98
cmobile_app_trx                        7.77
mcuenta_corriente                      5.39
mforex_sell                            4.25
cforex_sell                            4.25
cforex                                 4.14
ccajas_depositos                       3.98
matm_other                             3.96
catm_trx_other                         3.96
ccomisiones_mantenimiento              3.95
mcomisiones_mantenimiento              3.84
mtarjeta_master_consumo                3.49
ctarjeta_master_transacciones          3.49
ccallcenter_transacciones              3.33
tcallcenter                            3.33
mtransferencias_recibidas              3.23
ctransferencias_recibidas              3.23
ccajeros_propios_descuentos            3.15
mtransferencias_emitidas        

In [25]:
pct_ceros.loc[['Visa_Finiciomora', 'Master_fultimo_cierre',
               'Visa_fultimo_cierre', 'Visa_mpagado']].round(2)

foto_mes,202103,202104,202105,202106,202107,202108
Visa_Finiciomora,0.00,0.00,1.97,0.00,0.00,1.97
Master_fultimo_cierre,0.00,0.00,0.00,69.09,0.00,0.00
Visa_fultimo_cierre,0.00,0.00,0.00,69.04,0.00,0.00
Visa_mpagado,86.68,86.89,69.68,85.35,86.42,69.16


In [26]:
pct_nulos = pct_nulos.astype(float)
pct_ceros = pct_ceros.astype(float)

In [27]:
def resumir_cambios(tabla, umbral_pp=20):
    """Resume cambios respecto del comportamiento mensual típico."""
    mediana = tabla.median(axis=1)
    desvios = tabla.sub(mediana, axis=0).abs()

    resumen = pd.DataFrame({
        "rango_pp":
            (tabla.max(axis=1) - tabla.min(axis=1)).round(2),

        "max_desvio_pp":
            desvios.max(axis=1).round(2),

        "mes_mayor_desvio":
            desvios.idxmax(axis=1),
    })

    resumen["meses_alerta"] = [
        list(tabla.columns[desvios.loc[variable] > umbral_pp])
        for variable in tabla.index
    ]

    resumen["alerta"] = (
        resumen["meses_alerta"]
        .str.len()
        .gt(0)
    )

    return resumen.sort_values(
        "max_desvio_pp",
        ascending=False
    )


# ============================================================
# DETECTOR MEJORADO DE CEROS
# ============================================================

# Cantidad de valores no nulos de cada variable y mes
cantidad_observados = (
    (~es_nulo)
    .groupby(data["foto_mes"])
    .sum()
    .T
)

# Aseguramos que los porcentajes sean numéricos
pct_ceros_observados = pct_ceros_observados.astype(float)

# Comportamiento habitual de cada variable
mediana_ceros_total = pct_ceros.median(axis=1)

mediana_ceros_observados = (
    pct_ceros_observados.median(axis=1)
)

# Diferencia de cada mes respecto de la mediana
desvio_ceros_total = pct_ceros.sub(
    mediana_ceros_total,
    axis=0
).abs()

desvio_ceros_observados = (
    pct_ceros_observados.sub(
        mediana_ceros_observados,
        axis=0
    ).abs()
)

# Detecta cambios importantes sobre el total de clientes
alerta_total = desvio_ceros_total >= 10

# Detecta cambios entre los valores observados
# Solo se considera si existen al menos 500 observaciones
alerta_observados = (
    (desvio_ceros_observados >= 20)
    & (cantidad_observados >= 500)
)

# Alerta conjunta
alertas_ceros = (
    alerta_total
    | alerta_observados
)

# Resumen final
resumen_ceros_mejorado = pd.DataFrame({
    "max_desvio_total_pp":
        desvio_ceros_total.max(axis=1).round(2),

    "max_desvio_observados_pp":
        desvio_ceros_observados
        .where(cantidad_observados >= 500)
        .max(axis=1)
        .round(2),

    "meses_alerta":
        alertas_ceros.apply(
            lambda fila:
                fila.index[fila.fillna(False)].tolist(),
            axis=1
        ),

    "cantidad_meses_alerta":
        alertas_ceros.sum(axis=1)
})

# Mostramos solamente las variables con alertas
resumen_ceros_mejorado = (
    resumen_ceros_mejorado[
        resumen_ceros_mejorado[
            "cantidad_meses_alerta"
        ] > 0
    ]
    .sort_values(
        [
            "cantidad_meses_alerta",
            "max_desvio_total_pp"
        ],
        ascending=[True, False]
    )
)

display(resumen_ceros_mejorado)

,max_desvio_total_pp,max_desvio_observados_pp,meses_alerta,cantidad_meses_alerta
Master_fultimo_cierre,69.09,76.89,[202106],1
Visa_fultimo_cierre,69.04,72.64,[202106],1
Visa_mpagado,16.72,17.44,"[202105, 202108]",2
Visa_Finiciomora,1.97,78.04,"[202105, 202108]",2


In [28]:
resumen_nulos = resumir_cambios(pct_nulos)
resumen_nulos[resumen_nulos["alerta"]]

,rango_pp,max_desvio_pp,mes_mayor_desvio,meses_alerta,alerta


In [29]:
control = pct_nulos + pct_ceros
assert (control <= 100.0001).all().all(), "Nulos y ceros usan denominadores incompatibles"
print("Control aprobado: nulos + ceros no supera 100% en ninguna variable-mes.")

Control aprobado: nulos + ceros no supera 100% en ninguna variable-mes.


### Guardo nulos y ceros

In [30]:
pct_nulos.to_csv('dq_nulos.csv')
pct_ceros.to_csv('dq_ceros.csv')
pct_ceros_observados.to_csv('dq_ceros_sobre_observados.csv')

## PSI 


### Interpretación de los Valores PSI:

- **PSI bajo (< 0,10):** distribución relativamente estable.

- **PSI moderado (0,10 a 0,25):** señal para investigar y monitorear.

- **PSI alto (> 0,25):** cambio importante de distribución.

Estos umbrales son orientativos. El PSI no prueba que una variable esté rota: el cambio puede responder a estacionalidad, inflación o una transformación real del comportamiento. Las fechas se excluyen del PSI numérico crudo y la tasa de nulos se informa por separado.

In [31]:
def _psi_desde_conteos(
    expected_counts,
    actual_counts,
    alpha=0.5
):
    """
    Calcula PSI con suavizado de Laplace
    sobre bins que ya fueron definidos.
    """
    expected_counts = np.asarray(
        expected_counts,
        dtype=float
    )

    actual_counts = np.asarray(
        actual_counts,
        dtype=float
    )

    k = len(expected_counts)

    if (
        k == 0
        or expected_counts.sum() == 0
        or actual_counts.sum() == 0
    ):
        return np.nan

    expected_prop = (
        expected_counts + alpha
    ) / (
        expected_counts.sum() + alpha * k
    )

    actual_prop = (
        actual_counts + alpha
    ) / (
        actual_counts.sum() + alpha * k
    )

    return float(
        np.sum(
            (actual_prop - expected_prop)
            * np.log(actual_prop / expected_prop)
        )
    )


def psi(
    expected,
    actual,
    buckets=10,
    categorical_max_unique=20
):
    """
    Calcula PSI usando:

    - Variables categóricas como categorías.
    - Un bin exclusivo para ceros.
    - Bins por cuantiles para valores distintos de cero.
    - Los nulos se analizan por separado.
    """

    expected = pd.Series(expected).replace(
        [np.inf, -np.inf],
        np.nan
    )

    actual = pd.Series(actual).replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Sacamos los nulos porque se controlan por separado
    expected_obs = expected.dropna()
    actual_obs = actual.dropna()

    if expected_obs.empty or actual_obs.empty:
        return np.nan

    es_numerica = pd.api.types.is_numeric_dtype(
        expected_obs
    )

    baja_cardinalidad = (
        expected_obs.nunique()
        <= categorical_max_unique
    )

    # Variables categóricas o numéricas
    # con pocos valores posibles
    if not es_numerica or baja_cardinalidad:

        expected_cat = expected_obs.astype(str)
        actual_cat = actual_obs.astype(str)

        categorias = sorted(
            set(expected_cat.unique())
            | set(actual_cat.unique())
        )

        expected_counts = (
            expected_cat
            .value_counts()
            .reindex(
                categorias,
                fill_value=0
            )
        )

        actual_counts = (
            actual_cat
            .value_counts()
            .reindex(
                categorias,
                fill_value=0
            )
        )

        return _psi_desde_conteos(
            expected_counts,
            actual_counts
        )

    # ========================================================
    # VARIABLES NUMÉRICAS CONTINUAS
    # ========================================================

    # Los ceros forman un bin propio
    expected_es_cero = expected_obs.eq(0)
    actual_es_cero = actual_obs.eq(0)

    cantidad_ceros_expected = (
        expected_es_cero.sum()
    )

    cantidad_ceros_actual = (
        actual_es_cero.sum()
    )

    # Valores distintos de cero
    expected_no_cero = (
        expected_obs[~expected_es_cero]
        .astype(float)
    )

    actual_no_cero = (
        actual_obs[~actual_es_cero]
        .astype(float)
    )

    # Si la referencia solamente tiene ceros,
    # comparamos cero contra distinto de cero
    if expected_no_cero.empty:

        expected_counts = [
            cantidad_ceros_expected,
            len(expected_no_cero)
        ]

        actual_counts = [
            cantidad_ceros_actual,
            len(actual_no_cero)
        ]

        return _psi_desde_conteos(
            expected_counts,
            actual_counts
        )

    # Los bins se calculan solamente con los valores
    # distintos de cero del período de referencia
    try:
        _, edges = pd.qcut(
            expected_no_cero,
            q=min(
                buckets,
                expected_no_cero.nunique()
            ),
            duplicates="drop",
            retbins=True
        )

    except ValueError:

        # Si no se pueden construir cuantiles,
        # usamos cero contra distinto de cero
        expected_counts = [
            cantidad_ceros_expected,
            len(expected_no_cero)
        ]

        actual_counts = [
            cantidad_ceros_actual,
            len(actual_no_cero)
        ]

        return _psi_desde_conteos(
            expected_counts,
            actual_counts
        )

    edges = np.unique(
        np.asarray(edges, dtype=float)
    )

    # Con dos límites todavía podemos formar
    # un bin para los valores distintos de cero
    if len(edges) < 2:

        expected_counts = [
            cantidad_ceros_expected,
            len(expected_no_cero)
        ]

        actual_counts = [
            cantidad_ceros_actual,
            len(actual_no_cero)
        ]

        return _psi_desde_conteos(
            expected_counts,
            actual_counts
        )

    # Incluimos valores futuros fuera del rango original
    edges[0] = -np.inf
    edges[-1] = np.inf

    expected_no_cero_counts, _ = np.histogram(
        expected_no_cero,
        bins=edges
    )

    actual_no_cero_counts, _ = np.histogram(
        actual_no_cero,
        bins=edges
    )

    # Primer bin: cero.
    # Los siguientes: cuantiles de valores no cero.
    expected_counts = np.concatenate([
        [cantidad_ceros_expected],
        expected_no_cero_counts
    ])

    actual_counts = np.concatenate([
        [cantidad_ceros_actual],
        actual_no_cero_counts
    ])

    return _psi_desde_conteos(
        expected_counts,
        actual_counts
    )


def delta_nulos(expected, actual):
    """
    Cambio en la tasa de nulos,
    expresado en puntos porcentuales.
    """
    return 100 * (
        pd.Series(actual).isna().mean()
        - pd.Series(expected).isna().mean()
    )


# Las fechas se analizan mediante calendario,
# no mediante el PSI numérico general
COLUMNAS_FECHA = [
    columna
    for columna in features
    if columna.lower().endswith((
        "fvencimiento",
        "finiciomora",
        "fultimo_cierre",
        "fechaalta"
    ))
]

features_psi = [
    columna
    for columna in features
    if columna not in COLUMNAS_FECHA
]

print(
    "Variables excluidas del PSI por ser fechas:",
    COLUMNAS_FECHA
)

Variables excluidas del PSI por ser fechas: ['Master_Fvencimiento', 'Master_Finiciomora', 'Master_fultimo_cierre', 'Master_fechaalta', 'Visa_Fvencimiento', 'Visa_Finiciomora', 'Visa_fultimo_cierre', 'Visa_fechaalta']


La implementación de PSI queda definida una sola vez en la celda anterior.

## Todos contra marzo

Razón: el drift se acumula. Si cada mes se corre un poquito, mes contra mes te da siempre valores chicos y no ves nada. Marzo contra agosto te muestra todo el corrimiento junto, que es el que le va a pegar al modelo.

In [32]:
mes_ref = 202103
meses = [202104, 202105, 202106, 202107, 202108]

ref = data[data.foto_mes == mes_ref]

resultados_psi = {}
resultados_nulos = {}
for mes in meses:
    actual = data[data.foto_mes == mes]
    resultados_psi[mes] = {c: psi(ref[c], actual[c]) for c in features_psi}
    resultados_nulos[mes] = {c: delta_nulos(ref[c], actual[c]) for c in features_psi}

psi_tabla = pd.DataFrame(resultados_psi).round(3)
delta_nulos_tabla = pd.DataFrame(resultados_nulos).round(2)


def nivel(v):
    if pd.isna(v): return "NO_CALCULABLE"
    if v > 0.25: return "ALTO"
    if v > 0.10: return "MEDIO"
    return "BAJO"

psi_resumen = psi_tabla.copy()
psi_resumen["psi_max"] = psi_tabla.max(axis=1).round(3)
psi_resumen["mes_psi_max"] = psi_tabla.idxmax(axis=1)
psi_resumen["nivel"] = psi_resumen["psi_max"].apply(nivel)
psi_resumen["max_delta_nulos_pp"] = delta_nulos_tabla.abs().max(axis=1).round(2)

display(psi_resumen.sort_values("psi_max", ascending=False).head(25))
psi_resumen.to_csv("dq_psi_vs_202103.csv")

,202104,202105,202106,202107,202108,psi_max,mes_psi_max,nivel,max_delta_nulos_pp
ccajas_depositos,0.032,0.358,0.076,0.002,0.001,0.358,202105,ALTO,0.00
Visa_mpagado,0.000,0.235,0.005,0.002,0.251,0.251,202108,ALTO,0.45
mpayroll,0.002,0.014,0.249,0.046,0.056,0.249,202106,MEDIO,0.00
Visa_mconsumosdolares,0.047,0.114,0.119,0.177,0.216,0.216,202108,MEDIO,1.17
mcomisiones_mantenimiento,0.008,0.012,0.112,0.208,0.203,0.208,202107,MEDIO,0.00
cpayroll_trx,0.005,0.003,0.178,0.001,0.004,0.178,202106,MEDIO,0.00
ctarjeta_visa_debitos_automaticos,0.066,0.067,0.074,0.094,0.096,0.096,202108,BAJO,0.00
mcomisiones_otras,0.034,0.046,0.048,0.065,0.082,0.082,202108,BAJO,0.00
mcomisiones,0.035,0.047,0.050,0.069,0.081,0.081,202108,BAJO,0.00
mttarjeta_visa_debitos_automaticos,0.038,0.043,0.030,0.062,0.071,0.071,202108,BAJO,0.00


In [33]:
comparaciones_psi = [
    ("train_202103_vs_score_202105", [202103], 202105),
    ("train_202103_202104_vs_score_202106", [202103, 202104], 202106),
    ("train_final_vs_score_202108", [202103, 202104, 202105, 202106], 202108),
]

resultados_comparaciones = {}
for nombre, train, test in comparaciones_psi:
    ref = data[data.foto_mes.isin(train)]
    actual = data[data.foto_mes == test]
    resultados_comparaciones[nombre] = {
        c: psi(ref[c], actual[c]) for c in features_psi
    }

psi_comparaciones = pd.DataFrame(resultados_comparaciones).round(3)
psi_comparaciones["psi_max"] = psi_comparaciones.max(axis=1).round(3)
psi_comparaciones["nivel"] = psi_comparaciones["psi_max"].apply(nivel)
display(psi_comparaciones.sort_values("psi_max", ascending=False).head(25))

,train_202103_vs_score_202105,train_202103_202104_vs_score_202106,train_final_vs_score_202108,psi_max,nivel
ccajas_depositos,0.358,0.038,0.019,0.358,ALTO
mpayroll,0.014,0.236,0.017,0.236,MEDIO
Visa_mpagado,0.235,0.005,0.121,0.235,MEDIO
cpayroll_trx,0.003,0.209,0.018,0.209,MEDIO
mcomisiones_mantenimiento,0.012,0.119,0.126,0.126,MEDIO
Visa_mconsumosdolares,0.114,0.088,0.045,0.114,MEDIO
ctarjeta_visa_debitos_automaticos,0.067,0.019,0.012,0.067,BAJO
mcomisiones_otras,0.046,0.017,0.065,0.065,BAJO
mcomisiones,0.047,0.017,0.064,0.064,BAJO
mrentabilidad,0.008,0.009,0.057,0.057,BAJO


## Análisis 

Los porcentajes de nulos y ceros se calculan sobre el total de filas de cada mes, de modo que son comparables y nunca pueden sumar más de 100%. El porcentaje de ceros entre valores observados se conserva en una tabla separada.

Las variables de fecha se excluyen del PSI crudo porque el calendario cambia naturalmente entre meses. Deben estudiarse mediante reglas de validez —por ejemplo, fechas iguales a cero— o transformarse en antigüedad/recencia antes de incorporarlas a un modelo.

El PSI es una señal de cambio, no una orden de imputación. Antes de tratar una variable se debe distinguir una rotura puntual de un cambio económico o estacional.

### Pit stop: qué falta investigar


La regla general es: **no imputar, eliminar ni transformar todavía**. Primero debemos determinar si cada cambio es negocio, ausencia estructural o falla técnica.

### 1. Fechas

Variables:

- `Master/Visa_fultimo_cierre`
- `Master/Visa_Fvencimiento`
- `Master/Visa_Finiciomora`
- `Master/Visa_fechaalta`

Investigar:

- Confirmar la fecha de referencia usada para calcular los enteros.
- Reconstruir las fechas calendario.
- Verificar jueves y cohortes de cierre.
- Seguir al mismo cliente entre meses.
- Comprobar qué significa exactamente `Fvencimiento`.
- Confirmar si `Finiciomora = 0` significa “mora iniciada hoy”.

Decisión probable:

- `fultimo_cierre = 0` puede ser válido.
- Las fechas deben excluirse del detector general de ceros.
- No imputar fechas con media o mediana.

---

### 2. Nulos

Investigar cada nulo junto con:

- Tenencia de tarjeta.
- `status`.
- Límite de compra.
- Saldo.
- Consumos.
- Movimientos anteriores y posteriores del mismo cliente.

Clasificarlo como:

| Tipo de nulo | Ejemplo | Tratamiento probable |
|---|---|---|
| Estructural | No tiene tarjeta | Conservar `NaN` |
| De evento | Nunca estuvo en mora | Conservar `NaN` |
| Operativo | Información aún no procesada | Conservar y posiblemente crear indicador |
| Técnico | Falla masiva de carga | Marcar el período y evaluar excluir/corregir |

Hasta ahora no hay grandes saltos generales de nulos. El problema principal no parece estar allí.

---

### 3. Ceros

Aplicar el detector de ceros solamente a montos, saldos, cantidades y transacciones.

Investigar:

- Si cero significa ausencia de actividad.
- Si debería ser nulo.
- Si aparece concentrado en un único mes.
- Si afecta a los mismos clientes.
- Si otras variables relacionadas siguen informadas.

Casos prioritarios:

#### `ccajas_depositos`

- Mayo tiene 100% de ceros.
- Comparar con `ccajas_transacciones`, extracciones, consultas y otras operaciones.
- Determinar si fue una restricción operativa o una falla de carga.

#### `Visa_mpagado`

- Más valores positivos en mayo y agosto.
- Identificar si son los mismos clientes.
- Cruzar con pagos en pesos/dólares, saldo, consumo y cierre.
- Analizar su relación con la baja en mayo.

#### `Visa_Finiciomora`

- El cero no significa “sin mora”.
- Cruzarlo con `delinquency`, `status` y fecha reconstruida.
- Determinar si cero representa inicio de mora en la fecha de referencia.

---

### 4. PSI medio o alto

Para cada alerta hay que descubrir qué componente la produce:

- Cambio en ceros.
- Cambio en nulos.
- Cambio en valores positivos.
- Inflación.
- Estacionalidad.
- Cambio operativo.
- Rotura técnica.

| Variable | PSI | Investigación |
|---|---|---|
| `ccajas_depositos` | Alto en mayo | Operación de sucursales o falla |
| `Visa_mpagado` | Alto mayo/agosto | Ciclo de pago o carga periódica |
| `mpayroll` | Casi alto en junio | Aguinaldo; prácticamente explicado |
| `cpayroll_trx` | Medio en junio | Segunda acreditación; explicado |
| `Visa_mconsumosdolares` | Medio hacia agosto | Cantidad de usuarios, montos y valores extremos |
| `mcomisiones_mantenimiento` | Medio desde junio/julio | Cambio de tarifas o inflación |
| Débitos automáticos Visa | Cerca de medio | Cambio gradual de uso |

Para la competencia, la comparación principal es entrenamiento final contra agosto:

- `Visa_mpagado`: `0,121`.
- `mcomisiones_mantenimiento`: `0,126`.

Estas dos son las alertas más relevantes para el scoring.

---

### 5. Validación con el target

Una vez entendida cada variable, probar tres versiones del modelo:

1. Variable original.
2. Variable excluida.
3. Variable corregida o acompañada por un indicador.

Comparar:

- Ganancia.
- Estabilidad entre meses.
- Importancia de la variable.
- Resultado en mayo y junio.
- Sensibilidad del modelo al mes elegido.



## 1 Fechas 



### Fecha último cierre

In [34]:
%%sql

SELECT
    name AS variable,
    type AS tipo_de_dato,
    "notnull" AS obligatorio,
    pk AS clave_primaria

FROM pragma_table_info('competencia_01')

WHERE name IN (
    'Master_Fvencimiento',
    'Master_Finiciomora',
    'Master_fultimo_cierre',
    'Master_fechaalta',
    'Visa_Fvencimiento',
    'Visa_Finiciomora',
    'Visa_fultimo_cierre',
    'Visa_fechaalta'
)

ORDER BY name;

,variable,tipo_de_dato,obligatorio,clave_primaria
0,Master_Finiciomora,BIGINT,False,False
1,Master_Fvencimiento,BIGINT,False,False
2,Master_fechaalta,BIGINT,False,False
3,Master_fultimo_cierre,BIGINT,False,False
4,Visa_Finiciomora,BIGINT,False,False
5,Visa_Fvencimiento,BIGINT,False,False
6,Visa_fechaalta,BIGINT,False,False
7,Visa_fultimo_cierre,BIGINT,False,False


In [35]:
%%sql

SELECT
    numero_de_cliente,
    foto_mes,

    Master_Fvencimiento,
    Master_Finiciomora,
    Master_fultimo_cierre,
    Master_fechaalta,

    Visa_Fvencimiento,
    Visa_Finiciomora,
    Visa_fultimo_cierre,
    Visa_fechaalta

FROM competencia_01

ORDER BY
    foto_mes,
    numero_de_cliente

LIMIT 10;

,numero_de_cliente,foto_mes,Master_Fvencimiento,Master_Finiciomora,Master_fultimo_cierre,Master_fechaalta,Visa_Fvencimiento,Visa_Finiciomora,Visa_fultimo_cierre,Visa_fechaalta
0,12159854.0,202103,-2159,<NA>,1,3324,-2159,<NA>,1,4056
1,12159858.0,202103,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,12160484.0,202103,-578,<NA>,1,1632,-1674,<NA>,1,1632
3,12160591.0,202103,-1248,<NA>,1,595,-90,<NA>,1,2122
4,12160747.0,202103,-333,<NA>,1,5901,-1794,<NA>,1,5901
5,12160769.0,202103,<NA>,<NA>,<NA>,<NA>,-121,<NA>,1,3180
6,12160968.0,202103,-486,<NA>,1,2465,-1095,<NA>,1,4388
7,12161135.0,202103,<NA>,<NA>,<NA>,<NA>,-1095,<NA>,1,2206
8,12161416.0,202103,-517,<NA>,1,3503,-517,<NA>,1,3503
9,12161961.0,202103,-455,<NA>,14,7587,-486,<NA>,1,8665


Esta tabla confirma que las columnas no contienen fechas directas, sino cantidades de días relativas a alguna referencia.

In [36]:

%%sql
-- finicio mora debería contrastarla con delinquency y status
SELECT
    numero_de_cliente,
    foto_mes,

    Master_Finiciomora,
    Master_delinquency,
    Master_status,

    Visa_Finiciomora,
    Visa_delinquency,
    Visa_status

FROM competencia_01

WHERE
    Master_Finiciomora IS NOT NULL
    OR Visa_Finiciomora IS NOT NULL

ORDER BY
    foto_mes,
    numero_de_cliente

LIMIT 30;

,numero_de_cliente,foto_mes,Master_Finiciomora,Master_delinquency,Master_status,Visa_Finiciomora,Visa_delinquency,Visa_status
0,12184886.0,202103,<NA>,0,0,17,1,0
1,12188338.0,202103,<NA>,0,0,24,1,0
2,12220337.0,202103,<NA>,<NA>,<NA>,17,1,0
3,12242789.0,202103,16,1,0,<NA>,0,0
4,12272184.0,202103,<NA>,0,0,24,1,0
5,12292736.0,202103,<NA>,0,0,17,1,0
6,12307229.0,202103,58,1,7,24,1,0
7,12316480.0,202103,9,1,0,17,1,0
8,12401810.0,202103,58,1,7,<NA>,0,0
9,12416477.0,202103,<NA>,<NA>,<NA>,17,1,0


Hipótesis: Los cierres de las tarjetas bancarias, en argentina suelen ser los días jueves. Veamos si eso se cumple, de ser así se pueden imputar los datos faltantes. 



In [37]:
%%sql

WITH valores AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_fultimo_cierre AS valor_original
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_fultimo_cierre AS valor_original
    FROM competencia_01
)

SELECT
    foto_mes,
    marca,
    valor_original,
    COUNT(*) AS clientes

FROM valores

WHERE valor_original IS NOT NULL

GROUP BY
    foto_mes,
    marca,
    valor_original

ORDER BY
    foto_mes,
    marca,
    valor_original;

,foto_mes,marca,valor_original,clientes
0,202103,Master,1,112027
1,202103,Master,7,23712
2,202103,Master,14,2860
3,202103,Master,21,7399
4,202103,Master,28,5
5,202103,Master,35,3
6,202103,Master,63,4
7,202103,Master,70,2
8,202103,Master,91,1
9,202103,Master,126,1


Dada la progresión podría sostenerse que el cierre se da en un día específico de la semana, cada cartera tiene 7 días de diferencia salvo excepciones. 

In [38]:
%%sql

WITH cierres AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_fultimo_cierre AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_fultimo_cierre AS valor
    FROM competencia_01
),

fechas AS (

    SELECT
        foto_mes,
        marca,
        valor,

        CAST(
            STRPTIME(
                CAST(foto_mes AS VARCHAR),
                '%Y%m'
            )
            + INTERVAL 1 MONTH
            - valor * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_cierre

    FROM cierres

    WHERE
        valor IS NOT NULL
)

SELECT
    foto_mes,
    marca,
    valor,
    fecha_cierre,
    DAYNAME(fecha_cierre) AS dia_semana,
    COUNT(*) AS clientes

FROM fechas

GROUP BY
    foto_mes,
    marca,
    valor,
    fecha_cierre

ORDER BY
    marca,
    clientes DESC;

,foto_mes,marca,valor,fecha_cierre,dia_semana,clientes
0,202107,Master,3,2021-07-29,Thursday,114303
1,202108,Master,6,2021-08-26,Thursday,113597
2,202106,Master,0,2021-07-01,Thursday,113387
3,202105,Master,5,2021-05-27,Thursday,112815
4,202104,Master,2,2021-04-29,Thursday,112378
5,202103,Master,1,2021-03-31,Wednesday,112027
6,202108,Master,13,2021-08-19,Thursday,23912
7,202106,Master,7,2021-06-24,Thursday,23798
8,202105,Master,12,2021-05-20,Thursday,23788
9,202107,Master,10,2021-07-22,Thursday,23786


In [39]:
%%sql

WITH cierres AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_fultimo_cierre AS valor_original
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_fultimo_cierre AS valor_original
    FROM competencia_01
),

fechas AS (

    SELECT
        foto_mes,
        marca,

        CAST(
            STRPTIME(
                CAST(foto_mes AS VARCHAR),
                '%Y%m'
            )
            + INTERVAL 1 MONTH
            - valor_original * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_cierre

    FROM cierres

    WHERE valor_original IS NOT NULL
),

conteo AS (

    SELECT
        foto_mes,
        marca,
        DAYNAME(fecha_cierre) AS dia_semana,
        COUNT(*) AS clientes

    FROM fechas

    GROUP BY
        foto_mes,
        marca,
        dia_semana
)

SELECT
    foto_mes,
    marca,
    dia_semana,
    clientes,

    ROUND(
        100.0 * clientes
        / SUM(clientes) OVER (
            PARTITION BY foto_mes, marca
        ),
        2
    ) AS porcentaje

FROM conteo

ORDER BY
    foto_mes,
    marca,
    porcentaje DESC;

,foto_mes,marca,dia_semana,clientes,porcentaje
0,202103,Master,Wednesday,112027,76.72
1,202103,Master,Thursday,33987,23.28
2,202103,Visa,Wednesday,112152,72.36
3,202103,Visa,Thursday,42833,27.64
4,202104,Master,Thursday,146402,100.00
5,202104,Master,Wednesday,6,0.00
6,202104,Visa,Thursday,155253,100.00
7,202104,Visa,Wednesday,3,0.00
8,202105,Master,Thursday,146872,100.00
9,202105,Master,Wednesday,4,0.00


### Fecha de vencimiento (parece ser vencimiento del plástico y no del resumen)
Hipótesis inicial: podría ser una cantidad de días relativa a foto_mes, pero todavía no sabemos qué significa el signo ni cuál es la fecha de referencia.

In [40]:
%%sql

WITH vencimientos AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS valor
    FROM competencia_01
)

SELECT
    foto_mes,
    marca,
    COUNT(*) AS filas,

    COUNT(*) FILTER (WHERE valor IS NULL) AS nulos,
    COUNT(*) FILTER (WHERE valor = 0) AS ceros,
    COUNT(*) FILTER (WHERE valor < 0) AS negativos,
    COUNT(*) FILTER (WHERE valor > 0) AS positivos,

    COUNT(DISTINCT valor) AS valores_distintos,

    MIN(valor) AS minimo,
    QUANTILE_CONT(valor, 0.01) AS percentil_01,
    MEDIAN(valor) AS mediana,
    QUANTILE_CONT(valor, 0.99) AS percentil_99,
    MAX(valor) AS maximo

FROM vencimientos

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,filas,nulos,ceros,negativos,positivos,valores_distintos,minimo,percentil_01,mediana,percentil_99,maximo
0,202103,Master,162900,16847,0,145911,142,128,-2251,-2190.0,-882.0,-90.0,6850
1,202103,Visa,162900,7894,0,154883,123,123,-2914178,-2190.0,-1339.0,-121.0,5815
2,202104,Master,163284,16822,0,146317,145,130,-2251,-2190.0,-882.0,-91.0,6880
3,202104,Visa,163284,7987,0,155169,128,123,-2914148,-2190.0,-1309.0,-122.0,5845
4,202105,Master,163768,16750,0,146865,153,128,-2251,-2190.0,-882.0,-91.0,6911
5,202105,Visa,163768,8039,0,155594,135,125,-2914117,-2190.0,-1309.0,-121.0,5876
6,202106,Master,164114,16600,0,147364,150,128,-2252,-2190.0,-913.0,-91.0,6941
7,202106,Visa,164114,8122,0,155859,133,124,-2914087,-2190.0,-1279.0,-91.0,5906
8,202107,Master,164348,15921,0,148222,205,131,-2251,-2190.0,-941.0,-60.0,6972
9,202107,Visa,164348,7232,0,156933,183,124,-2914056,-2190.0,-1279.0,-60.0,5937


Esta celda busca responder:
- ¿Predominan los valores negativos?
- ¿Hay ceros o positivos?
- ¿Los extremos afectan solamente a casos raros?
- ¿La distribución cambia entre meses?
- ¿Visa y Master utilizan la misma codificación?


- Los negativos son normales, no errores.
- No hay ceros.
- Los positivos son muy pocos: posiblemente tarjetas cuyo vencimiento ya pasó.
- Hay pocos valores distintos, así que es posible que sea una codificación estructurada.
- Los máximos avanzan exactamente con el calendario: 6850 → 6880 → 6911 → 6941...

Hipótesis:  valor = fecha de referencia − fecha de vencimiento

| foto_mes | valor |
|---|---:|
| 202103 | -2159 |
| 202104 | -2129 |
| 202105 | -2098 |


El contador aumenta:
- Marzo → abril: +30
- Abril → mayo: +31
Exactamente los días que avanzaría una referencia ubicada al final del mes/inicio del siguiente. 
Si usamos:
fecha_vencimiento = fecha_referencia − valor 

abril 2021 − (-2159 días)
= abril 2021 + 2159 días


Al mes siguiente, la referencia avanza 30 días y el valor pasa de -2159 a -2129; ambos reconstruyen la misma fecha futura.
Esto sustenta que es una distancia a una fecha fija, pero todavía debemos confirmar:
1. Que ocurra para la mayoría de los mismos clientes.
2. Que las fechas reconstruidas sean vencimientos razonables.
3. Si la referencia exacta es el primer día del mes siguiente o tiene un desplazamiento.

¿Es vencimiento del plástico o del resumen? Si fuera vencimiento del resumen las fechas serrían parecidas a las de cierre. 

In [41]:
%%sql

-- Primero que nada, voy a consultar la cantidad de días que hay entre la 
-- fecha de cierre y lo que figura como fecha de vencimiento. Si es vencimiento 
-- del plástico también podemos ver si coincide con la baja como cliente. 


WITH tarjetas AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_fultimo_cierre AS cierre,
        Master_Fvencimiento AS vencimiento
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_fultimo_cierre AS cierre,
        Visa_Fvencimiento AS vencimiento
    FROM competencia_01
),

comparacion AS (

    SELECT
        foto_mes,
        marca,
        cierre - vencimiento AS dias_entre_cierre_y_vencimiento

    FROM tarjetas

    WHERE cierre IS NOT NULL
      AND vencimiento IS NOT NULL

      -- Conservamos los ciclos de cierre normales
      AND cierre BETWEEN 0 AND 31
)

SELECT
    foto_mes,
    marca,
    COUNT(*) AS casos,

    QUANTILE_CONT(
        dias_entre_cierre_y_vencimiento, 0.01
    ) AS percentil_01,

    MEDIAN(
        dias_entre_cierre_y_vencimiento
    ) AS mediana_dias,

    QUANTILE_CONT(
        dias_entre_cierre_y_vencimiento, 0.99
    ) AS percentil_99,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE dias_entre_cierre_y_vencimiento
                  BETWEEN 1 AND 45
        ) / COUNT(*),
        2
    ) AS pct_compatible_resumen,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE dias_entre_cierre_y_vencimiento > 365
        ) / COUNT(*),
        2
    ) AS pct_mas_de_un_anio

FROM comparacion

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,casos,percentil_01,mediana_dias,percentil_99,pct_compatible_resumen,pct_mas_de_un_anio
0,202103,Master,146003,91.0,883.0,2191.0,0.00,87.73
1,202103,Visa,154966,122.0,1340.0,2191.0,0.00,92.86
2,202104,Master,146392,93.0,884.0,2192.0,0.01,87.83
3,202104,Visa,155239,124.0,1311.0,2192.0,0.01,93.00
4,202105,Master,146847,96.0,887.0,2195.0,0.01,86.91
5,202105,Visa,155590,126.0,1314.0,2195.0,0.01,92.69
6,202106,Master,147429,91.0,913.0,2190.0,0.01,83.60
7,202106,Visa,155904,91.0,1286.0,2190.0,0.00,91.92
8,202107,Master,148370,63.0,944.0,2193.0,0.01,83.25
9,202107,Visa,157054,84.0,1282.0,2193.0,0.02,91.98


Claramente debe ser fecha de vencimiento del plástico por la cantidad de días que hay entre el cierre y el vencimiento. Ahora bien, esto es si asumimos que cierre y vencimiento usan la misma referencia. Si no la usan, este análisis no sirve. 

In [42]:
%%sql

WITH vencimientos AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS valor
    FROM competencia_01
)

SELECT
    numero_de_cliente,
    foto_mes,
    marca,
    valor AS valor_original,

    CAST(
        STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
        + INTERVAL 1 MONTH
        AS DATE
    ) AS referencia_candidata,

    CAST(
        STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
        + INTERVAL 1 MONTH
        - valor * INTERVAL 1 DAY
        AS DATE
    ) AS vencimiento_calculado

FROM vencimientos

WHERE numero_de_cliente = 12159854
  AND valor IS NOT NULL

ORDER BY
    marca,
    foto_mes;

,numero_de_cliente,foto_mes,marca,valor_original,referencia_candidata,vencimiento_calculado
0,12159854.0,202103,Master,-2159,2021-04-01,2027-02-28
1,12159854.0,202104,Master,-2129,2021-05-01,2027-02-28
2,12159854.0,202105,Master,-2098,2021-06-01,2027-02-28
3,12159854.0,202106,Master,-2068,2021-07-01,2027-02-28
4,12159854.0,202107,Master,-2037,2021-08-01,2027-02-28
5,12159854.0,202108,Master,-2006,2021-09-01,2027-02-28
6,12159854.0,202103,Visa,-2159,2021-04-01,2027-02-28
7,12159854.0,202104,Visa,-2129,2021-05-01,2027-02-28
8,12159854.0,202105,Visa,-2098,2021-06-01,2027-02-28
9,12159854.0,202106,Visa,-2068,2021-07-01,2027-02-28


La referencia avanzó 30 días y el contador aumentó 30. Parecería que cada mes se acerca treinta unidades a un número cero que puede ser el vencimiento del plástico. Si es un valor positivo la tarjeta ya habría vencido y no sido renovada (motivo de baja)

In [43]:
%%sql

WITH vencimientos AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS valor
    FROM competencia_01
),

serie AS (

    SELECT
        *,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            AS DATE
        ) AS referencia,

        LAG(valor) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS valor_anterior,

        LAG(
            CAST(
                STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
                + INTERVAL 1 MONTH
                AS DATE
            )
        ) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS referencia_anterior

    FROM vencimientos
),

pares AS (

    SELECT
        foto_mes,
        marca,
        valor - valor_anterior AS cambio_valor,

        DATE_DIFF(
            'day',
            referencia_anterior,
            referencia
        ) AS dias_transcurridos

    FROM serie

    WHERE valor IS NOT NULL
      AND valor_anterior IS NOT NULL
      AND DATE_DIFF(
            'month',
            referencia_anterior,
            referencia
          ) = 1
)

SELECT
    foto_mes,
    marca,
    COUNT(*) AS pares,

    COUNT(*) FILTER (
        WHERE cambio_valor = dias_transcurridos
    ) AS casos_fecha_fija_relativa,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE cambio_valor = dias_transcurridos
        ) / COUNT(*),
        2
    ) AS pct_fecha_fija_relativa,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE cambio_valor = 0
        ) / COUNT(*),
        2
    ) AS pct_valor_igual,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ABS(cambio_valor) <= 7
        ) / COUNT(*),
        2
    ) AS pct_cambio_pequeno

FROM pares

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,pares,casos_fecha_fija_relativa,pct_fecha_fija_relativa,pct_valor_igual,pct_cambio_pequeno
0,202104,Master,144998,143948,99.28,0.0,0.0
1,202104,Visa,153827,153078,99.51,0.0,0.0
2,202105,Master,145364,144439,99.36,0.0,0.0
3,202105,Visa,154072,153427,99.58,0.0,0.0
4,202106,Master,145844,144848,99.32,0.0,0.0
5,202106,Visa,154346,153596,99.51,0.0,0.0
6,202107,Master,146531,145362,99.20,0.0,0.0
7,202107,Visa,154899,154257,99.59,0.0,0.0
8,202108,Master,146270,144132,98.54,0.0,0.0
9,202108,Visa,154906,153477,99.08,0.0,0.0


foto_mes: mes actual de la comparación. 202104 compara marzo contra abril de 2021.

marca: tarjeta analizada, Master o Visa.

pares: cantidad de clientes que tienen Fvencimiento informado en el mes actual y en el mes anterior.

casos_fecha_fija_relativa: clientes cuyo valor cambió exactamente la misma cantidad de días que avanzó el calendario. Eso equivale a conservar la misma fecha de vencimiento reconstruida.

pct_fecha_fija_relativa: porcentaje de los pares que conservaron esa misma fecha. Se calcula como: casos_fecha_fija_relativa / pares × 100 

pct_valor_igual: porcentaje cuyo número original quedó exactamente igual. No significa “misma fecha”; significa “mismo número almacenado”.

pct_cambio_pequeno: porcentaje cuyo número original cambió como máximo siete días.


La evidencia es fuertemente compatible con que Fvencimiento sea el vencimiento del plástico; el significado comercial exacto debería confirmarse con el diccionario de datos.
Está expresada como distancia respecto de una fecha de referencia.
Negativo: todavía no venció.
Positivo: el vencimiento registrado ya pasó; puede ser información relevante.
En aproximadamente 99% de los casos, la fecha permanece fija entre meses.
No debemos considerar negativos o positivos como errores automáticamente.

In [44]:
%%sql

WITH vencimientos AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS valor
    FROM competencia_01
),

fechas AS (

    SELECT
        foto_mes,
        marca,
        valor,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            AS DATE
        ) AS fecha_referencia,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - valor * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_vencimiento

    FROM vencimientos

    WHERE valor IS NOT NULL
)

SELECT
    foto_mes,
    marca,

    COUNT(*) AS vencimientos_informados,

    COUNT(*) FILTER (
        WHERE fecha_vencimiento < fecha_referencia
    ) AS plasticos_ya_vencidos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE fecha_vencimiento < fecha_referencia
        ) / COUNT(*),
        2
    ) AS pct_plasticos_ya_vencidos,

    COUNT(*) FILTER (
        WHERE YEAR(fecha_vencimiento) < 9999
          AND fecha_vencimiento = LAST_DAY(fecha_vencimiento)
    ) AS vencimientos_en_fin_de_mes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE YEAR(fecha_vencimiento) < 9999
              AND fecha_vencimiento = LAST_DAY(fecha_vencimiento)
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE YEAR(fecha_vencimiento) < 9999
            ),
            0
        ),
        2
    ) AS pct_vencimientos_en_fin_de_mes,

    COUNT(*) FILTER (
        WHERE fecha_vencimiento = DATE '9999-12-31'
    ) AS posible_codigo_31129999,

    MIN(fecha_vencimiento) FILTER (
        WHERE YEAR(fecha_vencimiento) < 9999
    ) AS fecha_mas_antigua,

    MAX(fecha_vencimiento) FILTER (
        WHERE YEAR(fecha_vencimiento) < 9999
    ) AS fecha_mas_futura

FROM fechas

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,vencimientos_informados,plasticos_ya_vencidos,pct_plasticos_ya_vencidos,vencimientos_en_fin_de_mes,pct_vencimientos_en_fin_de_mes,posible_codigo_31129999,fecha_mas_antigua,fecha_mas_futura
0,202103,Master,146053,142,0.10,144798,99.14,0,2002-06-30,2027-05-31
1,202103,Visa,155006,123,0.08,152178,98.47,466,2005-04-30,2027-05-31
2,202104,Master,146462,145,0.10,145218,99.15,0,2002-06-30,2027-06-30
3,202104,Visa,155297,128,0.08,152479,98.48,463,2005-04-30,2027-06-30
4,202105,Master,147018,153,0.10,145787,99.16,0,2002-06-30,2027-07-31
5,202105,Visa,155729,135,0.09,152926,98.49,458,2005-04-30,2027-07-31
6,202106,Master,147514,150,0.10,146289,99.17,0,2002-06-30,2027-08-31
7,202106,Visa,155992,133,0.09,153212,98.50,454,2005-04-30,2027-08-31
8,202107,Master,148427,205,0.14,147201,99.17,0,2002-06-30,2027-09-30
9,202107,Visa,157116,183,0.12,154344,98.52,449,2005-04-30,2027-09-30


- vencimientos_informados: tarjetas cuyo campo no es nulo.
- plasticos_ya_vencidos: fecha reconstruida anterior a la referencia.
- pct_plasticos_ya_vencidos: importancia relativa de esos casos.
- vencimientos_en_fin_de_mes: fechas que caen el último día de un mes.
- pct_vencimientos_en_fin_de_mes: si se acerca al 100%, confirma tanto la interpretación como la referencia empleada.
- posible_codigo_31129999: casos que requieren tratamiento especial.
- fecha_mas_antigua y fecha_mas_futura: permiten detectar otras fechas extremas.

Quiero responder cuatro preguntas concretas:
1. ¿La reconstrucción produce fechas de fin de mes, como debería ocurrir con el vencimiento de un plástico? Esto es para confirmar que interpreto correctamente el campo y la referencia. Los plásticos expresan su vencimiento como mes/año y normalmente se representan con el último día del mes.
2. ¿Cuántas tarjetas ya estaban vencidas? Esto es para conservar esa señal potencialmente útil: tarjeta vencida, posiblemente no renovada o inactiva.
3. ¿Aparece 31/12/9999 como posible código especial? Para detectar un posible código técnico como “sin vencimiento informado”. Si lo tratamos como una fecha real, el modelo interpretaría que faltan casi 8.000 años
4. ¿Cuál es el rango real de fechas? Para encontrar otras fechas imposibles o sospechosas. Por ejemplo, vencimientos demasiado antiguos o demasiado lejanos que podrían ser códigos, información desactualizada o errores.

1. La interpretación queda fuertemente respaldada, aunque no demostrada por un diccionario.
   Entre 98,47% y 99,18% de las fechas reconstruidas caen exactamente al final de un mes. Esto es muy consistente con un vencimiento de plástico y con la referencia usada.
2. Las tarjetas vencidas son pocas, pero pueden ser informativas.
   Representan aproximadamente 0,08%–0,14%. No debemos eliminarlas ni imputarlas.
3. 31/12/9999 es un código especial.
   Aparece principalmente en Visa: unos 450 clientes por mes. No puede ser un vencimiento real. Todavía no sabemos si significa “sin vencimiento”, “dato desconocido” u otra condición técnica.
4. Hay vencimientos antiguos.
   Master llega hasta 2002 y Visa hasta 2005. Probablemente sean registros vencidos y desactualizados, pero debemos comprobar si esas tarjetas siguen activas.
5. Sin contar 9999, el futuro máximo llega a 2027.
   Eso resulta razonable para fotografías tomadas en 2021.
Entonces quedan tres grupos que debemos investigar:
- Las fechas 31/12/9999.
- Los plásticos ya vencidos.
- El 1%–1,5% que no cae al final del mes.

Queremos saber si 31/12/9999 aparece en tarjetas activas o en registros sin actividad. Lo cruzamos con status, consumo y cierre:

In [45]:
%%sql

WITH tarjetas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS vencimiento,
        Master_status AS status,
        Master_mconsumototal AS consumo,
        Master_fultimo_cierre AS ultimo_cierre
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS vencimiento,
        Visa_status AS status,
        Visa_mconsumototal AS consumo,
        Visa_fultimo_cierre AS ultimo_cierre
    FROM competencia_01
),

fechas AS (

    SELECT
        *,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - vencimiento * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_vencimiento

    FROM tarjetas

    WHERE vencimiento IS NOT NULL
)

SELECT
    foto_mes,
    marca,
    status,

    COUNT(*) AS clientes_con_codigo_9999,

    COUNT(*) FILTER (
        WHERE consumo IS NOT NULL
    ) AS clientes_con_consumo_informado,

    COUNT(*) FILTER (
        WHERE consumo > 0
    ) AS clientes_con_consumo_positivo,

    COUNT(*) FILTER (
        WHERE ultimo_cierre IS NOT NULL
    ) AS clientes_con_cierre_informado,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE consumo > 0
        ) / COUNT(*),
        2
    ) AS pct_con_consumo_positivo

FROM fechas

WHERE fecha_vencimiento = DATE '9999-12-31'

GROUP BY
    foto_mes,
    marca,
    status

ORDER BY
    foto_mes,
    marca,
    status;

,foto_mes,marca,status,clientes_con_codigo_9999,clientes_con_consumo_informado,clientes_con_consumo_positivo,clientes_con_cierre_informado,pct_con_consumo_positivo
0,202103,Visa,0,460,442,406,460,88.26
1,202103,Visa,7,3,3,3,3,100.00
2,202103,Visa,9,3,2,1,3,33.33
3,202104,Visa,0,453,435,393,453,86.75
4,202104,Visa,7,1,1,1,1,100.00
5,202104,Visa,9,9,9,7,9,77.78
6,202105,Visa,0,451,433,395,451,87.58
7,202105,Visa,6,1,1,0,1,0.00
8,202105,Visa,9,6,6,4,6,66.67
9,202106,Visa,0,447,431,390,447,87.25



- clientes_con_codigo_9999: cantidad que tiene ese código.
- clientes_con_consumo_informado: existe el dato de consumo, aunque sea cero.
- clientes_con_consumo_positivo: efectivamente usaron la tarjeta.
- clientes_con_cierre_informado: el sistema siguió generando información de cierre.
- pct_con_consumo_positivo: proporción que utilizó la tarjeta ese mes.
Si tienen cierre y consumo, 9999 no significa necesariamente tarjeta inactiva; podría significar “vencimiento no informado” o una fecha técnica.

Tomo la primera fila, de esos 460:
- Los 460 tienen cierre informado.
- 442 tienen consumo informado.
- 406 realizaron consumos.
- El 88,26% utilizó la tarjeta.
El mismo patrón continúa todos los meses: alrededor del 87% tiene consumo positivo.
Conclusión:
- 31/12/9999 no significa tarjeta inexistente o inactiva.
- Tampoco es una fecha real.
- Es un código técnico que probablemente significa “vencimiento no informado”, “no aplicable” o algún tipo particular de tarjeta.
- Su significado exacto no puede deducirse todavía.
- Debemos conservarlo como una categoría especial, no como miles de años hasta el vencimiento.
- status = 0 concentra casi todos los casos, pero todavía no conocemos la etiqueta comercial de ese código.

Habría que seguir a los mismos clientes para ver si son los mismos clientes los que conservan 9999 todos los meses o si pasan posteriormente a una fecha normal. Eso permitirá distinguir un atributo estable de una falta de información temporal. 
Seguimos a cada cliente durante los seis meses y contamos si tuvo:
- Siempre el código 9999.
- 9999 y luego una fecha normal.
- 9999 combinado con valores nulos.

In [46]:
%%sql

WITH tarjetas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS vencimiento
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS vencimiento
    FROM competencia_01
),

estados AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        marca,

        CASE
            WHEN vencimiento IS NULL
                THEN 'SIN_DATO'

            WHEN CAST(
                STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
                + INTERVAL 1 MONTH
                - vencimiento * INTERVAL 1 DAY
                AS DATE
            ) = DATE '9999-12-31'
                THEN 'CODIGO_9999'

            ELSE 'FECHA_NORMAL'
        END AS estado_vencimiento

    FROM tarjetas
),

resumen_por_cliente AS (

    SELECT
        numero_de_cliente,
        marca,

        COUNT(*) AS meses_en_dataset,

        COUNT(*) FILTER (
            WHERE estado_vencimiento = 'CODIGO_9999'
        ) AS meses_con_9999,

        COUNT(*) FILTER (
            WHERE estado_vencimiento = 'FECHA_NORMAL'
        ) AS meses_con_fecha_normal,

        COUNT(*) FILTER (
            WHERE estado_vencimiento = 'SIN_DATO'
        ) AS meses_sin_dato

    FROM estados

    GROUP BY
        numero_de_cliente,
        marca

    HAVING COUNT(*) FILTER (
        WHERE estado_vencimiento = 'CODIGO_9999'
    ) > 0
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN meses_con_fecha_normal > 0
                THEN '9999_Y_FECHA_NORMAL'

            WHEN meses_sin_dato > 0
                THEN '9999_Y_NULOS'

            ELSE 'SOLO_9999'
        END AS comportamiento

    FROM resumen_por_cliente
)

SELECT
    marca,
    comportamiento,

    COUNT(*) AS clientes,

    ROUND(AVG(meses_en_dataset), 2) AS promedio_meses_en_dataset,
    ROUND(AVG(meses_con_9999), 2) AS promedio_meses_con_9999,
    ROUND(AVG(meses_con_fecha_normal), 2) AS promedio_meses_con_fecha_normal,
    ROUND(AVG(meses_sin_dato), 2) AS promedio_meses_sin_dato

FROM clasificacion

GROUP BY
    marca,
    comportamiento

ORDER BY
    marca,
    clientes DESC;

,marca,comportamiento,clientes,promedio_meses_en_dataset,promedio_meses_con_9999,promedio_meses_con_fecha_normal,promedio_meses_sin_dato
0,Master,9999_Y_FECHA_NORMAL,1,6.00,1.00,5.00,0.00
1,Visa,SOLO_9999,446,5.90,5.90,0.00,0.00
2,Visa,9999_Y_FECHA_NORMAL,31,5.94,2.90,2.81,0.23
3,Visa,9999_Y_NULOS,4,5.50,3.25,0.00,2.25



- SOLO_9999: código estable para ese cliente.
- 9999_Y_FECHA_NORMAL: el código fue temporal o posteriormente corregido.
- 9999_Y_NULOS: el código convivió con meses sin información.

La salida es bastante clara:
- 446 clientes Visa tuvieron solamente 9999 durante casi los seis meses observados.
- Nunca tuvieron una fecha normal ni un nulo.
- Representan aproximadamente el 93% de los clientes Visa que alguna vez tuvieron 9999.
- Además, vimos que la mayoría consume y tiene cierre informado.
Esto demuestra que 9999 no es una falla mensual aleatoria. Es una condición estable para un grupo de clientes. Puede ser una característica de cierto producto o una ausencia estructural de vencimiento, pero no conocemos su significado exacto.
Los otros casos:
- 31 Visa alternaron entre 9999 y una fecha normal: posiblemente actualización, renovación o corrección.
- 4 Visa combinaron 9999 con nulos.
- El único Master tuvo cinco meses normales y solo uno con 9999: parece un caso excepcional.

Faltan tres controles:
1. Valores positivos: verificar si las tarjetas vencidas siguen activas o consumiendo.
2. Nulos: distinguir tarjeta inexistente/inactiva de información perdida.
3. Fechas que no caen al final del mes: entender ese 1%–1,5% excepcional.

In [47]:
%%sql

WITH tarjetas AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS vencimiento,
        Master_mconsumototal AS consumo,
        Master_fultimo_cierre AS ultimo_cierre
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS vencimiento,
        Visa_mconsumototal AS consumo,
        Visa_fultimo_cierre AS ultimo_cierre
    FROM competencia_01
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN vencimiento > 0
                THEN 'VENCIDA'

            WHEN CAST(
                STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
                + INTERVAL 1 MONTH
                - vencimiento * INTERVAL 1 DAY
                AS DATE
            ) = DATE '9999-12-31'
                THEN 'CODIGO_9999'

            WHEN vencimiento < 0
                THEN 'VIGENTE'

            ELSE 'SIN_DATO'
        END AS estado_plastico

    FROM tarjetas
)

SELECT
    foto_mes,
    marca,

    COUNT(*) FILTER (
        WHERE estado_plastico = 'VENCIDA'
    ) AS plasticos_vencidos,

    COUNT(*) FILTER (
        WHERE estado_plastico = 'VENCIDA'
          AND consumo > 0
    ) AS vencidos_con_consumo,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE estado_plastico = 'VENCIDA'
              AND consumo > 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE estado_plastico = 'VENCIDA'
            ),
            0
        ),
        2
    ) AS pct_vencidos_con_consumo,

    COUNT(*) FILTER (
        WHERE estado_plastico = 'VENCIDA'
          AND ultimo_cierre BETWEEN 0 AND 31
    ) AS vencidos_con_cierre_ultimos_31_dias,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE estado_plastico = 'VENCIDA'
              AND ultimo_cierre BETWEEN 0 AND 31
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE estado_plastico = 'VENCIDA'
            ),
            0
        ),
        2
    ) AS pct_vencidos_con_cierre_ultimos_31_dias,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE estado_plastico = 'VIGENTE'
              AND consumo > 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE estado_plastico = 'VIGENTE'
            ),
            0
        ),
        2
    ) AS pct_vigentes_con_consumo

FROM clasificacion

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,plasticos_vencidos,vencidos_con_consumo,pct_vencidos_con_consumo,vencidos_con_cierre_ultimos_31_dias,pct_vencidos_con_cierre_ultimos_31_dias,pct_vigentes_con_consumo
0,202103,Master,142,10,7.04,133,93.66,28.86
1,202103,Visa,123,25,20.33,111,90.24,80.83
2,202104,Master,145,11,7.59,131,90.34,30.53
3,202104,Visa,128,27,21.09,114,89.06,81.09
4,202105,Master,153,9,5.88,134,87.58,29.80
5,202105,Visa,135,25,18.52,116,85.93,80.79
6,202106,Master,150,10,6.67,129,86.00,31.85
7,202106,Visa,133,29,21.80,109,81.95,81.63
8,202107,Master,205,7,3.41,184,89.76,29.75
9,202107,Visa,183,29,15.85,157,85.79,80.35


pct_vencidos_con_consumo: cuántos plásticos con fecha vencida continúan utilizándose.
pct_vencidos_con_cierre: cuántos siguen generando información de cierre.
pct_vigentes_con_consumo: referencia para comparar con tarjetas normales.

Vencido no equivale a cuenta inactiva ni a tarjeta no renovada.
Un plástico vencido puede seguir generando resumen por:
- Cuotas pendientes.
- Saldos financiados e intereses.
- Débitos automáticos.
- Comisiones.
- Renovación o reemplazo del plástico dentro de la misma cuenta.

- La mayoría de los plásticos vencidos sigue teniendo un cierre reciente: 76%–94%.
- Sin embargo, consumen mucho menos que los vigentes.
- Puede haber cuotas, saldos financiados, intereses o débitos aun cuando el plástico haya vencido.
- Por eso no podemos traducir vencimiento > 0 como “cuenta inactiva” o “no renovada”.
La interpretación correcta es únicamente:
La fecha registrada del plástico ya pasó.

El valor debe conservarse y luego podemos crear el indicador plastico_vencido.

Veamos los nulos. Cuando falta el vencimiento, ¿faltan también los demás datos de la tarjeta o solamente ese campo? Pasame la salida.

In [48]:
%%sql

WITH tarjetas AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS vencimiento,
        Master_status AS status,
        Master_mconsumototal AS consumo,
        Master_fultimo_cierre AS ultimo_cierre
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS vencimiento,
        Visa_status AS status,
        Visa_mconsumototal AS consumo,
        Visa_fultimo_cierre AS ultimo_cierre
    FROM competencia_01
)

SELECT
    foto_mes,
    marca,

    COUNT(*) FILTER (
        WHERE vencimiento IS NULL
    ) AS vencimientos_nulos,

    COUNT(*) FILTER (
        WHERE vencimiento IS NULL
          AND status IS NULL
          AND consumo IS NULL
          AND ultimo_cierre IS NULL
    ) AS nulos_sin_otros_datos_de_tarjeta,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE vencimiento IS NULL
              AND status IS NULL
              AND consumo IS NULL
              AND ultimo_cierre IS NULL
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE vencimiento IS NULL
            ),
            0
        ),
        2
    ) AS pct_nulos_sin_otros_datos,

    COUNT(*) FILTER (
        WHERE vencimiento IS NULL
          AND (
              status IS NOT NULL
              OR consumo IS NOT NULL
              OR ultimo_cierre IS NOT NULL
          )
    ) AS nulos_con_otros_datos_de_tarjeta,

    COUNT(*) FILTER (
        WHERE vencimiento IS NULL
          AND consumo > 0
    ) AS nulos_con_consumo_positivo,

    COUNT(*) FILTER (
        WHERE vencimiento IS NULL
          AND ultimo_cierre BETWEEN 0 AND 31
    ) AS nulos_con_cierre_ultimos_31_dias

FROM tarjetas

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,vencimientos_nulos,nulos_sin_otros_datos_de_tarjeta,pct_nulos_sin_otros_datos,nulos_con_otros_datos_de_tarjeta,nulos_con_consumo_positivo,nulos_con_cierre_ultimos_31_dias
0,202103,Master,16847,16847,100.0,0,0,0
1,202103,Visa,7894,7894,100.0,0,0,0
2,202104,Master,16822,16822,100.0,0,0,0
3,202104,Visa,7987,7987,100.0,0,0,0
4,202105,Master,16750,16750,100.0,0,0,0
5,202105,Visa,8039,8039,100.0,0,0,0
6,202106,Master,16600,16600,100.0,0,0,0
7,202106,Visa,8122,8122,100.0,0,0,0
8,202107,Master,15921,15921,100.0,0,0,0
9,202107,Visa,7232,7232,100.0,0,0,0


La lectura es directa:
- En marzo faltan 16.847 vencimientos Master.
- En esos mismos 16.847 casos también faltan status, consumo y último cierre.
- Esto se repite al 100% en todos los meses y en ambas marcas.
Por lo tanto, el nulo no es un problema aislado de Fvencimiento. Es ausencia conjunta de información de esa tarjeta, probablemente porque el cliente no tiene esa marca o porque la fuente completa no informó.
Decisión:
- No imputar el vencimiento.
- Conservar el nulo.
- Más adelante crear indicadores como tiene_datos_master y tiene_datos_visa.
Esto también puede apoyar la idea de que la principalidad en ARG es de Visa y que hay más gente con Visa que con Master. 


In [49]:
%%sql

WITH vencimientos AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS valor
    FROM competencia_01
),

fechas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        marca,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - valor * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_vencimiento

    FROM vencimientos

    WHERE valor IS NOT NULL
),

excepciones AS (

    SELECT
        numero_de_cliente,
        marca,

        DATE_DIFF(
            'day',
            fecha_vencimiento,
            LAST_DAY(fecha_vencimiento)
        ) AS dias_antes_del_fin_de_mes

    FROM fechas

    WHERE YEAR(fecha_vencimiento) < 9999
      AND fecha_vencimiento != LAST_DAY(fecha_vencimiento)
),

conteo AS (

    SELECT
        marca,
        dias_antes_del_fin_de_mes,
        COUNT(*) AS registros_cliente_mes,
        COUNT(DISTINCT numero_de_cliente) AS clientes

    FROM excepciones

    GROUP BY
        marca,
        dias_antes_del_fin_de_mes
)

SELECT
    marca,
    dias_antes_del_fin_de_mes,
    registros_cliente_mes,
    clientes,

    ROUND(
        100.0 * registros_cliente_mes
        / SUM(registros_cliente_mes) OVER (
            PARTITION BY marca
        ),
        2
    ) AS pct_de_las_excepciones

FROM conteo

ORDER BY
    marca,
    registros_cliente_mes DESC;

,marca,dias_antes_del_fin_de_mes,registros_cliente_mes,clientes,pct_de_las_excepciones
0,Master,1,7392,1288,100.0
1,Visa,1,14016,2400,100.0


En los casos en los que no venció en el último día del mes hay una diferencia de un día. 

La salida muestra un patrón muy claro:
- No existen desviaciones variadas.
- El 100% de las excepciones cae exactamente un día antes del final del mes.
- Son 1.288 clientes Master y 2.400 Visa.
- Cada cliente aparece unas 5,7–5,8 veces, por lo que parecen ser casi siempre los mismos clientes durante los seis meses.
Esto apunta a una segunda convención de carga —posiblemente una diferencia inclusiva/exclusiva de un día— y no a fechas aleatoriamente dañadas.
Antes de sumarles un día, debemos comprobar si esos clientes mantienen siempre el mismo patrón:

In [50]:
%%sql

WITH vencimientos AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Fvencimiento AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Fvencimiento AS valor
    FROM competencia_01
),

fechas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        marca,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - valor * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_vencimiento

    FROM vencimientos

    WHERE valor IS NOT NULL
),

estados AS (

    SELECT
        numero_de_cliente,
        marca,

        CASE
            WHEN fecha_vencimiento = LAST_DAY(fecha_vencimiento)
                THEN 'FIN_DE_MES'

            WHEN fecha_vencimiento
                 = LAST_DAY(fecha_vencimiento) - INTERVAL 1 DAY
                THEN 'UN_DIA_ANTES'

            ELSE 'OTRA_FECHA'
        END AS ubicacion_fecha

    FROM fechas

    WHERE YEAR(fecha_vencimiento) < 9999
),

resumen_cliente AS (

    SELECT
        numero_de_cliente,
        marca,

        COUNT(*) FILTER (
            WHERE ubicacion_fecha = 'UN_DIA_ANTES'
        ) AS meses_un_dia_antes,

        COUNT(*) FILTER (
            WHERE ubicacion_fecha = 'FIN_DE_MES'
        ) AS meses_fin_de_mes,

        COUNT(*) FILTER (
            WHERE ubicacion_fecha = 'OTRA_FECHA'
        ) AS meses_otra_fecha

    FROM estados

    GROUP BY
        numero_de_cliente,
        marca

    HAVING COUNT(*) FILTER (
        WHERE ubicacion_fecha = 'UN_DIA_ANTES'
    ) > 0
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN meses_fin_de_mes = 0
             AND meses_otra_fecha = 0
                THEN 'SIEMPRE_UN_DIA_ANTES'

            WHEN meses_fin_de_mes > 0
                THEN 'CAMBIA_ENTRE_UN_DIA_ANTES_Y_FIN_DE_MES'

            ELSE 'UN_DIA_ANTES_Y_OTRAS_FECHAS'
        END AS comportamiento

    FROM resumen_cliente
)

SELECT
    marca,
    comportamiento,
    COUNT(*) AS clientes,

    ROUND(
        AVG(meses_un_dia_antes),
        2
    ) AS promedio_meses_un_dia_antes,

    ROUND(
        AVG(meses_fin_de_mes),
        2
    ) AS promedio_meses_fin_de_mes,

    ROUND(
        AVG(meses_otra_fecha),
        2
    ) AS promedio_meses_otra_fecha

FROM clasificacion

GROUP BY
    marca,
    comportamiento

ORDER BY
    marca,
    clientes DESC;

,marca,comportamiento,clientes,promedio_meses_un_dia_antes,promedio_meses_fin_de_mes,promedio_meses_otra_fecha
0,Master,SIEMPRE_UN_DIA_ANTES,1288,5.74,0.0,0.0
1,Visa,SIEMPRE_UN_DIA_ANTES,2399,5.84,0.0,0.0
2,Visa,CAMBIA_ENTRE_UN_DIA_ANTES_Y_FIN_DE_MES,1,1.00,2.0,0.0


La salida confirma que no es un error aleatorio:
- 1.288 clientes Master usan siempre la fecha un día antes del fin de mes.
- 2.399 clientes Visa hacen lo mismo.
- Nunca alternan con otras fechas.
- Solo un cliente Visa cambió de convención.
El promedio de 5,74/5,84 meses se debe a que algunos clientes no aparecen durante los seis meses completos, no a que cambien de patrón.
Conclusión: existen dos convenciones estables para guardar el vencimiento:
1. Último día del mes.
2. Un día antes del último día.
Como el plástico expresa vencimiento por mes/año, ambas representan el mismo mes. Para modelar podemos:
- Convertir ambas al último día del mes o trabajar con el mes de vencimiento.
- Conservar opcionalmente un indicador vencimiento_un_dia_antes.
- No tratarlas como datos corruptos.

Los negativos no son errores.
Los nulos son estructurales y no deben imputarse.
9999 es un código especial que no debe tratarse como una fecha real.

### Fecha alta

¿El valor representa una fecha absoluta o la cantidad de días transcurridos desde un alta fija?

In [51]:
%%sql

WITH altas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_fechaalta AS valor_original
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_fechaalta AS valor_original
    FROM competencia_01
)

SELECT
    numero_de_cliente,
    foto_mes,
    marca,
    valor_original,

    CAST(
        STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
        + INTERVAL 1 MONTH
        AS DATE
    ) AS referencia_candidata,

    CAST(
        STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
        + INTERVAL 1 MONTH
        - valor_original * INTERVAL 1 DAY
        AS DATE
    ) AS fecha_alta_calculada

FROM altas

WHERE numero_de_cliente = 12159854
  AND valor_original IS NOT NULL

ORDER BY
    marca,
    foto_mes;

,numero_de_cliente,foto_mes,marca,valor_original,referencia_candidata,fecha_alta_calculada
0,12159854.0,202103,Master,3324,2021-04-01,2012-02-24
1,12159854.0,202104,Master,3354,2021-05-01,2012-02-24
2,12159854.0,202105,Master,3385,2021-06-01,2012-02-24
3,12159854.0,202106,Master,3415,2021-07-01,2012-02-24
4,12159854.0,202107,Master,3446,2021-08-01,2012-02-24
5,12159854.0,202108,Master,3477,2021-09-01,2012-02-24
6,12159854.0,202103,Visa,4056,2021-04-01,2010-02-22
7,12159854.0,202104,Visa,4086,2021-05-01,2010-02-22
8,12159854.0,202105,Visa,4117,2021-06-01,2010-02-22
9,12159854.0,202106,Visa,4147,2021-07-01,2010-02-22


Para este cliente queda demostrado que fechaalta es un contador de antigüedad:
- Master aumenta 30/31 días cada mes, pero siempre reconstruye 24/02/2012.
- Visa hace lo mismo y siempre reconstruye 22/02/2010.
Además, no parece ser el alta del plástico actual. Ese cliente tiene fechas de alta de 2010/2012 y vencimiento del plástico en 2027. Lo más probable es que sea el alta de la cuenta o de la relación con cada marca, renovada luego con distintos plásticos. El significado comercial exacto requiere diccionario.
Ahora verificamos si ocurre en toda la población:

In [52]:
%%sql

WITH altas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_fechaalta AS valor
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_fechaalta AS valor
    FROM competencia_01
),

fechas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        marca,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            AS DATE
        ) AS referencia,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - valor * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_alta_calculada

    FROM altas

    WHERE valor IS NOT NULL
),

comparaciones AS (

    SELECT
        *,

        LAG(referencia) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS referencia_anterior,

        LAG(fecha_alta_calculada) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS fecha_alta_anterior

    FROM fechas
)

SELECT
    foto_mes,
    marca,

    COUNT(*) AS comparaciones_cliente_mes,

    COUNT(*) FILTER (
        WHERE fecha_alta_calculada = fecha_alta_anterior
    ) AS mantienen_misma_fecha_alta,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE fecha_alta_calculada = fecha_alta_anterior
        ) / COUNT(*),
        2
    ) AS pct_mantienen_misma_fecha_alta

FROM comparaciones

WHERE referencia_anterior IS NOT NULL
  AND DATE_DIFF(
        'month',
        referencia_anterior,
        referencia
      ) = 1

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,comparaciones_cliente_mes,mantienen_misma_fecha_alta,pct_mantienen_misma_fecha_alta
0,202104,Master,144998,144996,100.00
1,202104,Visa,153827,153818,99.99
2,202105,Master,145364,145362,100.00
3,202105,Visa,154072,154065,100.00
4,202106,Master,145844,145843,100.00
5,202106,Visa,154346,154339,100.00
6,202107,Master,146531,146527,100.00
7,202107,Visa,154899,154892,100.00
8,202108,Master,146270,146269,100.00
9,202108,Visa,154906,154885,99.99


La conclusión es muy sólida:
- Prácticamente todos los clientes conservan exactamente la misma fecha de alta.
- Los pocos casos diferentes son mínimos.
- 100,00% está redondeado: por ejemplo, Master abril tiene 2 excepciones sobre 144.998 comparaciones.
Entonces fechaalta es una antigüedad expresada en días respecto de una fecha fija. Probablemente sea el alta de la relación/cuenta de tarjeta de cada marca, no la emisión del plástico actual.
Para cerrar esta variable sin extendernos innecesariamente, falta comprobar dos cosas:
- Si sus nulos coinciden con los de Fvencimiento.
- Si las fechas reconstruidas tienen un rango razonable.

In [53]:
%%sql

WITH tarjetas AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_fechaalta AS fechaalta,
        Master_Fvencimiento AS vencimiento
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_fechaalta AS fechaalta,
        Visa_Fvencimiento AS vencimiento
    FROM competencia_01
),

fechas AS (

    SELECT
        *,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            AS DATE
        ) AS referencia,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - fechaalta * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_alta_calculada

    FROM tarjetas
)

SELECT
    foto_mes,
    marca,

    COUNT(*) FILTER (
        WHERE fechaalta IS NULL
    ) AS fechaalta_nula,

    COUNT(*) FILTER (
        WHERE fechaalta IS NULL
          AND vencimiento IS NULL
    ) AS ambos_nulos,

    COUNT(*) FILTER (
        WHERE fechaalta IS NULL
          AND vencimiento IS NOT NULL
    ) AS falta_solo_fechaalta,

    COUNT(*) FILTER (
        WHERE fechaalta IS NOT NULL
          AND vencimiento IS NULL
    ) AS fechaalta_sin_vencimiento,

    COUNT(*) FILTER (
        WHERE fechaalta = 0
    ) AS altas_en_fecha_de_referencia,

    COUNT(*) FILTER (
        WHERE fecha_alta_calculada > referencia
    ) AS fechas_de_alta_futuras,

    MIN(fecha_alta_calculada) AS fecha_alta_mas_antigua,
    MAX(fecha_alta_calculada) AS fecha_alta_mas_reciente

FROM fechas

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,fechaalta_nula,ambos_nulos,falta_solo_fechaalta,fechaalta_sin_vencimiento,altas_en_fecha_de_referencia,fechas_de_alta_futuras,fecha_alta_mas_antigua,fecha_alta_mas_reciente
0,202103,Master,16847,16847,0,0,0,0,2000-06-08,2021-03-31
1,202103,Visa,7894,7894,0,0,0,0,1986-10-04,2021-03-31
2,202104,Master,16822,16822,0,0,0,0,2000-06-08,2021-04-29
3,202104,Visa,7987,7987,0,0,0,0,1986-10-04,2021-04-29
4,202105,Master,16750,16750,0,0,0,0,2000-06-08,2021-05-31
5,202105,Visa,8039,8039,0,0,0,0,1986-10-04,2021-05-31
6,202106,Master,16600,16600,0,0,0,0,2000-06-08,2021-06-30
7,202106,Visa,8122,8122,0,0,0,0,1986-10-04,2021-06-30
8,202107,Master,15921,15921,0,0,0,0,2000-06-08,2021-07-29
9,202107,Visa,7232,7232,0,0,0,0,1986-10-04,2021-07-29


- Sus nulos coinciden exactamente con los de Fvencimiento.
- No existe ningún caso donde falte solo uno de los dos campos.
- Por lo tanto, no es una pérdida aislada de fechaalta: coincide con la ausencia conjunta observada en los campos de tarjeta que controlamos.
- No hay altas futuras ni valores cero.
- El rango es razonable:
  - Master: desde 2000.
  - Visa: desde 1986.
  - Las fechas más recientes corresponden al propio mes de la fotografía.
Las fechas máximas cambian porque pertenecen a clientes distintos que se incorporan cada mes; no porque una misma fecha de alta se modifique.
Decisión:
- No imputar los nulos.
- Conservar el valor, que ya funciona como antigüedad en días.
- Más adelante puede transformarse en antiguedad_en_años.
- No tiene sentido crearle deltas: aumenta mecánicamente con el calendario.

### Fecha Inicio Mora

¿Finiciomora = 0 significa “sin mora” o aparece cuando delinquency indica mora?

In [54]:
%%sql

WITH mora AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Finiciomora AS inicio_mora,
        Master_delinquency AS delinquency
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Finiciomora AS inicio_mora,
        Visa_delinquency AS delinquency
    FROM competencia_01
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN inicio_mora IS NULL
                THEN 'INICIO_MORA_NULO'

            WHEN inicio_mora = 0
                THEN 'INICIO_MORA_CERO'

            WHEN inicio_mora > 0
                THEN 'INICIO_MORA_POSITIVO'

            ELSE 'INICIO_MORA_NEGATIVO'
        END AS estado_inicio_mora

    FROM mora
)

SELECT
    marca,
    estado_inicio_mora,

    COUNT(*) AS registros_cliente_mes,

    COUNT(*) FILTER (
        WHERE delinquency IS NULL
    ) AS delinquency_nula,

    COUNT(*) FILTER (
        WHERE delinquency = 0
    ) AS delinquency_cero,

    COUNT(*) FILTER (
        WHERE delinquency > 0
    ) AS delinquency_positiva,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE delinquency > 0
        ) / COUNT(*),
        2
    ) AS pct_con_delinquency_positiva,

    MIN(delinquency) AS delinquency_minima,
    MEDIAN(delinquency) AS delinquency_mediana,
    MAX(delinquency) AS delinquency_maxima

FROM clasificacion

GROUP BY
    marca,
    estado_inicio_mora

ORDER BY
    marca,
    estado_inicio_mora;

,marca,estado_inicio_mora,registros_cliente_mes,delinquency_nula,delinquency_cero,delinquency_positiva,pct_con_delinquency_positiva,delinquency_minima,delinquency_mediana,delinquency_maxima
0,Master,INICIO_MORA_CERO,1,0,0,1,100.0,1,1.0,1
1,Master,INICIO_MORA_NULO,978240,99644,878596,0,0.0,0,0.0,0
2,Master,INICIO_MORA_POSITIVO,4820,0,0,4820,100.0,1,1.0,1
3,Visa,INICIO_MORA_CERO,6471,0,0,6471,100.0,1,1.0,1
4,Visa,INICIO_MORA_NULO,969739,47186,922553,0,0.0,0,0.0,0
5,Visa,INICIO_MORA_POSITIVO,6851,0,0,6851,100.0,1,1.0,1


- Cada registro representa un cliente en un mes.
- Si INICIO_MORA_CERO aparece junto con delinquency_positiva, entonces cero no significa “sin mora”.
- Si INICIO_MORA_NULO coincide con delinquency = 0, el nulo probablemente representa ausencia de mora.
- Los mínimos, medianas y máximos nos ayudarán a descubrir si delinquency parece una cantidad de días o una categoría.

Esta salida demuestra tres cosas importantes:
1. delinquency es binaria: solamente toma 0 o 1. No representa cantidad de días.
2. delinquency = 1 siempre coincide con Finiciomora informada, ya sea cero o positiva. Por eso 1 funciona como indicador de mora, no de fraude.
3. Finiciomora = 0 no significa “sin mora”: todos esos casos tienen delinquency = 1.
Cuando Finiciomora es nula hay dos situaciones:
- delinquency = 0: la tarjeta está informada, pero no está en mora; la fecha no corresponde.
- delinquency nula: hay una ausencia estructural probable en los campos de tarjeta controlados en este cruce.
Ahora falta distinguir si Finiciomora = 0 significa “la mora comenzó en la referencia” o algún código especial. Para eso seguimos a quienes permanecieron en mora dos meses consecutivos:

In [55]:
%%sql

WITH mora AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_Finiciomora AS inicio_mora,
        Master_delinquency AS delinquency
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_Finiciomora AS inicio_mora,
        Visa_delinquency AS delinquency
    FROM competencia_01
),

fechas AS (

    SELECT
        *,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            AS DATE
        ) AS referencia,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            - inicio_mora * INTERVAL 1 DAY
            AS DATE
        ) AS fecha_inicio_mora

    FROM mora
),

serie AS (

    SELECT
        *,

        LAG(referencia) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS referencia_anterior,

        LAG(inicio_mora) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS inicio_mora_anterior,

        LAG(delinquency) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS delinquency_anterior,

        LAG(fecha_inicio_mora) OVER (
            PARTITION BY numero_de_cliente, marca
            ORDER BY foto_mes
        ) AS fecha_inicio_mora_anterior

    FROM fechas
)

SELECT
    foto_mes,
    marca,

    COUNT(*) AS clientes_en_mora_dos_meses_seguidos,

    COUNT(*) FILTER (
        WHERE fecha_inicio_mora = fecha_inicio_mora_anterior
    ) AS mantienen_misma_fecha_inicio,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE fecha_inicio_mora = fecha_inicio_mora_anterior
        ) / COUNT(*),
        2
    ) AS pct_mantienen_misma_fecha_inicio,

    COUNT(*) FILTER (
        WHERE inicio_mora_anterior = 0
    ) AS venian_de_inicio_mora_cero,

    COUNT(*) FILTER (
        WHERE inicio_mora_anterior = 0
          AND fecha_inicio_mora = fecha_inicio_mora_anterior
    ) AS desde_cero_mantienen_misma_fecha

FROM serie

WHERE delinquency = 1
  AND delinquency_anterior = 1
  AND inicio_mora IS NOT NULL
  AND inicio_mora_anterior IS NOT NULL
  AND DATE_DIFF(
        'month',
        referencia_anterior,
        referencia
      ) = 1

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,clientes_en_mora_dos_meses_seguidos,mantienen_misma_fecha_inicio,pct_mantienen_misma_fecha_inicio,venian_de_inicio_mora_cero,desde_cero_mantienen_misma_fecha
0,202104,Master,388,274,70.62,0,0
1,202104,Visa,589,383,65.03,0,0
2,202105,Master,346,263,76.01,0,0
3,202105,Visa,578,387,66.96,0,0
4,202106,Master,307,237,77.20,0,0
5,202106,Visa,506,375,74.11,57,55
6,202107,Master,220,167,75.91,1,0
7,202107,Visa,360,264,73.33,0,0
8,202108,Master,203,154,75.86,0,0
9,202108,Visa,401,243,60.60,0,0


Esta salida no confirma una regla tan fuerte como en las fechas anteriores:
- Entre 60% y 77% de quienes continúan en mora conservan la misma fecha de inicio.
- El resto cambia de fecha aunque delinquency siga en 1. Podría haber una nueva mora entre ambas fotografías, una actualización del dato o alguna regla que todavía no conocemos.
- Solo tenemos 57 casos Visa donde el mes anterior era cero y la mora continuó; 55 conservaron la fecha. Es evidencia favorable, pero insuficiente para explicar los 6.471 ceros totales.
Además, que casi ningún cero aparezca como “mes anterior” sugiere que podrían estar concentrados en un mes determinado. Debemos verlo antes de interpretar el cero:

In [56]:
%%sql

WITH mora AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_Finiciomora AS inicio_mora,
        Master_delinquency AS delinquency
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_Finiciomora AS inicio_mora,
        Visa_delinquency AS delinquency
    FROM competencia_01
)

SELECT
    foto_mes,
    marca,

    COUNT(*) FILTER (
        WHERE delinquency = 1
    ) AS clientes_en_mora,

    COUNT(*) FILTER (
        WHERE delinquency = 1
          AND inicio_mora IS NULL
    ) AS mora_sin_fecha_inicio,

    COUNT(*) FILTER (
        WHERE delinquency = 1
          AND inicio_mora = 0
    ) AS mora_con_inicio_cero,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE delinquency = 1
              AND inicio_mora = 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE delinquency = 1
            ),
            0
        ),
        2
    ) AS pct_mora_con_inicio_cero,

    COUNT(*) FILTER (
        WHERE delinquency = 1
          AND inicio_mora > 0
    ) AS mora_con_inicio_positivo,

    MEDIAN(inicio_mora) FILTER (
        WHERE delinquency = 1
          AND inicio_mora > 0
    ) AS mediana_dias_positivos,

    MAX(inicio_mora) FILTER (
        WHERE delinquency = 1
          AND inicio_mora > 0
    ) AS maximo_dias_positivos

FROM mora

GROUP BY
    foto_mes,
    marca

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,clientes_en_mora,mora_sin_fecha_inicio,mora_con_inicio_cero,pct_mora_con_inicio_cero,mora_con_inicio_positivo,mediana_dias_positivos,maximo_dias_positivos
0,202103,Master,1127,0,0,0.00,1127,16.0,169
1,202103,Visa,1479,0,0,0.00,1479,17.0,178
2,202104,Master,972,0,0,0.00,972,18.0,207
3,202104,Visa,1427,0,0,0.00,1427,19.0,208
4,202105,Master,818,0,0,0.00,818,21.0,230
5,202105,Visa,4344,0,3222,74.17,1122,22.0,239
6,202106,Master,652,0,1,0.15,651,23.0,260
7,202106,Visa,928,0,0,0.00,928,24.0,269
8,202107,Master,685,0,0,0.00,685,19.0,291
9,202107,Visa,981,0,0,0.00,981,20.0,300


Dato bancario: cuando pasan ciertos días en mora se da un alta en boletín o algo así. Qué pasa? quizás la persona sale de mora, porque paga pero sigue en Boletin y no puede usar la tarjeta.

Esto cambia cómo debemos leer las variables.
Podrían representar estados diferentes:
- delinquency = 1: el cliente está actualmente en mora.
- Finiciomora: inicio del episodio de mora actual.
- status: situación administrativa de la tarjeta, que puede seguir bloqueada o en boletín aunque la deuda se regularice.
- consumo: puede seguir en cero porque el cliente ya no puede operar.
- fultimo_cierre: puede continuar actualizándose por cuotas, saldo financiado, intereses o comisiones.
También explica por qué algunos clientes tienen delinquency = 1 en dos fotos consecutivas, pero cambia Finiciomora: pudieron pagar y volver a entrar en mora entre ambas fotografías.
La investigación correcta ahora sería seguir estas transiciones:

- sin mora → entra en mora
- mora → continúa en mora
- mora → sale de mora
- sin mora → continúa sin mora

Y para quienes pasan de mora → sin mora, observar:
- Qué status tienen.
- Si vuelven a consumir.
- Si el cierre continúa.
- Si posteriormente vuelve la mora.

In [57]:
%%sql

WITH mayo AS (

    SELECT DISTINCT numero_de_cliente
    FROM competencia_01
    WHERE foto_mes = 202105
      AND Visa_delinquency = 1
      AND Visa_Finiciomora = 0
),

agosto AS (

    SELECT DISTINCT numero_de_cliente
    FROM competencia_01
    WHERE foto_mes = 202108
      AND Visa_delinquency = 1
      AND Visa_Finiciomora = 0
)

SELECT
    (SELECT COUNT(*) FROM mayo) AS clientes_cero_mayo,
    (SELECT COUNT(*) FROM agosto) AS clientes_cero_agosto,

    COUNT(*) AS clientes_cero_en_ambos_meses,

    ROUND(
        100.0 * COUNT(*)
        / NULLIF((SELECT COUNT(*) FROM mayo), 0),
        2
    ) AS pct_de_mayo_que_repite_en_agosto,

    ROUND(
        100.0 * COUNT(*)
        / NULLIF((SELECT COUNT(*) FROM agosto), 0),
        2
    ) AS pct_de_agosto_que_ya_estaba_en_mayo

FROM mayo
INNER JOIN agosto
    USING (numero_de_cliente);

,clientes_cero_mayo,clientes_cero_agosto,clientes_cero_en_ambos_meses,pct_de_mayo_que_repite_en_agosto,pct_de_agosto_que_ya_estaba_en_mayo
0,3222,3249,1860,57.73,57.25


El solapamiento de aproximadamente 58% es muy alto:
- 1.860 clientes tuvieron Finiciomora = 0 tanto en mayo como en agosto.
- Si fueran dos grupos aleatorios dentro de unos 155.000 clientes Visa, esperaríamos solo decenas de coincidencias, no 1.860.
- Esto apunta a un grupo recurrente afectado por un proceso trimestral o una condición particular de Visa.
- No parece una entrada en mora completamente aleatoria, pero todavía no distingue entre fenómeno bancario y proceso de datos.
La siguiente pregunta es qué ocurrió con los 3.222 clientes de mayo antes y después de ese cero:

In [58]:
%%sql

WITH cohorte_mayo AS (

    SELECT DISTINCT numero_de_cliente
    FROM competencia_01
    WHERE foto_mes = 202105
      AND Visa_delinquency = 1
      AND Visa_Finiciomora = 0
),

observaciones AS (

    SELECT
        c.numero_de_cliente,
        'ABRIL_ANTES' AS momento,
        a.numero_de_cliente IS NOT NULL AS cliente_aparece,
        a.Visa_delinquency AS delinquency

    FROM cohorte_mayo c

    LEFT JOIN competencia_01 a
        ON c.numero_de_cliente = a.numero_de_cliente
       AND a.foto_mes = 202104

    UNION ALL

    SELECT
        c.numero_de_cliente,
        'JUNIO_DESPUES' AS momento,
        j.numero_de_cliente IS NOT NULL AS cliente_aparece,
        j.Visa_delinquency AS delinquency

    FROM cohorte_mayo c

    LEFT JOIN competencia_01 j
        ON c.numero_de_cliente = j.numero_de_cliente
       AND j.foto_mes = 202106
),

estados AS (

    SELECT
        momento,

        CASE
            WHEN NOT cliente_aparece
                THEN 'CLIENTE_NO_APARECE'

            WHEN delinquency IS NULL
                THEN 'VISA_SIN_DATO'

            WHEN delinquency = 0
                THEN 'SIN_MORA'

            WHEN delinquency = 1
                THEN 'EN_MORA'

            ELSE 'OTRO_VALOR'
        END AS estado

    FROM observaciones
),

conteo AS (

    SELECT
        momento,
        estado,
        COUNT(*) AS clientes

    FROM estados

    GROUP BY
        momento,
        estado
)

SELECT
    momento,
    estado,
    clientes,

    ROUND(
        100.0 * clientes
        / SUM(clientes) OVER (
            PARTITION BY momento
        ),
        2
    ) AS porcentaje

FROM conteo

ORDER BY
    momento,
    clientes DESC;

,momento,estado,clientes,porcentaje
0,ABRIL_ANTES,SIN_MORA,3161,98.11
1,ABRIL_ANTES,EN_MORA,48,1.49
2,ABRIL_ANTES,CLIENTE_NO_APARECE,12,0.37
3,ABRIL_ANTES,VISA_SIN_DATO,1,0.03
4,JUNIO_DESPUES,SIN_MORA,3132,97.21
5,JUNIO_DESPUES,EN_MORA,57,1.77
6,JUNIO_DESPUES,CLIENTE_NO_APARECE,23,0.71
7,JUNIO_DESPUES,VISA_SIN_DATO,10,0.31


Esta tabla pasa el control: abril y junio suman exactamente 3.222 clientes y 100%.
El comportamiento de mayo es muy claro:
- 98,11% no estaba en mora en abril.
- En mayo aparece con delinquency = 1 y Finiciomora = 0.
- En junio, 97,21% vuelve a estar sin mora.
- Solo 1,77% continúa en mora.
En esta cohorte, Finiciomora = 0 muestra un patrón compatible con una mora nueva y transitoria en la fecha de la fotografía, no con una simple ausencia de información. Esto describe el comportamiento observado, pero no identifica todavía el significado exacto del código cero.
Pero que aparezca masivamente en mayo y agosto, y que 58% sean los mismos clientes, sugiere un efecto periódico: fecha de pago, proceso de Visa o momento de extracción. No debemos corregirlo ni convertirlo en nulo; puede ser una señal real del ciclo operativo.
La hipótesis de boletín podría aplicarse después: aunque en junio delinquency vuelva a cero, el status podría seguir bloqueado. Eso se analizará con status.

Control:
- Los clientes deben sumar 3.249.
- Los porcentajes deben sumar aproximadamente 100%.
Si alrededor del 98% estaba sin mora en julio, agosto reproduce el mismo patrón transitorio observado en mayo.

In [59]:
%%sql

WITH cohorte_agosto AS (

    SELECT DISTINCT numero_de_cliente
    FROM competencia_01
    WHERE foto_mes = 202108
      AND Visa_delinquency = 1
      AND Visa_Finiciomora = 0
),

estado_julio AS (

    SELECT
        c.numero_de_cliente,

        CASE
            WHEN j.numero_de_cliente IS NULL
                THEN 'CLIENTE_NO_APARECE'

            WHEN j.Visa_delinquency IS NULL
                THEN 'VISA_SIN_DATO'

            WHEN j.Visa_delinquency = 0
                THEN 'SIN_MORA'

            WHEN j.Visa_delinquency = 1
                THEN 'EN_MORA'

            ELSE 'OTRO_VALOR'
        END AS estado_en_julio

    FROM cohorte_agosto c

    LEFT JOIN competencia_01 j
        ON c.numero_de_cliente = j.numero_de_cliente
       AND j.foto_mes = 202107
),

conteo AS (

    SELECT
        estado_en_julio,
        COUNT(*) AS clientes

    FROM estado_julio

    GROUP BY
        estado_en_julio
)

SELECT
    estado_en_julio,
    clientes,

    ROUND(
        100.0 * clientes
        / SUM(clientes) OVER (),
        2
    ) AS porcentaje

FROM conteo

ORDER BY
    clientes DESC;

,estado_en_julio,clientes,porcentaje
0,SIN_MORA,3202,98.55
1,EN_MORA,32,0.98
2,CLIENTE_NO_APARECE,13,0.40
3,VISA_SIN_DATO,2,0.06


Agosto replica el comienzo del patrón de mayo:
- 98,55% estaba sin mora en julio.
- En agosto aparece con delinquency = 1 y Finiciomora = 0.
- Solo 0,98% ya estaba en mora.
No tenemos septiembre para comprobar si salen de mora, pero mayo mostró que casi todos regresaron a delinquency = 0. Como además 58% de los clientes se repite entre mayo y agosto, es muy probable que exista un efecto periódico del ciclo de Visa.
Decisión sobre Finiciomora:
- NULL con delinquency = 0: no corresponde una fecha; no imputar.
- NULL con delinquency = NULL: falta el bloque de Visa/Master.
- 0 con delinquency = 1: mora iniciada en la referencia o estado transitorio del ciclo; conservar.
- Positivo: antigüedad del episodio de mora.
- Crear más adelante mora_inicio_cero como indicador separado.

## Visa_mpagado

Primero queremos separar si el cambio de mayo y agosto proviene de nulos, ceros, positivos o negativos.

In [60]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    COUNT(Visa_mpagado) AS pagos_informados,

    COUNT(*) FILTER (
        WHERE Visa_mpagado IS NULL
    ) AS pagos_nulos,

    COUNT(*) FILTER (
        WHERE Visa_mpagado = 0
    ) AS pagos_cero,

    COUNT(*) FILTER (
        WHERE Visa_mpagado > 0
    ) AS pagos_positivos,

    COUNT(*) FILTER (
        WHERE Visa_mpagado < 0
    ) AS pagos_negativos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mpagado IS NULL
        ) / COUNT(*),
        2
    ) AS pct_nulos_sobre_total,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mpagado = 0
        ) / NULLIF(COUNT(Visa_mpagado), 0),
        2
    ) AS pct_ceros_entre_informados,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mpagado > 0
        ) / NULLIF(COUNT(Visa_mpagado), 0),
        2
    ) AS pct_positivos_entre_informados

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,pagos_informados,pagos_nulos,pagos_cero,pagos_positivos,pagos_negativos,pct_nulos_sobre_total,pct_ceros_entre_informados,pct_positivos_entre_informados
0,202103,162900,155006,7894,141194,13812,0,4.85,91.09,8.91
1,202104,163284,155297,7987,141872,13425,0,4.89,91.36,8.64
2,202105,163768,155729,8039,114107,41622,0,4.91,73.27,26.73
3,202106,164114,155992,8122,140067,15925,0,4.95,89.79,10.21
4,202107,164348,157116,7232,142032,15084,0,4.40,90.40,9.60
5,202108,164647,156735,7912,113872,42863,0,4.81,72.65,27.35


Esta primera tabla responde solamente: ¿qué tipo de valor reemplaza a los ceros en mayo y agosto?

La salida es clara:
- Los nulos permanecen estables alrededor del 5%: no explican el cambio.
- No hay valores negativos.
- En meses normales, solo 9–10% tiene pago positivo.
- En mayo y agosto sube a 27%.
- Aproximadamente 27.000 clientes pasan de cero a positivo en esos meses.
Es un patrón sistemático, pero todavía no sabemos si es comercial o de carga. Ahora comprobamos si también aparece en pagos en pesos o dólares:

In [61]:
%%sql

SELECT
    foto_mes,

    COUNT(*) FILTER (
        WHERE Visa_mpagado > 0
    ) AS mpagado_positivo,

    COUNT(*) FILTER (
        WHERE Visa_mpagospesos <> 0
    ) AS pagos_pesos_no_cero,

    COUNT(*) FILTER (
        WHERE Visa_mpagosdolares <> 0
    ) AS pagos_dolares_no_cero,

    COUNT(*) FILTER (
        WHERE Visa_mpagado > 0
          AND (
              COALESCE(Visa_mpagospesos, 0) <> 0
              OR COALESCE(Visa_mpagosdolares, 0) <> 0
          )
    ) AS mpagado_positivo_con_componente,

    COUNT(*) FILTER (
        WHERE Visa_mpagado > 0
          AND COALESCE(Visa_mpagospesos, 0) = 0
          AND COALESCE(Visa_mpagosdolares, 0) = 0
    ) AS mpagado_positivo_sin_componentes,

    COUNT(*) FILTER (
        WHERE Visa_mpagado = 0
          AND (
              COALESCE(Visa_mpagospesos, 0) <> 0
              OR COALESCE(Visa_mpagosdolares, 0) <> 0
          )
    ) AS mpagado_cero_con_componentes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mpagado > 0
              AND (
                  COALESCE(Visa_mpagospesos, 0) <> 0
                  OR COALESCE(Visa_mpagosdolares, 0) <> 0
              )
        )
        / NULLIF(
            COUNT(*) FILTER (WHERE Visa_mpagado > 0),
            0
        ),
        2
    ) AS pct_positivo_con_componente

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,mpagado_positivo,pagos_pesos_no_cero,pagos_dolares_no_cero,mpagado_positivo_con_componente,mpagado_positivo_sin_componentes,mpagado_cero_con_componentes,pct_positivo_con_componente
0,202103,13812,136664,15575,13465,347,123214,97.49
1,202104,13425,137784,13883,13143,282,124652,97.90
2,202105,41622,138256,14333,40891,731,97390,98.24
3,202106,15925,139189,14240,15646,279,123553,98.25
4,202107,15084,139998,17202,14785,299,125228,98.02
5,202108,42863,140324,15025,42245,618,98098,98.56


- Los pagos en pesos y dólares permanecen relativamente estables.
- El salto de mayo y agosto ocurre específicamente en Visa_mpagado.
- Visa_mpagado = 0 no significa “no realizó pagos”: entre 97.000 y 125.000 clientes tienen cero allí, pero valores distintos de cero en las otras variables.
- Por lo tanto, mpagospesos y mpagosdolares no son componentes que sumen mpagado; representan conceptos o ciclos diferentes.
- Todavía no sabemos si el patrón es comercial o del proceso de carga.

In [62]:
%%sql

WITH comportamiento AS (

    SELECT
        numero_de_cliente,

        MAX(
            CASE
                WHEN foto_mes = 202105
                 AND Visa_mpagado > 0
                THEN 1
                ELSE 0
            END
        ) AS positivo_mayo,

        MAX(
            CASE
                WHEN foto_mes = 202108
                 AND Visa_mpagado > 0
                THEN 1
                ELSE 0
            END
        ) AS positivo_agosto

    FROM competencia_01

    WHERE foto_mes IN (202105, 202108)

    GROUP BY numero_de_cliente
)

SELECT
    SUM(positivo_mayo) AS positivos_mayo,

    SUM(positivo_agosto) AS positivos_agosto,

    SUM(
        CASE
            WHEN positivo_mayo = 1
             AND positivo_agosto = 1
            THEN 1
            ELSE 0
        END
    ) AS positivos_en_ambos_meses,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN positivo_mayo = 1
                 AND positivo_agosto = 1
                THEN 1
                ELSE 0
            END
        ) / NULLIF(SUM(positivo_mayo), 0),
        2
    ) AS pct_mayo_que_repite_agosto,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN positivo_mayo = 1
                 AND positivo_agosto = 1
                THEN 1
                ELSE 0
            END
        ) / NULLIF(SUM(positivo_agosto), 0),
        2
    ) AS pct_agosto_que_ya_estaba_mayo

FROM comportamiento;

,positivos_mayo,positivos_agosto,positivos_en_ambos_meses,pct_mayo_que_repite_agosto,pct_agosto_que_ya_estaba_mayo
0,41622.0,42863.0,36681.0,88.13,85.58


- 36.681 clientes son positivos en ambos meses.
- El 88,13% de los positivos de mayo reaparece en agosto.
- De todos los clientes positivos en al menos uno de esos meses, aproximadamente 76,7% aparece en ambos.
Esto sugiere una cohorte estable de clientes, no dos aumentos independientes. Puede ser un segmento de clientes activos combinado con un ciclo trimestral o con consumo estacional. Todavía no permite elegir entre esas explicaciones.

In [63]:
%%sql

WITH cohorte_mayo_agosto AS (

    SELECT
        numero_de_cliente

    FROM competencia_01

    GROUP BY numero_de_cliente

    HAVING MAX(
        CASE
            WHEN foto_mes = 202105
             AND Visa_mpagado > 0
            THEN 1
            ELSE 0
        END
    ) = 1

    AND MAX(
        CASE
            WHEN foto_mes = 202108
             AND Visa_mpagado > 0
            THEN 1
            ELSE 0
        END
    ) = 1
)

SELECT
    d.foto_mes,

    COUNT(*) AS clientes_de_la_cohorte,

    COUNT(*) FILTER (
        WHERE d.Visa_mpagado IS NULL
    ) AS mpagado_nulo,

    COUNT(*) FILTER (
        WHERE d.Visa_mpagado = 0
    ) AS mpagado_cero,

    COUNT(*) FILTER (
        WHERE d.Visa_mpagado > 0
    ) AS mpagado_positivo,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE d.Visa_mpagado > 0
        ) / NULLIF(COUNT(d.Visa_mpagado), 0),
        2
    ) AS pct_mpagado_positivo,

    COUNT(*) FILTER (
        WHERE d.Visa_mconsumototal > 0
    ) AS consumo_positivo,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE d.Visa_mconsumototal > 0
        ) / NULLIF(COUNT(d.Visa_mconsumototal), 0),
        2
    ) AS pct_consumo_positivo

FROM competencia_01 d

INNER JOIN cohorte_mayo_agosto c
    USING (numero_de_cliente)

GROUP BY d.foto_mes

ORDER BY d.foto_mes;

,foto_mes,clientes_de_la_cohorte,mpagado_nulo,mpagado_cero,mpagado_positivo,pct_mpagado_positivo,consumo_positivo,pct_consumo_positivo
0,202103,36384,11,24304,12069,33.18,33784,93.52
1,202104,36550,5,24663,11882,32.51,33981,93.43
2,202105,36681,0,0,36681,100.00,34140,93.08
3,202106,36681,1,23320,13360,36.42,34261,93.41
4,202107,36681,0,23703,12978,35.38,34108,93.05
5,202108,36681,0,0,36681,100.00,34183,93.19


Hay dos lecturas importantes:
- El porcentaje con consumo positivo permanece prácticamente fijo en 93%. No aparecen más consumidores en mayo ni julio.
- Dentro de esta cohorte, mpagado es positivo en aproximadamente un tercio durante meses normales.
Importante: el 100% de mayo y agosto está garantizado porque construimos la cohorte eligiendo clientes positivos en ambos meses. No debemos tomar ese 100% como un descubrimiento.
La hipótesis de consumo por el aguinaldo o hot sale (aunque fue en mayo) todavía puede sostenerse si en julio aumentó el monto consumido, aunque no aumentara la cantidad de consumidores. Veamos la distribución de los importes:

In [64]:
%%sql

SELECT
    foto_mes,

    COUNT(Visa_mconsumototal) AS consumos_informados,

    COUNT(*) FILTER (
        WHERE Visa_mconsumototal > 0
    ) AS consumos_positivos,

    ROUND(
        MEDIAN(Visa_mconsumototal) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS mediana_consumo_positivo,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumototal, 0.75
        ) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS percentil_75,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumototal, 0.90
        ) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS percentil_90,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumototal, 0.99
        ) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS percentil_99

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,consumos_informados,consumos_positivos,mediana_consumo_positivo,percentil_75,percentil_90,percentil_99
0,202103,140975,125245,11822.66,28484.99,56711.51,172571.67
1,202104,141494,125874,12298.11,29729.27,58674.20,175743.03
2,202105,142130,125757,12339.32,29892.95,59512.74,187804.05
3,202106,142878,127287,14499.00,35144.97,69799.75,226669.92
4,202107,144147,126151,12934.09,31705.70,64000.06,204087.26
5,202108,143677,126984,13826.56,33821.01,67834.47,212544.71


In [65]:
%%sql

SELECT
    foto_mes,

    COUNT(Visa_mconsumototal) AS consumos_informados,

    COUNT(*) FILTER (
        WHERE Visa_mconsumototal > 0
    ) AS consumos_positivos,

    ROUND(
        MEDIAN(Visa_mconsumototal) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS mediana_consumo_positivo,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumototal, 0.75
        ) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS percentil_75,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumototal, 0.90
        ) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS percentil_90,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumototal, 0.99
        ) FILTER (
            WHERE Visa_mconsumototal > 0
        ),
        2
    ) AS percentil_99

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,consumos_informados,consumos_positivos,mediana_consumo_positivo,percentil_75,percentil_90,percentil_99
0,202103,140975,125245,11822.66,28484.99,56711.51,172571.67
1,202104,141494,125874,12298.11,29729.27,58674.20,175743.03
2,202105,142130,125757,12339.32,29892.95,59512.74,187804.05
3,202106,142878,127287,14499.00,35144.97,69799.75,226669.92
4,202107,144147,126151,12934.09,31705.70,64000.06,204087.26
5,202108,143677,126984,13826.56,33821.01,67834.47,212544.71


In [66]:
%%sql

WITH historia AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        Visa_mpagado,
        Visa_mconsumototal AS consumo_actual,

        LAG(foto_mes) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS mes_anterior,

        LAG(Visa_mconsumototal) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS consumo_anterior

    FROM competencia_01
)

SELECT
    foto_mes,

    CASE
        WHEN Visa_mpagado > 0 THEN 'MPAGADO_POSITIVO'
        WHEN Visa_mpagado = 0 THEN 'MPAGADO_CERO'
    END AS estado_mpagado,

    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE consumo_anterior > 0
        ) / NULLIF(COUNT(consumo_anterior), 0),
        2
    ) AS pct_con_consumo_anterior,

    ROUND(
        MEDIAN(consumo_anterior) FILTER (
            WHERE consumo_anterior > 0
        ),
        2
    ) AS mediana_consumo_anterior,

    ROUND(
        MEDIAN(consumo_actual) FILTER (
            WHERE consumo_actual > 0
        ),
        2
    ) AS mediana_consumo_actual

FROM historia

WHERE foto_mes BETWEEN 202104 AND 202108
  AND mes_anterior = foto_mes - 1
  AND Visa_mpagado IS NOT NULL

GROUP BY
    foto_mes,
    estado_mpagado

ORDER BY
    foto_mes,
    estado_mpagado;

,foto_mes,estado_mpagado,clientes,pct_con_consumo_anterior,mediana_consumo_anterior,mediana_consumo_actual
0,202104,MPAGADO_CERO,140562,88.58,11627.06,12051.07
1,202104,MPAGADO_POSITIVO,13378,92.96,14046.34,15190.38
2,202105,MPAGADO_CERO,112827,87.56,11457.62,11502.74
3,202105,MPAGADO_POSITIVO,41454,92.86,14604.91,14571.89
4,202106,MPAGADO_CERO,138649,88.19,12072.65,14178.00
5,202106,MPAGADO_POSITIVO,15855,92.50,15189.54,17495.72
6,202107,MPAGADO_CERO,141025,88.85,14194.20,12618.52
7,202107,MPAGADO_POSITIVO,15021,92.90,17619.28,16061.13
8,202108,MPAGADO_CERO,112672,86.69,11973.68,12905.90
9,202108,MPAGADO_POSITIVO,42664,92.18,15433.65,16218.54


In [67]:
%%sql

SELECT
    foto_mes,

    COUNT(*) FILTER (
        WHERE Visa_mpagado > 0
    ) AS pagos_positivos,

    ROUND(
        MEDIAN(Visa_mpagado) FILTER (
            WHERE Visa_mpagado > 0
        ),
        2
    ) AS mediana_pago_positivo,

    ROUND(
        QUANTILE_CONT(
            Visa_mpagado, 0.75
        ) FILTER (
            WHERE Visa_mpagado > 0
        ),
        2
    ) AS percentil_75,

    ROUND(
        QUANTILE_CONT(
            Visa_mpagado, 0.90
        ) FILTER (
            WHERE Visa_mpagado > 0
        ),
        2
    ) AS percentil_90,

    ROUND(
        QUANTILE_CONT(
            Visa_mpagado, 0.99
        ) FILTER (
            WHERE Visa_mpagado > 0
        ),
        2
    ) AS percentil_99

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,pagos_positivos,mediana_pago_positivo,percentil_75,percentil_90,percentil_99
0,202103,13812,24847.73,48983.58,87875.49,229319.69
1,202104,13425,25716.14,51037.96,89810.06,253218.47
2,202105,41622,24582.57,49485.02,88933.30,245323.77
3,202106,15925,29879.61,58316.64,105024.86,304323.54
4,202107,15084,28941.54,55997.22,99153.39,287115.11
5,202108,42863,28385.00,56322.52,102261.55,290594.29


La variación está en cuántos clientes tienen valor positivo, no en importes extraordinarios:
- Mayo triplica los positivos, pero su mediana ($24.583) es similar a marzo–abril.
- Agosto también triplica los positivos, pero su distribución es semejante a junio–julio.
- No hay evidencia de valores extremos o inflación como causa principal del PSI.
- Es una cohorte muy estable que reaparece cada tres meses.
- El consumo es mayor en esa cohorte, pero no explica por sí solo la periodicidad.
Decisión:
- Conservar Visa_mpagado.
- Mantener los ceros como ceros.
- No imputar ni eliminar la variable por su PSI alto.
- Más adelante crear o probar un indicador Visa_mpagado > 0.
- Usar mayo como período especialmente importante para validar un modelo destinado a agosto.
- Dejar la causa exacta como “patrón periódico posiblemente operativo/comercial”; no está demostrada.

## ccajas_depositos

In [68]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    COUNT(ccajas_depositos) AS valores_informados,

    COUNT(*) FILTER (
        WHERE ccajas_depositos IS NULL
    ) AS valores_nulos,

    COUNT(*) FILTER (
        WHERE ccajas_depositos = 0
    ) AS valores_cero,

    COUNT(*) FILTER (
        WHERE ccajas_depositos > 0
    ) AS valores_positivos,

    COUNT(*) FILTER (
        WHERE ccajas_depositos < 0
    ) AS valores_negativos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ccajas_depositos = 0
        ) / NULLIF(COUNT(ccajas_depositos), 0),
        2
    ) AS pct_ceros_entre_informados,

    ROUND(
        MEDIAN(ccajas_depositos) FILTER (
            WHERE ccajas_depositos > 0
        ),
        2
    ) AS mediana_entre_positivos,

    ROUND(
        QUANTILE_CONT(
            ccajas_depositos, 0.90
        ) FILTER (
            WHERE ccajas_depositos > 0
        ),
        2
    ) AS percentil_90_positivos,

    MAX(ccajas_depositos) AS maximo

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,valores_informados,valores_nulos,valores_cero,valores_positivos,valores_negativos,pct_ceros_entre_informados,mediana_entre_positivos,percentil_90_positivos,maximo
0,202103,162900,162900,0,156414,6486,0,96.02,1.0,2.0,46
1,202104,163284,163284,0,161109,2175,0,98.67,1.0,2.0,8
2,202105,163768,163768,0,163768,0,0,100.00,NaN,NaN,0
3,202106,164114,164114,0,163283,831,0,99.49,1.0,1.0,5
4,202107,164348,164348,0,159072,5276,0,96.79,1.0,2.0,41
5,202108,164647,164647,0,158819,5828,0,96.46,1.0,2.0,33


In [69]:
%%sql

SELECT
    foto_mes,

    COUNT(*) FILTER (
        WHERE ccajas_depositos > 0
    ) AS clientes_con_depositos,

    SUM(ccajas_depositos) AS cantidad_depositos,

    COUNT(*) FILTER (
        WHERE ccajas_extracciones > 0
    ) AS clientes_con_extracciones,

    SUM(ccajas_extracciones) AS cantidad_extracciones,

    COUNT(*) FILTER (
        WHERE ccajas_consultas > 0
    ) AS clientes_con_consultas,

    SUM(ccajas_consultas) AS cantidad_consultas,

    COUNT(*) FILTER (
        WHERE ccajas_otras > 0
    ) AS clientes_con_otras,

    SUM(ccajas_otras) AS cantidad_otras,

    COUNT(*) FILTER (
        WHERE ccajas_transacciones > 0
    ) AS clientes_con_transacciones,

    SUM(ccajas_transacciones) AS cantidad_transacciones

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes_con_depositos,cantidad_depositos,clientes_con_extracciones,cantidad_extracciones,clientes_con_consultas,cantidad_consultas,clientes_con_otras,cantidad_otras,clientes_con_transacciones,cantidad_transacciones
0,202103,6486,8949.0,8953,11895.0,6204,11246.0,2057,3644.0,16007,16007.0
1,202104,2175,2498.0,7976,10552.0,5474,9869.0,4973,7851.0,13893,13893.0
2,202105,0,0.0,7214,9602.0,4964,8860.0,6490,10389.0,13016,13016.0
3,202106,831,922.0,7321,9800.0,5050,9278.0,5966,9503.0,13211,13211.0
4,202107,5276,7134.0,8601,11384.0,5698,10109.0,1971,4406.0,14557,14557.0
5,202108,5828,7795.0,9497,12560.0,6227,10906.0,1899,3688.0,15788,15788.0


La caída no es general del canal de cajas:
- Extracciones y consultas continúan todos los meses.
- ccajas_transacciones baja moderadamente, pero nunca desaparece.
- Cuando depósitos cae, ccajas_otras sube fuertemente.
- En julio, depósitos reaparece y otras vuelve a bajar.
Esto sugiere una reclasificación temporal de operaciones entre depositos y otras, especialmente entre abril y junio.
Además, cantidad_transacciones coincide exactamente con clientes_con_transacciones. Eso indica que ccajas_transacciones solamente toma 0 o 1; probablemente funciona como indicador de uso del canal, aunque su nombre comience con c.
Comprobemos si depositos + otras se mantiene estable:

In [70]:
%%sql

SELECT
    foto_mes,

    SUM(ccajas_depositos) AS depositos,

    SUM(ccajas_otras) AS otras,

    SUM(
        COALESCE(ccajas_depositos, 0)
        + COALESCE(ccajas_otras, 0)
    ) AS depositos_mas_otras,

    COUNT(*) FILTER (
        WHERE ccajas_depositos > 0
           OR ccajas_otras > 0
    ) AS clientes_con_depositos_u_otras,

    COUNT(*) FILTER (
        WHERE ccajas_depositos > 0
          AND ccajas_otras > 0
    ) AS clientes_con_ambas,

    ROUND(
        100.0 * SUM(ccajas_depositos)
        / NULLIF(
            SUM(
                COALESCE(ccajas_depositos, 0)
                + COALESCE(ccajas_otras, 0)
            ),
            0
        ),
        2
    ) AS pct_clasificado_como_deposito

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,depositos,otras,depositos_mas_otras,clientes_con_depositos_u_otras,clientes_con_ambas,pct_clasificado_como_deposito
0,202103,8949.0,3644.0,12593.0,7939,604,71.06
1,202104,2498.0,7851.0,10349.0,6502,646,24.14
2,202105,0.0,10389.0,10389.0,6490,0,0.00
3,202106,922.0,9503.0,10425.0,6473,324,8.84
4,202107,7134.0,4406.0,11540.0,6704,543,61.82
5,202108,7795.0,3688.0,11483.0,7131,596,67.88


La evidencia es muy fuerte a favor de una reclasificación:
- La proporción catalogada como depósito pasa de 71% en marzo a 0% en mayo y vuelve a 68% en agosto.
- Sin embargo, depositos + otras se mantiene alrededor de 10.000–12.500 operaciones.
- La cantidad de clientes con alguna de las dos operaciones también permanece relativamente estable.
Por lo tanto, mayo no significa que nadie realizó depósitos. Probablemente esas operaciones fueron registradas temporalmente como otras.
Decisión para ccajas_depositos:
- No imputar.
- No interpretar el cero de mayo como comportamiento real del cliente.
- Marcar ccajas_depositos y ccajas_otras como variables con cambio de clasificación.
- Más adelante crear una variable estable: ccajas_depositos_otras = ccajas_depositos + ccajas_otras

Comparar el modelo usando la suma y excluyendo los dos componentes originales.
El PSI alto de ccajas_depositos queda explicado principalmente por esta reclasificación probable. Una reclasificación significa que la operación sigue existiendo, pero el sistema la guarda bajo otra categoría.
Ejemplo:
- En marzo, un depósito por ventanilla se registra en ccajas_depositos.
- En mayo, una operación equivalente podría haberse registrado en ccajas_otras.
- En julio vuelve a registrarse como depósito.

## Visa_mconsumosdolares

In [71]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    COUNT(Visa_mconsumosdolares) AS valores_informados,

    COUNT(*) FILTER (
        WHERE Visa_mconsumosdolares IS NULL
    ) AS valores_nulos,

    COUNT(*) FILTER (
        WHERE Visa_mconsumosdolares = 0
    ) AS valores_cero,

    COUNT(*) FILTER (
        WHERE Visa_mconsumosdolares > 0
    ) AS valores_positivos,

    COUNT(*) FILTER (
        WHERE Visa_mconsumosdolares < 0
    ) AS valores_negativos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares = 0
        ) / NULLIF(COUNT(Visa_mconsumosdolares), 0),
        2
    ) AS pct_ceros_entre_informados,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares <> 0
        ) / NULLIF(COUNT(Visa_mconsumosdolares), 0),
        2
    ) AS pct_no_cero_entre_informados,

    ROUND(
        MEDIAN(Visa_mconsumosdolares) FILTER (
            WHERE Visa_mconsumosdolares > 0
        ),
        2
    ) AS mediana_positivos,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumosdolares, 0.90
        ) FILTER (
            WHERE Visa_mconsumosdolares > 0
        ),
        2
    ) AS percentil_90_positivos,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumosdolares, 0.99
        ) FILTER (
            WHERE Visa_mconsumosdolares > 0
        ),
        2
    ) AS percentil_99_positivos

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,valores_informados,valores_nulos,valores_cero,valores_positivos,valores_negativos,pct_ceros_entre_informados,pct_no_cero_entre_informados,mediana_positivos,percentil_90_positivos,percentil_99_positivos
0,202103,162900,140975,21925,101534,39441,0,72.02,27.98,3.16,49.52,633.99
1,202104,163284,141494,21790,101418,40076,0,71.68,28.32,3.50,46.90,564.36
2,202105,163768,142130,21638,101692,40438,0,71.55,28.45,3.46,50.24,729.72
3,202106,164114,142878,21236,98735,44143,0,69.10,30.90,4.50,66.22,1021.47
4,202107,164348,144147,20201,102944,41203,0,71.42,28.58,4.42,53.44,939.54
5,202108,164647,143677,20970,102842,40835,0,71.58,28.42,4.39,58.30,981.50


Acá el PSI no parece provenir de una rotura:
- Los nulos bajan suavemente.
- La proporción de clientes con consumo en dólares permanece cerca de 28–31%.
- No hay negativos.
- Lo que cambia es el monto entre quienes consumen.
Comparando marzo con agosto:
- Mediana: USD 3,16 → USD 4,39, aproximadamente +39%.
- Percentil 90: USD 49,52 → USD 58,30.
- Percentil 99: USD 633,99 → USD 981,50, aproximadamente +55%.
El salto comienza en junio y luego disminuye un poco, pero no vuelve al nivel anterior. Como el campo está denominado en dólares, la inflación argentina no lo explica mecánicamente. Parece un cambio real en la distribución, aunque todavía debemos confirmar el significado con el diccionario.

In [72]:
%%sql

SELECT
    foto_mes,

    COUNT(*) FILTER (
        WHERE Visa_mconsumosdolares > 0
    ) AS consumidores_dolares,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 0
              AND Visa_mconsumosdolares <= 10
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumosdolares > 0
            ),
            0
        ),
        2
    ) AS pct_hasta_10,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 10
              AND Visa_mconsumosdolares <= 50
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumosdolares > 0
            ),
            0
        ),
        2
    ) AS pct_10_a_50,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 50
              AND Visa_mconsumosdolares <= 100
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumosdolares > 0
            ),
            0
        ),
        2
    ) AS pct_50_a_100,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 100
              AND Visa_mconsumosdolares <= 500
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumosdolares > 0
            ),
            0
        ),
        2
    ) AS pct_100_a_500,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 500
              AND Visa_mconsumosdolares <= 1000
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumosdolares > 0
            ),
            0
        ),
        2
    ) AS pct_500_a_1000,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 1000
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumosdolares > 0
            ),
            0
        ),
        2
    ) AS pct_mas_1000

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,consumidores_dolares,pct_hasta_10,pct_10_a_50,pct_50_a_100,pct_100_a_500,pct_500_a_1000,pct_mas_1000
0,202103,39441,70.48,19.57,4.19,4.45,0.76,0.55
1,202104,40076,69.57,21.00,4.08,4.22,0.62,0.50
2,202105,40438,69.02,20.96,4.27,4.32,0.72,0.71
3,202106,44143,65.13,22.62,5.03,5.21,0.99,1.03
4,202107,41203,68.72,20.77,4.18,4.57,0.81,0.94
5,202108,40835,67.19,21.75,4.32,4.88,0.88,0.99


El cambio no está concentrado únicamente en unos pocos valores extremos:
- Los consumos de hasta USD 10 bajan de 70,48% a 67,19%.
- Los rangos intermedios aumentan.
- Los superiores a USD 1.000 casi se duplican: 0,55% → 0,99%.
- Junio es el mes más intenso y luego hay una reversión parcial.
- Agosto conserva una distribución algo más alta que marzo.
Por eso el PSI detecta un desplazamiento real de toda la distribución. Los rangos amplios incluso ocultan movimientos dentro del grupo de hasta USD 10, donde se concentra cerca del 70%.

In [73]:
%%sql

SELECT
    foto_mes,
    'Visa' AS marca,

    COUNT(*) FILTER (
        WHERE Visa_mconsumosdolares > 0
    ) AS consumidores_dolares,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumosdolares > 0
        ) / NULLIF(COUNT(Visa_mconsumosdolares), 0),
        2
    ) AS pct_consumidores_entre_informados,

    ROUND(
        MEDIAN(Visa_mconsumosdolares) FILTER (
            WHERE Visa_mconsumosdolares > 0
        ),
        2
    ) AS mediana_positivos,

    ROUND(
        QUANTILE_CONT(
            Visa_mconsumosdolares, 0.90
        ) FILTER (
            WHERE Visa_mconsumosdolares > 0
        ),
        2
    ) AS percentil_90

FROM competencia_01

GROUP BY foto_mes

UNION ALL

SELECT
    foto_mes,
    'Master' AS marca,

    COUNT(*) FILTER (
        WHERE Master_mconsumosdolares > 0
    ) AS consumidores_dolares,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Master_mconsumosdolares > 0
        ) / NULLIF(COUNT(Master_mconsumosdolares), 0),
        2
    ) AS pct_consumidores_entre_informados,

    ROUND(
        MEDIAN(Master_mconsumosdolares) FILTER (
            WHERE Master_mconsumosdolares > 0
        ),
        2
    ) AS mediana_positivos,

    ROUND(
        QUANTILE_CONT(
            Master_mconsumosdolares, 0.90
        ) FILTER (
            WHERE Master_mconsumosdolares > 0
        ),
        2
    ) AS percentil_90

FROM competencia_01

GROUP BY foto_mes

ORDER BY
    foto_mes,
    marca;

,foto_mes,marca,consumidores_dolares,pct_consumidores_entre_informados,mediana_positivos,percentil_90
0,202103,Master,4620,7.06,10.53,168.95
1,202103,Visa,39441,27.98,3.16,49.52
2,202104,Master,4788,7.30,10.54,149.65
3,202104,Visa,40076,28.32,3.50,46.90
4,202105,Master,4896,7.36,10.95,205.93
5,202105,Visa,40438,28.45,3.46,50.24
6,202106,Master,5568,8.16,12.86,264.19
7,202106,Visa,44143,30.90,4.50,66.22
8,202107,Master,5038,7.28,11.70,200.84
9,202107,Visa,41203,28.58,4.42,53.44


El control con Master aclara bastante el diagnóstico:
- Ambas marcas aumentan en junio.
- Ambas retroceden parcialmente en julio.
- Ambas permanecen por encima de marzo–abril en agosto.
- También aumenta temporalmente la proporción de consumidores.
Por ejemplo, en junio:
- Visa: mediana USD 3,46 → USD 4,50.
- Master: mediana USD 10,95 → USD 12,86.
- Los percentiles 90 también suben fuertemente en ambas.
Esto hace poco probable una rotura exclusiva de Visa. Es más compatible con un fenómeno general de consumo, estacionalidad o contexto económico. No podemos atribuirlo específicamente al aguinaldo, pero el momento es compatible.
Decisión:
- Conservar Visa_mconsumosdolares y Master_mconsumosdolares.
- Mantener los ceros como ausencia de consumo en dólares.
- No imputar ni excluir por el PSI.
- Más adelante probar indicadores de consumo en dólares y un monto agregado entre marcas.
- Validar temporalmente su utilidad para BAJA+2.

## mcomisiones_mantenimiento

Analizamos mcomisiones_mantenimiento, cuyo PSI comienza a subir en junio y permanece medio en julio–agosto.
Primero distinguimos si cambió:
- La cantidad de clientes que paga mantenimiento.
- La proporción de ceros.
- El importe cobrado.

In [74]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    COUNT(mcomisiones_mantenimiento) AS valores_informados,

    COUNT(*) FILTER (
        WHERE mcomisiones_mantenimiento IS NULL
    ) AS valores_nulos,

    COUNT(*) FILTER (
        WHERE mcomisiones_mantenimiento = 0
    ) AS valores_cero,

    COUNT(*) FILTER (
        WHERE mcomisiones_mantenimiento > 0
    ) AS valores_positivos,

    COUNT(*) FILTER (
        WHERE mcomisiones_mantenimiento < 0
    ) AS valores_negativos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE mcomisiones_mantenimiento = 0
        ) / NULLIF(COUNT(mcomisiones_mantenimiento), 0),
        2
    ) AS pct_ceros_entre_informados,

    ROUND(
        MEDIAN(mcomisiones_mantenimiento) FILTER (
            WHERE mcomisiones_mantenimiento > 0
        ),
        2
    ) AS mediana_positivos,

    ROUND(
        QUANTILE_CONT(
            mcomisiones_mantenimiento, 0.75
        ) FILTER (
            WHERE mcomisiones_mantenimiento > 0
        ),
        2
    ) AS percentil_75,

    ROUND(
        QUANTILE_CONT(
            mcomisiones_mantenimiento, 0.90
        ) FILTER (
            WHERE mcomisiones_mantenimiento > 0
        ),
        2
    ) AS percentil_90,

    ROUND(
        QUANTILE_CONT(
            mcomisiones_mantenimiento, 0.99
        ) FILTER (
            WHERE mcomisiones_mantenimiento > 0
        ),
        2
    ) AS percentil_99

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,valores_informados,valores_nulos,valores_cero,valores_positivos,valores_negativos,pct_ceros_entre_informados,mediana_positivos,percentil_75,percentil_90,percentil_99
0,202103,162900,162900,0,109882,51800,1218,67.45,1774.04,2505.95,3413.33,3413.33
1,202104,163284,163284,0,108461,50337,4486,66.42,1774.04,2505.95,3413.33,3413.33
2,202105,163768,163768,0,112839,46414,4515,68.90,1774.04,2505.95,3413.33,3413.33
3,202106,164114,164114,0,112071,50726,1317,68.29,1947.36,2505.95,3413.33,3947.05
4,202107,164348,164348,0,113533,49601,1214,69.08,2128.85,3007.14,4095.99,4271.84
5,202108,164647,164647,0,115678,47957,1012,70.26,2128.85,3007.14,4095.99,4271.84


Esto mostrará si el drift corresponde a un aumento tarifario, a más clientes cobrados o a una anomalía de carga.

La salida es compatible con un cambio tarifario, no con una rotura:
- No hay nulos.
- La proporción de ceros cambia poco: 66–70%.
- La cantidad de clientes con comisión positiva baja moderadamente.
- Los importes forman escalones muy definidos.
El dato más claro:
Mediana marzo–mayo: 1.774,04
Mediana julio–agosto: 2.128,85

2.128,85 es exactamente un 20% mayor que 1.774,04. Lo mismo ocurre con otros niveles:
2.505,95 × 1,20 = 3.007,14
3.413,33 × 1,20 = 4.095,99
Junio parece ser el mes de transición. Los valores negativos podrían representar bonificaciones o reversos, pero no podemos afirmarlo sin diccionario.

Decisión:
- Conservar mcomisiones_mantenimiento.
- No imputar ni corregir.
- Considerar el PSI como cambio comercial/tarifario.
- Más adelante probar:
  - valor original;
  - indicador positivo/cero/negativo;
  - valor normalizado para reducir el efecto del cambio tarifario.

# EDA

In [75]:
%%sql
FROM pragma_table_info('competencia_01')

,cid,name,type,notnull,dflt_value,pk
0,0,active_quarter,BIGINT,False,None,False
1,1,cliente_vip,BIGINT,False,None,False
2,2,internet,BIGINT,False,None,False
3,3,cliente_edad,BIGINT,False,None,False
4,4,cliente_antiguedad,BIGINT,False,None,False
5,5,mrentabilidad,DOUBLE,False,None,False
6,6,mrentabilidad_annual,DOUBLE,False,None,False
7,7,mcomisiones,DOUBLE,False,None,False
8,8,mactivos_margen,DOUBLE,False,None,False
9,9,mpasivos_margen,DOUBLE,False,None,False


In [76]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'CONTINUA'
    ) AS continua,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+1'
    ) AS baja_1,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+2'
    ) AS baja_2,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE clase_ternaria = 'CONTINUA'
        ) / COUNT(*),
        3
    ) AS pct_continua,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+1'
        ) / COUNT(*),
        3
    ) AS pct_baja_1,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+2'
        ) / COUNT(*),
        3
    ) AS pct_baja_2,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE clase_ternaria IN ('BAJA+1', 'BAJA+2')
        ) / COUNT(*),
        3
    ) AS pct_bajas_totales

FROM competencia_01

WHERE foto_mes BETWEEN 202103 AND 202106

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,continua,baja_1,baja_2,pct_continua,pct_baja_1,pct_baja_2,pct_bajas_totales
0,202103,162900,160921,1019,960,98.785,0.626,0.589,1.215
1,202104,163284,161181,964,1139,98.712,0.590,0.698,1.288
2,202105,163768,161755,1143,870,98.771,0.698,0.531,1.229
3,202106,164114,162142,874,1098,98.798,0.533,0.669,1.202


- BAJA+2 es muy poco frecuente: entre 0,531% y 0,698%. El problema está fuertemente desbalanceado.
- Las bajas totales son muy estables: entre 1,202% y 1,288%.
- BAJA+1 y BAJA+2 parecen compensarse:
  - Abril y junio tienen más BAJA+2.
  - Marzo y mayo tienen más BAJA+1.
Esto sugiere que no cambia mucho la cantidad total de bajas, sino el momento en que ocurren.

Baja más dos como subconjunto de baja más uno

In [77]:
%%sql

WITH seguimiento_baja2 AS (

    SELECT
        origen.foto_mes AS mes_origen,

        COUNT(*) AS clientes_baja2,

        COUNT(*) FILTER (
            WHERE siguiente.clase_ternaria = 'BAJA+1'
        ) AS baja2_que_pasan_a_baja1

    FROM competencia_01 origen

    LEFT JOIN competencia_01 siguiente
        ON origen.numero_de_cliente
            = siguiente.numero_de_cliente
       AND siguiente.foto_mes
            = origen.foto_mes + 1

    WHERE origen.clase_ternaria = 'BAJA+2'
      AND origen.foto_mes BETWEEN 202103 AND 202106

    GROUP BY origen.foto_mes
),

baja1_mes_siguiente AS (

    SELECT
        foto_mes - 1 AS mes_origen,
        COUNT(*) AS clientes_baja1_mes_siguiente

    FROM competencia_01

    WHERE clase_ternaria = 'BAJA+1'
      AND foto_mes BETWEEN 202104 AND 202107

    GROUP BY foto_mes
)

SELECT
    s.mes_origen,
    s.clientes_baja2,
    s.baja2_que_pasan_a_baja1,

    b.clientes_baja1_mes_siguiente,

    b.clientes_baja1_mes_siguiente
        - s.baja2_que_pasan_a_baja1
        AS baja1_sin_baja2_previa,

    ROUND(
        100.0 * s.baja2_que_pasan_a_baja1
        / NULLIF(s.clientes_baja2, 0),
        2
    ) AS pct_baja2_que_pasa_a_baja1,

    ROUND(
        100.0 * s.baja2_que_pasan_a_baja1
        / NULLIF(
            b.clientes_baja1_mes_siguiente,
            0
        ),
        2
    ) AS pct_baja1_que_venia_de_baja2

FROM seguimiento_baja2 s

INNER JOIN baja1_mes_siguiente b
    USING (mes_origen)

ORDER BY s.mes_origen;

,mes_origen,clientes_baja2,baja2_que_pasan_a_baja1,clientes_baja1_mes_siguiente,baja1_sin_baja2_previa,pct_baja2_que_pasa_a_baja1,pct_baja1_que_venia_de_baja2
0,202103,960,960,964,4,100.0,99.59
1,202104,1139,1139,1143,4,100.0,99.65
2,202105,870,870,874,4,100.0,99.54
3,202106,1098,1098,1103,5,100.0,99.55


- El 100% de los BAJA+2 se convierte en BAJA+1 al mes siguiente.
- Entre 99,54% y 99,65% de los BAJA+1 venía de ser BAJA+2.
- Los 4 o 5 casos adicionales probablemente son clientes que ingresaron a la base y desaparecieron sin tener un mes previo observable.
Esto explica la tabla anterior:
BAJA+1 de abril ≈ BAJA+2 de marzo
BAJA+1 de mayo  ≈ BAJA+2 de abril
BAJA+1 de junio ≈ BAJA+2 de mayo
Por eso las dos clases parecen compensarse y las bajas totales son tan estables.
Consecuencias para el análisis:
- BAJA+1 y BAJA+2 no son dos poblaciones independientes: son casi siempre el mismo evento observado con un mes de diferencia.
- Para la competencia, el target sigue siendo únicamente BAJA+2.
- Si entrenáramos con ambas como positivas, contaríamos el mismo abandono en dos fotografías consecutivas.
- No debemos usar una división aleatoria de filas: el mismo cliente podría aparecer a ambos lados de la validación.
- En el EDA podemos comparar ambas clases para estudiar cómo evoluciona el cliente durante su último mes, pero sin interpretarlas como fenómenos separados.

### Mapa de variables: Tipos, cardinalidad, constantes, indicadores y cantidades

In [78]:
%%sql

SELECT
    type AS tipo_de_dato,
    COUNT(*) AS cantidad_variables

FROM pragma_table_info('competencia_01')

GROUP BY type

ORDER BY cantidad_variables DESC;

,tipo_de_dato,cantidad_variables
0,BIGINT,80
1,DOUBLE,74
2,VARCHAR,1


In [79]:
columnas_excluidas = {
    "numero_de_cliente",
    "foto_mes",
    "clase_ternaria"
}

columnas_fecha = {
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_fultimo_cierre",
    "Master_fechaalta",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_fultimo_cierre",
    "Visa_fechaalta"
}

variables = [
    columna
    for columna in data.columns
    if columna not in columnas_excluidas
]

cantidad_observados = data[variables].notna().sum()

perfil_variables = pd.DataFrame({
    "tipo_pandas": data[variables].dtypes.astype(str),
    "cantidad_observados": cantidad_observados,
    "pct_nulos": (
        data[variables].isna().mean().mul(100).round(2)
    ),
    "valores_distintos": (
        data[variables].nunique(dropna=True)
    )
})

def clasificar_cardinalidad(variable, distintos):

    if variable in columnas_fecha:
        return "FECHA_CODIFICADA"

    if distintos <= 1:
        return "CONSTANTE"

    if distintos == 2:
        return "DOS_VALORES"

    if distintos <= 20:
        return "3_A_20_VALORES"

    return "MAS_DE_20_VALORES"


perfil_variables["grupo_cardinalidad"] = [
    clasificar_cardinalidad(variable, distintos)
    for variable, distintos
    in perfil_variables["valores_distintos"].items()
]

resumen_cardinalidad = (
    perfil_variables
    .groupby("grupo_cardinalidad")
    .size()
    .rename("cantidad_variables")
    .sort_values(ascending=False)
    .to_frame()
)

display(resumen_cardinalidad)

,cantidad_variables
grupo_cardinalidad,
MAS_DE_20_VALORES,108
3_A_20_VALORES,25
DOS_VALORES,11
FECHA_CODIFICADA,8


Conclusiones iniciales:
- No hay variables constantes en toda la base.
- Hay 11 variables con exactamente dos valores: posibles indicadores.
- Hay 25 de baja cardinalidad: pueden ser códigos, estados o cantidades pequeñas.
- Las 8 fechas quedaron separadas correctamente.
- Que no haya constantes globales no impide que alguna sea constante dentro de un mes, como ccajas_depositos en mayo.

In [80]:
variables_dos_valores = (
    perfil_variables[
        perfil_variables["grupo_cardinalidad"]
        == "DOS_VALORES"
    ]
    .index
    .tolist()
)

detalle_dos_valores = pd.DataFrame({
    "variable": variables_dos_valores,

    "tipo_pandas": [
        str(data[variable].dtype)
        for variable in variables_dos_valores
    ],

    "pct_nulos": [
        round(data[variable].isna().mean() * 100, 2)
        for variable in variables_dos_valores
    ],

    "valores": [
        sorted(
            data[variable]
            .dropna()
            .unique()
            .tolist()
        )
        for variable in variables_dos_valores
    ]
})

display(detalle_dos_valores)

,variable,tipo_pandas,pct_nulos,valores
0,active_quarter,int64,0.00,"[0, 1]"
1,cliente_vip,int64,0.00,"[0, 1]"
2,tcuentas,int64,0.00,"[1, 2]"
3,cdescubierto_preacordado,int64,0.00,"[0, 1]"
4,tcallcenter,int64,0.00,"[0, 1]"
5,thomebanking,int64,0.00,"[0, 1]"
6,ccajas_transacciones,int64,0.00,"[0, 1]"
7,tmobile_app,int64,0.00,"[0, 1]"
8,cmobile_app_trx,int64,0.00,"[0, 1]"
9,Master_delinquency,Int64,10.14,"[0, 1]"


In [81]:
variables_baja_cardinalidad = (
    perfil_variables[
        perfil_variables["grupo_cardinalidad"]
        == "3_A_20_VALORES"
    ]
    .index
    .tolist()
)

detalle_baja_cardinalidad = pd.DataFrame({
    "variable": variables_baja_cardinalidad,

    "tipo_pandas": [
        str(data[variable].dtype)
        for variable in variables_baja_cardinalidad
    ],

    "pct_nulos": [
        round(data[variable].isna().mean() * 100, 2)
        for variable in variables_baja_cardinalidad
    ],

    "cantidad_valores": [
        data[variable].nunique(dropna=True)
        for variable in variables_baja_cardinalidad
    ],

    "valores": [
        sorted(
            data[variable]
            .dropna()
            .unique()
            .tolist()
        )
        for variable in variables_baja_cardinalidad
    ]
})

display(detalle_baja_cardinalidad)

,variable,tipo_pandas,pct_nulos,cantidad_valores,valores
0,internet,int64,0.00,5,"[0, 1, 2, 3, 4]"
1,cproductos,int64,0.00,15,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ..."
2,ccuenta_corriente,int64,0.00,3,"[1, 2, 3]"
3,ccaja_ahorro,int64,0.00,7,"[2, 3, 4, 5, 6, 7, 8]"
4,ctarjeta_debito,int64,0.00,15,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."
5,ctarjeta_visa,int64,0.00,4,"[0, 1, 2, 3]"
6,ctarjeta_master,int64,0.00,3,"[0, 1, 2]"
7,cinversion1,int64,0.00,12,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]"
8,cseguro_vida,int64,0.00,4,"[0, 1, 2, 3]"
9,cseguro_auto,int64,0.00,10,"[0, 1, 2, 3, 4, 5, 6, 8, 9, 11]"


En la salida hay dos grupos diferentes:
- Casi todas las variables que comienzan con c son cantidades pequeñas: cantidad de productos, tarjetas, seguros, operaciones, etc. Deben seguir tratándose como números.
- Master_status y Visa_status son códigos categóricos. El valor 9 no representa “más” que 7; son estados distintos.
- internet = [0,1,2,3,4] podría ser código, nivel o cantidad: queda pendiente de diccionario.
- Los adelantos de efectivo son cantidades, aunque tengan pocos valores.
- En status, el NaN agrega otro estado conceptual: tarjeta sin información.
Conclusión del mapa de cardinalidad:
- No hay constantes globales.
- Los indicadores 0/1 están identificados.
- La baja cardinalidad corresponde principalmente a conteos.
- Master_status y Visa_status requieren tratamiento categórico.
- No debemos convertir automáticamente todas estas variables en categorías.

In [82]:
variables_extremos = (
    perfil_variables[
        perfil_variables["grupo_cardinalidad"]
        == "MAS_DE_20_VALORES"
    ]
    .index
    .tolist()
)

filas_extremos = []

for variable in variables_extremos:

    serie = (
        pd.to_numeric(
            data[variable],
            errors="coerce"
        )
        .dropna()
        .astype(float)
    )

    cantidad_infinitos = np.isinf(serie).sum()
    serie_finita = serie[np.isfinite(serie)]

    if serie_finita.empty:
        continue

    cuantiles = serie_finita.quantile(
        [0.001, 0.01, 0.50, 0.99, 0.999]
    )

    maximo_absoluto = serie_finita.abs().max()
    percentil_99_absoluto = (
        serie_finita.abs().quantile(0.99)
    )

    if percentil_99_absoluto > 0:
        ratio_max_p99 = (
            maximo_absoluto
            / percentil_99_absoluto
        )
    elif maximo_absoluto > 0:
        ratio_max_p99 = np.inf
    else:
        ratio_max_p99 = 1.0

    filas_extremos.append({
        "variable": variable,
        "infinitos": cantidad_infinitos,
        "pct_nulos": round(
            data[variable].isna().mean() * 100,
            2
        ),
        "pct_ceros_observados": round(
            100.0
            * data[variable].eq(0).sum()
            / max(data[variable].notna().sum(), 1),
            2
        ),
        "minimo": serie_finita.min(),
        "percentil_001": cuantiles.loc[0.001],
        "percentil_01": cuantiles.loc[0.01],
        "mediana": cuantiles.loc[0.50],
        "percentil_99": cuantiles.loc[0.99],
        "percentil_999": cuantiles.loc[0.999],
        "maximo": serie_finita.max(),
        "max_abs_sobre_p99_abs": round(
            ratio_max_p99,
            2
        )
    })

perfil_extremos = (
    pd.DataFrame(filas_extremos)
    .set_index("variable")
    .sort_values(
        "max_abs_sobre_p99_abs",
        ascending=False
    )
)

display(perfil_extremos.head(25))

,infinitos,pct_nulos,pct_ceros_observados,minimo,percentil_001,percentil_01,mediana,percentil_99,percentil_999,maximo,max_abs_sobre_p99_abs
variable,,,,,,,,,,,
mforex_buy,0,0.00,99.65,0.00,0.00000,0.0,0.00,0.000000e+00,2.322500e+02,3.671490e+04,inf
mcheques_emitidos_rechazados,0,0.00,99.83,0.00,0.00000,0.0,0.00,0.000000e+00,2.873850e+04,3.796907e+06,inf
mplazo_fijo_pesos,0,0.00,99.56,0.00,0.00000,0.0,0.00,0.000000e+00,1.307216e+06,3.354967e+07,inf
mpayroll2,0,0.00,99.92,0.00,0.00000,0.0,0.00,0.000000e+00,0.000000e+00,1.290300e+06,inf
cpagodeservicios,0,0.00,99.12,0.00,0.00000,0.0,0.00,0.000000e+00,5.000000e+00,8.300000e+01,inf
mprestamos_prendarios,0,0.00,99.10,0.00,0.00000,0.0,0.00,0.000000e+00,1.217243e+06,2.044281e+07,inf
cprestamos_prendarios,0,0.00,99.08,0.00,0.00000,0.0,0.00,0.000000e+00,1.200000e+01,2.300000e+01,inf
mpagodeservicios,0,0.00,99.12,0.00,0.00000,0.0,0.00,0.000000e+00,4.194412e+04,1.354154e+07,inf
mcheques_depositados_rechazados,0,0.00,99.81,0.00,0.00000,0.0,0.00,0.000000e+00,4.226680e+04,6.543845e+06,inf


In [83]:
filas_extremos_no_cero = []

for variable in variables_extremos:

    serie = (
        pd.to_numeric(
            data[variable],
            errors="coerce"
        )
        .dropna()
        .astype(float)
    )

    serie = serie[np.isfinite(serie)]
    serie_no_cero = serie[serie != 0]
    absolutos_no_cero = serie_no_cero.abs()

    if absolutos_no_cero.empty:
        continue

    percentil_99_abs = (
        absolutos_no_cero.quantile(0.99)
    )

    maximo_abs = absolutos_no_cero.max()

    filas_extremos_no_cero.append({
        "variable": variable,

        "registros_no_cero":
            len(serie_no_cero),

        "pct_no_cero": round(
            100.0 * len(serie_no_cero)
            / max(len(serie), 1),
            3
        ),

        "minimo_original":
            serie_no_cero.min(),

        "mediana_abs_no_cero": round(
            absolutos_no_cero.median(),
            2
        ),

        "percentil_90_abs_no_cero": round(
            absolutos_no_cero.quantile(0.90),
            2
        ),

        "percentil_99_abs_no_cero": round(
            percentil_99_abs,
            2
        ),

        "maximo_original":
            serie_no_cero.max(),

        "maximo_abs":
            maximo_abs,

        "max_abs_sobre_p99_abs_no_cero": round(
            maximo_abs
            / percentil_99_abs,
            2
        )
    })

perfil_extremos_no_cero = (
    pd.DataFrame(filas_extremos_no_cero)
    .set_index("variable")
    .sort_values(
        "max_abs_sobre_p99_abs_no_cero",
        ascending=False
    )
)

display(perfil_extremos_no_cero.head(25))

,registros_no_cero,pct_no_cero,minimo_original,mediana_abs_no_cero,percentil_90_abs_no_cero,percentil_99_abs_no_cero,maximo_original,maximo_abs,max_abs_sobre_p99_abs_no_cero
variable,,,,,,,,,
mpagomiscuentas,481251,48.954,0.01,11043.56,63382.84,337396.26,2.306702e+08,2.306702e+08,683.68
mcuenta_debitos_automaticos,470788,47.890,0.01,3825.82,22119.03,78522.22,4.395548e+07,4.395548e+07,559.78
mtransferencias_recibidas,738261,75.098,0.01,74954.70,305496.12,1691639.99,5.313686e+08,5.313686e+08,314.11
Visa_mfinanciacion_limite,935047,99.912,505.56,202694.40,853584.27,2619363.99,6.623916e+08,6.623916e+08,252.88
mtransferencias_emitidas,601229,61.159,0.01,33354.25,194976.08,1299036.52,3.272050e+08,3.272050e+08,251.88
Visa_mpagominimo,808648,86.406,0.01,3131.91,17207.91,76445.11,1.141511e+07,1.141511e+07,149.32
Master_mfinanciacion_limite,882995,99.952,886.78,177357.60,773018.97,3705662.82,5.507904e+08,5.507904e+08,148.63
mcuentas_saldo,977028,99.386,-7179917.37,44415.79,441973.45,3129012.99,3.287445e+08,3.287445e+08,105.06
mcuenta_corriente,499085,50.768,-2987384.90,1753.03,28921.27,292595.08,2.687961e+07,2.687961e+07,91.87


Lo importante ahora es saber si esos extremos:
- pertenecen a un cliente real y se repiten;
- aparecen una sola vez;
- o se concentran masivamente en un mes.

In [84]:
# Variables monetarias con extremos muy alejados
variables_a_revisar = perfil_extremos_no_cero.index[
    perfil_extremos_no_cero[
        "max_abs_sobre_p99_abs_no_cero"
    ] > 100
].tolist()

# Agregamos conteos con máximos operativamente llamativos
for variable in [
    "cpagomiscuentas",
    "chomebanking_transacciones",
    "cpayroll_trx"
]:
    if variable not in variables_a_revisar:
        variables_a_revisar.append(variable)


filas_extremas = []

for variable in variables_a_revisar:

    serie = pd.to_numeric(
        data[variable],
        errors="coerce"
    )

    p99 = perfil_extremos_no_cero.loc[
        variable,
        "percentil_99_abs_no_cero"
    ]

    # Umbral exploratorio: diez veces el percentil 99
    umbral = 10 * p99

    mascara = serie.abs() > umbral

    bloque = data.loc[
        mascara,
        ["numero_de_cliente", "foto_mes"]
    ].copy()

    bloque["variable"] = variable
    bloque["valor_original"] = serie.loc[mascara].values
    bloque["umbral_10_x_p99"] = umbral

    filas_extremas.append(bloque)


extremos_10x = pd.concat(
    filas_extremas,
    ignore_index=True
)


# ¿Cuántos meses se repite cada cliente como extremo?
repeticion_clientes = (
    extremos_10x
    .groupby(["variable", "numero_de_cliente"])
    ["foto_mes"]
    .nunique()
    .reset_index(name="meses_como_extremo")
)


resumen_extremos = (
    extremos_10x
    .groupby("variable")
    .agg(
        umbral_10_x_p99=("umbral_10_x_p99", "first"),
        registros_extremos=("valor_original", "size"),
        clientes_extremos=("numero_de_cliente", "nunique"),
        meses_con_extremos=("foto_mes", "nunique"),
        maximo_absoluto=(
            "valor_original",
            lambda x: x.abs().max()
        )
    )
    .reset_index()
)


clientes_repetidos = (
    repeticion_clientes
    .groupby("variable")
    ["meses_como_extremo"]
    .apply(lambda x: (x > 1).sum())
    .reset_index(name="clientes_extremos_repetidos")
)


resumen_extremos = resumen_extremos.merge(
    clientes_repetidos,
    on="variable",
    how="left"
)


extremos_por_mes = (
    extremos_10x
    .groupby(["variable", "foto_mes"])
    .agg(
        registros_extremos=("valor_original", "size"),
        clientes_extremos=("numero_de_cliente", "nunique")
    )
    .reset_index()
)


display(
    resumen_extremos.sort_values(
        "registros_extremos",
        ascending=False
    )
)

display(extremos_por_mes)

,variable,umbral_10_x_p99,registros_extremos,clientes_extremos,meses_con_extremos,maximo_absoluto,clientes_extremos_repetidos
0,Master_mfinanciacion_limite,37056628.2,3589,3567,6,5.507904e+08,22
1,Visa_mfinanciacion_limite,26193639.9,1101,1101,6,6.623916e+08,0
10,mtransferencias_recibidas,16916399.9,314,148,6,5.313686e+08,63
8,mpagomiscuentas,3373962.6,273,173,6,2.306702e+08,68
6,mcuenta_debitos_automaticos,785222.2,262,78,6,4.395548e+07,62
7,mcuentas_saldo,31290129.9,196,60,6,3.287445e+08,33
9,mtransferencias_emitidas,12990365.2,142,75,6,3.272050e+08,26
2,Visa_mpagominimo,764451.1,101,70,6,1.141511e+07,15
5,cpayroll_trx,70.0,50,19,6,2.720000e+02,13
3,chomebanking_transacciones,3650.0,22,6,6,1.594000e+04,6


,variable,foto_mes,registros_extremos,clientes_extremos
0,Master_mfinanciacion_limite,202103,12,12
1,Master_mfinanciacion_limite,202104,3517,3517
2,Master_mfinanciacion_limite,202105,5,5
3,Master_mfinanciacion_limite,202106,14,14
4,Master_mfinanciacion_limite,202107,3,3
5,Master_mfinanciacion_limite,202108,38,38
6,Visa_mfinanciacion_limite,202103,15,15
7,Visa_mfinanciacion_limite,202104,19,19
8,Visa_mfinanciacion_limite,202105,7,7
9,Visa_mfinanciacion_limite,202106,21,21


- Master: 3517 de 3589 extremos están en 202104: 98%.
- Visa: 1035 de 1101 extremos están en 202108: 94%.
- Los demás montos extremos están repartidos entre meses y muchos clientes se repiten: parecen colas bancarias legítimas.
Además, la variable tiene una comprobación interna: normalmente el límite de financiación es aproximadamente el 90% del límite de compra. Veámoslo:

In [85]:
%%sql

WITH limites AS (

    SELECT
        foto_mes,
        'Master' AS marca,
        Master_mfinanciacion_limite AS limite_financiacion,
        Master_mlimitecompra AS limite_compra
    FROM competencia_01

    UNION ALL

    SELECT
        foto_mes,
        'Visa' AS marca,
        Visa_mfinanciacion_limite AS limite_financiacion,
        Visa_mlimitecompra AS limite_compra
    FROM competencia_01
),

relaciones AS (

    SELECT
        foto_mes,
        marca,

        ROUND(
            limite_financiacion
            / NULLIF(limite_compra, 0),
            1
        ) AS relacion_financiacion_compra,

        COUNT(*) AS clientes

    FROM limites

    WHERE limite_financiacion IS NOT NULL
      AND limite_compra IS NOT NULL
      AND limite_compra <> 0

    GROUP BY
        foto_mes,
        marca,
        relacion_financiacion_compra
),

ranking AS (

    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY foto_mes, marca
            ORDER BY clientes DESC
        ) AS orden

    FROM relaciones
)

SELECT
    foto_mes,
    marca,
    relacion_financiacion_compra,
    clientes

FROM ranking

WHERE orden <= 5

ORDER BY
    foto_mes,
    marca,
    clientes DESC;

,foto_mes,marca,relacion_financiacion_compra,clientes
0,202103,Master,0.9,145991
1,202103,Master,0.6,35
2,202103,Master,82.3,13
3,202103,Master,0.0,6
4,202103,Master,1.0,5
5,202103,Visa,0.9,154761
6,202103,Visa,0.5,103
7,202103,Visa,1.0,92
8,202103,Visa,0.6,14
9,202103,Visa,0.7,13


In [86]:
%%sql

WITH clientes_master_abril AS (

    SELECT DISTINCT
        numero_de_cliente

    FROM competencia_01

    WHERE foto_mes = 202104
      AND Master_mlimitecompra > 0
      AND (
          Master_mfinanciacion_limite
          / Master_mlimitecompra
      ) > 10
),

seguimiento AS (

    SELECT
        c.*

    FROM competencia_01 AS c

    INNER JOIN clientes_master_abril AS a
        USING (numero_de_cliente)

    WHERE c.foto_mes IN (202103, 202104, 202105)
)

SELECT
    foto_mes,

    COUNT(DISTINCT numero_de_cliente)
        AS clientes_de_la_cohorte_presentes,

    ROUND(
        MEDIAN(Master_mlimitecompra),
        2
    ) AS mediana_limite_compra,

    ROUND(
        MEDIAN(Master_mfinanciacion_limite),
        2
    ) AS mediana_limite_financiacion,

    ROUND(
        MEDIAN(
            Master_mfinanciacion_limite
            / NULLIF(Master_mlimitecompra, 0)
        ),
        2
    ) AS mediana_relacion_financiacion_compra,

    COUNT(DISTINCT numero_de_cliente) FILTER (
        WHERE Master_mlimitecompra > 0
          AND (
              Master_mfinanciacion_limite
              / Master_mlimitecompra
          ) > 10
    ) AS clientes_con_relacion_mayor_a_10

FROM seguimiento

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes_de_la_cohorte_presentes,mediana_limite_compra,mediana_limite_financiacion,mediana_relacion_financiacion_compra,clientes_con_relacion_mayor_a_10
0,202103,4158,578659.14,520717.70,0.9,1
1,202104,4159,493539.75,41309277.07,83.7,4159
2,202105,4151,498976.60,449078.94,0.9,0


sta salida confirma la rotura de Master:
- El límite de compra se mantiene alrededor de $500.000.
- El límite de financiación pasa de $520.718 a $41.309.277 en abril.
- La relación pasa de 0.9 a 83.7 y vuelve a 0.9 en mayo.
- Los 4159 clientes se alteran simultáneamente en abril.
Por lo tanto, no es un aumento real de límites: Master_mfinanciacion_limite está mal escalada para esa cohorte en 202104.

In [87]:
%%sql

WITH clientes_visa_agosto AS (

    SELECT DISTINCT
        numero_de_cliente

    FROM competencia_01

    WHERE foto_mes = 202108
      AND Visa_mlimitecompra > 0
      AND (
          Visa_mfinanciacion_limite
          / Visa_mlimitecompra
      ) > 10
),

seguimiento AS (

    SELECT
        c.*

    FROM competencia_01 AS c

    INNER JOIN clientes_visa_agosto AS a
        USING (numero_de_cliente)

    WHERE c.foto_mes IN (202107, 202108)
)

SELECT
    foto_mes,

    COUNT(DISTINCT numero_de_cliente)
        AS clientes_de_la_cohorte_presentes,

    ROUND(
        MEDIAN(Visa_mlimitecompra),
        2
    ) AS mediana_limite_compra,

    ROUND(
        MEDIAN(Visa_mfinanciacion_limite),
        2
    ) AS mediana_limite_financiacion,

    ROUND(
        MEDIAN(
            Visa_mfinanciacion_limite
            / NULLIF(Visa_mlimitecompra, 0)
        ),
        2
    ) AS mediana_relacion_financiacion_compra,

    COUNT(DISTINCT numero_de_cliente) FILTER (
        WHERE Visa_mlimitecompra > 0
          AND (
              Visa_mfinanciacion_limite
              / Visa_mlimitecompra
          ) > 10
    ) AS clientes_con_relacion_mayor_a_10

FROM seguimiento

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes_de_la_cohorte_presentes,mediana_limite_compra,mediana_limite_financiacion,mediana_relacion_financiacion_compra,clientes_con_relacion_mayor_a_10
0,202107,1035,729463.36,656460.34,0.9,0
1,202108,1035,514812.10,52163279.33,96.5,1035


Esto confirma una inconsistencia en Visa_mfinanciacion_limite durante agosto:
- Julio: compra $729.463, financiación $656.460, relación 0.9.
- Agosto: compra $514.812, financiación $52.163.279, relación 96.5.
- Los 1035 clientes presentan simultáneamente esa alteración.
El límite de compra baja, pero no lo suficiente para explicar una relación de 96.5. El que explota es el límite de financiación.
Ahora revisamos separadamente los 3334 clientes con relaciones 1.2 o 1.3:

In [88]:
%%sql

WITH clientes_visa_relacion_moderada AS (

    SELECT DISTINCT
        numero_de_cliente

    FROM competencia_01

    WHERE foto_mes = 202108
      AND Visa_mlimitecompra > 0
      AND ROUND(
          Visa_mfinanciacion_limite
          / Visa_mlimitecompra,
          1
      ) IN (1.2, 1.3)
),

seguimiento AS (

    SELECT
        c.*

    FROM competencia_01 AS c

    INNER JOIN clientes_visa_relacion_moderada AS a
        USING (numero_de_cliente)

    WHERE c.foto_mes IN (202107, 202108)
)

SELECT
    foto_mes,

    COUNT(DISTINCT numero_de_cliente)
        AS clientes_de_la_cohorte_presentes,

    COUNT(DISTINCT numero_de_cliente) FILTER (
        WHERE Visa_mlimitecompra > 0
          AND Visa_mfinanciacion_limite IS NOT NULL
    ) AS clientes_con_ambos_limites,

    ROUND(
        MEDIAN(Visa_mlimitecompra),
        2
    ) AS mediana_limite_compra,

    ROUND(
        MEDIAN(Visa_mfinanciacion_limite),
        2
    ) AS mediana_limite_financiacion,

    ROUND(
        MEDIAN(
            CASE
                WHEN Visa_mlimitecompra > 0
                THEN Visa_mfinanciacion_limite
                     / Visa_mlimitecompra
            END
        ),
        2
    ) AS mediana_relacion_financiacion_compra,

    COUNT(DISTINCT numero_de_cliente) FILTER (
        WHERE Visa_mlimitecompra > 0
          AND ROUND(
              Visa_mfinanciacion_limite
              / Visa_mlimitecompra,
              1
          ) IN (1.2, 1.3)
    ) AS clientes_con_relacion_1_2_o_1_3

FROM seguimiento

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes_de_la_cohorte_presentes,clientes_con_ambos_limites,mediana_limite_compra,mediana_limite_financiacion,mediana_relacion_financiacion_compra,clientes_con_relacion_1_2_o_1_3
0,202107,3332,3332,168912.0,152020.8,0.90,0
1,202108,3334,3334,112608.0,152020.8,1.29,3334


Este grupo es distinto al de 96.5:
- La financiación mediana queda exactamente en $152.020,80.
- El límite de compra baja de $168.912 a $112.608.
- Por eso la relación aumenta de 0.9 a 1.29.
Pero dos medianas iguales no garantizan que cada cliente conserve el mismo valor. Compare­mos cliente por cliente y separemos 1.2 de 1.3:

In [89]:
%%sql

WITH agosto AS (

    SELECT
        numero_de_cliente,

        ROUND(
            Visa_mfinanciacion_limite
            / NULLIF(Visa_mlimitecompra, 0),
            1
        ) AS grupo_ratio_agosto,

        Visa_mlimitecompra AS compra_agosto,
        Visa_mfinanciacion_limite AS financiacion_agosto

    FROM competencia_01

    WHERE foto_mes = 202108
      AND Visa_mlimitecompra > 0
      AND ROUND(
          Visa_mfinanciacion_limite
          / Visa_mlimitecompra,
          1
      ) IN (1.2, 1.3)
),

pares_julio_agosto AS (

    SELECT
        a.*,
        j.Visa_mlimitecompra AS compra_julio,
        j.Visa_mfinanciacion_limite AS financiacion_julio

    FROM agosto AS a

    INNER JOIN competencia_01 AS j
        USING (numero_de_cliente)

    WHERE j.foto_mes = 202107
)

SELECT
    grupo_ratio_agosto,

    COUNT(*) AS clientes,

    ROUND(MEDIAN(compra_julio), 2)
        AS mediana_compra_julio,

    ROUND(MEDIAN(compra_agosto), 2)
        AS mediana_compra_agosto,

    ROUND(
        MEDIAN(
            100.0 * (
                compra_agosto
                / NULLIF(compra_julio, 0) - 1
            )
        ),
        2
    ) AS mediana_cambio_pct_compra,

    ROUND(MEDIAN(financiacion_julio), 2)
        AS mediana_financiacion_julio,

    ROUND(MEDIAN(financiacion_agosto), 2)
        AS mediana_financiacion_agosto,

    ROUND(
        MEDIAN(
            100.0 * (
                financiacion_agosto
                / NULLIF(financiacion_julio, 0) - 1
            )
        ),
        2
    ) AS mediana_cambio_pct_financiacion

FROM pares_julio_agosto

GROUP BY grupo_ratio_agosto

ORDER BY grupo_ratio_agosto;

,grupo_ratio_agosto,clientes,mediana_compra_julio,mediana_compra_agosto,mediana_cambio_pct_compra,mediana_financiacion_julio,mediana_financiacion_agosto,mediana_cambio_pct_financiacion
0,1.2,605,112608.0,84456.0,-25.0,101347.2,101347.2,0.0
1,1.3,2727,168912.0,112608.0,-30.0,152020.8,152020.8,0.0


Buscamos comprobar si el límite de compra disminuyó mientras el de financiación quedó congelado. Si ocurre, no sería la misma rotura de escala que el grupo 96.5. Podría ser una actualización descoordinada o un cambio real de límites.

- Relaciones mayores a 10: anomalía técnica; luego crearemos una versión corregida y un indicador.
- Relaciones 1.2–1.3: conservar sin corregir. Puede ser una reducción real del límite o una actualización descoordinada, y potencialmente contiene información útil.
- No aplicamos recorte general a los outliers.
Nota: en el grupo 1.3, dividir las dos medianas da −33%, mientras la tabla muestra −30%. No es un error: esa columna calcula primero el cambio de cada cliente y después obtiene su mediana.

## veamos qué variables separan a BAJA+2 de CONTINUA, siempre comparando dentro del mismo mes.

Empezamos con el perfil general del cliente. Compararemos las tres clases dentro de cada mes para evitar mezclar cambios temporales.
Buscamos si quienes abandonan son:
- Más jóvenes o mayores.
- Más nuevos en el banco.
- Menos vinculados mediante productos.
- Menos activos.
- Menos rentables.
- Menos frecuentemente VIP.

In [90]:
%%sql

SELECT
    foto_mes,
    clase_ternaria,

    COUNT(*) AS clientes,

    ROUND(
        MEDIAN(cliente_edad),
        2
    ) AS mediana_edad,

    ROUND(
        MEDIAN(cliente_antiguedad),
        2
    ) AS mediana_antiguedad,

    ROUND(
        100.0 * AVG(cliente_vip),
        2
    ) AS pct_clientes_vip,

    ROUND(
        MEDIAN(cproductos),
        2
    ) AS mediana_productos,

    ROUND(
        MEDIAN(ctrx_quarter),
        2
    ) AS mediana_transacciones_trimestre,

    ROUND(
        MEDIAN(mrentabilidad),
        2
    ) AS mediana_rentabilidad

FROM competencia_01

WHERE foto_mes BETWEEN 202103 AND 202106
  AND clase_ternaria IS NOT NULL

GROUP BY
    foto_mes,
    clase_ternaria

ORDER BY
    foto_mes,

    CASE clase_ternaria
        WHEN 'CONTINUA' THEN 1
        WHEN 'BAJA+2'   THEN 2
        WHEN 'BAJA+1'   THEN 3
    END;

,foto_mes,clase_ternaria,clientes,mediana_edad,mediana_antiguedad,pct_clientes_vip,mediana_productos,mediana_transacciones_trimestre,mediana_rentabilidad
0,202103,CONTINUA,160921,45.0,127.0,0.26,7.0,105.0,1239.26
1,202103,BAJA+2,960,46.0,85.0,0.10,6.0,20.0,1713.22
2,202103,BAJA+1,1019,46.0,89.0,0.00,6.0,18.0,1754.35
3,202104,CONTINUA,161181,45.0,127.0,0.26,7.0,106.0,1112.37
4,202104,BAJA+2,1139,45.0,81.0,0.09,6.0,17.0,1791.85
5,202104,BAJA+1,964,46.0,86.0,0.10,6.0,18.0,1730.08
6,202105,CONTINUA,161755,45.0,128.0,0.27,7.0,106.0,1211.22
7,202105,BAJA+2,870,47.0,86.0,0.11,7.0,25.0,1706.14
8,202105,BAJA+1,1143,45.0,82.0,0.09,6.0,16.0,1822.39
9,202106,CONTINUA,162142,45.0,128.0,0.57,7.0,106.0,1350.80



- BAJA+2 es el objetivo que queremos anticipar.
- BAJA+1 es prácticamente ese mismo abandono un mes más tarde.
- Buscamos características que ya distingan al cliente cuando todavía es BAJA+2.
Empezamos con ctrx_quarter, porque su mediana era aproximadamente 20 para las bajas y 105 para quienes continúan.

In [91]:
%%sql

WITH base AS (

    SELECT
        clase_ternaria,
        ctrx_quarter,

        CASE
            WHEN ctrx_quarter IS NULL THEN 'NULO'
            WHEN ctrx_quarter = 0 THEN '0'
            WHEN ctrx_quarter <= 10 THEN '1 a 10'
            WHEN ctrx_quarter <= 25 THEN '11 a 25'
            WHEN ctrx_quarter <= 50 THEN '26 a 50'
            WHEN ctrx_quarter <= 100 THEN '51 a 100'
            WHEN ctrx_quarter <= 200 THEN '101 a 200'
            ELSE 'Más de 200'
        END AS banda_transacciones,

        CASE
            WHEN ctrx_quarter IS NULL THEN 0
            WHEN ctrx_quarter = 0 THEN 1
            WHEN ctrx_quarter <= 10 THEN 2
            WHEN ctrx_quarter <= 25 THEN 3
            WHEN ctrx_quarter <= 50 THEN 4
            WHEN ctrx_quarter <= 100 THEN 5
            WHEN ctrx_quarter <= 200 THEN 6
            ELSE 7
        END AS orden

    FROM competencia_01

    -- Solamente meses cuya clase está completamente observable
    WHERE foto_mes BETWEEN 202103 AND 202106
),

resumen AS (

    SELECT
        orden,
        banda_transacciones,

        COUNT(*) AS clientes,

        COUNT(*) FILTER (
            WHERE clase_ternaria = 'CONTINUA'
        ) AS continua,

        COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+1'
        ) AS baja_1,

        COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+2'
        ) AS baja_2

    FROM base

    GROUP BY
        orden,
        banda_transacciones
)

SELECT
    banda_transacciones,
    clientes,
    continua,
    baja_1,
    baja_2,

    ROUND(
        100.0 * baja_2 / NULLIF(clientes, 0),
        3
    ) AS pct_clientes_de_la_banda_que_son_baja2,

    ROUND(
        100.0 * baja_2
        / NULLIF(SUM(baja_2) OVER (), 0),
        2
    ) AS pct_de_todas_las_baja2_ubicadas_en_la_banda

FROM resumen

ORDER BY orden;

,banda_transacciones,clientes,continua,baja_1,baja_2,pct_clientes_de_la_banda_que_son_baja2,pct_de_todas_las_baja2_ubicadas_en_la_banda
0,0,6762,5768,482,512,7.572,12.59
1,1 a 10,18904,17088,951,865,4.576,21.27
2,11 a 25,35132,33292,923,917,2.610,22.55
3,26 a 50,73795,72304,724,767,1.039,18.86
4,51 a 100,176658,175507,553,598,0.339,14.70
5,101 a 200,254732,254085,314,333,0.131,8.19
6,Más de 200,88083,87955,53,75,0.085,1.84


Este es un predictor muy fuerte y ordenado:
- Tasa general de BAJA+2: aproximadamente 0,62%.
- Sin transacciones: 7,57%, unas 12 veces el promedio.
- De 1 a 10: 4,58%.
- De 11 a 25: 2,61%.
- Más de 200: apenas 0,085%.
Además:
- Los casos con hasta 25 transacciones son solo el 9,3% de los registros.
- Allí se concentra el 56,4% de todas las BAJA+2.
- Hasta 50 transacciones se concentra aproximadamente el 75% de las bajas.
Esto no demuestra causalidad, pero sí que la baja está fuertemente asociada con menor actividad. Los conteos son registros cliente-mes; el mismo cliente puede aparecer en varios meses.

In [92]:
%%sql

WITH base AS (

    SELECT
        foto_mes,
        clase_ternaria,

        CASE
            WHEN ctrx_quarter IS NULL THEN 'NULO'
            WHEN ctrx_quarter = 0 THEN '0'
            WHEN ctrx_quarter <= 10 THEN '1 a 10'
            WHEN ctrx_quarter <= 25 THEN '11 a 25'
            WHEN ctrx_quarter <= 50 THEN '26 a 50'
            WHEN ctrx_quarter <= 100 THEN '51 a 100'
            WHEN ctrx_quarter <= 200 THEN '101 a 200'
            ELSE 'Más de 200'
        END AS banda_transacciones,

        CASE
            WHEN ctrx_quarter IS NULL THEN 0
            WHEN ctrx_quarter = 0 THEN 1
            WHEN ctrx_quarter <= 10 THEN 2
            WHEN ctrx_quarter <= 25 THEN 3
            WHEN ctrx_quarter <= 50 THEN 4
            WHEN ctrx_quarter <= 100 THEN 5
            WHEN ctrx_quarter <= 200 THEN 6
            ELSE 7
        END AS orden

    FROM competencia_01

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
)

SELECT
    foto_mes,
    banda_transacciones,

    COUNT(*) AS casos_cliente_mes,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+2'
    ) AS casos_baja_2,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+2'
        )
        / COUNT(*),
        3
    ) AS tasa_baja_2_pct

FROM base

GROUP BY
    foto_mes,
    orden,
    banda_transacciones

ORDER BY
    foto_mes,
    orden;

,foto_mes,banda_transacciones,casos_cliente_mes,casos_baja_2,tasa_baja_2_pct
0,202103,0,1734,121,6.978
1,202103,1 a 10,4907,207,4.218
2,202103,11 a 25,8899,230,2.585
3,202103,26 a 50,18522,171,0.923
4,202103,51 a 100,43955,123,0.280
5,202103,101 a 200,63398,86,0.136
6,202103,Más de 200,21485,22,0.102
7,202104,0,1684,169,10.036
8,202104,1 a 10,4798,287,5.982
9,202104,11 a 25,8744,223,2.550


El patrón se mantiene en los cuatro meses y siempre es descendente: a mayor actividad, menor riesgo de BAJA+2.
Incluso cuando cambia la intensidad —por ejemplo, el riesgo sin transacciones va de 4,59% a 10,04%— nunca se invierte el orden. Por eso ctrx_quarter es una señal estable, aunque no alcanza sola: la mayoría de los clientes con baja actividad igualmente continúa.
La siguiente pregunta es: ¿también importa que la actividad venga cayendo respecto del mes anterior?

In [93]:
%%sql

WITH historial AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        clase_ternaria,
        ctrx_quarter,

        LAG(foto_mes) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS foto_mes_anterior,

        LAG(ctrx_quarter) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS transacciones_anteriores

    FROM competencia_01
),

consecutividad AS (

    SELECT
        *,

        CASE
            WHEN foto_mes = 202104
             AND foto_mes_anterior = 202103 THEN 1

            WHEN foto_mes = 202105
             AND foto_mes_anterior = 202104 THEN 1

            WHEN foto_mes = 202106
             AND foto_mes_anterior = 202105 THEN 1

            ELSE 0
        END AS tiene_mes_anterior_consecutivo

    FROM historial

    WHERE foto_mes BETWEEN 202104 AND 202106
      AND clase_ternaria IS NOT NULL
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN tiene_mes_anterior_consecutivo = 0
                THEN 'SIN MES ANTERIOR'

            WHEN transacciones_anteriores IS NULL
              OR ctrx_quarter IS NULL
                THEN 'SIN DATO DE ACTIVIDAD'

            WHEN transacciones_anteriores = 0
             AND ctrx_quarter = 0
                THEN 'SIGUE EN CERO'

            WHEN transacciones_anteriores = 0
             AND ctrx_quarter > 0
                THEN 'SALE DE CERO'

            WHEN ctrx_quarter <= 0.5 * transacciones_anteriores
                THEN 'CAÍDA DE 50% O MÁS'

            WHEN ctrx_quarter < 0.8 * transacciones_anteriores
                THEN 'CAÍDA ENTRE 20% Y 50%'

            WHEN ctrx_quarter <= 1.2 * transacciones_anteriores
                THEN 'ESTABLE: ±20%'

            ELSE 'SUBA MAYOR A 20%'
        END AS cambio_actividad

    FROM consecutividad
)

SELECT
    cambio_actividad,

    COUNT(*) AS casos_cliente_mes,

    ROUND(
        MEDIAN(transacciones_anteriores) FILTER (
            WHERE tiene_mes_anterior_consecutivo = 1
        ),
        1
    ) AS mediana_transacciones_mes_anterior,

    ROUND(
        MEDIAN(ctrx_quarter),
        1
    ) AS mediana_transacciones_actual,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+2'
    ) AS casos_baja_2,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+2'
        )
        / COUNT(*),
        3
    ) AS tasa_baja_2_pct

FROM clasificacion

GROUP BY cambio_actividad

ORDER BY CASE cambio_actividad
    WHEN 'SIN MES ANTERIOR' THEN 0
    WHEN 'SIN DATO DE ACTIVIDAD' THEN 1
    WHEN 'SIGUE EN CERO' THEN 2
    WHEN 'SALE DE CERO' THEN 3
    WHEN 'CAÍDA DE 50% O MÁS' THEN 4
    WHEN 'CAÍDA ENTRE 20% Y 50%' THEN 5
    WHEN 'ESTABLE: ±20%' THEN 6
    WHEN 'SUBA MAYOR A 20%' THEN 7
END;

,cambio_actividad,casos_cliente_mes,mediana_transacciones_mes_anterior,mediana_transacciones_actual,casos_baja_2,tasa_baja_2_pct
0,SIN MES ANTERIOR,4340,NaN,71.0,11,0.253
1,SIGUE EN CERO,4179,0.0,0.0,316,7.562
2,SALE DE CERO,531,0.0,2.0,41,7.721
3,CAÍDA DE 50% O MÁS,3467,11.0,4.0,229,6.605
4,CAÍDA ENTRE 20% Y 50%,20652,65.0,46.0,411,1.990
5,ESTABLE: ±20%,423346,113.0,113.0,1798,0.425
6,SUBA MAYOR A 20%,34651,52.0,70.0,301,0.869


La actividad actual parece ser la señal principal:
- SALE DE CERO sube, pero apenas hasta una mediana de 2 transacciones: riesgo 7,72%.
- Una caída fuerte deja la actividad en 4: riesgo 6,61%.
- La actividad estable permanece en 113: riesgo 0,43%.
- Quienes suben llegan a 70, todavía menos que los estables; por eso su riesgo de 0,87% no significa que aumentar actividad sea perjudicial.
Ahora necesitamos separar nivel actual de caída reciente. Compararemos clientes con actividad actual parecida, pero distinguiendo si cayeron al menos 20%.

In [94]:
%%sql

WITH historial AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        clase_ternaria,
        ctrx_quarter,

        LAG(foto_mes) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS foto_mes_anterior,

        LAG(ctrx_quarter) OVER (
            PARTITION BY numero_de_cliente
            ORDER BY foto_mes
        ) AS transacciones_anteriores

    FROM competencia_01
),

pares_consecutivos AS (

    SELECT
        *

    FROM historial

    WHERE clase_ternaria IS NOT NULL
      AND (
            (foto_mes = 202104 AND foto_mes_anterior = 202103)
         OR (foto_mes = 202105 AND foto_mes_anterior = 202104)
         OR (foto_mes = 202106 AND foto_mes_anterior = 202105)
      )
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN ctrx_quarter = 0 THEN '0'
            WHEN ctrx_quarter <= 10 THEN '1 a 10'
            WHEN ctrx_quarter <= 25 THEN '11 a 25'
            WHEN ctrx_quarter <= 50 THEN '26 a 50'
            WHEN ctrx_quarter <= 100 THEN '51 a 100'
            WHEN ctrx_quarter <= 200 THEN '101 a 200'
            ELSE 'Más de 200'
        END AS banda_actividad_actual,

        CASE
            WHEN ctrx_quarter = 0 THEN 1
            WHEN ctrx_quarter <= 10 THEN 2
            WHEN ctrx_quarter <= 25 THEN 3
            WHEN ctrx_quarter <= 50 THEN 4
            WHEN ctrx_quarter <= 100 THEN 5
            WHEN ctrx_quarter <= 200 THEN 6
            ELSE 7
        END AS orden_banda,

        CASE
            WHEN ctrx_quarter < 0.8 * transacciones_anteriores
                THEN 'CAYÓ 20% O MÁS'
            ELSE 'NO CAYÓ 20% O MÁS'
        END AS trayectoria

    FROM pares_consecutivos

    WHERE ctrx_quarter IS NOT NULL
      AND transacciones_anteriores IS NOT NULL
)

SELECT
    banda_actividad_actual,
    trayectoria,

    COUNT(*) AS casos_cliente_mes,

    ROUND(
        MEDIAN(transacciones_anteriores),
        1
    ) AS mediana_actividad_anterior,

    ROUND(
        MEDIAN(ctrx_quarter),
        1
    ) AS mediana_actividad_actual,

    COUNT(*) FILTER (
        WHERE clase_ternaria = 'BAJA+2'
    ) AS casos_baja_2,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE clase_ternaria = 'BAJA+2'
        )
        / COUNT(*),
        3
    ) AS tasa_baja_2_pct

FROM clasificacion

GROUP BY
    orden_banda,
    banda_actividad_actual,
    trayectoria

ORDER BY
    orden_banda,
    trayectoria;

,banda_actividad_actual,trayectoria,casos_cliente_mes,mediana_actividad_anterior,mediana_actividad_actual,casos_baja_2,tasa_baja_2_pct
0,0,CAYÓ 20% O MÁS,841,2.0,0.0,75,8.918
1,0,NO CAYÓ 20% O MÁS,4179,0.0,0.0,316,7.562
2,1 a 10,CAYÓ 20% O MÁS,3935,9.0,5.0,209,5.311
3,1 a 10,NO CAYÓ 20% O MÁS,9219,6.0,6.0,444,4.816
4,11 a 25,CAYÓ 20% O MÁS,4301,27.0,18.0,158,3.674
5,11 a 25,NO CAYÓ 20% O MÁS,21581,18.0,19.0,528,2.447
6,26 a 50,CAYÓ 20% O MÁS,5290,53.0,37.0,104,1.966
7,26 a 50,NO CAYÓ 20% O MÁS,49490,38.0,39.0,490,0.990
8,51 a 100,CAYÓ 20% O MÁS,6077,97.0,70.0,78,1.284
9,51 a 100,NO CAYÓ 20% O MÁS,125434,76.0,77.0,396,0.316


Esta tabla confirma que la trayectoria agrega información, incluso comparando clientes con actividad actual semejante.
Ejemplos:
- 26–50 transacciones: caída 1,97% vs sin caída 0,99%.
- 51–100: caída 1,28% vs 0,32%.
- 101–200: caída 0,44% vs 0,12%.
En todas las bandas, quienes cayeron al menos 20% tienen mayor riesgo. En “más de 200” hay solo 2 bajas en el grupo con caída, por lo que esa diferencia es inestable.
Conclusión para guardar:
BAJA+2 se relaciona tanto con un bajo nivel actual de actividad trimestral como con una caída respecto del mes anterior. Por eso, más adelante conviene crear lag y delta de ctrx_quarter.

In [95]:
%%sql

WITH columnas AS (

    SELECT
        name AS variable,
        type AS tipo_dato

    FROM pragma_table_info('competencia_01')

    WHERE name NOT IN (
        'numero_de_cliente',
        'foto_mes',
        'clase_ternaria'
    )
),

clasificacion AS (

    SELECT
        *,

        CASE
            WHEN REGEXP_MATCHES(variable, '^(Master_|Visa_)')
                THEN 'Tarjetas'

            WHEN variable IN (
                'active_quarter',
                'cliente_vip',
                'internet',
                'cliente_edad',
                'cliente_antiguedad',
                'cproductos',
                'tcuentas'
            )
                THEN 'Cliente y relación'

            WHEN REGEXP_MATCHES(
                variable,
                '^(mrentabilidad|mcomisiones$|mactivos_margen$|mpasivos_margen$|[cm]comisiones_)'
            )
                THEN 'Rentabilidad, márgenes y comisiones'

            WHEN REGEXP_MATCHES(
                variable,
                'cuenta_corriente|caja_ahorro|descubierto|^mcuentas_saldo$'
            )
                THEN 'Cuentas y saldos'

            WHEN variable IN (
                'ctarjeta_debito',
                'ctarjeta_debito_transacciones',
                'ctarjeta_visa',
                'ctarjeta_visa_transacciones',
                'mtarjeta_visa_consumo',
                'ctarjeta_master',
                'ctarjeta_master_transacciones',
                'mtarjeta_master_consumo'
            )
                THEN 'Tarjetas'

            WHEN REGEXP_MATCHES(
                variable,
                'prestamos|plazo_fijo|inversion'
            )
                THEN 'Préstamos e inversiones'

            WHEN REGEXP_MATCHES(
                variable,
                'seguro|^ccaja_seguridad$'
            )
                THEN 'Seguros y caja de seguridad'

            WHEN REGEXP_MATCHES(variable, 'payroll')
                THEN 'Payroll'

            WHEN REGEXP_MATCHES(
                variable,
                'debitos_automaticos|pagodeservicios|pagomiscuentas'
            )
                THEN 'Débitos y pagos recurrentes'

            WHEN REGEXP_MATCHES(variable, 'descuentos')
                THEN 'Descuentos'

            WHEN REGEXP_MATCHES(
                variable,
                'forex|transferencias|extraccion_autoservicio|cheques'
            )
                THEN 'Movimientos financieros'

            WHEN REGEXP_MATCHES(
                variable,
                'autoservicio|callcenter|homebanking|cajas_|atm|mobile_app|^ctrx_quarter$'
            )
                THEN 'Canales y actividad'

            ELSE 'SIN CLASIFICAR'
        END AS familia

    FROM columnas
)

SELECT
    familia,
    COUNT(*) AS cantidad_variables

FROM clasificacion

GROUP BY familia

ORDER BY cantidad_variables DESC;

,familia,cantidad_variables
0,Tarjetas,52
1,Movimientos financieros,19
2,Canales y actividad,17
3,Préstamos e inversiones,14
4,Débitos y pagos recurrentes,10
5,Cuentas y saldos,9
6,"Rentabilidad, márgenes y comisiones",9
7,Cliente y relación,7
8,Descuentos,6
9,Seguros y caja de seguridad,5


In [96]:
%%sql

SELECT
    cid,
    name AS variable,
    type AS tipo_dato

FROM pragma_table_info('competencia_01')

WHERE name NOT IN (
    'numero_de_cliente',
    'foto_mes',
    'clase_ternaria'
)
  AND (
      cid < 2
      OR cid > 153
  )

ORDER BY cid;

,cid,variable,tipo_dato
0,0,active_quarter,BIGINT
1,1,cliente_vip,BIGINT


Tarjetas: 52 variables, 34% del total.
Tarjetas + movimientos + canales: 88 variables, casi 58%.
Perfil general del cliente: solamente 7 variables.
Muchas columnas de Master y Visa están espejadas; más adelante revisaremos redundancia.

In [97]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    ROUND(
        QUANTILE_CONT(cliente_edad, 0.10),
        1
    ) AS edad_percentil_10,

    ROUND(
        MEDIAN(cliente_edad),
        1
    ) AS edad_mediana,

    ROUND(
        QUANTILE_CONT(cliente_edad, 0.90),
        1
    ) AS edad_percentil_90,

    ROUND(
        QUANTILE_CONT(cliente_antiguedad, 0.10),
        1
    ) AS antiguedad_percentil_10,

    ROUND(
        MEDIAN(cliente_antiguedad),
        1
    ) AS antiguedad_mediana,

    ROUND(
        QUANTILE_CONT(cliente_antiguedad, 0.90),
        1
    ) AS antiguedad_percentil_90,

    ROUND(
        100.0 * AVG(active_quarter),
        2
    ) AS pct_clientes_activos,

    ROUND(
        100.0 * AVG(cliente_vip),
        2
    ) AS pct_clientes_vip,

    ROUND(
        MEDIAN(cproductos),
        1
    ) AS mediana_productos,

    ROUND(
        MEDIAN(tcuentas),
        1
    ) AS mediana_cuentas

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,edad_percentil_10,edad_mediana,edad_percentil_90,antiguedad_percentil_10,antiguedad_mediana,antiguedad_percentil_90,pct_clientes_activos,pct_clientes_vip,mediana_productos,mediana_cuentas
0,202103,162900,31.0,45.0,65.0,30.0,126.0,259.0,98.57,0.26,7.0,1.0
1,202104,163284,31.0,45.0,65.0,30.0,127.0,260.0,98.69,0.26,7.0,1.0
2,202105,163768,31.0,45.0,65.0,30.0,127.0,261.0,98.73,0.27,7.0,1.0
3,202106,164114,31.0,45.0,65.0,30.0,128.0,261.0,98.77,0.56,7.0,1.0
4,202107,164348,31.0,45.0,65.0,30.0,128.0,262.0,98.75,0.27,7.0,1.0
5,202108,164647,31.0,45.0,65.0,30.0,129.0,263.0,98.66,0.28,7.0,1.0


In [98]:
%%sql

WITH indicadores AS (

    SELECT
        foto_mes,

        CASE WHEN COALESCE(ctarjeta_debito, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_debito,

        CASE WHEN COALESCE(ctarjeta_visa, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_visa,

        CASE WHEN COALESCE(ctarjeta_master, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_master,

        CASE WHEN
            COALESCE(cprestamos_personales, 0)
          + COALESCE(cprestamos_prendarios, 0)
          + COALESCE(cprestamos_hipotecarios, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_prestamo,

        CASE WHEN
            COALESCE(cplazo_fijo, 0)
          + COALESCE(cinversion1, 0)
          + COALESCE(cinversion2, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_inversion,

        CASE WHEN
            COALESCE(cseguro_vida, 0)
          + COALESCE(cseguro_auto, 0)
          + COALESCE(cseguro_vivienda, 0)
          + COALESCE(cseguro_accidentes_personales, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_seguro,

        CASE WHEN
               COALESCE(cpayroll_trx, 0) > 0
            OR COALESCE(cpayroll2_trx, 0) > 0
            OR COALESCE(mpayroll, 0) > 0
            OR COALESCE(mpayroll2, 0) > 0
            THEN 1 ELSE 0
        END AS tiene_movimiento_payroll,

        CASE WHEN
               COALESCE(thomebanking, 0) > 0
            OR COALESCE(tmobile_app, 0) > 0
            THEN 1 ELSE 0
        END AS usa_canal_digital

    FROM competencia_01
)

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(100.0 * AVG(tiene_debito), 2)
        AS pct_con_debito,

    ROUND(100.0 * AVG(tiene_visa), 2)
        AS pct_con_visa,

    ROUND(100.0 * AVG(tiene_master), 2)
        AS pct_con_master,

    ROUND(100.0 * AVG(tiene_prestamo), 2)
        AS pct_con_prestamo,

    ROUND(100.0 * AVG(tiene_inversion), 2)
        AS pct_con_inversion,

    ROUND(100.0 * AVG(tiene_seguro), 2)
        AS pct_con_seguro,

    ROUND(100.0 * AVG(tiene_movimiento_payroll), 2)
        AS pct_con_movimiento_payroll,

    ROUND(100.0 * AVG(usa_canal_digital), 2)
        AS pct_que_usa_canal_digital

FROM indicadores

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,pct_con_debito,pct_con_visa,pct_con_master,pct_con_prestamo,pct_con_inversion,pct_con_seguro,pct_con_movimiento_payroll,pct_que_usa_canal_digital
0,202103,162900,99.95,94.88,89.40,23.45,16.51,31.46,54.05,75.29
1,202104,163284,99.95,94.75,89.35,23.60,16.93,31.38,54.10,74.06
2,202105,163768,99.95,94.70,89.46,23.41,17.21,31.22,54.96,74.05
3,202106,164114,99.95,94.78,89.62,23.16,17.76,31.16,55.27,73.03
4,202107,164348,99.95,94.69,89.62,22.77,18.53,31.03,55.17,73.32
5,202108,164647,99.95,94.66,89.53,22.13,18.36,30.96,55.75,74.06


In [99]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjeta_visa > 0
              AND ctarjeta_master > 0
        ) / COUNT(*),
        2
    ) AS pct_con_ambas,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjeta_visa > 0
              AND ctarjeta_master = 0
        ) / COUNT(*),
        2
    ) AS pct_solo_visa,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjeta_visa = 0
              AND ctarjeta_master > 0
        ) / COUNT(*),
        2
    ) AS pct_solo_master,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjeta_visa = 0
              AND ctarjeta_master = 0
        ) / COUNT(*),
        2
    ) AS pct_sin_visa_ni_master,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjeta_visa > 0
              AND ctarjeta_visa_transacciones > 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE ctarjeta_visa > 0
            ),
            0
        ),
        2
    ) AS pct_tenedores_visa_con_transacciones,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjeta_master > 0
              AND ctarjeta_master_transacciones > 0
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE ctarjeta_master > 0
            ),
            0
        ),
        2
    ) AS pct_tenedores_master_con_transacciones,

    ROUND(
        MEDIAN(ctarjeta_visa_transacciones) FILTER (
            WHERE ctarjeta_visa_transacciones > 0
        ),
        1
    ) AS mediana_transacciones_usuarios_visa,

    ROUND(
        MEDIAN(ctarjeta_master_transacciones) FILTER (
            WHERE ctarjeta_master_transacciones > 0
        ),
        1
    ) AS mediana_transacciones_usuarios_master

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,pct_con_ambas,pct_solo_visa,pct_solo_master,pct_sin_visa_ni_master,pct_tenedores_visa_con_transacciones,pct_tenedores_master_con_transacciones,mediana_transacciones_usuarios_visa,mediana_transacciones_usuarios_master
0,202103,162900,88.55,6.33,0.85,4.27,89.53,39.68,10.0,3.0
1,202104,163284,88.50,6.25,0.85,4.40,89.82,41.53,11.0,3.0
2,202105,163768,88.57,6.13,0.89,4.41,89.93,41.64,11.0,3.0
3,202106,164114,88.72,6.05,0.90,4.32,90.37,43.45,11.0,4.0
4,202107,164348,88.73,5.96,0.89,4.43,90.44,42.77,11.0,3.0
5,202108,164647,88.83,5.82,0.69,4.65,90.55,43.53,11.0,3.0



- Tenencia: 88,5% tiene ambas marcas. Master no tiene pocos registros en esta base.
- Principalidad: Visa es claramente la tarjeta utilizada.
  - Cerca del 90% de los tenedores de Visa realiza transacciones.
  - Solo 40–44% de los tenedores de Master las realiza.
  - Entre usuarios activos, la mediana es 11 transacciones Visa contra 3 Master.
- Solo Visa ronda 6%; Solo Master, menos de 1%.

In [100]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE tcallcenter > 0
        ) / COUNT(*),
        2
    ) AS pct_usa_callcenter,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE thomebanking > 0
        ) / COUNT(*),
        2
    ) AS pct_usa_homebanking,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE tmobile_app > 0
        ) / COUNT(*),
        2
    ) AS pct_usa_mobile_app,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ccajas_transacciones > 0
        ) / COUNT(*),
        2
    ) AS pct_usa_sucursal,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE COALESCE(catm_trx, 0)
                + COALESCE(catm_trx_other, 0) > 0
        ) / COUNT(*),
        2
    ) AS pct_usa_atm

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,pct_usa_callcenter,pct_usa_homebanking,pct_usa_mobile_app,pct_usa_sucursal,pct_usa_atm
0,202103,162900,10.67,74.96,3.28,9.83,51.79
1,202104,163284,9.80,73.68,3.06,8.51,50.83
2,202105,163768,9.13,73.70,2.91,7.95,49.22
3,202106,164114,8.39,72.68,2.86,8.05,49.65
4,202107,164348,8.31,73.09,2.36,8.86,51.04
5,202108,164647,7.34,73.77,2.57,9.59,50.46


No aparece una migración clara hacia la aplicación móvil:
- Call center cae de 10,67% a 7,34%.
- Home banking sigue siendo el canal dominante; baja hasta junio y recupera parcialmente.
- tmobile_app permanece muy bajo y también cae.
- ATM se mantiene alrededor de 50%.
- Sucursal cae hasta mayo y después se recupera.

El próximo paso general es detectar variables que contienen casi la misma información. Para correlaciones conviene usar Python y un único mes, evitando que la tendencia temporal infle artificialmente los resultados:

In [101]:

variables_fecha = [
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_fultimo_cierre",
    "Master_fechaalta",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_fultimo_cierre",
    "Visa_fechaalta"
]

excluir = {
    "numero_de_cliente",
    "foto_mes",
    "clase_ternaria",
    *variables_fecha
}

# Usamos junio como una fotografía transversal completa
datos_correlacion = data.loc[
    data["foto_mes"] == 202106
].copy()

# Solo variables numéricas con más de 20 valores diferentes
variables_correlacion = [
    variable
    for variable in datos_correlacion.select_dtypes(
        include="number"
    ).columns
    if variable not in excluir
    and datos_correlacion[variable].nunique(dropna=True) > 20
]

matriz_correlacion = datos_correlacion[
    variables_correlacion
].corr(
    method="pearson",
    min_periods=1000
)

# Conservamos solo una mitad de la matriz para no duplicar pares
mascara_superior = np.triu(
    np.ones(matriz_correlacion.shape, dtype=bool),
    k=1
)

pares_correlacion = (
    matriz_correlacion
    .where(mascara_superior)
    .stack()
    .reset_index()
)

pares_correlacion.columns = [
    "variable_1",
    "variable_2",
    "correlacion"
]

pares_correlacion["correlacion_absoluta"] = (
    pares_correlacion["correlacion"].abs()
)

pares_muy_correlacionados = (
    pares_correlacion[
        pares_correlacion["correlacion_absoluta"] >= 0.98
    ]
    .sort_values(
        "correlacion_absoluta",
        ascending=False
    )
)

display(pares_muy_correlacionados.head(30))

,variable_1,variable_2,correlacion,correlacion_absoluta
4857,Master_mconsumospesos,Master_mconsumototal,1.000000,1.000000
5103,Visa_mconsumospesos,Visa_mconsumototal,1.000000,1.000000
443,mcomisiones,mcomisiones_otras,0.999385,0.999385
5060,Visa_msaldototal,Visa_msaldopesos,0.988111,0.988111


La redundancia fuerte no es generalizada: solo aparecen cuatro pares.
Lectura provisional:
- mconsumospesos y mconsumototal: podrían ser duplicados, pero correlación 1 no garantiza igualdad exacta.
- mcomisiones y mcomisiones_otras: probablemente el total sea “otras + mantenimiento”.
- Visa_msaldototal y Visa_msaldopesos: están relacionadas, pero 0,988 no permite tratarlas como duplicadas.
Verifiquemos igualdad exacta en todos los meses:

In [102]:
%%sql

SELECT
    foto_mes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Master_mconsumototal IS NOT NULL
              AND Master_mconsumospesos IS NOT NULL
              AND ABS(
                  Master_mconsumototal
                  - Master_mconsumospesos
              ) < 0.01
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Master_mconsumototal IS NOT NULL
                  AND Master_mconsumospesos IS NOT NULL
            ),
            0
        ),
        2
    ) AS pct_master_consumo_total_igual_pesos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_mconsumototal IS NOT NULL
              AND Visa_mconsumospesos IS NOT NULL
              AND ABS(
                  Visa_mconsumototal
                  - Visa_mconsumospesos
              ) < 0.01
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_mconsumototal IS NOT NULL
                  AND Visa_mconsumospesos IS NOT NULL
            ),
            0
        ),
        2
    ) AS pct_visa_consumo_total_igual_pesos,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ABS(
                mcomisiones
                - (
                    COALESCE(mcomisiones_mantenimiento, 0)
                    + COALESCE(mcomisiones_otras, 0)
                )
            ) < 0.01
        ) / COUNT(*),
        2
    ) AS pct_comisiones_igual_suma_componentes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE Visa_msaldototal IS NOT NULL
              AND Visa_msaldopesos IS NOT NULL
              AND ABS(
                  Visa_msaldototal
                  - Visa_msaldopesos
              ) < 0.01
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE Visa_msaldototal IS NOT NULL
                  AND Visa_msaldopesos IS NOT NULL
            ),
            0
        ),
        2
    ) AS pct_visa_saldo_total_igual_pesos

FROM competencia_01

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,pct_master_consumo_total_igual_pesos,pct_visa_consumo_total_igual_pesos,pct_comisiones_igual_suma_componentes,pct_visa_saldo_total_igual_pesos
0,202103,100.0,100.0,64.89,9.13
1,202104,100.0,100.0,63.80,8.90
2,202105,100.0,100.0,65.88,8.73
3,202106,100.0,100.0,65.16,8.32
4,202107,100.0,100.0,65.62,8.32
5,202108,100.0,100.0,66.89,8.32


La salida separa claramente los casos:
- Master_mconsumototal = Master_mconsumospesos: 100% en todos los meses.
- Visa_mconsumototal = Visa_mconsumospesos: 100% en todos los meses.
Son duplicados entre los valores informados. Semánticamente convendría conservar mconsumospesos y mconsumosdolares, y más adelante excluir mconsumototal.
Los otros pares no son redundantes:
- Comisiones: la fórmula coincide solo en 64–67%. La correlación era alta porque mcomisiones_otras domina el total.
- Saldo Visa: coincide solo en 8–9%. Pesos explica gran parte del total, pero no todo.
Falta confirmar que los duplicados también tengan los nulos en las mismas filas:

In [103]:
%%sql

SELECT
    COUNT(*) FILTER (
        WHERE
            (Master_mconsumototal IS NULL)
            <> (Master_mconsumospesos IS NULL)
    ) AS master_nulos_no_coincidentes,

    COUNT(*) FILTER (
        WHERE Master_mconsumototal IS NOT NULL
          AND Master_mconsumospesos IS NOT NULL
          AND ABS(
              Master_mconsumototal
              - Master_mconsumospesos
          ) >= 0.01
    ) AS master_valores_no_coincidentes,

    COUNT(*) FILTER (
        WHERE
            (Visa_mconsumototal IS NULL)
            <> (Visa_mconsumospesos IS NULL)
    ) AS visa_nulos_no_coincidentes,

    COUNT(*) FILTER (
        WHERE Visa_mconsumototal IS NOT NULL
          AND Visa_mconsumospesos IS NOT NULL
          AND ABS(
              Visa_mconsumototal
              - Visa_mconsumospesos
          ) >= 0.01
    ) AS visa_valores_no_coincidentes

FROM competencia_01;

,master_nulos_no_coincidentes,master_valores_no_coincidentes,visa_nulos_no_coincidentes,visa_valores_no_coincidentes
0,0,0,0,0


- Conservar Master_mconsumospesos y Visa_mconsumospesos.
- Excluir más adelante Master_mconsumototal y Visa_mconsumototal.
- No modificar todavía la tabla original.

In [104]:
%%sql

CREATE OR REPLACE VIEW competencia_01_analitica AS

SELECT
    c.* EXCLUDE (
        Master_mfinanciacion_limite,
        Visa_mfinanciacion_limite,
        Master_mconsumototal,
        Visa_mconsumototal
    ),

    CASE
        WHEN Master_mlimitecompra > 0
         AND Master_mfinanciacion_limite
             / NULLIF(Master_mlimitecompra, 0) > 10
        THEN 0.9 * Master_mlimitecompra
        ELSE Master_mfinanciacion_limite
    END AS Master_mfinanciacion_limite,

    CASE
        WHEN Visa_mlimitecompra > 0
         AND Visa_mfinanciacion_limite
             / NULLIF(Visa_mlimitecompra, 0) > 10
        THEN 0.9 * Visa_mlimitecompra
        ELSE Visa_mfinanciacion_limite
    END AS Visa_mfinanciacion_limite,

    CASE
        WHEN Master_mlimitecompra > 0
         AND Master_mfinanciacion_limite
             / NULLIF(Master_mlimitecompra, 0) > 10
        THEN 1
        ELSE 0
    END AS Master_limite_financiacion_fue_corregido,

    CASE
        WHEN Visa_mlimitecompra > 0
         AND Visa_mfinanciacion_limite
             / NULLIF(Visa_mlimitecompra, 0) > 10
        THEN 1
        ELSE 0
    END AS Visa_limite_financiacion_fue_corregido,

    ccajas_depositos + ccajas_otras
        AS ccajas_depositos_mas_otras

FROM competencia_01 AS c;

,Success


In [105]:
%%sql

SELECT
    table_name,
    table_type
FROM information_schema.tables
WHERE table_name = 'competencia_01_analitica';

,table_name,table_type
0,competencia_01_analitica,VIEW


In [106]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS registros,

    SUM(
        Master_limite_financiacion_fue_corregido
    ) AS master_limites_corregidos,

    SUM(
        Visa_limite_financiacion_fue_corregido
    ) AS visa_limites_corregidos,

    ROUND(
        MAX(
            ABS(
                Master_mfinanciacion_limite
                / NULLIF(Master_mlimitecompra, 0)
            )
        ),
        2
    ) AS maxima_relacion_master_despues,

    ROUND(
        MAX(
            ABS(
                Visa_mfinanciacion_limite
                / NULLIF(Visa_mlimitecompra, 0)
            )
        ),
        2
    ) AS maxima_relacion_visa_despues,

    COUNT(*) FILTER (
        WHERE Master_mlimitecompra > 0
          AND ABS(
              Master_mfinanciacion_limite
              / Master_mlimitecompra
          ) > 10
    ) AS anomalias_master_restantes,

    COUNT(*) FILTER (
        WHERE Visa_mlimitecompra > 0
          AND ABS(
              Visa_mfinanciacion_limite
              / Visa_mlimitecompra
          ) > 10
    ) AS anomalias_visa_restantes

FROM competencia_01_analitica

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,registros,master_limites_corregidos,visa_limites_corregidos,maxima_relacion_master_despues,maxima_relacion_visa_despues,anomalias_master_restantes,anomalias_visa_restantes
0,202103,162900,16.0,15.0,1.02,1.08,0,0
1,202104,163284,4159.0,19.0,1.02,1.17,0,0
2,202105,163768,7.0,7.0,0.99,1.58,0,0
3,202106,164114,20.0,21.0,1.17,1.00,0,0
4,202107,164348,4.0,4.0,1.17,1.58,0,0
5,202108,164647,39.0,1035.0,1.17,4.11,0,0


Primero armamos la población del ranking:
- Solo 202103–202106, porque tienen clase completa.
- BAJA+2 = 1.
- CONTINUA y BAJA+1 = 0, porque el premio corresponde únicamente a BAJA+2.

In [107]:
base_ranking = %sql SELECT * FROM competencia_01_analitica WHERE foto_mes BETWEEN 202103 AND 202106 AND clase_ternaria IS NOT NULL

base_ranking["target_baja2"] = (
    base_ranking["clase_ternaria"] == "BAJA+2"
).astype(int)

resumen_target = (
    base_ranking
    .groupby("foto_mes")
    .agg(
        clientes=("target_baja2", "size"),
        casos_baja2=("target_baja2", "sum")
    )
    .reset_index()
)

resumen_target["pct_baja2"] = (
    100
    * resumen_target["casos_baja2"]
    / resumen_target["clientes"]
).round(3)

resumen_target

,foto_mes,clientes,casos_baja2,pct_baja2
0,202103,162900,960,0.589
1,202104,163284,1139,0.698
2,202105,163768,870,0.531
3,202106,164114,1098,0.669


In [108]:
columnas_excluidas = {
    "numero_de_cliente",
    "foto_mes",
    "clase_ternaria",
    "target_baja2",

    # Fechas todavía codificadas
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_fultimo_cierre",
    "Master_fechaalta",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_fultimo_cierre",
    "Visa_fechaalta",

    # Indicadores técnicos
    "Master_limite_financiacion_fue_corregido",
    "Visa_limite_financiacion_fue_corregido",
}


variables_numericas = [
    variable
    for variable
    in base_ranking.select_dtypes(include="number").columns
    if variable not in columnas_excluidas
]


datos_descubrimiento = base_ranking[
    base_ranking["foto_mes"].between(202103, 202105)
]

datos_confirmacion = base_ranking[
    base_ranking["foto_mes"] == 202106
]


def calcular_auc(y, x):

    auxiliar = pd.DataFrame({
        "target": y,
        "variable": pd.to_numeric(x, errors="coerce")
    })

    auxiliar = (
        auxiliar
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if (
        auxiliar["target"].nunique() < 2
        or auxiliar["variable"].nunique() < 2
    ):
        return np.nan

    return roc_auc_score(
        auxiliar["target"],
        auxiliar["variable"]
    )


resultados = []

for variable in variables_numericas:

    # Señal contenida en el valor
    auc_valor = calcular_auc(
        datos_descubrimiento["target_baja2"],
        datos_descubrimiento[variable]
    )

    # Señal contenida en que el dato sea nulo
    auc_nulidad = calcular_auc(
        datos_descubrimiento["target_baja2"],
        datos_descubrimiento[variable].isna().astype(int)
    )

    candidatos = []

    if pd.notna(auc_valor):
        candidatos.append(("VALOR", auc_valor))

    if pd.notna(auc_nulidad):
        candidatos.append(("NULIDAD", auc_nulidad))

    if not candidatos:
        continue

    tipo_senal, auc_descubrimiento_cruda = max(
        candidatos,
        key=lambda resultado: max(
            resultado[1],
            1 - resultado[1]
        )
    )

    direccion_positiva = auc_descubrimiento_cruda >= 0.5

    auc_descubrimiento = max(
        auc_descubrimiento_cruda,
        1 - auc_descubrimiento_cruda
    )

    if tipo_senal == "VALOR":
        variable_confirmacion = datos_confirmacion[variable]
    else:
        variable_confirmacion = (
            datos_confirmacion[variable]
            .isna()
            .astype(int)
        )

    auc_confirmacion_cruda = calcular_auc(
        datos_confirmacion["target_baja2"],
        variable_confirmacion
    )

    if pd.isna(auc_confirmacion_cruda):
        auc_confirmacion = np.nan
    elif direccion_positiva:
        auc_confirmacion = auc_confirmacion_cruda
    else:
        auc_confirmacion = 1 - auc_confirmacion_cruda

    if tipo_senal == "VALOR":
        direccion = (
            "VALORES ALTOS = MÁS BAJA+2"
            if direccion_positiva
            else "VALORES BAJOS = MÁS BAJA+2"
        )
    else:
        direccion = (
            "SER NULO = MÁS BAJA+2"
            if direccion_positiva
            else "SER NULO = MENOS BAJA+2"
        )

    if pd.isna(auc_confirmacion):
        peor_auc = np.nan
        conserva_direccion = "NO EVALUABLE"
        diferencia_auc = np.nan
    else:
        peor_auc = min(
            auc_descubrimiento,
            auc_confirmacion
        )

        conserva_direccion = (
            "SÍ" if auc_confirmacion >= 0.5 else "NO"
        )

        diferencia_auc = abs(
            auc_descubrimiento
            - auc_confirmacion
        )

    resultados.append({
        "variable": variable,
        "senal_evaluada": tipo_senal,
        "direccion_en_marzo_mayo": direccion,
        "auc_marzo_mayo": auc_descubrimiento,
        "auc_junio": auc_confirmacion,
        "peor_auc": peor_auc,
        "diferencia_entre_periodos": diferencia_auc,
        "conserva_direccion_en_junio": conserva_direccion,
        "pct_nulos_marzo_mayo":
            100 * datos_descubrimiento[variable].isna().mean(),
        "valores_distintos_marzo_mayo":
            datos_descubrimiento[variable].nunique(dropna=True)
    })


ranking_variables = (
    pd.DataFrame(resultados)
    .sort_values(
        ["peor_auc", "auc_junio"],
        ascending=False
    )
    .round(3)
)

display(ranking_variables.head(20))

,variable,senal_evaluada,direccion_en_marzo_mayo,auc_marzo_mayo,auc_junio,peor_auc,diferencia_entre_periodos,conserva_direccion_en_junio,pct_nulos_marzo_mayo,valores_distintos_marzo_mayo
105,ctrx_quarter,VALOR,VALORES BAJOS = MÁS BAJA+2,0.837,0.844,0.837,0.007,SÍ,0.000,1056
16,mcaja_ahorro,VALOR,VALORES BAJOS = MÁS BAJA+2,0.806,0.811,0.806,0.005,SÍ,0.000,436317
9,mpasivos_margen,VALOR,VALORES BAJOS = MÁS BAJA+2,0.791,0.797,0.791,0.006,SÍ,0.000,251039
26,mtarjeta_visa_consumo,VALOR,VALORES BAJOS = MÁS BAJA+2,0.770,0.771,0.770,0.000,SÍ,0.000,381383
25,ctarjeta_visa_transacciones,VALOR,VALORES BAJOS = MÁS BAJA+2,0.766,0.769,0.766,0.003,SÍ,0.000,220
23,mautoservicio,VALOR,VALORES BAJOS = MÁS BAJA+2,0.761,0.775,0.761,0.015,SÍ,0.000,292182
22,ctarjeta_debito_transacciones,VALOR,VALORES BAJOS = MÁS BAJA+2,0.758,0.770,0.758,0.012,SÍ,0.000,139
20,mcuentas_saldo,VALOR,VALORES BAJOS = MÁS BAJA+2,0.758,0.780,0.758,0.022,SÍ,0.000,470693
71,ccomisiones_otras,VALOR,VALORES BAJOS = MÁS BAJA+2,0.750,0.751,0.750,0.001,SÍ,0.000,212
50,mpayroll,VALOR,VALORES BAJOS = MÁS BAJA+2,0.737,0.747,0.737,0.009,SÍ,0.000,208848


In [109]:
%%sql

WITH base AS (

    SELECT
        foto_mes,
        mcaja_ahorro,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
      AND mcaja_ahorro IS NOT NULL
),

deciles AS (

    SELECT
        *,

        NTILE(10) OVER (
            PARTITION BY foto_mes
            ORDER BY mcaja_ahorro
        ) AS decil_mcaja_ahorro

    FROM base
)

SELECT
    decil_mcaja_ahorro,

    COUNT(*) AS casos_cliente_mes,

    ROUND(
        MEDIAN(mcaja_ahorro),
        2
    ) AS mediana_mcaja_ahorro,

    SUM(es_baja2) AS casos_baja2,

    ROUND(
        100.0 * SUM(es_baja2) / COUNT(*),
        3
    ) AS tasa_baja2_pct

FROM deciles

GROUP BY decil_mcaja_ahorro

ORDER BY decil_mcaja_ahorro;

,decil_mcaja_ahorro,casos_cliente_mes,mediana_mcaja_ahorro,casos_baja2,tasa_baja2_pct
0,1,65408,0.00,2111.0,3.227
1,2,65408,1536.30,694.0,1.061
2,3,65408,5767.13,360.0,0.550
3,4,65408,11558.72,246.0,0.376
4,5,65406,19078.38,163.0,0.249
5,6,65406,29310.88,151.0,0.231
6,7,65406,44406.12,106.0,0.162
7,8,65406,69713.97,94.0,0.144
8,9,65405,123605.32,77.0,0.118
9,10,65405,321411.54,65.0,0.099


La relación es muy fuerte y perfectamente ordenada:
- El 10% con menor mcaja_ahorro tiene una tasa BAJA+2 de 3,227%.
- El 10% con mayor valor tiene 0,099%.
- El primer grupo tiene aproximadamente 33 veces más riesgo.
- El primer decil concentra 2.111 de 4.067 BAJA+2: 51,9%, aunque contiene solo el 10% de los registros.
- Los dos primeros deciles concentran aproximadamente 69% de los BAJA+2.
Todavía debemos confirmar que no sea un efecto producido por un solo mes:

In [110]:
%%sql

WITH base AS (

    SELECT
        foto_mes,
        mcaja_ahorro,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
      AND mcaja_ahorro IS NOT NULL
),

deciles AS (

    SELECT
        *,

        NTILE(10) OVER (
            PARTITION BY foto_mes
            ORDER BY mcaja_ahorro
        ) AS decil_mcaja_ahorro

    FROM base
)

SELECT
    foto_mes,

    ROUND(
        100.0 * AVG(es_baja2)
            FILTER (WHERE decil_mcaja_ahorro = 1),
        3
    ) AS tasa_baja2_10_pct_menor,

    ROUND(
        100.0 * AVG(es_baja2)
            FILTER (
                WHERE decil_mcaja_ahorro BETWEEN 2 AND 9
            ),
        3
    ) AS tasa_baja2_80_pct_intermedio,

    ROUND(
        100.0 * AVG(es_baja2)
            FILTER (WHERE decil_mcaja_ahorro = 10),
        3
    ) AS tasa_baja2_10_pct_mayor

FROM deciles

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,tasa_baja2_10_pct_menor,tasa_baja2_80_pct_intermedio,tasa_baja2_10_pct_mayor
0,202103,2.916,0.358,0.117
1,202104,3.730,0.392,0.110
2,202105,2.607,0.332,0.049
3,202106,3.656,0.364,0.122


La relación queda confirmada en todos los meses:
- Grupo inferior: tasa entre 2,61% y 3,73%.
- Grupo intermedio: alrededor de 0,33%–0,39%.
- Grupo superior: solamente 0,05%–0,12%.
Por lo tanto, no es una anomalía mensual. mcaja_ahorro queda como una señal fuerte y estable. Además, el decil 2 también presenta riesgo elevado, así que la relación no depende únicamente de los valores cero.

In [111]:
%%sql

WITH altas AS (

    SELECT
        numero_de_cliente,
        foto_mes,
        'Master' AS marca,
        Master_fechaalta AS valor_original
    FROM competencia_01

    UNION ALL

    SELECT
        numero_de_cliente,
        foto_mes,
        'Visa' AS marca,
        Visa_fechaalta AS valor_original
    FROM competencia_01
)

SELECT
    numero_de_cliente,
    foto_mes,
    marca,
    valor_original,

    CAST(
        STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
        + INTERVAL 1 MONTH
        AS DATE
    ) AS referencia_candidata,

    CAST(
        STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
        + INTERVAL 1 MONTH
        - valor_original * INTERVAL 1 DAY
        AS DATE
    ) AS fecha_alta_calculada

FROM altas

WHERE numero_de_cliente = 12159854
  AND valor_original IS NOT NULL

ORDER BY
    marca,
    foto_mes;

,numero_de_cliente,foto_mes,marca,valor_original,referencia_candidata,fecha_alta_calculada
0,12159854.0,202103,Master,3324,2021-04-01,2012-02-24
1,12159854.0,202104,Master,3354,2021-05-01,2012-02-24
2,12159854.0,202105,Master,3385,2021-06-01,2012-02-24
3,12159854.0,202106,Master,3415,2021-07-01,2012-02-24
4,12159854.0,202107,Master,3446,2021-08-01,2012-02-24
5,12159854.0,202108,Master,3477,2021-09-01,2012-02-24
6,12159854.0,202103,Visa,4056,2021-04-01,2010-02-22
7,12159854.0,202104,Visa,4086,2021-05-01,2010-02-22
8,12159854.0,202105,Visa,4117,2021-06-01,2010-02-22
9,12159854.0,202106,Visa,4147,2021-07-01,2010-02-22


In [112]:
%%sql

WITH base AS (

    SELECT
        ctarjeta_visa_transacciones,
        mtarjeta_visa_consumo,

        CASE
            WHEN clase_ternaria = 'BAJA+2' THEN 1
            ELSE 0
        END AS es_baja2,

        CASE
            WHEN ctarjeta_visa_transacciones = 0 THEN '0'
            WHEN ctarjeta_visa_transacciones <= 5 THEN '1 A 5'
            WHEN ctarjeta_visa_transacciones <= 10 THEN '6 A 10'
            WHEN ctarjeta_visa_transacciones <= 20 THEN '11 A 20'
            WHEN ctarjeta_visa_transacciones <= 50 THEN '21 A 50'
            ELSE 'MÁS DE 50'
        END AS banda_transacciones_visa,

        CASE
            WHEN ctarjeta_visa_transacciones = 0 THEN 1
            WHEN ctarjeta_visa_transacciones <= 5 THEN 2
            WHEN ctarjeta_visa_transacciones <= 10 THEN 3
            WHEN ctarjeta_visa_transacciones <= 20 THEN 4
            WHEN ctarjeta_visa_transacciones <= 50 THEN 5
            ELSE 6
        END AS orden_banda

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
)

SELECT
    banda_transacciones_visa,
    COUNT(*) AS casos_cliente_mes,

    ROUND(
        MEDIAN(mtarjeta_visa_consumo),
        2
    ) AS mediana_consumo_visa,

    SUM(es_baja2) AS casos_baja2,

    ROUND(
        100.0 * SUM(es_baja2) / COUNT(*),
        3
    ) AS tasa_baja2_pct

FROM base

GROUP BY
    orden_banda,
    banda_transacciones_visa

ORDER BY orden_banda;

,banda_transacciones_visa,casos_cliente_mes,mediana_consumo_visa,casos_baja2,tasa_baja2_pct
0,0,96700,0.00,2170.0,2.244
1,1 A 5,145215,6610.14,977.0,0.673
2,6 A 10,128234,17614.64,416.0,0.324
3,11 A 20,153743,32058.10,312.0,0.203
4,21 A 50,112081,62992.51,176.0,0.157
5,MÁS DE 50,18093,145953.26,16.0,0.088


La relación también es muy clara:
- Sin transacciones: tasa BAJA+2 de 2,244%.
- Más de 50 transacciones: 0,088%.
- El grupo sin transacciones tiene unas 25 veces más riesgo.
- Cero transacciones concentra 2.170 de 4.067 BAJA+2: 53,4%.
- La tasa disminuye en todas las bandas y el consumo mediano aumenta coherentemente.
Pero falta distinguir dos situaciones: no tener Visa y tener Visa pero no usarla.

In [113]:
%%sql

WITH base AS (

    SELECT
        CASE
            WHEN COALESCE(ctarjeta_visa, 0) = 0
                THEN 'SIN VISA'

            WHEN ctarjeta_visa_transacciones = 0
                THEN 'CON VISA, SIN TRANSACCIONES'

            ELSE 'CON VISA Y TRANSACCIONES'
        END AS estado_uso_visa,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2,

        mtarjeta_visa_consumo

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
)

SELECT
    estado_uso_visa,

    COUNT(*) AS casos_cliente_mes,

    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_de_la_base,

    ROUND(
        MEDIAN(mtarjeta_visa_consumo),
        2
    ) AS mediana_consumo_visa,

    SUM(es_baja2) AS casos_baja2,

    ROUND(
        100.0 * SUM(es_baja2) / COUNT(*),
        3
    ) AS tasa_baja2_pct

FROM base

GROUP BY estado_uso_visa

ORDER BY tasa_baja2_pct DESC;

,estado_uso_visa,casos_cliente_mes,pct_de_la_base,mediana_consumo_visa,casos_baja2,tasa_baja2_pct
0,SIN VISA,34163,5.22,0.00,1111.0,3.252
1,"CON VISA, SIN TRANSACCIONES",62537,9.56,0.00,1059.0,1.693
2,CON VISA Y TRANSACCIONES,557366,85.22,24675.43,1897.0,0.340


Acá aparecen dos señales distintas:
- Sin Visa: 5,22% de la base, pero concentra 27,3% de los BAJA+2. Tasa: 3,252%.
- Tiene Visa pero no la usa: tasa 1,693%.
- Tiene Visa y la usa: tasa 0,340%.
Entonces no es solamente tenencia. Tener la tarjeta sin utilizarla también está asociado con mayor baja. Esto refleja pérdida de vinculación o principalidad, aunque no demuestra causalidad.
Verificamos que las tres relaciones se mantengan mensualmente:

In [114]:
%%sql

WITH base AS (

    SELECT
        foto_mes,

        CASE
            WHEN COALESCE(ctarjeta_visa, 0) = 0
                THEN 'SIN VISA'
            WHEN ctarjeta_visa_transacciones = 0
                THEN 'CON VISA, SIN TRANSACCIONES'
            ELSE 'CON VISA Y TRANSACCIONES'
        END AS estado_uso_visa,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
)

SELECT
    foto_mes,

    ROUND(
        100.0 * AVG(es_baja2)
            FILTER (WHERE estado_uso_visa = 'SIN VISA'),
        3
    ) AS tasa_baja2_sin_visa,

    ROUND(
        100.0 * AVG(es_baja2)
            FILTER (
                WHERE estado_uso_visa =
                      'CON VISA, SIN TRANSACCIONES'
            ),
        3
    ) AS tasa_baja2_con_visa_sin_uso,

    ROUND(
        100.0 * AVG(es_baja2)
            FILTER (
                WHERE estado_uso_visa =
                      'CON VISA Y TRANSACCIONES'
            ),
        3
    ) AS tasa_baja2_con_visa_y_uso

FROM base

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,tasa_baja2_sin_visa,tasa_baja2_con_visa_sin_uso,tasa_baja2_con_visa_y_uso
0,202103,2.867,1.582,0.336
1,202104,4.104,2.012,0.338
2,202105,2.120,1.511,0.323
3,202106,3.921,1.669,0.364


In [115]:
%%sql

WITH base AS (

    SELECT
        foto_mes,
        cpayroll_trx,
        mpayroll,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2,

        CASE
            WHEN cpayroll_trx = 0 THEN 1
            WHEN cpayroll_trx = 1 THEN 2
            ELSE 3
        END AS orden_banda,

        CASE
            WHEN cpayroll_trx = 0
                THEN 'SIN MOVIMIENTOS PAYROLL'
            WHEN cpayroll_trx = 1
                THEN '1 MOVIMIENTO PAYROLL'
            ELSE '2 O MÁS MOVIMIENTOS PAYROLL'
        END AS banda_payroll

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
      AND cpayroll_trx IS NOT NULL
)

SELECT
    foto_mes,
    banda_payroll,

    COUNT(*) AS casos_cliente_mes,

    ROUND(
        MEDIAN(mpayroll),
        2
    ) AS mediana_monto_payroll,

    SUM(es_baja2) AS casos_baja2,

    ROUND(
        100.0 * SUM(es_baja2) / COUNT(*),
        3
    ) AS tasa_baja2_pct

FROM base

GROUP BY
    foto_mes,
    orden_banda,
    banda_payroll

ORDER BY
    foto_mes,
    orden_banda;

,foto_mes,banda_payroll,casos_cliente_mes,mediana_monto_payroll,casos_baja2,tasa_baja2_pct
0,202103,SIN MOVIMIENTOS PAYROLL,74880,0.00,887.0,1.185
1,202103,1 MOVIMIENTO PAYROLL,50185,88314.65,43.0,0.086
2,202103,2 O MÁS MOVIMIENTOS PAYROLL,37835,107772.14,30.0,0.079
3,202104,SIN MOVIMIENTOS PAYROLL,74945,0.00,1040.0,1.388
4,202104,1 MOVIMIENTO PAYROLL,54349,95419.33,62.0,0.114
5,202104,2 O MÁS MOVIMIENTOS PAYROLL,33990,107792.18,37.0,0.109
6,202105,SIN MOVIMIENTOS PAYROLL,73761,0.00,802.0,1.087
7,202105,1 MOVIMIENTO PAYROLL,54088,99898.43,39.0,0.072
8,202105,2 O MÁS MOVIMIENTOS PAYROLL,35919,112160.62,29.0,0.081
9,202106,SIN MOVIMIENTOS PAYROLL,73414,0.00,1023.0,1.393


Buscamos si la ausencia de payroll está asociada con mayor BAJA+2 y si el patrón continúa en junio, aun cuando payroll presentó drift.
Payroll muestra una señal estable:
- Sin payroll: tasa BAJA+2 entre 1,09% y 1,39%.
- Con payroll: aproximadamente 0,07%–0,11%.
- La diferencia principal es tener o no tener payroll; uno frente a dos movimientos cambia poco el riesgo.
En junio sucede algo importante: los clientes con un movimiento bajan de 54.088 a 26.648, mientras que los de dos o más suben de 35.919 a 64.052. El total con payroll casi no cambia. Es compatible con un movimiento adicional como aguinaldo, aunque la tabla no prueba la causa. Por eso el PSI de payroll no parece una rotura.

In [116]:
%%sql

WITH base AS (

    SELECT
        foto_mes,

        CASE
            WHEN COALESCE(ctransferencias_recibidas, 0) = 0
             AND COALESCE(ctransferencias_emitidas, 0) = 0
                THEN 'SIN TRANSFERENCIAS'

            WHEN COALESCE(ctransferencias_recibidas, 0) > 0
             AND COALESCE(ctransferencias_emitidas, 0) = 0
                THEN 'SOLO RECIBE'

            WHEN COALESCE(ctransferencias_recibidas, 0) = 0
             AND COALESCE(ctransferencias_emitidas, 0) > 0
                THEN 'SOLO EMITE'

            ELSE 'RECIBE Y EMITE'
        END AS estado_transferencias,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
),

mensual AS (

    SELECT
        foto_mes,
        estado_transferencias,
        COUNT(*) AS casos,
        SUM(es_baja2) AS bajas2,
        100.0 * SUM(es_baja2) / COUNT(*) AS tasa_baja2

    FROM base

    GROUP BY
        foto_mes,
        estado_transferencias
),

global AS (

    SELECT
        estado_transferencias,
        COUNT(*) AS casos_cliente_mes,
        SUM(es_baja2) AS casos_baja2,
        100.0 * SUM(es_baja2) / COUNT(*) AS tasa_global

    FROM base

    GROUP BY estado_transferencias
)

SELECT
    g.estado_transferencias,
    g.casos_cliente_mes,

    ROUND(
        100.0 * g.casos_cliente_mes
        / SUM(g.casos_cliente_mes) OVER (),
        2
    ) AS pct_de_la_base,

    g.casos_baja2,

    ROUND(g.tasa_global, 3)
        AS tasa_baja2_global_pct,

    ROUND(MIN(m.tasa_baja2), 3)
        AS menor_tasa_mensual_pct,

    ROUND(MAX(m.tasa_baja2), 3)
        AS mayor_tasa_mensual_pct

FROM global AS g

JOIN mensual AS m
    USING (estado_transferencias)

GROUP BY
    g.estado_transferencias,
    g.casos_cliente_mes,
    g.casos_baja2,
    g.tasa_global

ORDER BY tasa_baja2_global_pct DESC;

,estado_transferencias,casos_cliente_mes,pct_de_la_base,casos_baja2,tasa_baja2_global_pct,menor_tasa_mensual_pct,mayor_tasa_mensual_pct
0,SIN TRANSFERENCIAS,107973,16.51,2173.0,2.013,1.537,2.339
1,SOLO RECIBE,150323,22.98,950.0,0.632,0.580,0.693
2,SOLO EMITE,59421,9.08,160.0,0.269,0.251,0.312
3,RECIBE Y EMITE,336349,51.42,784.0,0.233,0.222,0.246


a señal es fuerte y estable:
- Sin transferencias: 16,5% de la base, pero concentra 53,4% de los BAJA+2.
- Su tasa de 2,013% es unas 8,6 veces la de quienes reciben y emiten.
- Solo recibir queda en una situación intermedia.
- Los rangos mensuales conservan el mismo orden.
Esto refuerza la historia de desvinculación, aunque puede solaparse con ctrx_quarter. El modelo determinará cuánto aporta por separado.

In [117]:
%%sql

WITH base AS (

    SELECT
        foto_mes,
        mpasivos_margen,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
      AND mpasivos_margen IS NOT NULL
),

quintiles AS (

    SELECT
        *,

        NTILE(5) OVER (
            PARTITION BY foto_mes
            ORDER BY mpasivos_margen
        ) AS quintil_margen

    FROM base
),

mensual AS (

    SELECT
        foto_mes,
        quintil_margen,
        100.0 * SUM(es_baja2) / COUNT(*) AS tasa_baja2

    FROM quintiles

    GROUP BY
        foto_mes,
        quintil_margen
),

global AS (

    SELECT
        quintil_margen,
        COUNT(*) AS casos_cliente_mes,
        MEDIAN(mpasivos_margen) AS mediana_margen,
        SUM(es_baja2) AS casos_baja2,
        100.0 * SUM(es_baja2) / COUNT(*) AS tasa_baja2

    FROM quintiles

    GROUP BY quintil_margen
)

SELECT
    g.quintil_margen,
    g.casos_cliente_mes,

    ROUND(g.mediana_margen, 2)
        AS mediana_mpasivos_margen,

    g.casos_baja2,

    ROUND(g.tasa_baja2, 3)
        AS tasa_baja2_global_pct,

    ROUND(MIN(m.tasa_baja2), 3)
        AS menor_tasa_mensual_pct,

    ROUND(MAX(m.tasa_baja2), 3)
        AS mayor_tasa_mensual_pct

FROM global AS g

JOIN mensual AS m
    USING (quintil_margen)

GROUP BY
    g.quintil_margen,
    g.casos_cliente_mes,
    g.mediana_margen,
    g.casos_baja2,
    g.tasa_baja2

ORDER BY g.quintil_margen;

,quintil_margen,casos_cliente_mes,mediana_mpasivos_margen,casos_baja2,tasa_baja2_global_pct,menor_tasa_mensual_pct,mayor_tasa_mensual_pct
0,1,130814,45.25,2757.0,2.108,1.704,2.428
1,2,130814,319.51,609.0,0.466,0.388,0.515
2,3,130814,677.66,324.0,0.248,0.218,0.266
3,4,130813,1320.77,212.0,0.162,0.141,0.192
4,5,130811,3668.44,165.0,0.126,0.107,0.138


La relación es nuevamente fuerte y estable:
- El 20% con menor mpasivos_margen concentra 2.757 de 4.067 BAJA+2: 67,8%.
- Su tasa es 2,108%, frente a 0,126% en el quintil superior: aproximadamente 17 veces mayor.
- La tasa cae ordenadamente en todos los quintiles.
- Los rangos mensuales no se superponen de forma relevante.
Esta familia queda confirmada. Puede estar relacionada con saldos y actividad, por lo que será el modelo quien determine su aporte adicional.v

In [118]:
%%sql

WITH base AS (

    SELECT
        foto_mes,
        cproductos,

        CASE
            WHEN clase_ternaria = 'BAJA+2'
            THEN 1
            ELSE 0
        END AS es_baja2,

        CASE
            WHEN cproductos <= 4 THEN 1
            WHEN cproductos <= 6 THEN 2
            WHEN cproductos <= 8 THEN 3
            WHEN cproductos <= 10 THEN 4
            ELSE 5
        END AS orden_banda,

        CASE
            WHEN cproductos <= 4 THEN 'HASTA 4'
            WHEN cproductos <= 6 THEN '5 A 6'
            WHEN cproductos <= 8 THEN '7 A 8'
            WHEN cproductos <= 10 THEN '9 A 10'
            ELSE '11 O MÁS'
        END AS banda_productos

    FROM competencia_01_analitica

    WHERE foto_mes BETWEEN 202103 AND 202106
      AND clase_ternaria IS NOT NULL
      AND cproductos IS NOT NULL
),

mensual AS (

    SELECT
        foto_mes,
        orden_banda,
        banda_productos,
        100.0 * SUM(es_baja2) / COUNT(*) AS tasa_baja2

    FROM base

    GROUP BY
        foto_mes,
        orden_banda,
        banda_productos
),

global AS (

    SELECT
        orden_banda,
        banda_productos,
        COUNT(*) AS casos_cliente_mes,
        SUM(es_baja2) AS casos_baja2,
        100.0 * SUM(es_baja2) / COUNT(*) AS tasa_baja2

    FROM base

    GROUP BY
        orden_banda,
        banda_productos
)

SELECT
    g.banda_productos,
    g.casos_cliente_mes,

    ROUND(
        100.0 * g.casos_cliente_mes
        / SUM(g.casos_cliente_mes) OVER (),
        2
    ) AS pct_de_la_base,

    g.casos_baja2,

    ROUND(g.tasa_baja2, 3)
        AS tasa_baja2_global_pct,

    ROUND(MIN(m.tasa_baja2), 3)
        AS menor_tasa_mensual_pct,

    ROUND(MAX(m.tasa_baja2), 3)
        AS mayor_tasa_mensual_pct

FROM global AS g

JOIN mensual AS m
    USING (
        orden_banda,
        banda_productos
    )

GROUP BY
    g.orden_banda,
    g.banda_productos,
    g.casos_cliente_mes,
    g.casos_baja2,
    g.tasa_baja2

ORDER BY g.orden_banda;

,banda_productos,casos_cliente_mes,pct_de_la_base,casos_baja2,tasa_baja2_global_pct,menor_tasa_mensual_pct,mayor_tasa_mensual_pct
0,HASTA 4,9057,1.38,431.0,4.759,2.822,7.060
1,5 A 6,147708,22.58,1747.0,1.183,0.983,1.342
2,7 A 8,350507,53.59,1624.0,0.463,0.427,0.504
3,9 A 10,121796,18.62,243.0,0.200,0.171,0.235
4,11 O MÁS,24998,3.82,22.0,0.088,0.048,0.143


Hasta 4 productos: tasa BAJA+2 de 4,759%.
5–6 productos: 1,183%.
7–8 productos: 0,463%.
9–10 productos: 0,200%.
11 o más: 0,088%.

BAJA+2 está asociada de manera estable con menor vinculación:
- Menos actividad.
- Menores saldos y márgenes.
- Sin Visa o con Visa sin uso.
- Sin payroll.
- Sin transferencias.
- Menos productos.

# Modelo Base 

In [119]:

# Abril-mayo se usarán para entrenar.
# Junio se usará como validación temporal.
base_modelo = %sql SELECT * FROM competencia_01_analitica WHERE foto_mes BETWEEN 202104 AND 202106 AND clase_ternaria IS NOT NULL ORDER BY foto_mes, numero_de_cliente


# Target que realmente queremos predecir.
base_modelo["target_baja2"] = (
    base_modelo["clase_ternaria"] == "BAJA+2"
).astype("int8")


# No entran al modelo.
columnas_excluidas_base = {
    "numero_de_cliente",
    "foto_mes",
    "clase_ternaria",
    "target_baja2",

    # Primero las transformaremos en variables interpretables.
    "Master_Fvencimiento",
    "Master_Finiciomora",
    "Master_fultimo_cierre",
    "Master_fechaalta",
    "Visa_Fvencimiento",
    "Visa_Finiciomora",
    "Visa_fultimo_cierre",
    "Visa_fechaalta",

    # Indican una corrección técnica, no comportamiento bancario.
    "Master_limite_financiacion_fue_corregido",
    "Visa_limite_financiacion_fue_corregido",
}


features_base = [
    columna
    for columna in base_modelo.columns
    if columna not in columnas_excluidas_base
]


# Auditoría de los meses y del target.
resumen_base = (
    base_modelo
    .groupby("foto_mes")
    .agg(
        clientes=("numero_de_cliente", "size"),
        casos_baja2=("target_baja2", "sum")
    )
)

resumen_base["tasa_baja2_pct"] = (
    100 * resumen_base["casos_baja2"]
    / resumen_base["clientes"]
).round(3)


print("Cantidad de features del modelo base:", len(features_base))
print(
    "Columnas no numéricas:",
    base_modelo[features_base]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

display(resumen_base)

Cantidad de features del modelo base: 143
Columnas no numéricas: []


,clientes,casos_baja2,tasa_baja2_pct
foto_mes,,,
202104,163284,1139,0.698
202105,163768,870,0.531
202106,164114,1098,0.669


In [124]:
# LightGBM admite nulos, pero no infinitos.
base_modelo[features_base] = (
    base_modelo[features_base]
    .replace([np.inf, -np.inf], np.nan)
)


folds_temporales = [
    {
        "entrenamiento": [202103],
        "validacion": 202105
    },
    {
        "entrenamiento": [202103, 202104],
        "validacion": 202106
    }
]


def objective_base(trial):

    params = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "max_bin": 31,

        "num_leaves": trial.suggest_int(
            "num_leaves", 8, 100
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.005, 0.30,
            log=True
        ),

        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf", 20, 1000,
            log=True
        ),

        "feature_fraction": trial.suggest_float(
            "feature_fraction", 0.20, 1.00
        ),

        "bagging_fraction": trial.suggest_float(
            "bagging_fraction", 0.50, 1.00
        ),

        # Sin esto, bagging_fraction no tendría efecto.
        "bagging_freq": 1,

        "seed": semillas[0],
        "feature_fraction_seed": semillas[0],
        "bagging_seed": semillas[0],
    }

    ganancias_folds = []
    auc_folds = []
    iteraciones_folds = []

    for fold in folds_temporales:

        mascara_train = base_modelo["foto_mes"].isin(
            fold["entrenamiento"]
        )

        mascara_valid = (
            base_modelo["foto_mes"]
            == fold["validacion"]
        )

        X_train = base_modelo.loc[
            mascara_train, features_base
        ]

        y_train = base_modelo.loc[
            mascara_train, "target_baja2"
        ]

        X_valid = base_modelo.loc[
            mascara_valid, features_base
        ]

        y_valid = base_modelo.loc[
            mascara_valid, "target_baja2"
        ]

        clases_valid = base_modelo.loc[
            mascara_valid, "clase_ternaria"
        ]

        ganancia_fila_valid = ganancia_por_fila(
            clases_valid
        )

        dataset_train = lgb.Dataset(
            X_train,
            label=y_train
        )

        dataset_valid = lgb.Dataset(
            X_valid,
            label=y_valid,
            reference=dataset_train
        )

        modelo = lgb.train(
            params=params,
            train_set=dataset_train,
            num_boost_round=1500,
            valid_sets=[dataset_valid],
            valid_names=["validacion"],
            feval=feval_ganancia(
                ganancia_fila_valid
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=100,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        probabilidades = modelo.predict(
            X_valid,
            num_iteration=modelo.best_iteration
        )

        ganancia_fold = ganancia(
            probabilidades,
            ganancia_fila_valid
        )

        auc_fold = roc_auc_score(
            y_valid,
            probabilidades
        )

        ganancias_folds.append(float(ganancia_fold))
        auc_folds.append(float(auc_fold))
        iteraciones_folds.append(
            int(modelo.best_iteration)
        )

        mes = fold["validacion"]

        trial.set_user_attr(
            f"ganancia_{mes}",
            float(ganancia_fold)
        )

        trial.set_user_attr(
            f"auc_{mes}",
            float(auc_fold)
        )

        trial.set_user_attr(
            f"iteraciones_{mes}",
            int(modelo.best_iteration)
        )

    trial.set_user_attr(
        "peor_ganancia",
        min(ganancias_folds)
    )

    return float(np.mean(ganancias_folds))

In [126]:
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

sampler = optuna.samplers.TPESampler(
    seed=semillas[0]
)

study_base = optuna.create_study(
    direction="maximize",
    study_name="baseline_temporal_v1",
    storage="sqlite:///optuna_competencia.db",
    load_if_exists=True,
    sampler=sampler
)

# Por ahora, un solo trial para comprobar el circuito.
""" study_base.optimize(
    objective_base,
    n_trials=1
)

mejor_trial = study_base.best_trial

print(
    "Ganancia temporal promedio:",
    f"{mejor_trial.value:,.0f}"
)

print("Hiperparámetros:")
display(mejor_trial.params)

resultados_temporales = pd.DataFrame([
    {
        "validacion": 202105,
        "ganancia": mejor_trial.user_attrs["ganancia_202105"],
        "auc": mejor_trial.user_attrs["auc_202105"],
        "iteraciones": mejor_trial.user_attrs["iteraciones_202105"]
    },
    {
        "validacion": 202106,
        "ganancia": mejor_trial.user_attrs["ganancia_202106"],
        "auc": mejor_trial.user_attrs["auc_202106"],
        "iteraciones": mejor_trial.user_attrs["iteraciones_202106"]
    }
])

display(resultados_temporales)"""

' study_base.optimize(\n    objective_base,\n    n_trials=1\n)\n\nmejor_trial = study_base.best_trial\n\nprint(\n    "Ganancia temporal promedio:",\n    f"{mejor_trial.value:,.0f}"\n)\n\nprint("Hiperparámetros:")\ndisplay(mejor_trial.params)\n\nresultados_temporales = pd.DataFrame([\n    {\n        "validacion": 202105,\n        "ganancia": mejor_trial.user_attrs["ganancia_202105"],\n        "auc": mejor_trial.user_attrs["auc_202105"],\n        "iteraciones": mejor_trial.user_attrs["iteraciones_202105"]\n    },\n    {\n        "validacion": 202106,\n        "ganancia": mejor_trial.user_attrs["ganancia_202106"],\n        "auc": mejor_trial.user_attrs["auc_202106"],\n        "iteraciones": mejor_trial.user_attrs["iteraciones_202106"]\n    }\n])\n\ndisplay(resultados_temporales)'

In [127]:
# ============================================================
# OPTIMIZACIÓN REAL DEL MODELO BASE
# ============================================================

def objective_base_v2(trial):

    params = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,

        "max_depth": trial.suggest_categorical(
            "max_depth",
            [-1, 4, 6, 8, 10, 12]
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            8,
            256,
            log=True
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.005,
            0.20,
            log=True
        ),

        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf",
            20,
            2000,
            log=True
        ),

        "feature_fraction": trial.suggest_float(
            "feature_fraction",
            0.30,
            1.00
        ),

        "bagging_fraction": trial.suggest_float(
            "bagging_fraction",
            0.50,
            1.00
        ),

        "bagging_freq": 1,

        "lambda_l1": trial.suggest_float(
            "lambda_l1",
            1e-8,
            100.0,
            log=True
        ),

        "lambda_l2": trial.suggest_float(
            "lambda_l2",
            1e-8,
            100.0,
            log=True
        ),

        "min_gain_to_split": trial.suggest_float(
            "min_gain_to_split",
            0.0,
            5.0
        ),

        "max_bin": trial.suggest_categorical(
            "max_bin",
            [31, 63, 127, 255]
        ),

        "seed": semillas[0],
        "feature_fraction_seed": semillas[0],
        "bagging_seed": semillas[0],
    }

    ganancias_folds = []
    auc_folds = []

    for fold in folds_temporales:

        mascara_train = base_modelo["foto_mes"].isin(
            fold["entrenamiento"]
        )

        mascara_valid = (
            base_modelo["foto_mes"]
            == fold["validacion"]
        )

        X_train = base_modelo.loc[
            mascara_train,
            features_base
        ]

        y_train = base_modelo.loc[
            mascara_train,
            "target_baja2"
        ]

        X_valid = base_modelo.loc[
            mascara_valid,
            features_base
        ]

        y_valid = base_modelo.loc[
            mascara_valid,
            "target_baja2"
        ]

        clases_valid = base_modelo.loc[
            mascara_valid,
            "clase_ternaria"
        ]

        ganancia_fila_valid = ganancia_por_fila(
            clases_valid
        )

        dataset_train = lgb.Dataset(
            X_train,
            label=y_train
        )

        dataset_valid = lgb.Dataset(
            X_valid,
            label=y_valid,
            reference=dataset_train
        )

        modelo = lgb.train(
            params=params,
            train_set=dataset_train,
            num_boost_round=3000,
            valid_sets=[dataset_valid],
            valid_names=["validacion"],
            feval=feval_ganancia(
                ganancia_fila_valid
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        probabilidades = modelo.predict(
            X_valid,
            num_iteration=modelo.best_iteration
        )

        ganancia_fold = ganancia(
            probabilidades,
            ganancia_fila_valid
        )

        auc_fold = roc_auc_score(
            y_valid,
            probabilidades
        )

        mes_validacion = fold["validacion"]

        ganancias_folds.append(
            float(ganancia_fold)
        )

        auc_folds.append(
            float(auc_fold)
        )

        trial.set_user_attr(
            f"ganancia_{mes_validacion}",
            float(ganancia_fold)
        )

        trial.set_user_attr(
            f"auc_{mes_validacion}",
            float(auc_fold)
        )

        trial.set_user_attr(
            f"iteraciones_{mes_validacion}",
            int(modelo.best_iteration)
        )

    trial.set_user_attr(
        "peor_ganancia",
        min(ganancias_folds)
    )

    trial.set_user_attr(
        "auc_promedio",
        float(np.mean(auc_folds))
    )

    return float(
        np.mean(ganancias_folds)
    )


# El estudio se guarda automáticamente en el archivo SQLite.
sampler_v2 = optuna.samplers.TPESampler(
    seed=semillas[0]
)

study_base_v2 = optuna.create_study(
    direction="maximize",
    study_name="baseline_temporal_v2",
    storage="sqlite:///optuna_competencia.db",
    load_if_exists=True,
    sampler=sampler_v2
)


# Si se interrumpe o volvemos a ejecutar Run All,
# continúa hasta completar un total de 30 trials.
trials_completos = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in study_base_v2.trials
)

trials_faltantes = max(
    0,
    30 - trials_completos
)

print(
    "Trials completos:",
    trials_completos
)

print(
    "Trials que se ejecutarán:",
    trials_faltantes
)

if trials_faltantes > 0:

    study_base_v2.optimize(
        objective_base_v2,
        n_trials=trials_faltantes,
        show_progress_bar=True
    )


# ============================================================
# RESULTADO DEL MEJOR TRIAL
# ============================================================

mejor_trial_base = study_base_v2.best_trial

print()
print(
    "Mejor trial:",
    mejor_trial_base.number
)

print(
    "Mejor ganancia temporal promedio:",
    f"{mejor_trial_base.value:,.0f}"
)

print(
    "Peor ganancia temporal:",
    f"{mejor_trial_base.user_attrs['peor_ganancia']:,.0f}"
)

print(
    "AUC promedio:",
    round(
        mejor_trial_base.user_attrs["auc_promedio"],
        4
    )
)

print()
print("Mejores hiperparámetros:")

display(
    mejor_trial_base.params
)


resultados_mejor_trial = pd.DataFrame([
    {
        "validacion": 202105,
        "ganancia":
            mejor_trial_base.user_attrs[
                "ganancia_202105"
            ],
        "auc":
            mejor_trial_base.user_attrs[
                "auc_202105"
            ],
        "iteraciones":
            mejor_trial_base.user_attrs[
                "iteraciones_202105"
            ]
    },
    {
        "validacion": 202106,
        "ganancia":
            mejor_trial_base.user_attrs[
                "ganancia_202106"
            ],
        "auc":
            mejor_trial_base.user_attrs[
                "auc_202106"
            ],
        "iteraciones":
            mejor_trial_base.user_attrs[
                "iteraciones_202106"
            ]
    }
])

display(
    resultados_mejor_trial
)

Trials completos: 30
Trials que se ejecutarán: 0

Mejor trial: 16
Mejor ganancia temporal promedio: 311,382,500
Peor ganancia temporal: 263,835,000
AUC promedio: 0.8932

Mejores hiperparámetros:


{'max_depth': 8,
 'num_leaves': 124,
 'learning_rate': 0.005037451408350684,
 'min_data_in_leaf': 898,
 'feature_fraction': 0.6483445217575807,
 'bagging_fraction': 0.8255294262037027,
 'lambda_l1': 0.0005389599836272209,
 'lambda_l2': 0.000645134086407597,
 'min_gain_to_split': 1.0175159546614605,
 'max_bin': 255}

,validacion,ganancia,auc,iteraciones
0,202105,263835000.0,0.887552,419
1,202106,358930000.0,0.898861,1123


In [129]:
# ============================================================
# BASELINE CORREGIDO: RESPETA LOS DOS MESES DEL TARGET BAJA+2
# ============================================================

# Incorporamos marzo porque ahora será el primer mes de entrenamiento.
base_modelo = %sql SELECT * FROM competencia_01_analitica WHERE foto_mes BETWEEN 202103 AND 202106 AND clase_ternaria IS NOT NULL ORDER BY foto_mes, numero_de_cliente


base_modelo["target_baja2"] = (
    base_modelo["clase_ternaria"] == "BAJA+2"
).astype("int8")


features_base = [
    columna
    for columna in base_modelo.columns
    if columna not in columnas_excluidas_base
]


base_modelo[features_base] = (
    base_modelo[features_base]
    .replace([np.inf, -np.inf], np.nan)
)


# Para predecir un mes solamente usamos targets
# que ya habrían completado sus dos meses futuros.
folds_temporales = [
    {
        "entrenamiento": [202103],
        "validacion": 202105
    },
    {
        "entrenamiento": [202103, 202104],
        "validacion": 202106
    }
]


print("División temporal corregida:")

for fold in folds_temporales:

    cantidad_train = base_modelo[
        base_modelo["foto_mes"].isin(
            fold["entrenamiento"]
        )
    ].shape[0]

    cantidad_valid = base_modelo[
        base_modelo["foto_mes"]
        == fold["validacion"]
    ].shape[0]

    print(
        f"Entrena con {fold['entrenamiento']} "
        f"({cantidad_train:,} filas) | "
        f"Valida en {fold['validacion']} "
        f"({cantidad_valid:,} filas)"
    )


# Creamos un estudio nuevo para no mezclarlo con el anterior.
sampler_v3 = optuna.samplers.TPESampler(
    seed=semillas[0]
)

study_base_v3 = optuna.create_study(
    direction="maximize",
    study_name="baseline_temporal_v3_sin_fuga",
    storage="sqlite:///optuna_competencia.db",
    load_if_exists=True,
    sampler=sampler_v3
)


# Aprovechamos los mejores parámetros anteriores
# como primera combinación candidata.
if len(study_base_v3.trials) == 0:

    study_base_v3.enqueue_trial(
        study_base_v2.best_params
    )


trials_completos_v3 = sum(
    trial.state == optuna.trial.TrialState.COMPLETE
    for trial in study_base_v3.trials
)

trials_faltantes_v3 = max(
    0,
    30 - trials_completos_v3
)

print()
print(
    "Trials completos del estudio corregido:",
    trials_completos_v3
)

print(
    "Trials que se ejecutarán:",
    trials_faltantes_v3
)


if trials_faltantes_v3 > 0:

    study_base_v3.optimize(
        objective_base_v2,
        n_trials=trials_faltantes_v3,
        show_progress_bar=True
    )


# ============================================================
# RESULTADO DEL BASELINE CORREGIDO
# ============================================================

mejor_trial_base_v3 = study_base_v3.best_trial

print()
print(
    "Mejor trial:",
    mejor_trial_base_v3.number
)

print(
    "Mejor ganancia temporal promedio:",
    f"{mejor_trial_base_v3.value:,.0f}"
)

print(
    "Peor ganancia temporal:",
    f"{mejor_trial_base_v3.user_attrs['peor_ganancia']:,.0f}"
)

print(
    "AUC promedio:",
    round(
        mejor_trial_base_v3.user_attrs["auc_promedio"],
        4
    )
)

print()
print("Mejores hiperparámetros:")

display(
    mejor_trial_base_v3.params
)


resultados_mejor_trial_v3 = pd.DataFrame([
    {
        "entrenamiento": "202103",
        "validacion": 202105,
        "ganancia":
            mejor_trial_base_v3.user_attrs[
                "ganancia_202105"
            ],
        "auc":
            mejor_trial_base_v3.user_attrs[
                "auc_202105"
            ],
        "iteraciones":
            mejor_trial_base_v3.user_attrs[
                "iteraciones_202105"
            ]
    },
    {
        "entrenamiento": "202103-202104",
        "validacion": 202106,
        "ganancia":
            mejor_trial_base_v3.user_attrs[
                "ganancia_202106"
            ],
        "auc":
            mejor_trial_base_v3.user_attrs[
                "auc_202106"
            ],
        "iteraciones":
            mejor_trial_base_v3.user_attrs[
                "iteraciones_202106"
            ]
    }
])

display(
    resultados_mejor_trial_v3
)

División temporal corregida:
Entrena con [202103] (162,900 filas) | Valida en 202105 (163,768 filas)
Entrena con [202103, 202104] (326,184 filas) | Valida en 202106 (164,114 filas)

Trials completos del estudio corregido: 0
Trials que se ejecutarán: 30


Best trial: 20. Best value: 3.0998e+08: 100%|██████████| 30/30 [11:55<00:00, 23.86s/it] 


Mejor trial: 20
Mejor ganancia temporal promedio: 309,980,000
Peor ganancia temporal: 252,945,000
AUC promedio: 0.8933

Mejores hiperparámetros:


{'max_depth': 8,
 'num_leaves': 48,
 'learning_rate': 0.011968361180129191,
 'min_data_in_leaf': 405,
 'feature_fraction': 0.30152141273199784,
 'bagging_fraction': 0.7006936995574339,
 'lambda_l1': 8.320444953703719e-05,
 'lambda_l2': 0.0006696837335733366,
 'min_gain_to_split': 1.5614099380945257,
 'max_bin': 255}

,entrenamiento,validacion,ganancia,auc,iteraciones
0,202103,202105,252945000.0,0.888242,259
1,202103-202104,202106,367015000.0,0.898366,715


In [130]:
# ============================================================
# BASELINE CORREGIDO CON CINCO SEMILLAS
# ============================================================

def indice_mes(foto_mes):
    anio = foto_mes // 100
    mes = foto_mes % 100
    return anio * 12 + mes


# Control automático contra fuga temporal.
for fold in folds_temporales:

    ultimo_mes_train = max(
        fold["entrenamiento"]
    )

    distancia = (
        indice_mes(fold["validacion"])
        - indice_mes(ultimo_mes_train)
    )

    assert distancia >= 2, (
        "Fuga temporal: el target necesita "
        "dos meses para madurar. "
        f"Train={ultimo_mes_train}, "
        f"validación={fold['validacion']}."
    )


mejores_parametros_base_v3 = (
    study_base_v3.best_params.copy()
)

resultados_semillas_base_v3 = []


for semilla_actual in semillas:

    params_semilla = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "bagging_freq": 1,

        **mejores_parametros_base_v3,

        "seed": semilla_actual,
        "feature_fraction_seed": semilla_actual,
        "bagging_seed": semilla_actual,
        "data_random_seed": semilla_actual,
    }

    for fold in folds_temporales:

        mascara_train = base_modelo["foto_mes"].isin(
            fold["entrenamiento"]
        )

        mascara_valid = (
            base_modelo["foto_mes"]
            == fold["validacion"]
        )

        X_train = base_modelo.loc[
            mascara_train,
            features_base
        ]

        y_train = base_modelo.loc[
            mascara_train,
            "target_baja2"
        ]

        X_valid = base_modelo.loc[
            mascara_valid,
            features_base
        ]

        y_valid = base_modelo.loc[
            mascara_valid,
            "target_baja2"
        ]

        clases_valid = base_modelo.loc[
            mascara_valid,
            "clase_ternaria"
        ]

        ganancia_fila_valid = ganancia_por_fila(
            clases_valid
        )

        dataset_train = lgb.Dataset(
            X_train,
            label=y_train
        )

        dataset_valid = lgb.Dataset(
            X_valid,
            label=y_valid,
            reference=dataset_train
        )

        modelo = lgb.train(
            params=params_semilla,
            train_set=dataset_train,
            num_boost_round=3000,
            valid_sets=[dataset_valid],
            valid_names=["validacion"],
            feval=feval_ganancia(
                ganancia_fila_valid
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        probabilidades = modelo.predict(
            X_valid,
            num_iteration=modelo.best_iteration
        )

        ganancia_fold = ganancia(
            probabilidades,
            ganancia_fila_valid
        )

        auc_fold = roc_auc_score(
            y_valid,
            probabilidades
        )

        resultados_semillas_base_v3.append({
            "semilla": semilla_actual,
            "entrenamiento":
                "-".join(
                    str(mes)
                    for mes in fold["entrenamiento"]
                ),
            "validacion": fold["validacion"],
            "ganancia": float(ganancia_fold),
            "auc": float(auc_fold),
            "iteraciones":
                int(modelo.best_iteration)
        })

        print(
            f"Semilla {semilla_actual} | "
            f"Validación {fold['validacion']} | "
            f"Ganancia {ganancia_fold:,.0f} | "
            f"AUC {auc_fold:.4f}"
        )


resultados_semillas_base_v3 = pd.DataFrame(
    resultados_semillas_base_v3
)


resultado_base_v3_por_semilla = (
    resultados_semillas_base_v3
    .groupby("semilla")
    .agg(
        ganancia_promedio=("ganancia", "mean"),
        ganancia_minima=("ganancia", "min"),
        auc_promedio=("auc", "mean")
    )
    .round(4)
    .reset_index()
)


resumen_base_v3_por_mes = (
    resultados_semillas_base_v3
    .groupby("validacion")
    .agg(
        ganancia_media=("ganancia", "mean"),
        ganancia_desvio=("ganancia", "std"),
        ganancia_minima=("ganancia", "min"),
        ganancia_maxima=("ganancia", "max"),
        auc_medio=("auc", "mean"),
        iteraciones_mediana=("iteraciones", "median")
    )
    .round(4)
)


ganancia_base_v3_promedio = (
    resultados_semillas_base_v3[
        "ganancia"
    ].mean()
)

ganancia_base_v3_minima = (
    resultados_semillas_base_v3[
        "ganancia"
    ].min()
)


print()
print(
    "Ganancia promedio general:",
    f"{ganancia_base_v3_promedio:,.0f}"
)

print(
    "Peor ganancia observada:",
    f"{ganancia_base_v3_minima:,.0f}"
)

print()
print("Resultado por semilla:")
display(resultado_base_v3_por_semilla)

print("Resumen por mes:")
display(resumen_base_v3_por_mes)


resultados_semillas_base_v3.to_csv(
    "resultados_baseline_v3_semillas.csv",
    index=False
)

Semilla 100129 | Validación 202105 | Ganancia 252,945,000 | AUC 0.8882
Semilla 100129 | Validación 202106 | Ganancia 356,757,500 | AUC 0.8992
Semilla 100183 | Validación 202105 | Ganancia 250,222,500 | AUC 0.8870
Semilla 100183 | Validación 202106 | Ganancia 355,300,000 | AUC 0.8975
Semilla 100151 | Validación 202105 | Ganancia 249,205,000 | AUC 0.8897
Semilla 100151 | Validación 202106 | Ganancia 346,500,000 | AUC 0.8987
Semilla 500041 | Validación 202105 | Ganancia 248,957,500 | AUC 0.8890
Semilla 500041 | Validación 202106 | Ganancia 350,790,000 | AUC 0.8990
Semilla 900007 | Validación 202105 | Ganancia 254,017,500 | AUC 0.8886
Semilla 900007 | Validación 202106 | Ganancia 351,725,000 | AUC 0.8985

Ganancia promedio general: 301,642,000
Peor ganancia observada: 248,957,500

Resultado por semilla:


,semilla,ganancia_promedio,ganancia_minima,auc_promedio
0,100129,304851250.0,252945000.0,0.8937
1,100151,297852500.0,249205000.0,0.8942
2,100183,302761250.0,250222500.0,0.8922
3,500041,299873750.0,248957500.0,0.8940
4,900007,302871250.0,254017500.0,0.8935


Resumen por mes:


,ganancia_media,ganancia_desvio,ganancia_minima,ganancia_maxima,auc_medio,iteraciones_mediana
validacion,,,,,,
202105,251069500.0,2.283775e+06,248957500.0,254017500.0,0.8885,259.0
202106,352214500.0,4.033717e+06,346500000.0,356757500.0,0.8986,546.0


Empezamos con un bloque pequeño de vinculación y anclaje bancario. Todavía no entrenamos: primero creamos y auditamos las variables.

In [131]:
%%sql

CREATE OR REPLACE VIEW competencia_01_fe_anclaje AS

WITH calculadas AS (

    SELECT
        a.*,

        CASE
            WHEN COALESCE(cpayroll_trx, 0) > 0
              OR COALESCE(mpayroll, 0) > 0
            THEN 1
            ELSE 0
        END AS tiene_payroll,

        COALESCE(ccuenta_debitos_automaticos, 0)
        + COALESCE(ctarjeta_visa_debitos_automaticos, 0)
        + COALESCE(ctarjeta_master_debitos_automaticos, 0)
        + COALESCE(cpagomiscuentas, 0)
        + COALESCE(cpagodeservicios, 0)
            AS coperaciones_recurrentes_total,

        CASE
            WHEN COALESCE(ctarjeta_visa, 0) > 0
            THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN COALESCE(ctarjeta_master, 0) > 0
            THEN 1
            ELSE 0
        END AS cantidad_marcas_tarjeta,

        CASE
            WHEN COALESCE(ctarjeta_visa_transacciones, 0) > 0
            THEN 1
            ELSE 0
        END
        +
        CASE
            WHEN COALESCE(ctarjeta_master_transacciones, 0) > 0
            THEN 1
            ELSE 0
        END AS cantidad_marcas_tarjeta_con_uso,

        CASE
            WHEN COALESCE(ctarjeta_visa, 0) > 0
             AND COALESCE(ctarjeta_visa_transacciones, 0) = 0
            THEN 1
            ELSE 0
        END AS visa_sin_uso,

        CASE
            WHEN COALESCE(ctarjeta_master, 0) > 0
             AND COALESCE(ctarjeta_master_transacciones, 0) = 0
            THEN 1
            ELSE 0
        END AS master_sin_uso,

        CASE
            WHEN COALESCE(chomebanking_transacciones, 0) > 0
            THEN 1 ELSE 0
        END
        +
        CASE
            WHEN COALESCE(cmobile_app_trx, 0) > 0
            THEN 1 ELSE 0
        END
        +
        CASE
            WHEN COALESCE(catm_trx, 0)
               + COALESCE(catm_trx_other, 0) > 0
            THEN 1 ELSE 0
        END
        +
        CASE
            WHEN COALESCE(ccajas_transacciones, 0) > 0
            THEN 1 ELSE 0
        END
        +
        CASE
            WHEN COALESCE(ccallcenter_transacciones, 0) > 0
            THEN 1 ELSE 0
        END AS cantidad_canales_usados,

        CASE
            WHEN COALESCE(chomebanking_transacciones, 0) > 0
              OR COALESCE(cmobile_app_trx, 0) > 0
            THEN 1
            ELSE 0
        END AS usa_canal_digital

    FROM competencia_01_analitica AS a
)

SELECT
    *,

    CASE
        WHEN coperaciones_recurrentes_total > 0
        THEN 1
        ELSE 0
    END AS tiene_operaciones_recurrentes,

    CASE
        WHEN cantidad_marcas_tarjeta
           > cantidad_marcas_tarjeta_con_uso
        THEN 1
        ELSE 0
    END AS tiene_tarjeta_sin_uso,

    CASE
        WHEN tiene_payroll = 1
         AND coperaciones_recurrentes_total > 0
         AND cantidad_marcas_tarjeta > 0
        THEN 1
        ELSE 0
    END AS cliente_anclado

FROM calculadas;

,Success


In [132]:
%%sql

SELECT
    foto_mes,

    COUNT(*) AS clientes,

    ROUND(
        100.0 * AVG(tiene_payroll),
        2
    ) AS pct_con_payroll,

    ROUND(
        MEDIAN(coperaciones_recurrentes_total),
        1
    ) AS mediana_operaciones_recurrentes,

    ROUND(
        100.0 * AVG(tiene_operaciones_recurrentes),
        2
    ) AS pct_con_operaciones_recurrentes,

    ROUND(
        MEDIAN(cantidad_marcas_tarjeta),
        1
    ) AS mediana_marcas_tarjeta,

    ROUND(
        100.0 * AVG(tiene_tarjeta_sin_uso),
        2
    ) AS pct_con_alguna_tarjeta_sin_uso,

    ROUND(
        MEDIAN(cantidad_canales_usados),
        1
    ) AS mediana_canales_usados,

    ROUND(
        100.0 * AVG(usa_canal_digital),
        2
    ) AS pct_usa_canal_digital,

    ROUND(
        100.0 * AVG(cliente_anclado),
        2
    ) AS pct_cliente_anclado

FROM competencia_01_fe_anclaje

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,pct_con_payroll,mediana_operaciones_recurrentes,pct_con_operaciones_recurrentes,mediana_marcas_tarjeta,pct_con_alguna_tarjeta_sin_uso,mediana_canales_usados,pct_usa_canal_digital,pct_cliente_anclado
0,202103,162900,54.05,4.0,84.29,2.0,55.78,2.0,93.97,46.57
1,202104,163284,54.10,5.0,85.80,2.0,54.11,2.0,93.93,47.22
2,202105,163768,54.96,5.0,85.60,2.0,54.01,2.0,94.13,47.88
3,202106,164114,55.27,5.0,85.62,2.0,52.46,2.0,94.17,48.16
4,202107,164348,55.17,5.0,86.10,2.0,53.01,2.0,94.25,48.31
5,202108,164647,55.75,5.0,86.26,2.0,52.23,2.0,94.44,48.74


In [133]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    COUNT(*) FILTER (
        WHERE cpayroll_trx IS NULL
           OR mpayroll IS NULL
    ) AS filas_con_nulos_payroll,

    COUNT(*) FILTER (
        WHERE ccuenta_debitos_automaticos IS NULL
           OR ctarjeta_visa_debitos_automaticos IS NULL
           OR ctarjeta_master_debitos_automaticos IS NULL
           OR cpagomiscuentas IS NULL
           OR cpagodeservicios IS NULL
    ) AS filas_con_nulos_operaciones_recurrentes,

    COUNT(*) FILTER (
        WHERE ctarjeta_visa IS NULL
           OR ctarjeta_master IS NULL
           OR ctarjeta_visa_transacciones IS NULL
           OR ctarjeta_master_transacciones IS NULL
    ) AS filas_con_nulos_tarjetas,

    COUNT(*) FILTER (
        WHERE chomebanking_transacciones IS NULL
           OR cmobile_app_trx IS NULL
           OR catm_trx IS NULL
           OR catm_trx_other IS NULL
           OR ccajas_transacciones IS NULL
           OR ccallcenter_transacciones IS NULL
    ) AS filas_con_nulos_canales,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN COALESCE(thomebanking, 0) > 0
                  OR COALESCE(tmobile_app, 0) > 0
                THEN 1 ELSE 0
            END
        ),
        2
    ) AS pct_digital_segun_flags,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN COALESCE(chomebanking_transacciones, 0) > 0
                  OR COALESCE(cmobile_app_trx, 0) > 0
                THEN 1 ELSE 0
            END
        ),
        2
    ) AS pct_digital_segun_transacciones,

    ROUND(
        100.0 * AVG(
            CASE
                WHEN
                    (
                        COALESCE(thomebanking, 0) > 0
                        OR COALESCE(tmobile_app, 0) > 0
                    )
                    <>
                    (
                        COALESCE(chomebanking_transacciones, 0) > 0
                        OR COALESCE(cmobile_app_trx, 0) > 0
                    )
                THEN 1 ELSE 0
            END
        ),
        2
    ) AS pct_discrepancia_digital

FROM competencia_01_analitica

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,clientes,filas_con_nulos_payroll,filas_con_nulos_operaciones_recurrentes,filas_con_nulos_tarjetas,filas_con_nulos_canales,pct_digital_segun_flags,pct_digital_segun_transacciones,pct_discrepancia_digital
0,202103,162900,0,0,0,0,75.29,93.97,19.28
1,202104,163284,0,0,0,0,74.06,93.93,20.44
2,202105,163768,0,0,0,0,74.05,94.13,20.70
3,202106,164114,0,0,0,0,73.03,94.17,21.76
4,202107,164348,0,0,0,0,73.32,94.25,21.41
5,202108,164647,0,0,0,0,74.06,94.44,20.94


In [134]:
%%sql

SELECT
    foto_mes,

    COUNT(*) FILTER (
        WHERE tiene_payroll IS NULL
           OR coperaciones_recurrentes_total IS NULL
           OR cantidad_marcas_tarjeta IS NULL
           OR cantidad_marcas_tarjeta_con_uso IS NULL
           OR visa_sin_uso IS NULL
           OR master_sin_uso IS NULL
           OR cantidad_canales_usados IS NULL
           OR usa_canal_digital IS NULL
           OR tiene_operaciones_recurrentes IS NULL
           OR tiene_tarjeta_sin_uso IS NULL
           OR cliente_anclado IS NULL
    ) AS filas_con_nulos_nuevos,

    COUNT(*) FILTER (
        WHERE cantidad_marcas_tarjeta NOT BETWEEN 0 AND 2
           OR cantidad_marcas_tarjeta_con_uso NOT BETWEEN 0 AND 2
           OR cantidad_canales_usados NOT BETWEEN 0 AND 5
           OR coperaciones_recurrentes_total < 0
    ) AS filas_con_valores_imposibles,

    MIN(coperaciones_recurrentes_total)
        AS minimo_operaciones_recurrentes,

    MEDIAN(coperaciones_recurrentes_total)
        AS mediana_operaciones_recurrentes,

    ROUND(
        QUANTILE_CONT(
            coperaciones_recurrentes_total,
            0.99
        ),
        1
    ) AS percentil_99_operaciones_recurrentes,

    MAX(coperaciones_recurrentes_total)
        AS maximo_operaciones_recurrentes,

    MIN(cantidad_canales_usados)
        AS minimo_canales,

    MAX(cantidad_canales_usados)
        AS maximo_canales

FROM competencia_01_fe_anclaje

GROUP BY foto_mes

ORDER BY foto_mes;

,foto_mes,filas_con_nulos_nuevos,filas_con_valores_imposibles,minimo_operaciones_recurrentes,mediana_operaciones_recurrentes,percentil_99_operaciones_recurrentes,maximo_operaciones_recurrentes,minimo_canales,maximo_canales
0,202103,0,0,0,4.0,25.0,678,0,5
1,202104,0,0,0,5.0,28.0,680,0,5
2,202105,0,0,0,5.0,27.0,759,0,5
3,202106,0,0,0,5.0,28.0,954,0,5
4,202107,0,0,0,5.0,28.0,1037,0,5
5,202108,0,0,0,5.0,29.0,1310,0,5


La auditoría general está bien:
- No hay nulos en las variables originales utilizadas.
- No hay nulos ni valores imposibles en las features nuevas.
- El percentil 99 de operaciones recurrentes es estable: 25–29.
- Los máximos son casos extremos aislados; por ahora no los corregimos.
Lo que falta resolver es la definición digital. Hay una discrepancia estable de 19–22% entre los flags y las transacciones. Veamos qué dirección tiene y qué canal la genera:

In [135]:
%%sql

WITH comparacion AS (

    SELECT
        foto_mes,
        'HOMEBANKING' AS canal,

        CASE
            WHEN thomebanking > 0
            THEN 1 ELSE 0
        END AS tiene_flag,

        CASE
            WHEN chomebanking_transacciones > 0
            THEN 1 ELSE 0
        END AS tiene_transacciones

    FROM competencia_01_analitica

    UNION ALL

    SELECT
        foto_mes,
        'MOBILE_APP' AS canal,

        CASE
            WHEN tmobile_app > 0
            THEN 1 ELSE 0
        END AS tiene_flag,

        CASE
            WHEN cmobile_app_trx > 0
            THEN 1 ELSE 0
        END AS tiene_transacciones

    FROM competencia_01_analitica
),

clasificacion AS (

    SELECT
        foto_mes,
        canal,

        CASE
            WHEN tiene_flag = 1
             AND tiene_transacciones = 1
                THEN 'FLAG_Y_TRANSACCIONES'

            WHEN tiene_flag = 1
             AND tiene_transacciones = 0
                THEN 'SOLO_FLAG'

            WHEN tiene_flag = 0
             AND tiene_transacciones = 1
                THEN 'SOLO_TRANSACCIONES'

            ELSE 'NINGUNO'
        END AS estado

    FROM comparacion
)

SELECT
    foto_mes,
    canal,
    estado,

    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(*)
        / SUM(COUNT(*)) OVER (
            PARTITION BY foto_mes, canal
        ),
        2
    ) AS porcentaje

FROM clasificacion

GROUP BY
    foto_mes,
    canal,
    estado

ORDER BY
    foto_mes,
    canal,
    estado;

,foto_mes,canal,estado,clientes,porcentaje
0,202103,HOMEBANKING,FLAG_Y_TRANSACCIONES,119850,73.57
1,202103,HOMEBANKING,NINGUNO,23342,14.33
2,202103,HOMEBANKING,SOLO_FLAG,2254,1.38
3,202103,HOMEBANKING,SOLO_TRANSACCIONES,17454,10.71
4,202103,MOBILE_APP,FLAG_Y_TRANSACCIONES,4126,2.53
5,202103,MOBILE_APP,NINGUNO,45704,28.06
6,202103,MOBILE_APP,SOLO_FLAG,1225,0.75
7,202103,MOBILE_APP,SOLO_TRANSACCIONES,111845,68.66
8,202104,HOMEBANKING,FLAG_Y_TRANSACCIONES,118151,72.36
9,202104,HOMEBANKING,NINGUNO,24054,14.73


La diferencia viene principalmente de tmobile_app:
- Solo 2–3% tiene simultáneamente flag y transacciones.
- Entre 69% y 77% tiene transacciones aunque el flag sea cero.
- El patrón es gradual y estable, no parece una rotura puntual.
Conclusión: no podemos interpretar tmobile_app como “tiene la aplicación”. Conservamos el campo original, pero para medir uso digital utilizamos las transacciones.

In [136]:
%%sql

CREATE OR REPLACE VIEW competencia_01_fe_anclaje_final AS

SELECT
    * EXCLUDE (usa_canal_digital),

    CASE
        WHEN chomebanking_transacciones > 0
        THEN 1
        ELSE 0
    END AS usa_homebanking_transaccional,

    CASE
        WHEN cmobile_app_trx > 0
        THEN 1
        ELSE 0
    END AS usa_mobile_app_transaccional,

    usa_canal_digital
        AS actividad_digital_transaccional

FROM competencia_01_fe_anclaje;

,Success


In [137]:
# ============================================================
# PREPARACIÓN DEL BLOQUE DE ANCLAJE
# ============================================================

base_anclaje = %sql SELECT * FROM competencia_01_fe_anclaje_final WHERE foto_mes BETWEEN 202103 AND 202106 AND clase_ternaria IS NOT NULL ORDER BY foto_mes, numero_de_cliente


base_anclaje["target_baja2"] = (
    base_anclaje["clase_ternaria"] == "BAJA+2"
).astype("int8")


features_anclaje = [
    columna
    for columna in base_anclaje.columns
    if columna not in columnas_excluidas_base
]


features_nuevas_anclaje = [
    columna
    for columna in features_anclaje
    if columna not in features_base
]


base_anclaje[features_anclaje] = (
    base_anclaje[features_anclaje]
    .replace([np.inf, -np.inf], np.nan)
)


# Las filas deben ser exactamente las mismas que en el baseline.
assert len(base_anclaje) == len(base_modelo)

assert base_anclaje[
    ["numero_de_cliente", "foto_mes"]
].equals(
    base_modelo[
        ["numero_de_cliente", "foto_mes"]
    ]
)

assert base_anclaje["target_baja2"].equals(
    base_modelo["target_baja2"]
)


print(
    "Features del baseline:",
    len(features_base)
)

print(
    "Features con bloque de anclaje:",
    len(features_anclaje)
)

print(
    "Features nuevas:",
    len(features_nuevas_anclaje)
)

display(
    pd.DataFrame({
        "feature_nueva": features_nuevas_anclaje
    })
)

Features del baseline: 143
Features con bloque de anclaje: 156
Features nuevas: 13


,feature_nueva
0,tiene_payroll
1,coperaciones_recurrentes_total
2,cantidad_marcas_tarjeta
3,cantidad_marcas_tarjeta_con_uso
4,visa_sin_uso
5,master_sin_uso
6,cantidad_canales_usados
7,tiene_operaciones_recurrentes
8,tiene_tarjeta_sin_uso
9,cliente_anclado


In [143]:
%%sql

CREATE OR REPLACE VIEW competencia_01_fe_tarjetas AS

WITH totales AS (

    SELECT
        a.*,

        CASE
            WHEN ctarjeta_visa_transacciones IS NULL
             AND ctarjeta_master_transacciones IS NULL
            THEN NULL
            ELSE COALESCE(ctarjeta_visa_transacciones, 0)
               + COALESCE(ctarjeta_master_transacciones, 0)
        END AS ctarjetas_credito_transacciones_total,

        CASE
            WHEN mtarjeta_visa_consumo IS NULL
             AND mtarjeta_master_consumo IS NULL
            THEN NULL
            ELSE COALESCE(mtarjeta_visa_consumo, 0)
               + COALESCE(mtarjeta_master_consumo, 0)
        END AS mtarjetas_credito_consumo_total,

        CASE
            WHEN Visa_msaldopesos IS NULL
             AND Master_msaldopesos IS NULL
            THEN NULL
            ELSE COALESCE(Visa_msaldopesos, 0)
               + COALESCE(Master_msaldopesos, 0)
        END AS tarjetas_saldo_pesos_total,

        CASE
            WHEN Visa_msaldodolares IS NULL
             AND Master_msaldodolares IS NULL
            THEN NULL
            ELSE COALESCE(Visa_msaldodolares, 0)
               + COALESCE(Master_msaldodolares, 0)
        END AS tarjetas_saldo_dolares_total,

        CASE
            WHEN Visa_mlimitecompra IS NULL
             AND Master_mlimitecompra IS NULL
            THEN NULL
            ELSE COALESCE(Visa_mlimitecompra, 0)
               + COALESCE(Master_mlimitecompra, 0)
        END AS tarjetas_limite_compra_total,

        CASE
            WHEN Visa_mfinanciacion_limite IS NULL
             AND Master_mfinanciacion_limite IS NULL
            THEN NULL
            ELSE COALESCE(Visa_mfinanciacion_limite, 0)
               + COALESCE(Master_mfinanciacion_limite, 0)
        END AS tarjetas_limite_financiacion_total,

        CASE
            WHEN Visa_mconsumospesos IS NULL
             AND Master_mconsumospesos IS NULL
            THEN NULL
            ELSE COALESCE(Visa_mconsumospesos, 0)
               + COALESCE(Master_mconsumospesos, 0)
        END AS tarjetas_consumo_pesos_total,

        CASE
            WHEN Visa_mconsumosdolares IS NULL
             AND Master_mconsumosdolares IS NULL
            THEN NULL
            ELSE COALESCE(Visa_mconsumosdolares, 0)
               + COALESCE(Master_mconsumosdolares, 0)
        END AS tarjetas_consumo_dolares_total,

        CASE
            WHEN Visa_cconsumos IS NULL
             AND Master_cconsumos IS NULL
            THEN NULL
            ELSE COALESCE(Visa_cconsumos, 0)
               + COALESCE(Master_cconsumos, 0)
        END AS tarjetas_cconsumos_total,

        CASE
            WHEN Visa_mpagominimo IS NULL
             AND Master_mpagominimo IS NULL
            THEN NULL
            ELSE COALESCE(Visa_mpagominimo, 0)
               + COALESCE(Master_mpagominimo, 0)
        END AS tarjetas_pago_minimo_total

    FROM competencia_01_fe_anclaje_final AS a
)

SELECT
    *,

    CASE
        WHEN tarjetas_limite_compra_total > 0
        THEN tarjetas_saldo_pesos_total
             / tarjetas_limite_compra_total
    END AS tarjetas_utilizacion_limite,

    CASE
        WHEN tarjetas_limite_compra_total > 0
        THEN mtarjetas_credito_consumo_total
             / tarjetas_limite_compra_total
    END AS tarjetas_consumo_sobre_limite,

    CASE
        WHEN tarjetas_limite_compra_total > 0
        THEN tarjetas_limite_financiacion_total
             / tarjetas_limite_compra_total
    END AS tarjetas_financiacion_sobre_limite,

    CASE
        WHEN tarjetas_limite_compra_total IS NOT NULL
         AND tarjetas_saldo_pesos_total IS NOT NULL
        THEN tarjetas_limite_compra_total
             - tarjetas_saldo_pesos_total
    END AS tarjetas_holgura_compra,

    CASE
        WHEN ctarjetas_credito_transacciones_total > 0
        THEN mtarjetas_credito_consumo_total
             / ctarjetas_credito_transacciones_total
    END AS tarjetas_ticket_promedio,

    CASE
        WHEN COALESCE(ctarjeta_visa_transacciones, 0) >= 0
         AND COALESCE(ctarjeta_master_transacciones, 0) >= 0
         AND COALESCE(ctarjeta_visa_transacciones, 0)
             + COALESCE(ctarjeta_master_transacciones, 0) > 0
        THEN 1.0 * COALESCE(ctarjeta_visa_transacciones, 0)
             / (
                 COALESCE(ctarjeta_visa_transacciones, 0)
                 + COALESCE(ctarjeta_master_transacciones, 0)
             )
    END AS share_visa_transacciones,

    CASE
        WHEN COALESCE(mtarjeta_visa_consumo, 0) >= 0
         AND COALESCE(mtarjeta_master_consumo, 0) >= 0
         AND COALESCE(mtarjeta_visa_consumo, 0)
             + COALESCE(mtarjeta_master_consumo, 0) > 0
        THEN COALESCE(mtarjeta_visa_consumo, 0)
             / (
                 COALESCE(mtarjeta_visa_consumo, 0)
                 + COALESCE(mtarjeta_master_consumo, 0)
             )
    END AS share_visa_consumo

FROM totales;

,Success


In [144]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE tarjetas_limite_compra_total IS NULL
        ) / COUNT(*),
        2
    ) AS pct_nulos_limite_total,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE tarjetas_utilizacion_limite IS NULL
        ) / COUNT(*),
        2
    ) AS pct_nulos_utilizacion,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE tarjetas_ticket_promedio IS NULL
        ) / COUNT(*),
        2
    ) AS pct_nulos_ticket,

    COUNT(*) FILTER (
        WHERE share_visa_transacciones < 0
           OR share_visa_transacciones > 1
    ) AS shares_transacciones_fuera_de_rango,

    COUNT(*) FILTER (
        WHERE share_visa_consumo < 0
           OR share_visa_consumo > 1
    ) AS shares_consumo_fuera_de_rango,

    ROUND(
        MEDIAN(tarjetas_utilizacion_limite),
        4
    ) AS mediana_utilizacion,

    ROUND(
        QUANTILE_CONT(tarjetas_utilizacion_limite, 0.99),
        4
    ) AS p99_utilizacion,

    ROUND(
        MEDIAN(tarjetas_consumo_sobre_limite),
        4
    ) AS mediana_consumo_sobre_limite,

    ROUND(
        QUANTILE_CONT(tarjetas_consumo_sobre_limite, 0.99),
        4
    ) AS p99_consumo_sobre_limite,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE mtarjetas_credito_consumo_total IS NOT NULL
              AND tarjetas_consumo_pesos_total IS NOT NULL
              AND ABS(
                  mtarjetas_credito_consumo_total
                  - tarjetas_consumo_pesos_total
              ) < 0.01
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE mtarjetas_credito_consumo_total IS NOT NULL
                  AND tarjetas_consumo_pesos_total IS NOT NULL
            ),
            0
        ),
        2
    ) AS pct_consumos_macro_igual_resumen,

    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE ctarjetas_credito_transacciones_total IS NOT NULL
              AND tarjetas_cconsumos_total IS NOT NULL
              AND ctarjetas_credito_transacciones_total
                  = tarjetas_cconsumos_total
        )
        / NULLIF(
            COUNT(*) FILTER (
                WHERE ctarjetas_credito_transacciones_total IS NOT NULL
                  AND tarjetas_cconsumos_total IS NOT NULL
            ),
            0
        ),
        2
    ) AS pct_transacciones_macro_igual_resumen

FROM competencia_01_fe_tarjetas

GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,pct_nulos_limite_total,pct_nulos_utilizacion,pct_nulos_ticket,shares_transacciones_fuera_de_rango,shares_consumo_fuera_de_rango,mediana_utilizacion,p99_utilizacion,mediana_consumo_sobre_limite,p99_consumo_sobre_limite,pct_consumos_macro_igual_resumen,pct_transacciones_macro_igual_resumen
0,202103,162900,4.03,4.03,13.79,0,0,0.0570,0.8915,0.0515,0.5171,10.17,12.05
1,202104,163284,4.07,4.07,13.57,0,0,0.0604,0.8978,0.0552,0.5463,9.23,11.56
2,202105,163768,4.07,4.07,13.54,0,0,0.0621,0.9284,0.0569,0.5579,8.93,11.22
3,202106,164114,4.08,4.08,13.02,0,0,0.0694,0.9526,0.0647,0.6254,8.34,10.70
4,202107,164348,3.62,3.62,13.09,0,0,0.0660,0.9442,0.0606,0.5699,9.00,11.10
5,202108,164647,4.10,4.10,13.06,0,0,0.0703,1.0331,0.0642,0.6159,8.37,10.46


In [145]:
# ============================================================
# PREPARACIÓN Y EVALUACIÓN DEL BLOQUE DE TARJETAS
# ============================================================

base_tarjetas = %sql SELECT * FROM competencia_01_fe_tarjetas WHERE foto_mes BETWEEN 202103 AND 202106 AND clase_ternaria IS NOT NULL ORDER BY foto_mes, numero_de_cliente


base_tarjetas["target_baja2"] = (
    base_tarjetas["clase_ternaria"] == "BAJA+2"
).astype("int8")


features_tarjetas = [
    columna
    for columna in base_tarjetas.columns
    if columna not in columnas_excluidas_base
]


features_nuevas_tarjetas = [
    columna
    for columna in features_tarjetas
    if columna not in features_anclaje
]


base_tarjetas[features_tarjetas] = (
    base_tarjetas[features_tarjetas]
    .replace([np.inf, -np.inf], np.nan)
)


# Las filas y el target deben ser idénticos al experimento anterior.
assert len(base_tarjetas) == len(base_anclaje)

assert base_tarjetas[
    ["numero_de_cliente", "foto_mes"]
].equals(
    base_anclaje[
        ["numero_de_cliente", "foto_mes"]
    ]
)

assert base_tarjetas["target_baja2"].equals(
    base_anclaje["target_baja2"]
)

columnas_no_numericas = (
    base_tarjetas[features_tarjetas]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

assert columnas_no_numericas == [], (
    f"Hay columnas no numéricas: {columnas_no_numericas}"
)


print("Features con anclaje:", len(features_anclaje))
print("Features con anclaje + tarjetas:", len(features_tarjetas))
print("Features nuevas de tarjetas:", len(features_nuevas_tarjetas))

display(
    pd.DataFrame({
        "feature_nueva": features_nuevas_tarjetas
    })
)


# Mismos hiperparámetros, folds y semillas que en el baseline.
resultados_tarjetas = evaluar_features_temporal(
    datos=base_tarjetas,
    features=features_tarjetas,
    nombre_experimento="ANCLAJE+TARJETAS"
)


# Comparación emparejada contra el bloque de anclaje.
comparacion_tarjetas = (
    resultados_anclaje
    .merge(
        resultados_tarjetas,
        on=["semilla", "validacion"],
        suffixes=("_anclaje", "_tarjetas")
    )
)


comparacion_tarjetas["delta_ganancia"] = (
    comparacion_tarjetas["ganancia_tarjetas"]
    - comparacion_tarjetas["ganancia_anclaje"]
)

comparacion_tarjetas["delta_auc"] = (
    comparacion_tarjetas["auc_tarjetas"]
    - comparacion_tarjetas["auc_anclaje"]
)


resumen_tarjetas = (
    comparacion_tarjetas
    .groupby("validacion")
    .agg(
        ganancia_anclaje=("ganancia_anclaje", "mean"),
        ganancia_tarjetas=("ganancia_tarjetas", "mean"),
        mejora_media=("delta_ganancia", "mean"),
        mejora_minima=("delta_ganancia", "min"),
        mejora_maxima=("delta_ganancia", "max"),
        cambio_auc_medio=("delta_auc", "mean"),
        semillas_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum())
        )
    )
    .round(4)
)


ganancia_tarjetas_promedio = (
    resultados_tarjetas["ganancia"].mean()
)

mejora_tarjetas_promedio = (
    comparacion_tarjetas["delta_ganancia"].mean()
)

mejora_tarjetas_pct = (
    100.0
    * mejora_tarjetas_promedio
    / resultados_anclaje["ganancia"].mean()
)

combinaciones_favorables = int(
    (comparacion_tarjetas["delta_ganancia"] > 0).sum()
)


print()
print(
    "Ganancia promedio con anclaje:",
    f"{resultados_anclaje['ganancia'].mean():,.0f}"
)

print(
    "Ganancia promedio con anclaje + tarjetas:",
    f"{ganancia_tarjetas_promedio:,.0f}"
)

print(
    "Mejora incremental:",
    f"{mejora_tarjetas_promedio:,.0f}",
    f"({mejora_tarjetas_pct:.2f}%)"
)

print(
    "Combinaciones favorables:",
    f"{combinaciones_favorables}/10"
)

display(resumen_tarjetas)


resultados_tarjetas.to_csv(
    "resultados_bloque_tarjetas.csv",
    index=False
)

Features con anclaje: 156
Features con anclaje + tarjetas: 173
Features nuevas de tarjetas: 17


,feature_nueva
0,ctarjetas_credito_transacciones_total
1,mtarjetas_credito_consumo_total
2,tarjetas_saldo_pesos_total
3,tarjetas_saldo_dolares_total
4,tarjetas_limite_compra_total
5,tarjetas_limite_financiacion_total
6,tarjetas_consumo_pesos_total
7,tarjetas_consumo_dolares_total
8,tarjetas_cconsumos_total
9,tarjetas_pago_minimo_total


ANCLAJE+TARJETAS | Semilla 100129 | Validación 202105 | Ganancia 247,830,000
ANCLAJE+TARJETAS | Semilla 100129 | Validación 202106 | Ganancia 346,500,000
ANCLAJE+TARJETAS | Semilla 100183 | Validación 202105 | Ganancia 253,137,500
ANCLAJE+TARJETAS | Semilla 100183 | Validación 202106 | Ganancia 344,575,000
ANCLAJE+TARJETAS | Semilla 100151 | Validación 202105 | Ganancia 255,997,500
ANCLAJE+TARJETAS | Semilla 100151 | Validación 202106 | Ganancia 359,287,500
ANCLAJE+TARJETAS | Semilla 500041 | Validación 202105 | Ganancia 248,270,000
ANCLAJE+TARJETAS | Semilla 500041 | Validación 202106 | Ganancia 344,877,500
ANCLAJE+TARJETAS | Semilla 900007 | Validación 202105 | Ganancia 255,062,500
ANCLAJE+TARJETAS | Semilla 900007 | Validación 202106 | Ganancia 356,895,000

Ganancia promedio con anclaje: 305,841,250
Ganancia promedio con anclaje + tarjetas: 301,243,250
Mejora incremental: -4,598,000 (-1.50%)
Combinaciones favorables: 3/10


,ganancia_anclaje,ganancia_tarjetas,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables
validacion,,,,,,,
202105,254732500.0,252059500.0,-2673000.0,-7562500.0,1017500.0,0.0001,1
202106,356950000.0,350427000.0,-6523000.0,-22797500.0,935000.0,-0.0001,2


El bloque completo de tarjetas empeoró de manera bastante consistente:
- −$4,60 M promedio.
- Resultado negativo en ambos meses.
- Solo mejora en 3 de 10 combinaciones.
- El AUC prácticamente no cambia.
Esto significa que las 17 juntas alteran desfavorablemente la parte superior del ranking, aunque la discriminación global sea similar. No las incorporamos todavía.

In [146]:
# ============================================================
# ABLACIÓN DEL BLOQUE DE TARJETAS
# ============================================================

grupos_tarjetas = {

    "TARJETAS_TOTALES": [
        "ctarjetas_credito_transacciones_total",
        "mtarjetas_credito_consumo_total",
        "tarjetas_saldo_pesos_total",
        "tarjetas_saldo_dolares_total",
        "tarjetas_limite_compra_total",
        "tarjetas_limite_financiacion_total",
        "tarjetas_consumo_pesos_total",
        "tarjetas_consumo_dolares_total",
        "tarjetas_cconsumos_total",
        "tarjetas_pago_minimo_total",
    ],

    "TARJETAS_RATIOS": [
        "tarjetas_utilizacion_limite",
        "tarjetas_consumo_sobre_limite",
        "tarjetas_financiacion_sobre_limite",
        "tarjetas_holgura_compra",
        "tarjetas_ticket_promedio",
    ],

    "TARJETAS_SHARES": [
        "share_visa_transacciones",
        "share_visa_consumo",
    ],
}


resultados_ablacion_tarjetas = []

for nombre_grupo, features_grupo in grupos_tarjetas.items():

    features_experimento = (
        features_anclaje
        + features_grupo
    )

    resultado_grupo = evaluar_features_temporal(
        datos=base_tarjetas,
        features=features_experimento,
        nombre_experimento=nombre_grupo
    )

    resultados_ablacion_tarjetas.append(
        resultado_grupo
    )


resultados_ablacion_tarjetas = pd.concat(
    resultados_ablacion_tarjetas,
    ignore_index=True
)


# Referencia: anclaje sin ninguna feature nueva de tarjetas.
referencia_anclaje = (
    resultados_anclaje[
        ["semilla", "validacion", "ganancia", "auc"]
    ]
    .rename(columns={
        "ganancia": "ganancia_anclaje",
        "auc": "auc_anclaje",
    })
)


comparacion_ablacion_tarjetas = (
    resultados_ablacion_tarjetas
    .merge(
        referencia_anclaje,
        on=["semilla", "validacion"],
        how="inner"
    )
)


comparacion_ablacion_tarjetas["delta_ganancia"] = (
    comparacion_ablacion_tarjetas["ganancia"]
    - comparacion_ablacion_tarjetas["ganancia_anclaje"]
)

comparacion_ablacion_tarjetas["delta_auc"] = (
    comparacion_ablacion_tarjetas["auc"]
    - comparacion_ablacion_tarjetas["auc_anclaje"]
)


# Resultado separado por mes.
resumen_ablacion_por_mes = (
    comparacion_ablacion_tarjetas
    .groupby(["experimento", "validacion"])
    .agg(
        ganancia_media=("ganancia", "mean"),
        mejora_media=("delta_ganancia", "mean"),
        mejora_minima=("delta_ganancia", "min"),
        mejora_maxima=("delta_ganancia", "max"),
        cambio_auc_medio=("delta_auc", "mean"),
        semillas_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum())
        )
    )
    .round(4)
)


# Resultado conjunto de los dos meses y cinco semillas.
resumen_ablacion_general = (
    comparacion_ablacion_tarjetas
    .groupby("experimento")
    .agg(
        ganancia_media=("ganancia", "mean"),
        mejora_media=("delta_ganancia", "mean"),
        mejora_minima=("delta_ganancia", "min"),
        mejora_maxima=("delta_ganancia", "max"),
        cambio_auc_medio=("delta_auc", "mean"),
        combinaciones_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum())
        )
    )
    .sort_values("mejora_media", ascending=False)
    .round(4)
)


resumen_ablacion_general["mejora_pct"] = (
    100.0
    * resumen_ablacion_general["mejora_media"]
    / resultados_anclaje["ganancia"].mean()
).round(2)


print("Resultado general:")
display(resumen_ablacion_general)

print("Resultado por mes:")
display(resumen_ablacion_por_mes)


resultados_ablacion_tarjetas.to_csv(
    "resultados_ablacion_tarjetas.csv",
    index=False
)

TARJETAS_TOTALES | Semilla 100129 | Validación 202105 | Ganancia 251,597,500
TARJETAS_TOTALES | Semilla 100129 | Validación 202106 | Ganancia 353,237,500
TARJETAS_TOTALES | Semilla 100183 | Validación 202105 | Ganancia 257,097,500
TARJETAS_TOTALES | Semilla 100183 | Validación 202106 | Ganancia 365,585,000
TARJETAS_TOTALES | Semilla 100151 | Validación 202105 | Ganancia 256,052,500
TARJETAS_TOTALES | Semilla 100151 | Validación 202106 | Ganancia 355,162,500
TARJETAS_TOTALES | Semilla 500041 | Validación 202105 | Ganancia 252,945,000
TARJETAS_TOTALES | Semilla 500041 | Validación 202106 | Ganancia 349,415,000
TARJETAS_TOTALES | Semilla 900007 | Validación 202105 | Ganancia 247,472,500
TARJETAS_TOTALES | Semilla 900007 | Validación 202106 | Ganancia 351,642,500
TARJETAS_RATIOS | Semilla 100129 | Validación 202105 | Ganancia 258,637,500
TARJETAS_RATIOS | Semilla 100129 | Validación 202106 | Ganancia 348,122,500
TARJETAS_RATIOS | Semilla 100183 | Validación 202105 | Ganancia 256,492,500
TA

,ganancia_media,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,combinaciones_favorables,mejora_pct
experimento,,,,,,,
TARJETAS_SHARES,304540500.0,-1300750.0,-15675000.0,14657500.0,-0.0001,4,-0.43
TARJETAS_TOTALES,304020750.0,-1820500.0,-15152500.0,4620000.0,0.0004,4,-0.60
TARJETAS_RATIOS,303971250.0,-1870000.0,-18865000.0,17352500.0,0.0009,5,-0.61


Resultado por mes:


ganancia_media  mejora_media  mejora_minima  \
experimento      validacion                                                
TARJETAS_RATIOS  202105         253632500.0    -1100000.0    -10065000.0   
                 202106         354310000.0    -2640000.0    -18865000.0   
TARJETAS_SHARES  202105         252884500.0    -1848000.0    -11550000.0   
                 202106         356196500.0     -753500.0    -15675000.0   
TARJETAS_TOTALES 202105         253033000.0    -1699500.0    -15152500.0   
                 202106         355008500.0    -1941500.0     -4950000.0   

                             mejora_maxima  cambio_auc_medio  \
experimento      validacion                                    
TARJETAS_RATIOS  202105          4180000.0            0.0015   
                 202106         17352500.0            0.0002   
TARJETAS_SHARES  202105          7672500.0            0.0002   
                 202106         14657500.0           -0.0004   
TARJETAS_TOTALES 202105          4620000.0            0.0007   
                 202106           330000.0            0.0001   

                             semillas_favorables  
experimento      validacion                       
TARJETAS_RATIOS  202105                        3  
                 202106                        2  
TARJETAS_SHARES  202105                        2  
                 202106                        2  
TARJETAS_TOTALES 202105                        3  
                 202106                        1

El resultado es claro: descartamos las 17 features de tarjetas en su forma estática. Ninguno de los tres subgrupos mejora la ganancia en promedio; el modelo vigente sigue siendo anclaje con 156 variables. Conservo la vista solo porque más adelante algunos ratios pueden ser útiles como deltas, pero no entran ahora.

El resultado confirma que no conviene incorporar ninguna de estas features estáticas de tarjetas:
- Los tres subgrupos pierden ganancia promedio.
- Ninguno mejora en ambos meses.
- Los ratios ganan AUC, pero pierden ganancia y tienen mucha variabilidad.
- Probar las 17 individualmente sería selección oportunista y aumentaría el riesgo de sobreajuste.
Decisión: el modelo vigente continúa siendo anclaje, con 156 features y $305,84 M de ganancia media. Conservamos la vista de tarjetas únicamente porque después podremos calcular cambios de utilización, saldo o consumo; sus deltas podrían aportar aunque sus niveles actuales no lo hagan.
El siguiente bloque será fechas derivadas, porque las ocho variables codificadas siguen excluidas del modelo. Crearemos:
- días hasta el vencimiento del plástico;
- plástico vencido;
- código especial 31/12/9999;
- antigüedad de la tarjeta;
- días desde el último cierre;
- antigüedad de la mora;
- indicador de mora con inicio cero;
- consolidados entre Visa y Master.

# Verificación para el modelo 

In [138]:
# ============================================================
# FUNCIÓN REUTILIZABLE PARA EVALUAR GRUPOS DE FEATURES
# ============================================================

def evaluar_features_temporal(
    datos,
    features,
    nombre_experimento
):

    resultados = []

    for fold in folds_temporales:

        ultimo_mes_train = max(
            fold["entrenamiento"]
        )

        distancia = (
            indice_mes(fold["validacion"])
            - indice_mes(ultimo_mes_train)
        )

        assert distancia >= 2, (
            "Fuga temporal detectada: "
            f"train={ultimo_mes_train}, "
            f"validación={fold['validacion']}"
        )

    for semilla_actual in semillas:

        params_semilla = {
            "objective": "binary",
            "metric": "None",
            "boosting_type": "gbdt",
            "boost_from_average": True,
            "feature_pre_filter": False,
            "first_metric_only": True,
            "verbosity": -1,
            "bagging_freq": 1,

            **study_base_v3.best_params,

            "seed": semilla_actual,
            "feature_fraction_seed": semilla_actual,
            "bagging_seed": semilla_actual,
            "data_random_seed": semilla_actual,
        }

        for fold in folds_temporales:

            mascara_train = datos["foto_mes"].isin(
                fold["entrenamiento"]
            )

            mascara_valid = (
                datos["foto_mes"]
                == fold["validacion"]
            )

            X_train = datos.loc[
                mascara_train,
                features
            ]

            y_train = datos.loc[
                mascara_train,
                "target_baja2"
            ]

            X_valid = datos.loc[
                mascara_valid,
                features
            ]

            y_valid = datos.loc[
                mascara_valid,
                "target_baja2"
            ]

            clases_valid = datos.loc[
                mascara_valid,
                "clase_ternaria"
            ]

            ganancia_fila_valid = ganancia_por_fila(
                clases_valid
            )

            dataset_train = lgb.Dataset(
                X_train,
                label=y_train
            )

            dataset_valid = lgb.Dataset(
                X_valid,
                label=y_valid,
                reference=dataset_train
            )

            modelo = lgb.train(
                params=params_semilla,
                train_set=dataset_train,
                num_boost_round=3000,
                valid_sets=[dataset_valid],
                valid_names=["validacion"],
                feval=feval_ganancia(
                    ganancia_fila_valid
                ),
                callbacks=[
                    lgb.early_stopping(
                        stopping_rounds=150,
                        verbose=False
                    ),
                    lgb.log_evaluation(period=0)
                ]
            )

            probabilidades = modelo.predict(
                X_valid,
                num_iteration=modelo.best_iteration
            )

            ganancia_fold = ganancia(
                probabilidades,
                ganancia_fila_valid
            )

            auc_fold = roc_auc_score(
                y_valid,
                probabilidades
            )

            resultados.append({
                "experimento": nombre_experimento,
                "semilla": semilla_actual,
                "validacion": fold["validacion"],
                "ganancia": float(ganancia_fold),
                "auc": float(auc_fold),
                "iteraciones":
                    int(modelo.best_iteration)
            })

            print(
                f"{nombre_experimento} | "
                f"Semilla {semilla_actual} | "
                f"Validación {fold['validacion']} | "
                f"Ganancia {ganancia_fold:,.0f}"
            )

    return pd.DataFrame(resultados)


# Evaluamos las 156 variables.
resultados_anclaje = evaluar_features_temporal(
    datos=base_anclaje,
    features=features_anclaje,
    nombre_experimento="ANCLAJE"
)


# Comparación emparejada:
# misma semilla y mismo mes para ambos modelos.
comparacion_anclaje = (
    resultados_semillas_base_v3
    .merge(
        resultados_anclaje,
        on=["semilla", "validacion"],
        suffixes=("_base", "_anclaje")
    )
)


comparacion_anclaje["delta_ganancia"] = (
    comparacion_anclaje["ganancia_anclaje"]
    - comparacion_anclaje["ganancia_base"]
)

comparacion_anclaje["delta_auc"] = (
    comparacion_anclaje["auc_anclaje"]
    - comparacion_anclaje["auc_base"]
)


resumen_comparacion_anclaje = (
    comparacion_anclaje
    .groupby("validacion")
    .agg(
        ganancia_base=("ganancia_base", "mean"),
        ganancia_anclaje=("ganancia_anclaje", "mean"),
        mejora_media=("delta_ganancia", "mean"),
        mejora_minima=("delta_ganancia", "min"),
        mejora_maxima=("delta_ganancia", "max"),
        cambio_auc_medio=("delta_auc", "mean")
    )
    .round(4)
)


ganancia_anclaje_promedio = (
    resultados_anclaje["ganancia"].mean()
)

mejora_anclaje_promedio = (
    comparacion_anclaje["delta_ganancia"].mean()
)

mejora_anclaje_pct = (
    100.0
    * mejora_anclaje_promedio
    / resultados_semillas_base_v3["ganancia"].mean()
)


print()
print(
    "Ganancia promedio baseline:",
    f"{ganancia_base_v3_promedio:,.0f}"
)

print(
    "Ganancia promedio con anclaje:",
    f"{ganancia_anclaje_promedio:,.0f}"
)

print(
    "Mejora promedio:",
    f"{mejora_anclaje_promedio:,.0f}",
    f"({mejora_anclaje_pct:.2f}%)"
)

display(
    resumen_comparacion_anclaje
)


resultados_anclaje.to_csv(
    "resultados_bloque_anclaje.csv",
    index=False
)

ANCLAJE | Semilla 100129 | Validación 202105 | Ganancia 254,457,500
ANCLAJE | Semilla 100129 | Validación 202106 | Ganancia 353,347,500
ANCLAJE | Semilla 100183 | Validación 202105 | Ganancia 253,275,000
ANCLAJE | Semilla 100183 | Validación 202106 | Ganancia 367,372,500
ANCLAJE | Semilla 100151 | Validación 202105 | Ganancia 254,980,000
ANCLAJE | Semilla 100151 | Validación 202106 | Ganancia 358,352,500
ANCLAJE | Semilla 500041 | Validación 202105 | Ganancia 248,325,000
ANCLAJE | Semilla 500041 | Validación 202106 | Ganancia 349,085,000
ANCLAJE | Semilla 900007 | Validación 202105 | Ganancia 262,625,000
ANCLAJE | Semilla 900007 | Validación 202106 | Ganancia 356,592,500

Ganancia promedio baseline: 301,642,000
Ganancia promedio con anclaje: 305,841,250
Mejora promedio: 4,199,250 (1.39%)


,ganancia_base,ganancia_anclaje,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio
validacion,,,,,,
202105,251069500.0,254732500.0,3663000.0,-632500.0,8607500.0,0.0007
202106,352214500.0,356950000.0,4735500.0,-3410000.0,12072500.0,0.0012


El bloque de anclaje aporta señal, aunque la mejora es moderada:
- Ganancia: 301,64 M → 305,84 M
- Mejora: +4,20 M (+1,39%)
- Mejora promedio positiva en mayo y junio.
- Resultado positivo en 7 de las 10 combinaciones mes-semilla.
- En 3 combinaciones empeoró, por lo que no es una mejora totalmente estable.
- El AUC también mejoró ligeramente en ambos meses.
Decisión: conservar provisionalmente las 13 features. Más adelante haremos una ablación para comprobar que sigan aportando cuando existan lags y otros agregados.
Quedan dos referencias:
- Baseline original: 301,64 M
- Modelo acumulado con anclaje: 305,84 M
El próximo bloque será tarjetas consolidadas: sumar Visa y Master correctamente y construir utilización, holgura, consumo relativo, transacciones totales y principalidad Visa. Primero lo construiremos y auditaremos, sin mezclar pesos con dólares ni usar todavía mpagado.




In [139]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
)

plot_optimization_history(study_base_v3).show()
plot_param_importances(study_base_v3).show()
plot_parallel_coordinate(study_base_v3).show()
plot_slice(study_base_v3).show()

In [141]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
    plot_parallel_coordinate,
)

print("Trials completos:", len([
    trial for trial in study_base_v3.trials
    if trial.state.name == "COMPLETE"
]))

display(
    study_base_v3.trials_dataframe()
    .sort_values("value", ascending=False)
    .head(10)
)

plot_optimization_history(study_base_v3).show()
plot_param_importances(study_base_v3).show()
plot_slice(study_base_v3).show()
plot_parallel_coordinate(study_base_v3).show()

Trials completos: 30


,number,value,datetime_start,datetime_complete,duration,params_bagging_fraction,params_feature_fraction,params_lambda_l1,params_lambda_l2,params_learning_rate,...,user_attrs_auc_202105,user_attrs_auc_202106,user_attrs_auc_promedio,user_attrs_ganancia_202105,user_attrs_ganancia_202106,user_attrs_iteraciones_202105,user_attrs_iteraciones_202106,user_attrs_peor_ganancia,system_attrs_fixed_params,state
20,20,309980000.0,2026-10-04 20:21:01.461411,2026-10-04 20:21:34.656613,0 days 00:00:33.195202,0.700694,0.301521,0.000083,0.000670,0.011968,...,0.888242,0.898366,0.893304,252945000.0,367015000.0,259,715,252945000.0,NaN,COMPLETE
21,21,309498750.0,2026-10-04 20:21:34.665623,2026-10-04 20:21:58.850911,0 days 00:00:24.185288,0.701390,0.318271,0.000051,0.000847,0.012118,...,0.888641,0.899396,0.894019,257867500.0,361130000.0,202,408,257867500.0,NaN,COMPLETE
22,22,308192500.0,2026-10-04 20:21:58.857533,2026-10-04 20:22:28.773368,0 days 00:00:29.915835,0.698423,0.307127,0.000031,0.017112,0.011827,...,0.888773,0.899016,0.893895,257015000.0,359370000.0,296,516,257015000.0,NaN,COMPLETE
29,29,307216250.0,2026-10-04 20:24:51.570222,2026-10-04 20:25:23.873143,0 days 00:00:32.302921,0.580100,0.349272,0.050355,0.000577,0.011147,...,0.889862,0.899566,0.894714,262212500.0,352220000.0,449,531,262212500.0,NaN,COMPLETE
16,16,306996250.0,2026-10-04 20:18:21.646085,2026-10-04 20:19:06.943008,0 days 00:00:45.296923,0.645311,0.421878,0.000003,0.005008,0.008269,...,0.888220,0.897428,0.892824,255557500.0,358435000.0,290,1032,255557500.0,NaN,COMPLETE
23,23,306666250.0,2026-10-04 20:22:28.780190,2026-10-04 20:23:00.895776,0 days 00:00:32.115586,0.712556,0.308291,0.000056,0.042423,0.012729,...,0.889663,0.898789,0.894226,254952500.0,358380000.0,253,798,254952500.0,NaN,COMPLETE
25,25,305360000.0,2026-10-04 20:23:25.683478,2026-10-04 20:23:48.167942,0 days 00:00:22.484464,0.781885,0.366386,0.000083,0.023044,0.021879,...,0.886995,0.895753,0.891374,251707500.0,359012500.0,142,341,251707500.0,NaN,COMPLETE
0,0,304095000.0,2026-10-04 20:13:28.174638,2026-10-04 20:14:14.426477,0 days 00:00:46.251839,0.825529,0.648345,0.000539,0.000645,0.005037,...,0.888436,0.896991,0.892713,257675000.0,350515000.0,455,1225,257675000.0,"{'max_depth': 8, 'num_leaves': 124, 'learning_...",COMPLETE
11,11,303201250.0,2026-10-04 20:16:30.541731,2026-10-04 20:16:47.396145,0 days 00:00:16.854414,0.877404,0.679187,0.001335,0.008071,0.016752,...,0.890575,0.894622,0.892599,253660000.0,352742500.0,343,554,253660000.0,NaN,COMPLETE
13,13,303173750.0,2026-10-04 20:17:16.982849,2026-10-04 20:17:38.317280,0 days 00:00:21.334431,0.867200,0.755690,0.000226,0.000007,0.015705,...,0.885532,0.894341,0.889937,257317500.0,349030000.0,112,263,257317500.0,NaN,COMPLETE


- Optuna encontró una zona estable: los trials 20, 21 y 22 tienen parámetros muy parecidos y ganancias cercanas. El mejor resultado no parece un accidente aislado.
- Trial 20 maximiza la ganancia promedio: $309,98 M.
- Trial 29 tiene mejor AUC promedio (0,8947) y mejor peor mes, pero menor ganancia promedio ($307,22 M).
- Esto confirma que mayor AUC no implica necesariamente mayor ganancia, que es la métrica real de la competencia.
- Los 30 trials alcanzan para congelar un baseline de comparación. Haremos una optimización más amplia recién con el conjunto final de features.

# veamos las fechas


In [147]:
%%sql

CREATE OR REPLACE VIEW competencia_01_fe_fechas AS

WITH fechas_reconstruidas AS (

    SELECT
        a.*,

        CAST(
            STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
            + INTERVAL 1 MONTH
            AS DATE
        ) AS fecha_referencia,

        CASE
            WHEN Master_Fvencimiento IS NOT NULL
            THEN CAST(
                STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
                + INTERVAL 1 MONTH
                - Master_Fvencimiento * INTERVAL 1 DAY
                AS DATE
            )
        END AS master_fecha_vencimiento_calculada,

        CASE
            WHEN Visa_Fvencimiento IS NOT NULL
            THEN CAST(
                STRPTIME(CAST(foto_mes AS VARCHAR), '%Y%m')
                + INTERVAL 1 MONTH
                - Visa_Fvencimiento * INTERVAL 1 DAY
                AS DATE
            )
        END AS visa_fecha_vencimiento_calculada

    FROM competencia_01_fe_anclaje_final AS a
),

por_marca AS (

    SELECT
        * EXCLUDE (
            fecha_referencia,
            master_fecha_vencimiento_calculada,
            visa_fecha_vencimiento_calculada
        ),

        -- ====================================================
        -- MASTER
        -- ====================================================

        CASE
            WHEN master_fecha_vencimiento_calculada IS NULL
              OR master_fecha_vencimiento_calculada
                    = DATE '9999-12-31'
            THEN NULL
            ELSE DATE_DIFF(
                'day',
                fecha_referencia,
                master_fecha_vencimiento_calculada
            )
        END AS master_dias_hasta_vencimiento,

        CASE
            WHEN master_fecha_vencimiento_calculada IS NULL
              OR master_fecha_vencimiento_calculada
                    = DATE '9999-12-31'
            THEN NULL
            WHEN master_fecha_vencimiento_calculada
                    < fecha_referencia
            THEN 1
            ELSE 0
        END AS master_plastico_vencido,

        CASE
            WHEN master_fecha_vencimiento_calculada IS NULL
            THEN NULL
            WHEN master_fecha_vencimiento_calculada
                    = DATE '9999-12-31'
            THEN 1
            ELSE 0
        END AS master_vencimiento_codigo_9999,

        Master_fechaalta
            AS master_antiguedad_tarjeta_dias,

        Master_fultimo_cierre
            AS master_dias_desde_ultimo_cierre,

        CASE
            WHEN Master_delinquency = 1
            THEN Master_Finiciomora
        END AS master_dias_mora,

        CASE
            WHEN Master_delinquency IS NULL
            THEN NULL
            WHEN Master_delinquency = 1
             AND Master_Finiciomora = 0
            THEN 1
            ELSE 0
        END AS master_mora_inicio_cero,

        -- ====================================================
        -- VISA
        -- ====================================================

        CASE
            WHEN visa_fecha_vencimiento_calculada IS NULL
              OR visa_fecha_vencimiento_calculada
                    = DATE '9999-12-31'
            THEN NULL
            ELSE DATE_DIFF(
                'day',
                fecha_referencia,
                visa_fecha_vencimiento_calculada
            )
        END AS visa_dias_hasta_vencimiento,

        CASE
            WHEN visa_fecha_vencimiento_calculada IS NULL
              OR visa_fecha_vencimiento_calculada
                    = DATE '9999-12-31'
            THEN NULL
            WHEN visa_fecha_vencimiento_calculada
                    < fecha_referencia
            THEN 1
            ELSE 0
        END AS visa_plastico_vencido,

        CASE
            WHEN visa_fecha_vencimiento_calculada IS NULL
            THEN NULL
            WHEN visa_fecha_vencimiento_calculada
                    = DATE '9999-12-31'
            THEN 1
            ELSE 0
        END AS visa_vencimiento_codigo_9999,

        Visa_fechaalta
            AS visa_antiguedad_tarjeta_dias,

        Visa_fultimo_cierre
            AS visa_dias_desde_ultimo_cierre,

        CASE
            WHEN Visa_delinquency = 1
            THEN Visa_Finiciomora
        END AS visa_dias_mora,

        CASE
            WHEN Visa_delinquency IS NULL
            THEN NULL
            WHEN Visa_delinquency = 1
             AND Visa_Finiciomora = 0
            THEN 1
            ELSE 0
        END AS visa_mora_inicio_cero

    FROM fechas_reconstruidas
)

SELECT
    *,

    CASE
        WHEN master_antiguedad_tarjeta_dias IS NULL
        THEN visa_antiguedad_tarjeta_dias
        WHEN visa_antiguedad_tarjeta_dias IS NULL
        THEN master_antiguedad_tarjeta_dias
        ELSE GREATEST(
            master_antiguedad_tarjeta_dias,
            visa_antiguedad_tarjeta_dias
        )
    END AS tarjetas_antiguedad_maxima_dias,

    CASE
        WHEN master_dias_mora IS NULL
        THEN visa_dias_mora
        WHEN visa_dias_mora IS NULL
        THEN master_dias_mora
        ELSE GREATEST(
            master_dias_mora,
            visa_dias_mora
        )
    END AS tarjetas_dias_mora_maximos,

    CASE
        WHEN master_dias_desde_ultimo_cierre IS NULL
        THEN visa_dias_desde_ultimo_cierre
        WHEN visa_dias_desde_ultimo_cierre IS NULL
        THEN master_dias_desde_ultimo_cierre
        ELSE LEAST(
            master_dias_desde_ultimo_cierre,
            visa_dias_desde_ultimo_cierre
        )
    END AS tarjetas_dias_desde_cierre_mas_reciente,

    CASE
        WHEN master_dias_hasta_vencimiento IS NULL
        THEN visa_dias_hasta_vencimiento
        WHEN visa_dias_hasta_vencimiento IS NULL
        THEN master_dias_hasta_vencimiento
        ELSE LEAST(
            master_dias_hasta_vencimiento,
            visa_dias_hasta_vencimiento
        )
    END AS tarjetas_dias_hasta_vencimiento_minimo,

    COALESCE(master_plastico_vencido, 0)
        + COALESCE(visa_plastico_vencido, 0)
        AS cantidad_plasticos_vencidos,

    COALESCE(master_mora_inicio_cero, 0)
        + COALESCE(visa_mora_inicio_cero, 0)
        AS cantidad_moras_inicio_cero

FROM por_marca;

,Success


In [148]:
%%sql

SELECT
    foto_mes,
    COUNT(*) AS clientes,

    SUM(master_vencimiento_codigo_9999)
        AS master_codigos_9999,

    SUM(visa_vencimiento_codigo_9999)
        AS visa_codigos_9999,

    SUM(master_plastico_vencido)
        AS master_plasticos_vencidos,

    SUM(visa_plastico_vencido)
        AS visa_plasticos_vencidos,

    SUM(master_mora_inicio_cero)
        AS master_moras_inicio_cero,

    SUM(visa_mora_inicio_cero)
        AS visa_moras_inicio_cero,

    COUNT(*) FILTER (
        WHERE master_antiguedad_tarjeta_dias < 0
           OR visa_antiguedad_tarjeta_dias < 0
           OR master_dias_desde_ultimo_cierre < 0
           OR visa_dias_desde_ultimo_cierre < 0
           OR master_dias_mora < 0
           OR visa_dias_mora < 0
    ) AS filas_con_valores_imposibles,

    ROUND(
        MEDIAN(tarjetas_antiguedad_maxima_dias),
        1
    ) AS mediana_antiguedad_maxima_dias,

    ROUND(
        MEDIAN(tarjetas_dias_desde_cierre_mas_reciente),
        1
    ) AS mediana_dias_desde_cierre,

    ROUND(
        MEDIAN(tarjetas_dias_hasta_vencimiento_minimo),
        1
    ) AS mediana_dias_hasta_vencimiento,

    ROUND(
        QUANTILE_CONT(
            tarjetas_dias_mora_maximos,
            0.90
        ),
        1
    ) AS p90_dias_mora

FROM competencia_01_fe_fechas

GROUP BY foto_mes
ORDER BY foto_mes;

,foto_mes,clientes,master_codigos_9999,visa_codigos_9999,master_plasticos_vencidos,visa_plasticos_vencidos,master_moras_inicio_cero,visa_moras_inicio_cero,filas_con_valores_imposibles,mediana_antiguedad_maxima_dias,mediana_dias_desde_cierre,mediana_dias_hasta_vencimiento,p90_dias_mora
0,202103,162900,0.0,466.0,142.0,123.0,0.0,0.0,0,2984.0,1.0,790.0,52.0
1,202104,163284,0.0,463.0,145.0,128.0,0.0,0.0,0,2991.0,2.0,790.0,47.0
2,202105,163768,0.0,458.0,153.0,135.0,0.0,3222.0,0,3008.0,5.0,790.0,49.0
3,202106,164114,0.0,454.0,150.0,133.0,1.0,0.0,0,3005.0,0.0,791.0,87.0
4,202107,164348,0.0,449.0,205.0,183.0,0.0,0.0,0,3015.0,3.0,790.0,66.0
5,202108,164647,1.0,446.0,161.0,149.0,0.0,3249.0,0,3039.0,6.0,790.0,28.8


La auditoría reproduce exactamente lo investigado antes:
- Códigos 9999: 466→446 en Visa y prácticamente ninguno en Master.
- Plásticos vencidos: mismos conteos anteriores.
- Mora con inicio cero: anomalía concentrada en mayo y agosto.
- Ningún valor imposible.
- Antigüedad, cierre y vencimiento tienen distribuciones coherentes.

In [149]:
# ============================================================
# PREPARACIÓN Y EVALUACIÓN DEL BLOQUE DE FECHAS
# ============================================================

base_fechas = %sql SELECT * FROM competencia_01_fe_fechas WHERE foto_mes BETWEEN 202103 AND 202106 AND clase_ternaria IS NOT NULL ORDER BY foto_mes, numero_de_cliente


base_fechas["target_baja2"] = (
    base_fechas["clase_ternaria"] == "BAJA+2"
).astype("int8")


features_fechas = [
    columna
    for columna in base_fechas.columns
    if columna not in columnas_excluidas_base
]


features_nuevas_fechas = [
    columna
    for columna in features_fechas
    if columna not in features_anclaje
]


base_fechas[features_fechas] = (
    base_fechas[features_fechas]
    .replace([np.inf, -np.inf], np.nan)
)


# Auditoría de identidad de filas y target.
assert len(base_fechas) == len(base_anclaje)

assert base_fechas[
    ["numero_de_cliente", "foto_mes"]
].equals(
    base_anclaje[
        ["numero_de_cliente", "foto_mes"]
    ]
)

assert base_fechas["target_baja2"].equals(
    base_anclaje["target_baja2"]
)

columnas_no_numericas = (
    base_fechas[features_fechas]
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

assert columnas_no_numericas == [], (
    f"Hay columnas no numéricas: {columnas_no_numericas}"
)


print("Features con anclaje:", len(features_anclaje))
print("Features con anclaje + fechas:", len(features_fechas))
print("Features nuevas de fechas:", len(features_nuevas_fechas))

display(
    pd.DataFrame({
        "feature_nueva": features_nuevas_fechas
    })
)


resultados_fechas = evaluar_features_temporal(
    datos=base_fechas,
    features=features_fechas,
    nombre_experimento="ANCLAJE+FECHAS"
)


# Comparación emparejada contra anclaje.
comparacion_fechas = (
    resultados_anclaje
    .merge(
        resultados_fechas,
        on=["semilla", "validacion"],
        suffixes=("_anclaje", "_fechas")
    )
)


comparacion_fechas["delta_ganancia"] = (
    comparacion_fechas["ganancia_fechas"]
    - comparacion_fechas["ganancia_anclaje"]
)

comparacion_fechas["delta_auc"] = (
    comparacion_fechas["auc_fechas"]
    - comparacion_fechas["auc_anclaje"]
)


resumen_fechas = (
    comparacion_fechas
    .groupby("validacion")
    .agg(
        ganancia_anclaje=("ganancia_anclaje", "mean"),
        ganancia_fechas=("ganancia_fechas", "mean"),
        mejora_media=("delta_ganancia", "mean"),
        mejora_minima=("delta_ganancia", "min"),
        mejora_maxima=("delta_ganancia", "max"),
        cambio_auc_medio=("delta_auc", "mean"),
        semillas_favorables=(
            "delta_ganancia",
            lambda x: int((x > 0).sum())
        )
    )
    .round(4)
)


ganancia_fechas_promedio = (
    resultados_fechas["ganancia"].mean()
)

mejora_fechas_promedio = (
    comparacion_fechas["delta_ganancia"].mean()
)

mejora_fechas_pct = (
    100.0
    * mejora_fechas_promedio
    / resultados_anclaje["ganancia"].mean()
)

combinaciones_favorables = int(
    (comparacion_fechas["delta_ganancia"] > 0).sum()
)


print()
print(
    "Ganancia promedio con anclaje:",
    f"{resultados_anclaje['ganancia'].mean():,.0f}"
)

print(
    "Ganancia promedio con anclaje + fechas:",
    f"{ganancia_fechas_promedio:,.0f}"
)

print(
    "Mejora incremental:",
    f"{mejora_fechas_promedio:,.0f}",
    f"({mejora_fechas_pct:.2f}%)"
)

print(
    "Combinaciones favorables:",
    f"{combinaciones_favorables}/10"
)

display(resumen_fechas)


resultados_fechas.to_csv(
    "resultados_bloque_fechas.csv",
    index=False
)

Features con anclaje: 156
Features con anclaje + fechas: 176
Features nuevas de fechas: 20


,feature_nueva
0,master_dias_hasta_vencimiento
1,master_plastico_vencido
2,master_vencimiento_codigo_9999
3,master_antiguedad_tarjeta_dias
4,master_dias_desde_ultimo_cierre
5,master_dias_mora
6,master_mora_inicio_cero
7,visa_dias_hasta_vencimiento
8,visa_plastico_vencido
9,visa_vencimiento_codigo_9999


ANCLAJE+FECHAS | Semilla 100129 | Validación 202105 | Ganancia 254,870,000
ANCLAJE+FECHAS | Semilla 100129 | Validación 202106 | Ganancia 360,030,000
ANCLAJE+FECHAS | Semilla 100183 | Validación 202105 | Ganancia 259,022,500
ANCLAJE+FECHAS | Semilla 100183 | Validación 202106 | Ganancia 358,517,500
ANCLAJE+FECHAS | Semilla 100151 | Validación 202105 | Ganancia 249,012,500
ANCLAJE+FECHAS | Semilla 100151 | Validación 202106 | Ganancia 349,002,500
ANCLAJE+FECHAS | Semilla 500041 | Validación 202105 | Ganancia 254,897,500
ANCLAJE+FECHAS | Semilla 500041 | Validación 202106 | Ganancia 356,427,500
ANCLAJE+FECHAS | Semilla 900007 | Validación 202105 | Ganancia 256,355,000
ANCLAJE+FECHAS | Semilla 900007 | Validación 202106 | Ganancia 347,545,000

Ganancia promedio con anclaje: 305,841,250
Ganancia promedio con anclaje + fechas: 304,568,000
Mejora incremental: -1,273,250 (-0.42%)
Combinaciones favorables: 5/10


,ganancia_anclaje,ganancia_fechas,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables
validacion,,,,,,,
202105,254732500.0,254831500.0,99000.0,-6270000.0,6572500.0,0.0012,3
202106,356950000.0,354304500.0,-2645500.0,-9350000.0,7342500.0,0.0004,2



- Las fechas estáticas no mejoran de forma robusta: −0,42%, con 5/10 resultados favorables.
- Mejoran algo el AUC, pero no la ganancia.
- Por ahora no agregamos esas 20 variables al modelo campeón.
- Conservamos la vista para construir cambios temporales.
Algunas fechas probablemente tengan más valor como evento que como nivel:
- cambio anormal en vencimiento → renovación del plástico;
- reinicio de antigüedad → posible reemplazo/alta nueva;
- aumento o reinicio de días en mora;
- cambio en días desde el último cierre;
- aparición/desaparición de mora_inicio_cero.

# Lags y deltas UNO

In [150]:
# ============================================================
# LAG1 Y DELTA1: ACTIVIDAD, SALDOS Y VINCULACIÓN
# ============================================================

variables_lag1 = [
    "ctrx_quarter",
    "coperaciones_recurrentes_total",
    "cantidad_canales_usados",
    "cproductos",
    "mcaja_ahorro",
    "mcuentas_saldo",
    "mpasivos_margen",
    "mrentabilidad",
    "ctarjeta_debito_transacciones",
    "mautoservicio",
    "ctarjeta_visa_transacciones",
    "mtarjeta_visa_consumo",
    "chomebanking_transacciones",
    "cmobile_app_trx",
    "catm_trx",
    "catm_trx_other",
    "cpayroll_trx",
    "mpayroll",
    "mtransferencias_recibidas",
    "mtransferencias_emitidas",
]


# Partimos del modelo campeón: anclaje con 156 features.
base_lag1 = (
    base_anclaje
    .copy()
    .sort_values(
        ["numero_de_cliente", "foto_mes"]
    )
    .reset_index(drop=True)
)


# Índice mensual para comprobar que el mes anterior
# sea realmente consecutivo.
anio = base_lag1["foto_mes"] // 100
mes = base_lag1["foto_mes"] % 100

base_lag1["_indice_mes"] = (
    anio * 12 + mes
)

base_lag1["_indice_mes_anterior"] = (
    base_lag1
    .groupby(
        "numero_de_cliente",
        sort=False
    )["_indice_mes"]
    .shift(1)
)

historia_valida = (
    base_lag1["_indice_mes"]
    - base_lag1["_indice_mes_anterior"]
    == 1
)

# Solo para auditoría; no entrará al modelo.
base_lag1["_tiene_historia_lag1"] = (
    historia_valida.astype("int8")
)


# Construcción conjunta para evitar fragmentar el DataFrame.
nuevas_columnas = {}
lags = {}
deltas = {}

for variable in variables_lag1:

    lag_crudo = (
        base_lag1
        .groupby(
            "numero_de_cliente",
            sort=False
        )[variable]
        .shift(1)
    )

    lag_valido = lag_crudo.where(
        historia_valida
    )

    delta_valido = (
        base_lag1[variable]
        - lag_valido
    )

    nombre_lag = f"{variable}_lag1"
    nombre_delta = f"{variable}_delta1"

    nuevas_columnas[nombre_lag] = lag_valido
    nuevas_columnas[nombre_delta] = delta_valido

    lags[variable] = lag_valido
    deltas[variable] = delta_valido


def crear_evento(condicion):
    """
    Devuelve NaN cuando no existe un mes anterior consecutivo.
    En los restantes casos devuelve 0 o 1.
    """
    condicion_limpia = (
        condicion
        .fillna(False)
        .astype("int8")
    )

    return pd.Series(
        np.where(
            historia_valida,
            condicion_limpia,
            np.nan
        ),
        index=base_lag1.index
    )


# Dejó de recibir payroll.
nuevas_columnas["apago_payroll_lag1"] = crear_evento(
    (lags["cpayroll_trx"] > 0)
    & (base_lag1["cpayroll_trx"] == 0)
)

# Dejó de tener operaciones recurrentes.
nuevas_columnas["apago_recurrentes_lag1"] = crear_evento(
    (lags["coperaciones_recurrentes_total"] > 0)
    & (
        base_lag1[
            "coperaciones_recurrentes_total"
        ] == 0
    )
)

# Dejó de usar Visa.
nuevas_columnas["apago_visa_lag1"] = crear_evento(
    (lags["ctarjeta_visa_transacciones"] > 0)
    & (
        base_lag1[
            "ctarjeta_visa_transacciones"
        ] == 0
    )
)

# Dejó de operar por home banking.
nuevas_columnas["apago_homebanking_lag1"] = crear_evento(
    (lags["chomebanking_transacciones"] > 0)
    & (
        base_lag1[
            "chomebanking_transacciones"
        ] == 0
    )
)

# Baja cajeros propios y sube cajeros de otros bancos.
nuevas_columnas["migracion_atm_ajeno_lag1"] = crear_evento(
    (deltas["catm_trx"] < 0)
    & (deltas["catm_trx_other"] > 0)
)

# Caída mensual de al menos 20% en actividad trimestral.
nuevas_columnas["caida_ctrx_20pct_lag1"] = crear_evento(
    (lags["ctrx_quarter"] > 0)
    & (
        base_lag1["ctrx_quarter"]
        < 0.80 * lags["ctrx_quarter"]
    )
)

# Redujo la cantidad de canales utilizados.
nuevas_columnas["caida_canales_lag1"] = crear_evento(
    base_lag1["cantidad_canales_usados"]
    < lags["cantidad_canales_usados"]
)


base_lag1 = pd.concat(
    [
        base_lag1,
        pd.DataFrame(
            nuevas_columnas,
            index=base_lag1.index
        )
    ],
    axis=1
)


features_nuevas_lag1 = list(
    nuevas_columnas.keys()
)

features_lag1 = (
    features_anclaje
    + features_nuevas_lag1
)


base_lag1[features_lag1] = (
    base_lag1[features_lag1]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# AUDITORÍA
# ============================================================

resumen_historia_lag1 = (
    base_lag1
    .groupby("foto_mes")
    .agg(
        clientes=(
            "numero_de_cliente",
            "size"
        ),
        clientes_con_historia_lag1=(
            "_tiene_historia_lag1",
            "sum"
        )
    )
)

resumen_historia_lag1[
    "pct_con_historia_lag1"
] = (
    100.0
    * resumen_historia_lag1[
        "clientes_con_historia_lag1"
    ]
    / resumen_historia_lag1["clientes"]
).round(3)


mascara_train_junio = (
    base_lag1["foto_mes"].isin(
        [202103, 202104]
    )
)

features_lag_con_dato_en_train_junio = sum(
    base_lag1.loc[
        mascara_train_junio,
        feature
    ].notna().any()
    for feature in features_nuevas_lag1
)


print("Variables originales lagueadas:", len(variables_lag1))
print("Features nuevas lag/delta/evento:", len(features_nuevas_lag1))
print("Features totales del experimento:", len(features_lag1))
print(
    "Features temporales con algún dato en el entrenamiento de junio:",
    features_lag_con_dato_en_train_junio
)

display(resumen_historia_lag1)

display(
    pd.DataFrame({
        "feature_nueva": features_nuevas_lag1
    })
)

Variables originales lagueadas: 20
Features nuevas lag/delta/evento: 47
Features totales del experimento: 203
Features temporales con algún dato en el entrenamiento de junio: 47


,clientes,clientes_con_historia_lag1,pct_con_historia_lag1
foto_mes,,,
202103,162900,0,0.000
202104,163284,161881,99.141
202105,163768,162320,99.116
202106,164114,162625,99.093


,feature_nueva
0,ctrx_quarter_lag1
1,ctrx_quarter_delta1
2,coperaciones_recurrentes_total_lag1
3,coperaciones_recurrentes_total_delta1
4,cantidad_canales_usados_lag1
5,cantidad_canales_usados_delta1
6,cproductos_lag1
7,cproductos_delta1
8,mcaja_ahorro_lag1
9,mcaja_ahorro_delta1


In [151]:
# ============================================================
# EVALUACIÓN DE LAG1 Y DELTA1
# ÚNICO FOLD INFORMATIVO: JUNIO
# ============================================================

meses_train_lag1 = [202103, 202104]
mes_validacion_lag1 = 202106

mascara_train_lag1 = (
    base_lag1["foto_mes"]
    .isin(meses_train_lag1)
)

mascara_valid_lag1 = (
    base_lag1["foto_mes"]
    == mes_validacion_lag1
)


# Control temporal del target.
assert (
    indice_mes(mes_validacion_lag1)
    - indice_mes(max(meses_train_lag1))
    >= 2
), "Fuga temporal en el experimento de lags"


X_train_lag1 = base_lag1.loc[
    mascara_train_lag1,
    features_lag1
]

y_train_lag1 = base_lag1.loc[
    mascara_train_lag1,
    "target_baja2"
]

X_valid_lag1 = base_lag1.loc[
    mascara_valid_lag1,
    features_lag1
]

y_valid_lag1 = base_lag1.loc[
    mascara_valid_lag1,
    "target_baja2"
]

clases_valid_lag1 = base_lag1.loc[
    mascara_valid_lag1,
    "clase_ternaria"
]

ganancia_fila_valid_lag1 = ganancia_por_fila(
    clases_valid_lag1
)


print(
    "Entrenamiento:",
    meses_train_lag1,
    "| Filas:",
    len(X_train_lag1)
)

print(
    "Validación:",
    mes_validacion_lag1,
    "| Filas:",
    len(X_valid_lag1)
)

print(
    "Features:",
    len(features_lag1)
)


resultados_lag1_junio = []
importancias_lag1 = []


for semilla_actual in semillas:

    params_semilla = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "bagging_freq": 1,

        **study_base_v3.best_params,

        "seed": semilla_actual,
        "feature_fraction_seed": semilla_actual,
        "bagging_seed": semilla_actual,
        "data_random_seed": semilla_actual,
    }


    dataset_train = lgb.Dataset(
        X_train_lag1,
        label=y_train_lag1
    )

    dataset_valid = lgb.Dataset(
        X_valid_lag1,
        label=y_valid_lag1,
        reference=dataset_train
    )


    modelo_lag1 = lgb.train(
        params=params_semilla,
        train_set=dataset_train,
        num_boost_round=3000,
        valid_sets=[dataset_valid],
        valid_names=["validacion"],
        feval=feval_ganancia(
            ganancia_fila_valid_lag1
        ),
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=150,
                verbose=False
            ),
            lgb.log_evaluation(period=0)
        ]
    )


    probabilidades_lag1 = modelo_lag1.predict(
        X_valid_lag1,
        num_iteration=modelo_lag1.best_iteration
    )


    ganancia_lag1 = ganancia(
        probabilidades_lag1,
        ganancia_fila_valid_lag1
    )

    auc_lag1 = roc_auc_score(
        y_valid_lag1,
        probabilidades_lag1
    )


    resultados_lag1_junio.append({
        "experimento": "ANCLAJE+LAG1",
        "semilla": semilla_actual,
        "validacion": mes_validacion_lag1,
        "ganancia": float(ganancia_lag1),
        "auc": float(auc_lag1),
        "iteraciones": int(
            modelo_lag1.best_iteration
        ),
    })


    # Importancia normalizada para saber si las nuevas
    # variables fueron utilizadas por el modelo.
    ganancias_importancia = (
        modelo_lag1.feature_importance(
            importance_type="gain"
        )
    )

    ganancia_importancia_total = (
        ganancias_importancia.sum()
    )

    for feature, importancia in zip(
        features_lag1,
        ganancias_importancia
    ):

        if feature in features_nuevas_lag1:

            importancias_lag1.append({
                "semilla": semilla_actual,
                "feature": feature,
                "importancia_gain": importancia,
                "importancia_pct": (
                    100.0 * importancia
                    / ganancia_importancia_total
                    if ganancia_importancia_total > 0
                    else 0.0
                ),
            })


    print(
        f"LAG1 | Semilla {semilla_actual} | "
        f"Ganancia {ganancia_lag1:,.0f} | "
        f"AUC {auc_lag1:.4f} | "
        f"Iteraciones {modelo_lag1.best_iteration}"
    )


resultados_lag1_junio = pd.DataFrame(
    resultados_lag1_junio
)

importancias_lag1 = pd.DataFrame(
    importancias_lag1
)


# ============================================================
# COMPARACIÓN EMPAREJADA CONTRA ANCLAJE
# ============================================================

referencia_anclaje_junio = (
    resultados_anclaje.loc[
        resultados_anclaje["validacion"] == 202106,
        ["semilla", "validacion", "ganancia", "auc"]
    ]
    .rename(columns={
        "ganancia": "ganancia_anclaje",
        "auc": "auc_anclaje",
    })
)


comparacion_lag1 = (
    resultados_lag1_junio
    .merge(
        referencia_anclaje_junio,
        on=["semilla", "validacion"],
        how="inner"
    )
)


comparacion_lag1["delta_ganancia"] = (
    comparacion_lag1["ganancia"]
    - comparacion_lag1["ganancia_anclaje"]
)

comparacion_lag1["delta_auc"] = (
    comparacion_lag1["auc"]
    - comparacion_lag1["auc_anclaje"]
)


resumen_lag1 = pd.DataFrame({
    "ganancia_anclaje": [
        comparacion_lag1[
            "ganancia_anclaje"
        ].mean()
    ],
    "ganancia_lag1": [
        comparacion_lag1[
            "ganancia"
        ].mean()
    ],
    "mejora_media": [
        comparacion_lag1[
            "delta_ganancia"
        ].mean()
    ],
    "mejora_minima": [
        comparacion_lag1[
            "delta_ganancia"
        ].min()
    ],
    "mejora_maxima": [
        comparacion_lag1[
            "delta_ganancia"
        ].max()
    ],
    "cambio_auc_medio": [
        comparacion_lag1[
            "delta_auc"
        ].mean()
    ],
    "semillas_favorables": [
        int(
            (
                comparacion_lag1[
                    "delta_ganancia"
                ] > 0
            ).sum()
        )
    ],
})


resumen_lag1["mejora_pct"] = (
    100.0
    * resumen_lag1["mejora_media"]
    / resumen_lag1["ganancia_anclaje"]
)


resumen_importancias_lag1 = (
    importancias_lag1
    .groupby("feature")
    .agg(
        importancia_media_pct=(
            "importancia_pct",
            "mean"
        ),
        semillas_en_que_se_uso=(
            "importancia_gain",
            lambda x: int((x > 0).sum())
        )
    )
    .sort_values(
        "importancia_media_pct",
        ascending=False
    )
)


print()
print("Comparación por semilla:")
display(
    comparacion_lag1[
        [
            "semilla",
            "ganancia_anclaje",
            "ganancia",
            "delta_ganancia",
            "auc_anclaje",
            "auc",
            "delta_auc",
        ]
    ]
)

print("Resumen:")
display(resumen_lag1.round(4))

print("Principales features temporales:")
display(
    resumen_importancias_lag1.head(20)
)


resultados_lag1_junio.to_csv(
    "resultados_lag1_junio.csv",
    index=False
)

importancias_lag1.to_csv(
    "importancias_lag1_junio.csv",
    index=False
)

Entrenamiento: [202103, 202104] | Filas: 326184
Validación: 202106 | Filas: 164114
Features: 203
LAG1 | Semilla 100129 | Ganancia 370,095,000 | AUC 0.9001 | Iteraciones 173
LAG1 | Semilla 100183 | Ganancia 369,737,500 | AUC 0.9015 | Iteraciones 354
LAG1 | Semilla 100151 | Ganancia 365,612,500 | AUC 0.8996 | Iteraciones 121
LAG1 | Semilla 500041 | Ganancia 352,880,000 | AUC 0.8998 | Iteraciones 415
LAG1 | Semilla 900007 | Ganancia 353,567,500 | AUC 0.9010 | Iteraciones 219

Comparación por semilla:


,semilla,ganancia_anclaje,ganancia,delta_ganancia,auc_anclaje,auc,delta_auc
0,100129,353347500.0,370095000.0,16747500.0,0.900405,0.900140,-0.000265
1,100183,367372500.0,369737500.0,2365000.0,0.899936,0.901504,0.001569
2,100151,358352500.0,365612500.0,7260000.0,0.898650,0.899627,0.000977
3,500041,349085000.0,352880000.0,3795000.0,0.899546,0.899846,0.000300
4,900007,356592500.0,353567500.0,-3025000.0,0.900540,0.901050,0.000510


Resumen:


,ganancia_anclaje,ganancia_lag1,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables,mejora_pct
0,356950000.0,362378500.0,5428500.0,-3025000.0,16747500.0,0.0006,4,1.5208


Principales features temporales:


,importancia_media_pct,semillas_en_que_se_uso
feature,,
ctrx_quarter_lag1,1.802016,5
mcuentas_saldo_lag1,1.019588,5
mrentabilidad_delta1,0.706244,5
mrentabilidad_lag1,0.698676,5
mcuentas_saldo_delta1,0.573559,5
mcaja_ahorro_lag1,0.526365,5
mpasivos_margen_delta1,0.494129,5
mpasivos_margen_lag1,0.486763,5
chomebanking_transacciones_delta1,0.459328,5


Este bloque sí aporta señal:
- Mejora media en junio: +$5,43 M (+1,52%).
- Mejora en 4 de 5 semillas.
- AUC: +0,0006.
- Las variables temporales principales fueron utilizadas en las cinco semillas.
- ctrx_quarter_lag1, saldos, rentabilidad y sus deltas aparecen arriba, coherentemente con el EDA.
Además, los resultados muestran otra vez que AUC y ganancia no son equivalentes:
- Semilla 100129: baja levemente el AUC, pero gana $16,75 M.
- Semilla 900007: mejora el AUC, pero pierde $3,03 M.
Decisión: conservar provisionalmente las 47 features temporales. No podemos declararlas definitivas porque solo junio permite validarlas, pero la evidencia es suficientemente positiva.

In [153]:
# ============================================================
# EVENTOS TEMPORALES DE FECHAS
# ============================================================

columnas_fecha_necesarias = [
    "master_dias_hasta_vencimiento",
    "master_antiguedad_tarjeta_dias",
    "master_dias_desde_ultimo_cierre",
    "master_dias_mora",
    "master_mora_inicio_cero",
    "visa_dias_hasta_vencimiento",
    "visa_antiguedad_tarjeta_dias",
    "visa_dias_desde_ultimo_cierre",
    "visa_dias_mora",
    "visa_mora_inicio_cero",
]


# Incorporamos las fechas derivadas a la base con lags,
# pero no entrarán automáticamente al modelo.
base_lag1_fechas = (
    base_lag1
    .merge(
        base_fechas[
            [
                "numero_de_cliente",
                "foto_mes",
                *columnas_fecha_necesarias
            ]
        ],
        on=["numero_de_cliente", "foto_mes"],
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        ["numero_de_cliente", "foto_mes"]
    )
    .reset_index(drop=True)
)


historia_valida_fecha = (
    base_lag1_fechas["_tiene_historia_lag1"]
    == 1
)


# Fecha de referencia: primer día del mes siguiente.
fecha_referencia = (
    pd.to_datetime(
        base_lag1_fechas["foto_mes"].astype(str),
        format="%Y%m"
    )
    + pd.offsets.MonthBegin(1)
)

fecha_referencia_anterior = (
    fecha_referencia
    .groupby(
        base_lag1_fechas["numero_de_cliente"]
    )
    .shift(1)
)

dias_entre_fotos = (
    fecha_referencia
    - fecha_referencia_anterior
).dt.days.where(
    historia_valida_fecha
)


def lag1_consecutivo(variable):

    lag_crudo = (
        base_lag1_fechas
        .groupby(
            "numero_de_cliente",
            sort=False
        )[variable]
        .shift(1)
    )

    return lag_crudo.where(
        historia_valida_fecha
    )


def evento_fecha(
    condicion,
    aplicable
):
    """
    Conserva NaN cuando no hay historia consecutiva
    o cuando el evento no puede evaluarse.
    """

    mascara = (
        historia_valida_fecha
        & aplicable.fillna(False)
    )

    condicion_limpia = (
        condicion
        .fillna(False)
        .astype("int8")
    )

    return pd.Series(
        np.where(
            mascara,
            condicion_limpia,
            np.nan
        ),
        index=base_lag1_fechas.index
    )


nuevas_fechas_temporales = {}


for marca, columna_delinquency in [
    ("master", "Master_delinquency"),
    ("visa", "Visa_delinquency"),
]:

    dias_vencimiento = (
        base_lag1_fechas[
            f"{marca}_dias_hasta_vencimiento"
        ]
    )

    antiguedad_tarjeta = (
        base_lag1_fechas[
            f"{marca}_antiguedad_tarjeta_dias"
        ]
    )

    dias_desde_cierre = (
        base_lag1_fechas[
            f"{marca}_dias_desde_ultimo_cierre"
        ]
    )

    dias_mora = (
        base_lag1_fechas[
            f"{marca}_dias_mora"
        ]
    )

    mora_inicio_cero = (
        base_lag1_fechas[
            f"{marca}_mora_inicio_cero"
        ]
    )

    delinquency = (
        base_lag1_fechas[
            columna_delinquency
        ]
    )


    lag_dias_vencimiento = lag1_consecutivo(
        f"{marca}_dias_hasta_vencimiento"
    )

    lag_antiguedad = lag1_consecutivo(
        f"{marca}_antiguedad_tarjeta_dias"
    )

    lag_dias_cierre = lag1_consecutivo(
        f"{marca}_dias_desde_ultimo_cierre"
    )

    lag_dias_mora = lag1_consecutivo(
        f"{marca}_dias_mora"
    )

    lag_mora_inicio_cero = lag1_consecutivo(
        f"{marca}_mora_inicio_cero"
    )

    lag_delinquency = lag1_consecutivo(
        columna_delinquency
    )


    # Si el vencimiento real no cambió:
    # delta de días + días transcurridos = 0.
    cambio_fecha_vencimiento = (
        dias_vencimiento
        - lag_dias_vencimiento
        + dias_entre_fotos
    )

    # Si la fecha de alta no cambió:
    # delta de antigüedad - días transcurridos = 0.
    cambio_fecha_alta = (
        antiguedad_tarjeta
        - lag_antiguedad
        - dias_entre_fotos
    )


    nuevas_fechas_temporales[
        f"{marca}_dias_hasta_vencimiento_lag1"
    ] = lag_dias_vencimiento

    nuevas_fechas_temporales[
        f"{marca}_cambio_fecha_vencimiento_dias"
    ] = cambio_fecha_vencimiento

    nuevas_fechas_temporales[
        f"{marca}_renovo_plastico_lag1"
    ] = evento_fecha(
        cambio_fecha_vencimiento > 15,
        dias_vencimiento.notna()
        & lag_dias_vencimiento.notna()
    )

    nuevas_fechas_temporales[
        f"{marca}_cambio_fecha_alta_dias"
    ] = cambio_fecha_alta

    nuevas_fechas_temporales[
        f"{marca}_reemitio_tarjeta_lag1"
    ] = evento_fecha(
        cambio_fecha_alta < -15,
        antiguedad_tarjeta.notna()
        & lag_antiguedad.notna()
    )

    nuevas_fechas_temporales[
        f"{marca}_delta_dias_desde_ultimo_cierre"
    ] = (
        dias_desde_cierre
        - lag_dias_cierre
    )

    nuevas_fechas_temporales[
        f"{marca}_delinquency_lag1"
    ] = lag_delinquency

    nuevas_fechas_temporales[
        f"{marca}_entro_mora_lag1"
    ] = evento_fecha(
        (lag_delinquency == 0)
        & (delinquency == 1),
        lag_delinquency.notna()
        & delinquency.notna()
    )

    nuevas_fechas_temporales[
        f"{marca}_salio_mora_lag1"
    ] = evento_fecha(
        (lag_delinquency == 1)
        & (delinquency == 0),
        lag_delinquency.notna()
        & delinquency.notna()
    )

    nuevas_fechas_temporales[
        f"{marca}_delta_dias_mora"
    ] = (
        dias_mora
        - lag_dias_mora
    )

    nuevas_fechas_temporales[
        f"{marca}_nueva_mora_inicio_cero_lag1"
    ] = evento_fecha(
        (mora_inicio_cero == 1)
        & (lag_mora_inicio_cero != 1),
        mora_inicio_cero.notna()
        & lag_mora_inicio_cero.notna()
    )


base_lag1_fechas = pd.concat(
    [
        base_lag1_fechas,
        pd.DataFrame(
            nuevas_fechas_temporales,
            index=base_lag1_fechas.index
        )
    ],
    axis=1
)


features_nuevas_fechas_temporales = list(
    nuevas_fechas_temporales.keys()
)

features_lag1_fechas = (
    features_lag1
    + features_nuevas_fechas_temporales
)


base_lag1_fechas[features_lag1_fechas] = (
    base_lag1_fechas[
        features_lag1_fechas
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# AUDITORÍA
# ============================================================

mascara_train_junio = (
    base_lag1_fechas["foto_mes"]
    .isin([202103, 202104])
)

features_fecha_con_dato_train = sum(
    base_lag1_fechas.loc[
        mascara_train_junio,
        feature
    ].notna().any()
    for feature
    in features_nuevas_fechas_temporales
)


columnas_eventos_fecha = [
    "master_renovo_plastico_lag1",
    "visa_renovo_plastico_lag1",
    "master_reemitio_tarjeta_lag1",
    "visa_reemitio_tarjeta_lag1",
    "master_entro_mora_lag1",
    "visa_entro_mora_lag1",
    "master_salio_mora_lag1",
    "visa_salio_mora_lag1",
    "master_nueva_mora_inicio_cero_lag1",
    "visa_nueva_mora_inicio_cero_lag1",
]


resumen_eventos_fecha = (
    base_lag1_fechas
    .groupby("foto_mes")[
        columnas_eventos_fecha
    ]
    .sum()
    .astype("int64")
)


print(
    "Features del candidato lag1:",
    len(features_lag1)
)

print(
    "Features temporales de fechas nuevas:",
    len(features_nuevas_fechas_temporales)
)

print(
    "Features totales del nuevo candidato:",
    len(features_lag1_fechas)
)

print(
    "Features de fechas con datos en train de junio:",
    features_fecha_con_dato_train
)

display(
    pd.DataFrame({
        "feature_nueva":
            features_nuevas_fechas_temporales
    })
)

display(resumen_eventos_fecha)

Features del candidato lag1: 203
Features temporales de fechas nuevas: 22
Features totales del nuevo candidato: 225
Features de fechas con datos en train de junio: 22


,feature_nueva
0,master_dias_hasta_vencimiento_lag1
1,master_cambio_fecha_vencimiento_dias
2,master_renovo_plastico_lag1
3,master_cambio_fecha_alta_dias
4,master_reemitio_tarjeta_lag1
5,master_delta_dias_desde_ultimo_cierre
6,master_delinquency_lag1
7,master_entro_mora_lag1
8,master_salio_mora_lag1
9,master_delta_dias_mora


,master_renovo_plastico_lag1,visa_renovo_plastico_lag1,master_reemitio_tarjeta_lag1,visa_reemitio_tarjeta_lag1,master_entro_mora_lag1,visa_entro_mora_lag1,master_salio_mora_lag1,visa_salio_mora_lag1,master_nueva_mora_inicio_cero_lag1,visa_nueva_mora_inicio_cero_lag1
foto_mes,,,,,,,,,,
202103,0,0,0,0,0,0,0,0,0,0
202104,1050,744,2,7,567,813,542,630,0,0
202105,924,637,2,5,461,3734,383,504,0,3209
202106,996,743,1,3,312,381,326,3521,1,0


In [154]:
# ============================================================
# FUNCIÓN REUTILIZABLE PARA CANDIDATOS TEMPORALES EN JUNIO
# ============================================================

def evaluar_candidato_junio(
    datos,
    features,
    features_nuevas,
    nombre_experimento
):

    meses_train = [202103, 202104]
    mes_validacion = 202106

    mascara_train = (
        datos["foto_mes"].isin(meses_train)
    )

    mascara_valid = (
        datos["foto_mes"] == mes_validacion
    )


    X_train = datos.loc[
        mascara_train,
        features
    ]

    y_train = datos.loc[
        mascara_train,
        "target_baja2"
    ]

    X_valid = datos.loc[
        mascara_valid,
        features
    ]

    y_valid = datos.loc[
        mascara_valid,
        "target_baja2"
    ]

    clases_valid = datos.loc[
        mascara_valid,
        "clase_ternaria"
    ]


    columnas_no_numericas = (
        X_train
        .select_dtypes(exclude="number")
        .columns
        .tolist()
    )

    assert columnas_no_numericas == [], (
        f"Columnas no numéricas: "
        f"{columnas_no_numericas}"
    )

    assert (
        indice_mes(mes_validacion)
        - indice_mes(max(meses_train))
        >= 2
    ), "Fuga temporal detectada"


    ganancia_fila_valid = ganancia_por_fila(
        clases_valid
    )

    resultados = []
    importancias = []


    for semilla_actual in semillas:

        params_semilla = {
            "objective": "binary",
            "metric": "None",
            "boosting_type": "gbdt",
            "boost_from_average": True,
            "feature_pre_filter": False,
            "first_metric_only": True,
            "verbosity": -1,
            "bagging_freq": 1,

            **study_base_v3.best_params,

            "seed": semilla_actual,
            "feature_fraction_seed":
                semilla_actual,
            "bagging_seed":
                semilla_actual,
            "data_random_seed":
                semilla_actual,
        }


        dataset_train = lgb.Dataset(
            X_train,
            label=y_train
        )

        dataset_valid = lgb.Dataset(
            X_valid,
            label=y_valid,
            reference=dataset_train
        )


        modelo = lgb.train(
            params=params_semilla,
            train_set=dataset_train,
            num_boost_round=3000,
            valid_sets=[dataset_valid],
            valid_names=["validacion"],
            feval=feval_ganancia(
                ganancia_fila_valid
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )


        probabilidades = modelo.predict(
            X_valid,
            num_iteration=modelo.best_iteration
        )


        ganancia_actual = ganancia(
            probabilidades,
            ganancia_fila_valid
        )

        auc_actual = roc_auc_score(
            y_valid,
            probabilidades
        )


        resultados.append({
            "experimento": nombre_experimento,
            "semilla": semilla_actual,
            "validacion": mes_validacion,
            "ganancia": float(
                ganancia_actual
            ),
            "auc": float(auc_actual),
            "iteraciones": int(
                modelo.best_iteration
            ),
        })


        importancia_gain = (
            modelo.feature_importance(
                importance_type="gain"
            )
        )

        importancia_total = (
            importancia_gain.sum()
        )

        for feature, importancia in zip(
            features,
            importancia_gain
        ):

            if feature in features_nuevas:

                importancias.append({
                    "semilla":
                        semilla_actual,
                    "feature":
                        feature,
                    "importancia_gain":
                        importancia,
                    "importancia_pct": (
                        100.0
                        * importancia
                        / importancia_total
                        if importancia_total > 0
                        else 0.0
                    ),
                })


        print(
            f"{nombre_experimento} | "
            f"Semilla {semilla_actual} | "
            f"Ganancia "
            f"{ganancia_actual:,.0f} | "
            f"AUC {auc_actual:.4f} | "
            f"Iteraciones "
            f"{modelo.best_iteration}"
        )


    return (
        pd.DataFrame(resultados),
        pd.DataFrame(importancias)
    )


# ============================================================
# EVALUACIÓN: LAG1 + EVENTOS DE FECHAS
# ============================================================

(
    resultados_lag1_fechas_junio,
    importancias_lag1_fechas
) = evaluar_candidato_junio(
    datos=base_lag1_fechas,
    features=features_lag1_fechas,
    features_nuevas=(
        features_nuevas_fechas_temporales
    ),
    nombre_experimento="LAG1+EVENTOS_FECHAS"
)


# Comparación contra el candidato lag1 sin fechas.
referencia_lag1 = (
    resultados_lag1_junio[
        [
            "semilla",
            "validacion",
            "ganancia",
            "auc"
        ]
    ]
    .rename(columns={
        "ganancia": "ganancia_lag1",
        "auc": "auc_lag1",
    })
)


comparacion_lag1_fechas = (
    resultados_lag1_fechas_junio
    .merge(
        referencia_lag1,
        on=["semilla", "validacion"],
        how="inner"
    )
)


comparacion_lag1_fechas[
    "delta_ganancia"
] = (
    comparacion_lag1_fechas["ganancia"]
    - comparacion_lag1_fechas[
        "ganancia_lag1"
    ]
)

comparacion_lag1_fechas[
    "delta_auc"
] = (
    comparacion_lag1_fechas["auc"]
    - comparacion_lag1_fechas["auc_lag1"]
)


resumen_lag1_fechas = pd.DataFrame({
    "ganancia_lag1": [
        comparacion_lag1_fechas[
            "ganancia_lag1"
        ].mean()
    ],
    "ganancia_con_fechas": [
        comparacion_lag1_fechas[
            "ganancia"
        ].mean()
    ],
    "mejora_media": [
        comparacion_lag1_fechas[
            "delta_ganancia"
        ].mean()
    ],
    "mejora_minima": [
        comparacion_lag1_fechas[
            "delta_ganancia"
        ].min()
    ],
    "mejora_maxima": [
        comparacion_lag1_fechas[
            "delta_ganancia"
        ].max()
    ],
    "cambio_auc_medio": [
        comparacion_lag1_fechas[
            "delta_auc"
        ].mean()
    ],
    "semillas_favorables": [
        int(
            (
                comparacion_lag1_fechas[
                    "delta_ganancia"
                ] > 0
            ).sum()
        )
    ],
})


resumen_lag1_fechas["mejora_pct"] = (
    100.0
    * resumen_lag1_fechas[
        "mejora_media"
    ]
    / resumen_lag1_fechas[
        "ganancia_lag1"
    ]
)


resumen_importancias_fechas = (
    importancias_lag1_fechas
    .groupby("feature")
    .agg(
        importancia_media_pct=(
            "importancia_pct",
            "mean"
        ),
        semillas_en_que_se_uso=(
            "importancia_gain",
            lambda x: int((x > 0).sum())
        )
    )
    .sort_values(
        "importancia_media_pct",
        ascending=False
    )
)


display(
    comparacion_lag1_fechas[
        [
            "semilla",
            "ganancia_lag1",
            "ganancia",
            "delta_ganancia",
            "auc_lag1",
            "auc",
            "delta_auc",
        ]
    ]
)

display(
    resumen_lag1_fechas.round(4)
)

display(
    resumen_importancias_fechas.head(15)
)


resultados_lag1_fechas_junio.to_csv(
    "resultados_lag1_eventos_fechas_junio.csv",
    index=False
)

LAG1+EVENTOS_FECHAS | Semilla 100129 | Ganancia 359,067,500 | AUC 0.9017 | Iteraciones 455
LAG1+EVENTOS_FECHAS | Semilla 100183 | Ganancia 366,657,500 | AUC 0.8996 | Iteraciones 169
LAG1+EVENTOS_FECHAS | Semilla 100151 | Ganancia 356,455,000 | AUC 0.9007 | Iteraciones 367
LAG1+EVENTOS_FECHAS | Semilla 500041 | Ganancia 380,462,500 | AUC 0.9020 | Iteraciones 129
LAG1+EVENTOS_FECHAS | Semilla 900007 | Ganancia 377,190,000 | AUC 0.9018 | Iteraciones 240


,semilla,ganancia_lag1,ganancia,delta_ganancia,auc_lag1,auc,delta_auc
0,100129,370095000.0,359067500.0,-11027500.0,0.900140,0.901708,0.001568
1,100183,369737500.0,366657500.0,-3080000.0,0.901504,0.899600,-0.001904
2,100151,365612500.0,356455000.0,-9157500.0,0.899627,0.900705,0.001078
3,500041,352880000.0,380462500.0,27582500.0,0.899846,0.902024,0.002179
4,900007,353567500.0,377190000.0,23622500.0,0.901050,0.901801,0.000751


,ganancia_lag1,ganancia_con_fechas,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables,mejora_pct
0,362378500.0,367966500.0,5588000.0,-11027500.0,27582500.0,0.0007,2,1.542


,importancia_media_pct,semillas_en_que_se_uso
feature,,
master_dias_hasta_vencimiento_lag1,0.502057,5
visa_dias_hasta_vencimiento_lag1,0.297678,5
visa_cambio_fecha_vencimiento_dias,0.072513,1
master_delta_dias_desde_ultimo_cierre,0.004791,4
visa_delta_dias_desde_ultimo_cierre,0.004059,4
master_cambio_fecha_alta_dias,0.003350,1
master_delinquency_lag1,0.002408,1
visa_entro_mora_lag1,0.001675,2
visa_delinquency_lag1,0.001502,2


El promedio mejora, pero el resultado no es robusto:
- +$5,59 M promedio.
- Solo 2 de 5 semillas mejoran.
- Tres semillas pierden entre $3,08 M y $11,03 M.
- La mejora está impulsada por dos resultados excepcionalmente altos.
- Casi toda la señal nueva proviene de solo dos variables de vencimiento, utilizadas en las cinco semillas.
- La mayoría de los eventos de mora, renovación y reemisión no fue utilizada.

In [155]:
# ============================================================
# FECHAS REDUCIDAS: SOLO LAS DOS VARIABLES ESTABLES
# ============================================================

features_fechas_reducidas = [
    "master_dias_hasta_vencimiento_lag1",
    "visa_dias_hasta_vencimiento_lag1",
]

features_lag1_fechas_reducidas = (
    features_lag1
    + features_fechas_reducidas
)


(
    resultados_fechas_reducidas_junio,
    importancias_fechas_reducidas
) = evaluar_candidato_junio(
    datos=base_lag1_fechas,
    features=features_lag1_fechas_reducidas,
    features_nuevas=features_fechas_reducidas,
    nombre_experimento="LAG1+VENCIMIENTO_LAG1"
)


comparacion_fechas_reducidas = (
    resultados_fechas_reducidas_junio
    .merge(
        referencia_lag1,
        on=["semilla", "validacion"],
        how="inner"
    )
)


comparacion_fechas_reducidas[
    "delta_ganancia"
] = (
    comparacion_fechas_reducidas["ganancia"]
    - comparacion_fechas_reducidas[
        "ganancia_lag1"
    ]
)

comparacion_fechas_reducidas[
    "delta_auc"
] = (
    comparacion_fechas_reducidas["auc"]
    - comparacion_fechas_reducidas["auc_lag1"]
)


resumen_fechas_reducidas = pd.DataFrame({
    "ganancia_lag1": [
        comparacion_fechas_reducidas[
            "ganancia_lag1"
        ].mean()
    ],
    "ganancia_con_vencimiento_lag1": [
        comparacion_fechas_reducidas[
            "ganancia"
        ].mean()
    ],
    "mejora_media": [
        comparacion_fechas_reducidas[
            "delta_ganancia"
        ].mean()
    ],
    "mejora_minima": [
        comparacion_fechas_reducidas[
            "delta_ganancia"
        ].min()
    ],
    "mejora_maxima": [
        comparacion_fechas_reducidas[
            "delta_ganancia"
        ].max()
    ],
    "cambio_auc_medio": [
        comparacion_fechas_reducidas[
            "delta_auc"
        ].mean()
    ],
    "semillas_favorables": [
        int(
            (
                comparacion_fechas_reducidas[
                    "delta_ganancia"
                ] > 0
            ).sum()
        )
    ],
})


resumen_fechas_reducidas["mejora_pct"] = (
    100.0
    * resumen_fechas_reducidas[
        "mejora_media"
    ]
    / resumen_fechas_reducidas[
        "ganancia_lag1"
    ]
)


display(
    comparacion_fechas_reducidas[
        [
            "semilla",
            "ganancia_lag1",
            "ganancia",
            "delta_ganancia",
            "auc_lag1",
            "auc",
            "delta_auc",
        ]
    ]
)

display(
    resumen_fechas_reducidas.round(4)
)


resultados_fechas_reducidas_junio.to_csv(
    "resultados_vencimiento_lag1_reducido.csv",
    index=False
)

LAG1+VENCIMIENTO_LAG1 | Semilla 100129 | Ganancia 371,277,500 | AUC 0.9023 | Iteraciones 317
LAG1+VENCIMIENTO_LAG1 | Semilla 100183 | Ganancia 352,440,000 | AUC 0.8998 | Iteraciones 594
LAG1+VENCIMIENTO_LAG1 | Semilla 100151 | Ganancia 364,155,000 | AUC 0.9023 | Iteraciones 335
LAG1+VENCIMIENTO_LAG1 | Semilla 500041 | Ganancia 360,442,500 | AUC 0.9025 | Iteraciones 449
LAG1+VENCIMIENTO_LAG1 | Semilla 900007 | Ganancia 377,190,000 | AUC 0.9007 | Iteraciones 155


,semilla,ganancia_lag1,ganancia,delta_ganancia,auc_lag1,auc,delta_auc
0,100129,370095000.0,371277500.0,1182500.0,0.900140,0.902300,0.002160
1,100183,369737500.0,352440000.0,-17297500.0,0.901504,0.899836,-0.001669
2,100151,365612500.0,364155000.0,-1457500.0,0.899627,0.902289,0.002662
3,500041,352880000.0,360442500.0,7562500.0,0.899846,0.902453,0.002607
4,900007,353567500.0,377190000.0,23622500.0,0.901050,0.900742,-0.000308


,ganancia_lag1,ganancia_con_vencimiento_lag1,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables,mejora_pct
0,362378500.0,365101000.0,2722500.0,-17297500.0,23622500.0,0.0011,3,0.7513


In [156]:
# ============================================================
# COMISIONES Y ENDEUDAMIENTO SOBRE PAYROLL
# ============================================================

base_carga_financiera = (
    base_lag1
    .copy()
    .sort_values(
        ["numero_de_cliente", "foto_mes"]
    )
    .reset_index(drop=True)
)

historia_valida_carga = (
    base_carga_financiera[
        "_tiene_historia_lag1"
    ] == 1
)


def suma_sin_null_series(series):

    tabla = pd.concat(
        series,
        axis=1
    )

    resultado = (
        tabla
        .fillna(0)
        .sum(axis=1)
    )

    return resultado.mask(
        tabla.isna().all(axis=1)
    )


def lag1_serie(serie):

    return (
        serie
        .groupby(
            base_carga_financiera[
                "numero_de_cliente"
            ],
            sort=False
        )
        .shift(1)
        .where(historia_valida_carga)
    )


def ratio_seguro(
    numerador,
    denominador
):
    return (
        numerador
        / denominador.where(
            denominador > 0
        )
    )


# ------------------------------------------------------------
# Denominadores de ingreso
# ------------------------------------------------------------

payroll_actual = (
    base_carga_financiera["mpayroll"]
)

payroll_lag1 = lag1_serie(
    payroll_actual
)

payroll_positivo_actual = (
    payroll_actual.where(
        payroll_actual > 0
    )
)

payroll_positivo_lag1 = (
    payroll_lag1.where(
        payroll_lag1 > 0
    )
)


# Promedio de los payroll positivos disponibles.
# Reduce el efecto de un único mes excepcional.
tabla_payroll_referencia = pd.concat(
    [
        payroll_positivo_actual,
        payroll_positivo_lag1
    ],
    axis=1
)

payroll_referencia_2m = (
    tabla_payroll_referencia
    .mean(axis=1, skipna=True)
    .mask(
        tabla_payroll_referencia
        .isna()
        .all(axis=1)
    )
)


# ------------------------------------------------------------
# Numeradores de carga financiera
# ------------------------------------------------------------

saldos_tarjeta_positivos = (
    base_carga_financiera[
        [
            "Visa_msaldopesos",
            "Master_msaldopesos"
        ]
    ]
    .clip(lower=0)
)

deuda_tarjetas_pesos = (
    suma_sin_null_series([
        saldos_tarjeta_positivos[
            "Visa_msaldopesos"
        ],
        saldos_tarjeta_positivos[
            "Master_msaldopesos"
        ],
    ])
)


pago_minimo_tarjetas = (
    suma_sin_null_series([
        base_carga_financiera[
            "Visa_mpagominimo"
        ],
        base_carga_financiera[
            "Master_mpagominimo"
        ],
    ])
)


prestamos_totales = (
    suma_sin_null_series([
        base_carga_financiera[
            "mprestamos_personales"
        ],
        base_carga_financiera[
            "mprestamos_prendarios"
        ],
        base_carga_financiera[
            "mprestamos_hipotecarios"
        ],
    ])
)


descubierto_utilizado = (
    -base_carga_financiera[
        "mcuenta_corriente"
    ]
).clip(lower=0)


deuda_total_pesos = (
    suma_sin_null_series([
        deuda_tarjetas_pesos,
        prestamos_totales,
        descubierto_utilizado,
    ])
)


numeradores_carga = {
    "comisiones":
        base_carga_financiera[
            "mcomisiones"
        ],

    "comisiones_mantenimiento":
        base_carga_financiera[
            "mcomisiones_mantenimiento"
        ],

    "deuda_tarjetas_pesos":
        deuda_tarjetas_pesos,

    "pago_minimo_tarjetas":
        pago_minimo_tarjetas,

    "prestamos_totales":
        prestamos_totales,

    "descubierto_utilizado":
        descubierto_utilizado,

    "deuda_total_pesos":
        deuda_total_pesos,

    "saldo_cuentas":
        base_carga_financiera[
            "mcuentas_saldo"
        ],
}


# ------------------------------------------------------------
# Features construidas
# ------------------------------------------------------------

nuevas_carga_financiera = {
    "payroll_referencia_2m":
        payroll_referencia_2m,

    "sin_payroll_monto_actual":
        (
            payroll_actual
            .fillna(0)
            .le(0)
            .astype("int8")
        ),

    "payroll_cambio_relativo_lag1":
        (
            payroll_actual
            - payroll_lag1
        )
        / payroll_lag1.where(
            payroll_lag1 > 0
        ),
}


for nombre, numerador_actual in (
    numeradores_carga.items()
):

    numerador_lag1 = lag1_serie(
        numerador_actual
    )

    ratio_actual = ratio_seguro(
        numerador_actual,
        payroll_actual
    )

    ratio_referencia = ratio_seguro(
        numerador_actual,
        payroll_referencia_2m
    )

    ratio_lag1 = ratio_seguro(
        numerador_lag1,
        payroll_lag1
    )

    delta_ratio = (
        ratio_actual
        - ratio_lag1
    )


    nuevas_carga_financiera[
        f"{nombre}_sobre_payroll_actual"
    ] = ratio_actual

    nuevas_carga_financiera[
        f"{nombre}_sobre_payroll_referencia_2m"
    ] = ratio_referencia

    nuevas_carga_financiera[
        f"{nombre}_sobre_payroll_lag1"
    ] = ratio_lag1

    nuevas_carga_financiera[
        f"{nombre}_sobre_payroll_delta1"
    ] = delta_ratio


base_carga_financiera = pd.concat(
    [
        base_carga_financiera,
        pd.DataFrame(
            nuevas_carga_financiera,
            index=base_carga_financiera.index
        )
    ],
    axis=1
)


features_nuevas_carga = list(
    nuevas_carga_financiera.keys()
)

features_lag1_carga = (
    features_lag1
    + features_nuevas_carga
)


base_carga_financiera[
    features_lag1_carga
] = (
    base_carga_financiera[
        features_lag1_carga
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# AUDITORÍA
# ============================================================

filas_resumen = []

for foto_mes, grupo in (
    base_carga_financiera
    .groupby("foto_mes")
):

    payroll_grupo = grupo["mpayroll"]
    referencia_grupo = grupo[
        "payroll_referencia_2m"
    ]

    filas_resumen.append({
        "foto_mes": foto_mes,
        "clientes": len(grupo),

        "pct_payroll_actual_positivo":
            round(
                100.0
                * (payroll_grupo > 0).mean(),
                2
            ),

        "pct_payroll_referencia_disponible":
            round(
                100.0
                * referencia_grupo.notna().mean(),
                2
            ),

        "mediana_payroll_positivo":
            round(
                payroll_grupo[
                    payroll_grupo > 0
                ].median(),
                2
            ),

        "mediana_payroll_referencia":
            round(
                referencia_grupo.median(),
                2
            ),

        "mediana_comisiones_sobre_payroll":
            round(
                grupo[
                    "comisiones_sobre_payroll_referencia_2m"
                ].median(),
                4
            ),

        "p99_deuda_tarjetas_sobre_payroll":
            round(
                grupo[
                    "deuda_tarjetas_pesos_sobre_payroll_referencia_2m"
                ].quantile(0.99),
                4
            ),

        "p99_deuda_total_sobre_payroll":
            round(
                grupo[
                    "deuda_total_pesos_sobre_payroll_referencia_2m"
                ].quantile(0.99),
                4
            ),
    })


resumen_carga_financiera = (
    pd.DataFrame(filas_resumen)
    .set_index("foto_mes")
)


cantidad_infinitos = int(
    np.isinf(
        base_carga_financiera[
            features_nuevas_carga
        ].to_numpy(dtype="float64")
    ).sum()
)


print("Features del candidato lag1:", len(features_lag1))
print("Features nuevas de carga:", len(features_nuevas_carga))
print("Features totales:", len(features_lag1_carga))
print("Valores infinitos:", cantidad_infinitos)

display(
    pd.DataFrame({
        "feature_nueva":
            features_nuevas_carga
    })
)

display(resumen_carga_financiera)

Features del candidato lag1: 203
Features nuevas de carga: 35
Features totales: 238
Valores infinitos: 0


,feature_nueva
0,payroll_referencia_2m
1,sin_payroll_monto_actual
2,payroll_cambio_relativo_lag1
3,comisiones_sobre_payroll_actual
4,comisiones_sobre_payroll_referencia_2m
5,comisiones_sobre_payroll_lag1
6,comisiones_sobre_payroll_delta1
7,comisiones_mantenimiento_sobre_payroll_actual
8,comisiones_mantenimiento_sobre_payroll_referen...
9,comisiones_mantenimiento_sobre_payroll_lag1


,clientes,pct_payroll_actual_positivo,pct_payroll_referencia_disponible,mediana_payroll_positivo,mediana_payroll_referencia,mediana_comisiones_sobre_payroll,p99_deuda_tarjetas_sobre_payroll,p99_deuda_total_sobre_payroll
foto_mes,,,,,,,,
202103,162900,53.45,53.45,98080.32,98080.32,0.0052,7.4342,21.0705
202104,163284,53.51,55.11,101399.98,100091.20,0.0056,7.3361,19.5085
202105,163768,54.38,55.50,105773.49,103991.15,0.0054,7.2205,19.5330
202106,164114,54.69,56.01,156529.81,130983.34,0.0045,8.2014,19.5464


In [157]:
# ============================================================
# EVALUACIÓN: LAG1 + CARGA FINANCIERA
# ============================================================

(
    resultados_carga_junio,
    importancias_carga_junio
) = evaluar_candidato_junio(
    datos=base_carga_financiera,
    features=features_lag1_carga,
    features_nuevas=features_nuevas_carga,
    nombre_experimento="LAG1+CARGA_FINANCIERA"
)


comparacion_carga = (
    resultados_carga_junio
    .merge(
        referencia_lag1,
        on=["semilla", "validacion"],
        how="inner"
    )
)


comparacion_carga["delta_ganancia"] = (
    comparacion_carga["ganancia"]
    - comparacion_carga["ganancia_lag1"]
)

comparacion_carga["delta_auc"] = (
    comparacion_carga["auc"]
    - comparacion_carga["auc_lag1"]
)


resumen_carga = pd.DataFrame({
    "ganancia_lag1": [
        comparacion_carga[
            "ganancia_lag1"
        ].mean()
    ],
    "ganancia_con_carga": [
        comparacion_carga[
            "ganancia"
        ].mean()
    ],
    "mejora_media": [
        comparacion_carga[
            "delta_ganancia"
        ].mean()
    ],
    "mejora_minima": [
        comparacion_carga[
            "delta_ganancia"
        ].min()
    ],
    "mejora_maxima": [
        comparacion_carga[
            "delta_ganancia"
        ].max()
    ],
    "cambio_auc_medio": [
        comparacion_carga[
            "delta_auc"
        ].mean()
    ],
    "semillas_favorables": [
        int(
            (
                comparacion_carga[
                    "delta_ganancia"
                ] > 0
            ).sum()
        )
    ],
})


resumen_carga["mejora_pct"] = (
    100.0
    * resumen_carga["mejora_media"]
    / resumen_carga["ganancia_lag1"]
)


resumen_importancias_carga = (
    importancias_carga_junio
    .groupby("feature")
    .agg(
        importancia_media_pct=(
            "importancia_pct",
            "mean"
        ),
        semillas_en_que_se_uso=(
            "importancia_gain",
            lambda x: int((x > 0).sum())
        )
    )
    .sort_values(
        "importancia_media_pct",
        ascending=False
    )
)


print("Comparación por semilla:")

display(
    comparacion_carga[
        [
            "semilla",
            "ganancia_lag1",
            "ganancia",
            "delta_ganancia",
            "auc_lag1",
            "auc",
            "delta_auc",
        ]
    ]
)


print("Resumen:")

display(
    resumen_carga.round(4)
)


print("Principales features de carga financiera:")

display(
    resumen_importancias_carga.head(20)
)


resultados_carga_junio.to_csv(
    "resultados_carga_financiera_junio.csv",
    index=False
)

importancias_carga_junio.to_csv(
    "importancias_carga_financiera_junio.csv",
    index=False
)

LAG1+CARGA_FINANCIERA | Semilla 100129 | Ganancia 373,835,000 | AUC 0.9010 | Iteraciones 168
LAG1+CARGA_FINANCIERA | Semilla 100183 | Ganancia 384,917,500 | AUC 0.8992 | Iteraciones 129
LAG1+CARGA_FINANCIERA | Semilla 100151 | Ganancia 368,637,500 | AUC 0.9004 | Iteraciones 430
LAG1+CARGA_FINANCIERA | Semilla 500041 | Ganancia 385,632,500 | AUC 0.9019 | Iteraciones 227
LAG1+CARGA_FINANCIERA | Semilla 900007 | Ganancia 381,645,000 | AUC 0.9016 | Iteraciones 457
Comparación por semilla:


,semilla,ganancia_lag1,ganancia,delta_ganancia,auc_lag1,auc,delta_auc
0,100129,370095000.0,373835000.0,3740000.0,0.900140,0.901015,0.000875
1,100183,369737500.0,384917500.0,15180000.0,0.901504,0.899247,-0.002258
2,100151,365612500.0,368637500.0,3025000.0,0.899627,0.900391,0.000764
3,500041,352880000.0,385632500.0,32752500.0,0.899846,0.901926,0.002080
4,900007,353567500.0,381645000.0,28077500.0,0.901050,0.901598,0.000548


Resumen:


,ganancia_lag1,ganancia_con_carga,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables,mejora_pct
0,362378500.0,378933500.0,16555000.0,3025000.0,32752500.0,0.0004,5,4.5684


Principales features de carga financiera:


,importancia_media_pct,semillas_en_que_se_uso
feature,,
comisiones_mantenimiento_sobre_payroll_actual,2.888110,5
comisiones_mantenimiento_sobre_payroll_referencia_2m,1.975880,5
comisiones_sobre_payroll_actual,1.077362,5
pago_minimo_tarjetas_sobre_payroll_actual,0.847151,5
payroll_referencia_2m,0.685145,5
deuda_tarjetas_pesos_sobre_payroll_actual,0.551914,5
comisiones_sobre_payroll_referencia_2m,0.452855,5
descubierto_utilizado_sobre_payroll_actual,0.405750,5
saldo_cuentas_sobre_payroll_actual,0.358004,5


Candidato: ANCLAJE + LAG1 + CARGA FINANCIERA
Features: 238
Ganancia media en junio: 378.933.500

In [158]:
# ============================================================
# FLUJO DE FONDOS Y COMPORTAMIENTO TRANSACCIONAL
# ============================================================

base_flujo = (
    base_carga_financiera
    .copy()
    .sort_values(
        ["numero_de_cliente", "foto_mes"]
    )
    .reset_index(drop=True)
)

historia_valida_flujo = (
    base_flujo["_tiene_historia_lag1"]
    == 1
)


def lag1_flujo(serie):

    return (
        serie
        .groupby(
            base_flujo["numero_de_cliente"],
            sort=False
        )
        .shift(1)
        .where(historia_valida_flujo)
    )


# ------------------------------------------------------------
# Entradas y salidas de dinero
# ------------------------------------------------------------

ingresos_fondos = suma_sin_null_series([
    base_flujo["mpayroll"],
    base_flujo["mtransferencias_recibidas"],
    base_flujo["mcheques_depositados"],
])


egresos_fondos = suma_sin_null_series([
    base_flujo["mtransferencias_emitidas"],
    base_flujo["mextraccion_autoservicio"],
    base_flujo["matm"],
    base_flujo["matm_other"],
    base_flujo["mcheques_emitidos"],
    base_flujo["mpagomiscuentas"],
    base_flujo["mpagodeservicios"],
])


flujo_neto = (
    ingresos_fondos
    - egresos_fondos
)


# ------------------------------------------------------------
# Composición de canales
# ------------------------------------------------------------

transacciones_digitales = (
    base_flujo["chomebanking_transacciones"]
    + base_flujo["cmobile_app_trx"]
)

transacciones_atm = (
    base_flujo["catm_trx"]
    + base_flujo["catm_trx_other"]
)

transacciones_canales_total = (
    transacciones_digitales
    + transacciones_atm
    + base_flujo["ccajas_transacciones"]
    + base_flujo["ccallcenter_transacciones"]
)

monto_atm_total = (
    base_flujo["matm"]
    + base_flujo["matm_other"]
)


# ------------------------------------------------------------
# Indicadores actuales
# ------------------------------------------------------------

indicadores_flujo_actuales = {

    "ingresos_fondos":
        ingresos_fondos,

    "egresos_fondos":
        egresos_fondos,

    "flujo_neto":
        flujo_neto,

    "egresos_sobre_ingresos":
        ratio_seguro(
            egresos_fondos,
            ingresos_fondos
        ),

    "transferencias_emitidas_sobre_ingresos":
        ratio_seguro(
            base_flujo[
                "mtransferencias_emitidas"
            ],
            ingresos_fondos
        ),

    "velocidad_vaciado_sobre_payroll":
        ratio_seguro(
            egresos_fondos,
            base_flujo[
                "payroll_referencia_2m"
            ]
        ),

    "saldo_cuentas_sobre_ingresos":
        ratio_seguro(
            base_flujo["mcuentas_saldo"],
            ingresos_fondos
        ),

    "uso_atm_ajeno_monto":
        ratio_seguro(
            base_flujo["matm_other"],
            monto_atm_total
        ),

    "uso_atm_ajeno_cantidad":
        ratio_seguro(
            base_flujo["catm_trx_other"],
            transacciones_atm
        ),

    "share_digital_transacciones":
        ratio_seguro(
            transacciones_digitales,
            transacciones_canales_total
        ),

    "share_sucursal_transacciones":
        ratio_seguro(
            base_flujo[
                "ccajas_transacciones"
            ],
            transacciones_canales_total
        ),

    "desaceleracion_rentabilidad":
        (
            base_flujo["mrentabilidad"]
            - base_flujo[
                "mrentabilidad_annual"
            ] / 12.0
        ),

    "rentabilidad_sobre_ingresos":
        ratio_seguro(
            base_flujo["mrentabilidad"],
            ingresos_fondos
        ),

    "comisiones_sobre_rentabilidad":
        ratio_seguro(
            base_flujo["mcomisiones"],
            base_flujo["mrentabilidad"]
        ),
}


# ------------------------------------------------------------
# Nivel, lag1 y delta1
# ------------------------------------------------------------

nuevas_flujo = {}

for nombre, valor_actual in (
    indicadores_flujo_actuales.items()
):

    valor_lag1 = lag1_flujo(
        valor_actual
    )

    nuevas_flujo[nombre] = (
        valor_actual
    )

    nuevas_flujo[
        f"{nombre}_lag1"
    ] = valor_lag1

    nuevas_flujo[
        f"{nombre}_delta1"
    ] = (
        valor_actual
        - valor_lag1
    )


base_flujo = pd.concat(
    [
        base_flujo,
        pd.DataFrame(
            nuevas_flujo,
            index=base_flujo.index
        )
    ],
    axis=1
)


features_nuevas_flujo = list(
    nuevas_flujo.keys()
)

features_candidato_flujo = (
    features_lag1_carga
    + features_nuevas_flujo
)


base_flujo[
    features_candidato_flujo
] = (
    base_flujo[
        features_candidato_flujo
    ]
    .replace([np.inf, -np.inf], np.nan)
)


# ============================================================
# AUDITORÍA
# ============================================================

columnas_share = [
    "uso_atm_ajeno_monto",
    "uso_atm_ajeno_cantidad",
    "share_digital_transacciones",
    "share_sucursal_transacciones",
]

valores_fuera_de_rango = int(
    sum(
        (
            (base_flujo[columna] < 0)
            | (base_flujo[columna] > 1)
        ).sum()
        for columna in columnas_share
    )
)

cantidad_infinitos = int(
    np.isinf(
        base_flujo[
            features_nuevas_flujo
        ].to_numpy(dtype="float64")
    ).sum()
)


filas_resumen_flujo = []

for foto_mes, grupo in (
    base_flujo.groupby("foto_mes")
):

    filas_resumen_flujo.append({
        "foto_mes": foto_mes,
        "clientes": len(grupo),

        "mediana_ingresos":
            round(
                grupo[
                    "ingresos_fondos"
                ].median(),
                2
            ),

        "mediana_egresos":
            round(
                grupo[
                    "egresos_fondos"
                ].median(),
                2
            ),

        "mediana_flujo_neto":
            round(
                grupo[
                    "flujo_neto"
                ].median(),
                2
            ),

        "pct_flujo_neto_negativo":
            round(
                100.0
                * (
                    grupo["flujo_neto"] < 0
                ).mean(),
                2
            ),

        "mediana_egresos_sobre_ingresos":
            round(
                grupo[
                    "egresos_sobre_ingresos"
                ].median(),
                4
            ),

        "p99_egresos_sobre_ingresos":
            round(
                grupo[
                    "egresos_sobre_ingresos"
                ].quantile(0.99),
                4
            ),

        "mediana_uso_atm_ajeno":
            round(
                grupo[
                    "uso_atm_ajeno_cantidad"
                ].median(),
                4
            ),

        "mediana_share_digital":
            round(
                grupo[
                    "share_digital_transacciones"
                ].median(),
                4
            ),
    })


resumen_flujo = (
    pd.DataFrame(filas_resumen_flujo)
    .set_index("foto_mes")
)


print(
    "Features del candidato con carga:",
    len(features_lag1_carga)
)

print(
    "Indicadores actuales de flujo:",
    len(indicadores_flujo_actuales)
)

print(
    "Features nuevas nivel/lag/delta:",
    len(features_nuevas_flujo)
)

print(
    "Features totales:",
    len(features_candidato_flujo)
)

print(
    "Valores infinitos:",
    cantidad_infinitos
)

print(
    "Shares fuera de [0, 1]:",
    valores_fuera_de_rango
)

display(
    pd.DataFrame({
        "feature_nueva":
            features_nuevas_flujo
    })
)

display(resumen_flujo)

Features del candidato con carga: 238
Indicadores actuales de flujo: 14
Features nuevas nivel/lag/delta: 42
Features totales: 280
Valores infinitos: 0
Shares fuera de [0, 1]: 0


,feature_nueva
0,ingresos_fondos
1,ingresos_fondos_lag1
2,ingresos_fondos_delta1
3,egresos_fondos
4,egresos_fondos_lag1
5,egresos_fondos_delta1
6,flujo_neto
7,flujo_neto_lag1
8,flujo_neto_delta1
9,egresos_sobre_ingresos


,clientes,mediana_ingresos,mediana_egresos,mediana_flujo_neto,pct_flujo_neto_negativo,mediana_egresos_sobre_ingresos,p99_egresos_sobre_ingresos,mediana_uso_atm_ajeno,mediana_share_digital
foto_mes,,,,,,,,,
202103,162900,110079.00,57975.56,29325.00,25.21,0.4989,20.0000,0.0,0.9730
202104,163284,107195.00,55463.22,27827.30,25.68,0.4994,22.5527,0.0,0.9744
202105,163768,111171.02,55074.98,29325.00,25.31,0.4898,22.1484,0.0,0.9785
202106,164114,156526.88,61372.90,59003.36,19.64,0.3918,17.3293,0.0,0.9762


In [159]:
# ============================================================
# EVALUACIÓN: CARGA FINANCIERA + FLUJO DE FONDOS
# ============================================================

(
    resultados_flujo_junio,
    importancias_flujo_junio
) = evaluar_candidato_junio(
    datos=base_flujo,
    features=features_candidato_flujo,
    features_nuevas=features_nuevas_flujo,
    nombre_experimento="CARGA+FLUJO"
)


referencia_carga = (
    resultados_carga_junio[
        [
            "semilla",
            "validacion",
            "ganancia",
            "auc"
        ]
    ]
    .rename(columns={
        "ganancia": "ganancia_carga",
        "auc": "auc_carga",
    })
)


comparacion_flujo = (
    resultados_flujo_junio
    .merge(
        referencia_carga,
        on=["semilla", "validacion"],
        how="inner"
    )
)


comparacion_flujo["delta_ganancia"] = (
    comparacion_flujo["ganancia"]
    - comparacion_flujo["ganancia_carga"]
)

comparacion_flujo["delta_auc"] = (
    comparacion_flujo["auc"]
    - comparacion_flujo["auc_carga"]
)


resumen_flujo_modelo = pd.DataFrame({
    "ganancia_carga": [
        comparacion_flujo[
            "ganancia_carga"
        ].mean()
    ],
    "ganancia_con_flujo": [
        comparacion_flujo[
            "ganancia"
        ].mean()
    ],
    "mejora_media": [
        comparacion_flujo[
            "delta_ganancia"
        ].mean()
    ],
    "mejora_minima": [
        comparacion_flujo[
            "delta_ganancia"
        ].min()
    ],
    "mejora_maxima": [
        comparacion_flujo[
            "delta_ganancia"
        ].max()
    ],
    "cambio_auc_medio": [
        comparacion_flujo[
            "delta_auc"
        ].mean()
    ],
    "semillas_favorables": [
        int(
            (
                comparacion_flujo[
                    "delta_ganancia"
                ] > 0
            ).sum()
        )
    ],
})


resumen_flujo_modelo["mejora_pct"] = (
    100.0
    * resumen_flujo_modelo[
        "mejora_media"
    ]
    / resumen_flujo_modelo[
        "ganancia_carga"
    ]
)


resumen_importancias_flujo = (
    importancias_flujo_junio
    .groupby("feature")
    .agg(
        importancia_media_pct=(
            "importancia_pct",
            "mean"
        ),
        semillas_en_que_se_uso=(
            "importancia_gain",
            lambda x: int((x > 0).sum())
        )
    )
    .sort_values(
        "importancia_media_pct",
        ascending=False
    )
)


print("Comparación por semilla:")

display(
    comparacion_flujo[
        [
            "semilla",
            "ganancia_carga",
            "ganancia",
            "delta_ganancia",
            "auc_carga",
            "auc",
            "delta_auc",
        ]
    ]
)


print("Resumen:")

display(
    resumen_flujo_modelo.round(4)
)


print("Principales features de flujo:")

display(
    resumen_importancias_flujo.head(20)
)


resultados_flujo_junio.to_csv(
    "resultados_flujo_junio.csv",
    index=False
)

importancias_flujo_junio.to_csv(
    "importancias_flujo_junio.csv",
    index=False
)

CARGA+FLUJO | Semilla 100129 | Ganancia 378,070,000 | AUC 0.9029 | Iteraciones 396
CARGA+FLUJO | Semilla 100183 | Ganancia 355,987,500 | AUC 0.9020 | Iteraciones 417
CARGA+FLUJO | Semilla 100151 | Ganancia 383,735,000 | AUC 0.9024 | Iteraciones 315
CARGA+FLUJO | Semilla 500041 | Ganancia 383,597,500 | AUC 0.9014 | Iteraciones 153
CARGA+FLUJO | Semilla 900007 | Ganancia 369,985,000 | AUC 0.9011 | Iteraciones 198
Comparación por semilla:


,semilla,ganancia_carga,ganancia,delta_ganancia,auc_carga,auc,delta_auc
0,100129,373835000.0,378070000.0,4235000.0,0.901015,0.902853,0.001838
1,100183,384917500.0,355987500.0,-28930000.0,0.899247,0.902014,0.002768
2,100151,368637500.0,383735000.0,15097500.0,0.900391,0.902390,0.002000
3,500041,385632500.0,383597500.0,-2035000.0,0.901926,0.901411,-0.000514
4,900007,381645000.0,369985000.0,-11660000.0,0.901598,0.901084,-0.000514


Resumen:


,ganancia_carga,ganancia_con_flujo,mejora_media,mejora_minima,mejora_maxima,cambio_auc_medio,semillas_favorables,mejora_pct
0,378933500.0,374275000.0,-4658500.0,-28930000.0,15097500.0,0.0011,2,-1.2294


Principales features de flujo:


,importancia_media_pct,semillas_en_que_se_uso
feature,,
ingresos_fondos,5.170024,5
egresos_fondos,2.806745,5
desaceleracion_rentabilidad,1.713374,5
saldo_cuentas_sobre_ingresos,1.619245,5
rentabilidad_sobre_ingresos,1.478852,5
egresos_sobre_ingresos,1.049404,5
comisiones_sobre_rentabilidad,0.999619,5
share_sucursal_transacciones,0.871803,5
flujo_neto,0.612603,5


El bloque de flujo se descarta completo. Aunque el AUC mejora +0.0011, la métrica relevante —ganancia— empeora:
- Ganancia: 378.933.500 → 374.275.000
- Diferencia: −4.658.500 (−1,23%)
- Solo mejora en 2/5 semillas.
- Peor caso: −28.930.000.
Esto significa que las variables de flujo ayudan un poco a ordenar globalmente los clientes, pero perjudican la parte superior del ranking donde hacemos los envíos rentables. Que tengan importancia alta no significa que aporten ganancia incremental.
Tampoco elegiría ahora solamente las variables de flujo que aparecen arriba: sería seleccionarlas mirando el mismo junio con el que evaluamos y aumentaría el sobreajuste.
Nuestro candidato queda congelado así:
- Anclaje.
- Lag 1 y deltas de actividad.
- Carga financiera sobre payroll.
- 238 variables.
- Sin fechas de tarjetas.
- Sin totales/ratios estáticos de tarjetas.
- Sin bloque de flujo.

In [160]:
# ============================================================
# CONGELAR CANDIDATO DE FEATURE ENGINEERING
# ============================================================

base_candidato_fe = base_carga_financiera.copy()
features_candidato_fe = features_lag1_carga.copy()

features_descartadas_flujo = features_nuevas_flujo.copy()

pd.Series(
    features_candidato_fe,
    name="feature"
).to_csv(
    "features_candidato_fe.csv",
    index=False
)

resultados_carga_junio.to_csv(
    "resultados_candidato_fe_junio.csv",
    index=False
)

print("Candidato de feature engineering congelado")
print("Cantidad de features:", len(features_candidato_fe))
print("Cantidad de filas:", len(base_candidato_fe))
print("Meses disponibles:", sorted(base_candidato_fe["foto_mes"].unique()))
print("Features de flujo descartadas:", len(features_descartadas_flujo))

Candidato de feature engineering congelado
Cantidad de features: 238
Cantidad de filas: 654066
Meses disponibles: [np.int64(202103), np.int64(202104), np.int64(202105), np.int64(202106)]
Features de flujo descartadas: 42


In [161]:
# ============================================================
# VALIDAR LA FUNCIÓN REPRODUCIBLE EN UNA MUESTRA
# ============================================================

import importlib
import feature_engineering

importlib.reload(feature_engineering)

features_congeladas = (
    pd.read_csv("features_candidato_fe.csv")["feature"]
    .tolist()
)

clientes_muestra = (
    base_anclaje["numero_de_cliente"]
    .drop_duplicates()
    .sample(
        n=min(
            2000,
            base_anclaje["numero_de_cliente"].nunique()
        ),
        random_state=100129
    )
)

muestra_entrada = (
    base_anclaje[
        base_anclaje["numero_de_cliente"].isin(clientes_muestra)
    ]
    .copy()
)

(
    muestra_reconstruida,
    features_lag_reconstruidas,
    features_carga_reconstruidas
) = feature_engineering.construir_features_candidato(
    muestra_entrada
)

muestra_referencia = (
    base_carga_financiera[
        base_carga_financiera["numero_de_cliente"].isin(clientes_muestra)
    ]
    .sort_values(["numero_de_cliente", "foto_mes"])
    .reset_index(drop=True)
)

claves = ["numero_de_cliente", "foto_mes"]

assert muestra_reconstruida[claves].equals(
    muestra_referencia[claves]
)

discrepancias = []

for feature in features_congeladas:

    valor_reconstruido = pd.to_numeric(
        muestra_reconstruida[feature],
        errors="coerce"
    ).to_numpy(dtype="float64")

    valor_referencia = pd.to_numeric(
        muestra_referencia[feature],
        errors="coerce"
    ).to_numpy(dtype="float64")

    if not np.allclose(
        valor_reconstruido,
        valor_referencia,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    ):
        discrepancias.append(feature)

print("Filas verificadas:", len(muestra_reconstruida))
print("Features lag/delta/evento:", len(features_lag_reconstruidas))
print("Features de carga financiera:", len(features_carga_reconstruidas))
print("Features congeladas:", len(features_congeladas))
print("Discrepancias:", len(discrepancias))

if discrepancias:
    display(
        pd.DataFrame({
            "feature_con_discrepancia": discrepancias
        })
    )
else:
    print("REPRODUCCIÓN EXACTA: la función genera el candidato validado.")

Filas verificadas: 7823
Features lag/delta/evento: 47
Features de carga financiera: 35
Features congeladas: 238
Discrepancias: 0
REPRODUCCIÓN EXACTA: la función genera el candidato validado.


In [162]:
# ============================================================
# CONSTRUIR EL CANDIDATO PARA LOS SEIS MESES
# ============================================================

import gc
import importlib
import feature_engineering

importlib.reload(feature_engineering)

# ------------------------------------------------------------
# 1. Liberar copias grandes anteriores de la memoria
#    No elimina archivos ni celdas del notebook.
# ------------------------------------------------------------

dataframes_liberados = []

for nombre, objeto in list(globals().items()):

    if (
        isinstance(objeto, pd.DataFrame)
        and len(objeto) >= 100_000
    ):
        dataframes_liberados.append(nombre)
        del globals()[nombre]

gc.collect()

print(
    "DataFrames anteriores liberados:",
    len(dataframes_liberados)
)

# ------------------------------------------------------------
# 2. Recuperar la lista congelada de 238 features
# ------------------------------------------------------------

features_candidato_final = (
    pd.read_csv("features_candidato_fe.csv")["feature"]
    .tolist()
)

features_temporales_generadas = (
    features_lag_reconstruidas
    + features_carga_reconstruidas
)

features_de_entrada = [
    feature
    for feature in features_candidato_final
    if feature not in set(features_temporales_generadas)
]

assert len(features_de_entrada) == 156
assert len(features_temporales_generadas) == 82
assert len(features_candidato_final) == 238

# ------------------------------------------------------------
# 3. Leer los seis meses
# ------------------------------------------------------------

base_anclaje_6m = %sql SELECT * FROM competencia_01_fe_anclaje_final ORDER BY numero_de_cliente, foto_mes

columnas_necesarias = (
    ["numero_de_cliente", "foto_mes", "clase_ternaria"]
    + features_de_entrada
)

columnas_necesarias = list(
    dict.fromkeys(columnas_necesarias)
)

base_anclaje_6m = (
    base_anclaje_6m[columnas_necesarias]
    .copy()
)

# Target disponible solamente en meses maduros.
base_anclaje_6m["target_baja2"] = pd.Series(
    pd.NA,
    index=base_anclaje_6m.index,
    dtype="Int8"
)

mascara_target_disponible = (
    base_anclaje_6m["clase_ternaria"].notna()
)

base_anclaje_6m.loc[
    mascara_target_disponible,
    "target_baja2"
] = (
    base_anclaje_6m.loc[
        mascara_target_disponible,
        "clase_ternaria"
    ]
    .eq("BAJA+2")
    .astype("int8")
)

# ------------------------------------------------------------
# 4. Aplicar la función validada
# ------------------------------------------------------------

(
    base_candidato_6m,
    features_lag_6m,
    features_carga_6m
) = feature_engineering.construir_features_candidato(
    base_anclaje_6m
)

del base_anclaje_6m
gc.collect()

# ------------------------------------------------------------
# 5. Auditoría
# ------------------------------------------------------------

features_faltantes = [
    feature
    for feature in features_candidato_final
    if feature not in base_candidato_6m.columns
]

cantidad_infinitos = int(
    np.isinf(
        base_candidato_6m[
            features_candidato_final
        ].to_numpy(dtype="float64")
    ).sum()
)

resumen_base_6m = (
    base_candidato_6m
    .groupby("foto_mes")
    .agg(
        clientes=("numero_de_cliente", "size"),
        clientes_con_historia=(
            "_tiene_historia_lag1",
            "sum"
        ),
        targets_disponibles=(
            "target_baja2",
            "count"
        ),
        casos_baja2=(
            "target_baja2",
            "sum"
        )
    )
)

resumen_base_6m["pct_con_historia"] = (
    100.0
    * resumen_base_6m["clientes_con_historia"]
    / resumen_base_6m["clientes"]
).round(3)

print("Filas totales:", len(base_candidato_6m))
print("Features finales:", len(features_candidato_final))
print("Features faltantes:", len(features_faltantes))
print("Valores infinitos:", cantidad_infinitos)
print("Meses:", sorted(base_candidato_6m["foto_mes"].unique()))

display(resumen_base_6m)

assert len(base_candidato_6m) == 983_061
assert len(features_candidato_final) == 238
assert len(features_faltantes) == 0
assert cantidad_infinitos == 0

print("BASE COMPLETA CONSTRUIDA CORRECTAMENTE.")

DataFrames anteriores liberados: 25
Filas totales: 983061
Features finales: 238
Features faltantes: 0
Valores infinitos: 0
Meses: [np.int64(202103), np.int64(202104), np.int64(202105), np.int64(202106), np.int64(202107), np.int64(202108)]


,clientes,clientes_con_historia,targets_disponibles,casos_baja2,pct_con_historia
foto_mes,,,,,
202103,162900,0,162900,960,0.000
202104,163284,161881,163284,1139,99.141
202105,163768,162320,163768,870,99.116
202106,164114,162625,164114,1098,99.093
202107,164348,163240,1103,0,99.326
202108,164647,163245,0,0,99.148


BASE COMPLETA CONSTRUIDA CORRECTAMENTE.


In [163]:
# ============================================================
# CORREGIR DISPONIBILIDAD TEMPORAL DEL TARGET
# ============================================================

meses_con_target_completo = [
    202103,
    202104,
    202105,
    202106
]

base_candidato_6m["target_baja2"] = pd.Series(
    pd.NA,
    index=base_candidato_6m.index,
    dtype="Int8"
)

mascara_target_completo = (
    base_candidato_6m["foto_mes"]
    .isin(meses_con_target_completo)
    & base_candidato_6m["clase_ternaria"].notna()
)

base_candidato_6m.loc[
    mascara_target_completo,
    "target_baja2"
] = (
    base_candidato_6m.loc[
        mascara_target_completo,
        "clase_ternaria"
    ]
    .eq("BAJA+2")
    .astype("int8")
)

auditoria_target = (
    base_candidato_6m
    .groupby("foto_mes")
    .agg(
        clientes=("numero_de_cliente", "size"),
        targets_disponibles=("target_baja2", "count"),
        casos_baja2=("target_baja2", "sum")
    )
)

display(auditoria_target)

assert (
    auditoria_target.loc[202103:202106, "targets_disponibles"].sum()
    == 654_066
)

assert (
    auditoria_target.loc[202103:202106, "casos_baja2"].sum()
    == 4_067
)

assert auditoria_target.loc[202107, "targets_disponibles"] == 0
assert auditoria_target.loc[202108, "targets_disponibles"] == 0

print("TARGET TEMPORAL CORREGIDO.")

,clientes,targets_disponibles,casos_baja2
foto_mes,,,
202103,162900,162900,960
202104,163284,163284,1139
202105,163768,163768,870
202106,164114,164114,1098
202107,164348,0,0
202108,164647,0,0


TARGET TEMPORAL CORREGIDO.


In [164]:
# ============================================================
# OPTUNA FINAL: 238 FEATURES, JUNIO, TRES SEMILLAS
# ============================================================

import gc
import optuna
import lightgbm as lgb

from sklearn.metrics import roc_auc_score


# ------------------------------------------------------------
# 1. División temporal correcta
# ------------------------------------------------------------

meses_train_optuna = [202103, 202104]
mes_validacion_optuna = 202106

mascara_train_optuna = (
    base_candidato_6m["foto_mes"]
    .isin(meses_train_optuna)
)

mascara_valid_optuna = (
    base_candidato_6m["foto_mes"]
    == mes_validacion_optuna
)

X_train_optuna = base_candidato_6m.loc[
    mascara_train_optuna,
    features_candidato_final
]

y_train_optuna = (
    base_candidato_6m.loc[
        mascara_train_optuna,
        "target_baja2"
    ]
    .astype("int8")
    .to_numpy()
)

X_valid_optuna = base_candidato_6m.loc[
    mascara_valid_optuna,
    features_candidato_final
]

y_valid_optuna = (
    base_candidato_6m.loc[
        mascara_valid_optuna,
        "target_baja2"
    ]
    .astype("int8")
    .to_numpy()
)

clases_valid_optuna = (
    base_candidato_6m.loc[
        mascara_valid_optuna,
        "clase_ternaria"
    ]
    .to_numpy()
)

ganancia_fila_valid_optuna = (
    ganancia_por_fila(
        clases_valid_optuna
    )
)

columnas_no_numericas = (
    X_train_optuna
    .select_dtypes(exclude="number")
    .columns
    .tolist()
)

assert columnas_no_numericas == []
assert len(X_train_optuna) == 326_184
assert len(X_valid_optuna) == 164_114

assert (
    indice_mes(mes_validacion_optuna)
    - indice_mes(max(meses_train_optuna))
    == 2
)

print(
    "Entrenamiento:",
    meses_train_optuna,
    "| Filas:",
    len(X_train_optuna)
)

print(
    "Validación:",
    mes_validacion_optuna,
    "| Filas:",
    len(X_valid_optuna)
)

print(
    "Features:",
    len(features_candidato_final)
)


# ------------------------------------------------------------
# 2. Semillas utilizadas dentro de cada trial
# ------------------------------------------------------------

semillas_optuna_final = [
    100129,
    100183,
    100151
]


# ------------------------------------------------------------
# 3. Función objetivo
# ------------------------------------------------------------

def objective_candidato_final(trial):

    params_trial = {

        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "bagging_freq": 1,

        "max_depth": trial.suggest_categorical(
            "max_depth",
            [-1, 4, 6, 8, 10, 12]
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            8,
            256,
            log=True
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.003,
            0.10,
            log=True
        ),

        "min_data_in_leaf": trial.suggest_int(
            "min_data_in_leaf",
            50,
            3000,
            log=True
        ),

        "feature_fraction": trial.suggest_float(
            "feature_fraction",
            0.25,
            1.00
        ),

        "bagging_fraction": trial.suggest_float(
            "bagging_fraction",
            0.50,
            1.00
        ),

        "lambda_l1": trial.suggest_float(
            "lambda_l1",
            1e-8,
            100.0,
            log=True
        ),

        "lambda_l2": trial.suggest_float(
            "lambda_l2",
            1e-8,
            100.0,
            log=True
        ),

        "min_gain_to_split": trial.suggest_float(
            "min_gain_to_split",
            0.0,
            5.0
        ),

        "max_bin": trial.suggest_categorical(
            "max_bin",
            [31, 63, 127, 255]
        ),
    }

    ganancias = []
    aucs = []
    iteraciones = []

    for semilla_actual in semillas_optuna_final:

        params_semilla = {
            **params_trial,

            "seed": semilla_actual,
            "feature_fraction_seed": semilla_actual,
            "bagging_seed": semilla_actual,
            "data_random_seed": semilla_actual,
        }

        dataset_train = lgb.Dataset(
            X_train_optuna,
            label=y_train_optuna
        )

        dataset_valid = lgb.Dataset(
            X_valid_optuna,
            label=y_valid_optuna,
            reference=dataset_train
        )

        modelo = lgb.train(
            params=params_semilla,
            train_set=dataset_train,
            num_boost_round=3000,
            valid_sets=[dataset_valid],
            valid_names=["validacion"],
            feval=feval_ganancia(
                ganancia_fila_valid_optuna
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        probabilidades = modelo.predict(
            X_valid_optuna,
            num_iteration=modelo.best_iteration
        )

        ganancia_actual = float(
            ganancia(
                probabilidades,
                ganancia_fila_valid_optuna
            )
        )

        auc_actual = float(
            roc_auc_score(
                y_valid_optuna,
                probabilidades
            )
        )

        ganancias.append(ganancia_actual)
        aucs.append(auc_actual)
        iteraciones.append(
            int(modelo.best_iteration)
        )

        trial.set_user_attr(
            f"ganancia_semilla_{semilla_actual}",
            ganancia_actual
        )

        trial.set_user_attr(
            f"auc_semilla_{semilla_actual}",
            auc_actual
        )

        trial.set_user_attr(
            f"iteraciones_semilla_{semilla_actual}",
            int(modelo.best_iteration)
        )

        del modelo
        del dataset_train
        del dataset_valid
        del probabilidades

    ganancia_media = float(
        np.mean(ganancias)
    )

    ganancia_minima = float(
        np.min(ganancias)
    )

    ganancia_desvio = float(
        np.std(ganancias)
    )

    auc_media = float(
        np.mean(aucs)
    )

    trial.set_user_attr(
        "ganancia_media",
        ganancia_media
    )

    trial.set_user_attr(
        "ganancia_minima",
        ganancia_minima
    )

    trial.set_user_attr(
        "ganancia_desvio",
        ganancia_desvio
    )

    trial.set_user_attr(
        "auc_media",
        auc_media
    )

    trial.set_user_attr(
        "iteraciones_mediana",
        int(np.median(iteraciones))
    )

    gc.collect()

    return ganancia_media


# ------------------------------------------------------------
# 4. Estudio persistente
# ------------------------------------------------------------

sampler_final = optuna.samplers.TPESampler(
    seed=100129
)

study_candidato_final = optuna.create_study(
    direction="maximize",
    study_name="candidato_238_junio_3semillas_v1",
    storage="sqlite:///optuna_competencia.db",
    load_if_exists=True,
    sampler=sampler_final
)


# Incorporar como primer candidato los parámetros anteriores.
parametros_referencia = {
    "max_depth": 8,
    "num_leaves": 48,
    "learning_rate": 0.011968361180129191,
    "min_data_in_leaf": 405,
    "feature_fraction": 0.30152141273199784,
    "bagging_fraction": 0.7006936995574339,
    "lambda_l1": 8.320444953703719e-05,
    "lambda_l2": 0.0006696837335733366,
    "min_gain_to_split": 1.5614099380945257,
    "max_bin": 255,
}

if len(study_candidato_final.trials) == 0:

    study_candidato_final.enqueue_trial(
        parametros_referencia
    )


trials_completos = sum(
    trial.state
    == optuna.trial.TrialState.COMPLETE
    for trial in study_candidato_final.trials
)

trials_a_ejecutar = max(
    0,
    30 - trials_completos
)

print()
print("Trials completos:", trials_completos)
print("Trials que se ejecutarán:", trials_a_ejecutar)


if trials_a_ejecutar > 0:

    study_candidato_final.optimize(
        objective_candidato_final,
        n_trials=trials_a_ejecutar,
        show_progress_bar=True
    )


# ------------------------------------------------------------
# 5. Resultado
# ------------------------------------------------------------

mejor_trial_final = (
    study_candidato_final.best_trial
)

print()
print("Mejor trial:", mejor_trial_final.number)

print(
    "Ganancia media:",
    f"{mejor_trial_final.user_attrs['ganancia_media']:,.0f}"
)

print(
    "Ganancia mínima:",
    f"{mejor_trial_final.user_attrs['ganancia_minima']:,.0f}"
)

print(
    "Desvío de ganancia:",
    f"{mejor_trial_final.user_attrs['ganancia_desvio']:,.0f}"
)

print(
    "AUC medio:",
    round(
        mejor_trial_final.user_attrs["auc_media"],
        4
    )
)

print(
    "Iteraciones medianas:",
    mejor_trial_final.user_attrs[
        "iteraciones_mediana"
    ]
)

print()
print("Mejores hiperparámetros:")

display(mejor_trial_final.params)

tabla_trials_final = (
    study_candidato_final
    .trials_dataframe()
    .sort_values(
        "value",
        ascending=False
    )
)

display(
    tabla_trials_final.head(10)
)

tabla_trials_final.to_csv(
    "optuna_candidato_238_trials.csv",
    index=False
)

Entrenamiento: [202103, 202104] | Filas: 326184
Validación: 202106 | Filas: 164114
Features: 238

Trials completos: 0
Trials que se ejecutarán: 30


Best trial: 25. Best value: 3.98814e+08: 100%|██████████| 30/30 [17:44<00:00, 35.48s/it]


Mejor trial: 25
Ganancia media: 398,814,167
Ganancia mínima: 387,172,500
Desvío de ganancia: 9,180,480
AUC medio: 0.9004
Iteraciones medianas: 130

Mejores hiperparámetros:


{'max_depth': 12,
 'num_leaves': 57,
 'learning_rate': 0.014449396240655157,
 'min_data_in_leaf': 1614,
 'feature_fraction': 0.2613620757159252,
 'bagging_fraction': 0.5276823217951291,
 'lambda_l1': 6.957248085997317e-05,
 'lambda_l2': 4.968141077185522e-06,
 'min_gain_to_split': 1.5074952598260436,
 'max_bin': 31}

,number,value,datetime_start,datetime_complete,duration,params_bagging_fraction,params_feature_fraction,params_lambda_l1,params_lambda_l2,params_learning_rate,...,user_attrs_ganancia_minima,user_attrs_ganancia_semilla_100129,user_attrs_ganancia_semilla_100151,user_attrs_ganancia_semilla_100183,user_attrs_iteraciones_mediana,user_attrs_iteraciones_semilla_100129,user_attrs_iteraciones_semilla_100151,user_attrs_iteraciones_semilla_100183,system_attrs_fixed_params,state
25,25,3.988142e+08,2026-10-05 17:40:26.787050,2026-10-05 17:40:49.364452,0 days 00:00:22.577402,0.527682,0.261362,6.957248e-05,0.000005,0.014449,...,387172500.0,409612500.0,387172500.0,399657500.0,130,129,150,130,NaN,COMPLETE
22,22,3.959175e+08,2026-10-05 17:37:11.953748,2026-10-05 17:37:57.584148,0 days 00:00:45.630400,0.550156,0.258730,1.130115e-04,0.000129,0.013222,...,391710000.0,398200000.0,391710000.0,397842500.0,225,149,376,225,NaN,COMPLETE
27,27,3.942308e+08,2026-10-05 17:41:27.377703,2026-10-05 17:41:51.856232,0 days 00:00:24.478529,0.570269,0.255346,4.770640e-07,0.000004,0.013850,...,374522500.0,404690000.0,374522500.0,403480000.0,169,141,169,178,NaN,COMPLETE
29,29,3.927183e+08,2026-10-05 17:42:19.918585,2026-10-05 17:42:49.999053,0 days 00:00:30.080468,0.661167,0.310290,5.760647e-05,0.000895,0.007144,...,385770000.0,404910000.0,385770000.0,387475000.0,271,295,271,248,NaN,COMPLETE
15,15,3.911325e+08,2026-10-05 17:33:26.350243,2026-10-05 17:33:54.919893,0 days 00:00:28.569650,0.505696,0.254881,7.768171e-06,0.000006,0.022526,...,379830000.0,390170000.0,379830000.0,403397500.0,131,102,150,131,NaN,COMPLETE
28,28,3.907383e+08,2026-10-05 17:41:51.861707,2026-10-05 17:42:19.911980,0 days 00:00:28.050273,0.575274,0.318178,1.416555e-07,0.192110,0.013470,...,375045000.0,408897500.0,375045000.0,388272500.0,183,183,241,165,NaN,COMPLETE
13,13,3.868333e+08,2026-10-05 17:32:13.005847,2026-10-05 17:32:41.538462,0 days 00:00:28.532615,0.534572,0.276582,7.361564e-04,0.000009,0.022750,...,373312500.0,393387500.0,373312500.0,393800000.0,124,124,90,127,NaN,COMPLETE
12,12,3.847250e+08,2026-10-05 17:31:47.822189,2026-10-05 17:32:12.997414,0 days 00:00:25.175225,0.555507,0.261219,2.915926e-04,0.000094,0.016377,...,371195000.0,399740000.0,371195000.0,383240000.0,138,140,125,138,NaN,COMPLETE
21,21,3.844958e+08,2026-10-05 17:36:37.765555,2026-10-05 17:37:11.947617,0 days 00:00:34.182062,0.540421,0.269894,4.031078e-04,0.000073,0.017834,...,378812500.0,387997500.0,378812500.0,386677500.0,125,125,501,124,NaN,COMPLETE
11,11,3.815992e+08,2026-10-05 17:31:29.328744,2026-10-05 17:31:47.814571,0 days 00:00:18.485827,0.562852,0.265656,3.889767e-04,0.000811,0.031666,...,379417500.0,384340000.0,381040000.0,379417500.0,51,42,51,113,NaN,COMPLETE


In [165]:
# ============================================================
# AUDITORÍA: TRIAL 25 VS TRIAL 22, CINCO SEMILLAS
# ============================================================

trials_por_numero = {
    trial.number: trial
    for trial in study_candidato_final.trials
}

candidatos_auditoria = {
    "TRIAL_25_MEJOR_MEDIA": trials_por_numero[25],
    "TRIAL_22_MAS_ROBUSTO": trials_por_numero[22],
}

semillas_auditoria = [
    100129,
    100183,
    100151,
    500041,
    900007
]

filas_auditoria = []

for nombre_candidato, trial_candidato in candidatos_auditoria.items():

    print()
    print(nombre_candidato)

    for semilla_actual in semillas_auditoria:

        params_semilla = {
            "objective": "binary",
            "metric": "None",
            "boosting_type": "gbdt",
            "boost_from_average": True,
            "feature_pre_filter": False,
            "first_metric_only": True,
            "verbosity": -1,
            "bagging_freq": 1,

            **trial_candidato.params,

            "seed": semilla_actual,
            "feature_fraction_seed": semilla_actual,
            "bagging_seed": semilla_actual,
            "data_random_seed": semilla_actual,
        }

        dataset_train = lgb.Dataset(
            X_train_optuna,
            label=y_train_optuna
        )

        dataset_valid = lgb.Dataset(
            X_valid_optuna,
            label=y_valid_optuna,
            reference=dataset_train
        )

        modelo = lgb.train(
            params=params_semilla,
            train_set=dataset_train,
            num_boost_round=3000,
            valid_sets=[dataset_valid],
            valid_names=["validacion"],
            feval=feval_ganancia(
                ganancia_fila_valid_optuna
            ),
            callbacks=[
                lgb.early_stopping(
                    stopping_rounds=150,
                    verbose=False
                ),
                lgb.log_evaluation(period=0)
            ]
        )

        probabilidades = modelo.predict(
            X_valid_optuna,
            num_iteration=modelo.best_iteration
        )

        ganancia_actual = float(
            ganancia(
                probabilidades,
                ganancia_fila_valid_optuna
            )
        )

        auc_actual = float(
            roc_auc_score(
                y_valid_optuna,
                probabilidades
            )
        )

        filas_auditoria.append({
            "candidato": nombre_candidato,
            "trial": trial_candidato.number,
            "semilla": semilla_actual,
            "ganancia": ganancia_actual,
            "auc": auc_actual,
            "iteraciones": int(modelo.best_iteration),
        })

        print(
            f"Semilla {semilla_actual} | "
            f"Ganancia {ganancia_actual:,.0f} | "
            f"AUC {auc_actual:.4f} | "
            f"Iteraciones {modelo.best_iteration}"
        )

        del modelo
        del dataset_train
        del dataset_valid
        del probabilidades

    gc.collect()


# ------------------------------------------------------------
# Comparación con el modelo anterior de carga financiera
# ------------------------------------------------------------

resultados_auditoria = pd.DataFrame(
    filas_auditoria
)

referencia_carga = (
    pd.read_csv(
        "resultados_carga_financiera_junio.csv"
    )[
        ["semilla", "ganancia", "auc"]
    ]
    .rename(columns={
        "ganancia": "ganancia_referencia",
        "auc": "auc_referencia"
    })
)

comparacion_auditoria = (
    resultados_auditoria
    .merge(
        referencia_carga,
        on="semilla",
        how="left"
    )
)

comparacion_auditoria["delta_ganancia"] = (
    comparacion_auditoria["ganancia"]
    - comparacion_auditoria["ganancia_referencia"]
)

comparacion_auditoria["delta_auc"] = (
    comparacion_auditoria["auc"]
    - comparacion_auditoria["auc_referencia"]
)


# ------------------------------------------------------------
# Resumen de robustez
# ------------------------------------------------------------

resumen_auditoria = (
    comparacion_auditoria
    .groupby(["candidato", "trial"])
    .agg(
        ganancia_media=("ganancia", "mean"),
        ganancia_minima=("ganancia", "min"),
        ganancia_maxima=("ganancia", "max"),
        ganancia_desvio=("ganancia", "std"),

        mejora_media=("delta_ganancia", "mean"),
        mejora_minima=("delta_ganancia", "min"),
        mejora_maxima=("delta_ganancia", "max"),

        auc_medio=("auc", "mean"),
        cambio_auc_medio=("delta_auc", "mean"),

        semillas_favorables=(
            "delta_ganancia",
            lambda valores: int((valores > 0).sum())
        ),

        iteraciones_mediana=(
            "iteraciones",
            "median"
        )
    )
    .sort_values(
        "ganancia_media",
        ascending=False
    )
)


display(
    comparacion_auditoria[
        [
            "candidato",
            "semilla",
            "ganancia_referencia",
            "ganancia",
            "delta_ganancia",
            "auc_referencia",
            "auc",
            "delta_auc",
            "iteraciones"
        ]
    ]
)

display(
    resumen_auditoria.round(4)
)

comparacion_auditoria.to_csv(
    "auditoria_optuna_final_5_semillas.csv",
    index=False
)

print("AUDITORÍA FINALIZADA.")


TRIAL_25_MEJOR_MEDIA
Semilla 100129 | Ganancia 409,612,500 | AUC 0.9010 | Iteraciones 129
Semilla 100183 | Ganancia 399,657,500 | AUC 0.9003 | Iteraciones 130
Semilla 100151 | Ganancia 387,172,500 | AUC 0.8999 | Iteraciones 150
Semilla 500041 | Ganancia 410,932,500 | AUC 0.9012 | Iteraciones 164
Semilla 900007 | Ganancia 411,345,000 | AUC 0.9010 | Iteraciones 143

TRIAL_22_MAS_ROBUSTO
Semilla 100129 | Ganancia 398,200,000 | AUC 0.9023 | Iteraciones 149
Semilla 100183 | Ganancia 397,842,500 | AUC 0.9031 | Iteraciones 225
Semilla 100151 | Ganancia 391,710,000 | AUC 0.9019 | Iteraciones 376
Semilla 500041 | Ganancia 391,820,000 | AUC 0.9019 | Iteraciones 185
Semilla 900007 | Ganancia 403,287,500 | AUC 0.9026 | Iteraciones 214


,candidato,semilla,ganancia_referencia,ganancia,delta_ganancia,auc_referencia,auc,delta_auc,iteraciones
0,TRIAL_25_MEJOR_MEDIA,100129,373835000.0,409612500.0,35777500.0,0.901015,0.901029,0.000014,129
1,TRIAL_25_MEJOR_MEDIA,100183,384917500.0,399657500.0,14740000.0,0.899247,0.900284,0.001038,130
2,TRIAL_25_MEJOR_MEDIA,100151,368637500.0,387172500.0,18535000.0,0.900391,0.899939,-0.000451,150
3,TRIAL_25_MEJOR_MEDIA,500041,385632500.0,410932500.0,25300000.0,0.901926,0.901247,-0.000679,164
4,TRIAL_25_MEJOR_MEDIA,900007,381645000.0,411345000.0,29700000.0,0.901598,0.901040,-0.000558,143
5,TRIAL_22_MAS_ROBUSTO,100129,373835000.0,398200000.0,24365000.0,0.901015,0.902276,0.001261,149
6,TRIAL_22_MAS_ROBUSTO,100183,384917500.0,397842500.0,12925000.0,0.899247,0.903083,0.003837,225
7,TRIAL_22_MAS_ROBUSTO,100151,368637500.0,391710000.0,23072500.0,0.900391,0.901918,0.001527,376
8,TRIAL_22_MAS_ROBUSTO,500041,385632500.0,391820000.0,6187500.0,0.901926,0.901906,-0.000019,185
9,TRIAL_22_MAS_ROBUSTO,900007,381645000.0,403287500.0,21642500.0,0.901598,0.902596,0.000998,214


,,ganancia_media,ganancia_minima,ganancia_maxima,ganancia_desvio,mejora_media,mejora_minima,mejora_maxima,auc_medio,cambio_auc_medio,semillas_favorables,iteraciones_mediana
candidato,trial,,,,,,,,,,,
TRIAL_25_MEJOR_MEDIA,25,403744000.0,387172500.0,411345000.0,1.043076e+07,24810500.0,14740000.0,35777500.0,0.9007,-0.0001,5,143.0
TRIAL_22_MAS_ROBUSTO,22,396572000.0,391710000.0,403287500.0,4.888329e+06,17638500.0,6187500.0,24365000.0,0.9024,0.0015,5,214.0


AUDITORÍA FINALIZADA.


In [166]:
# ============================================================
# CONGELAR TRIAL 25 Y MOSTRAR GRÁFICOS DE OPTUNA
# ============================================================

import json

from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_slice,
    plot_parallel_coordinate
)


# ------------------------------------------------------------
# Configuración elegida
# ------------------------------------------------------------

trial_final_elegido = trials_por_numero[25]

parametros_finales = (
    trial_final_elegido.params.copy()
)

iteraciones_finales_por_semilla = (
    comparacion_auditoria[
        comparacion_auditoria["trial"] == 25
    ]
    .set_index("semilla")["iteraciones"]
    .astype(int)
    .to_dict()
)

configuracion_modelo_final = {
    "trial": 25,
    "features": len(features_candidato_final),
    "meses_entrenamiento_final": [
        202103,
        202104,
        202105,
        202106
    ],
    "mes_prediccion": 202108,
    "semillas": semillas_auditoria,
    "parametros": parametros_finales,
    "iteraciones_por_semilla": {
        str(semilla): int(iteraciones)
        for semilla, iteraciones
        in iteraciones_finales_por_semilla.items()
    }
}

with open(
    "configuracion_modelo_final.json",
    "w",
    encoding="utf-8"
) as archivo:

    json.dump(
        configuracion_modelo_final,
        archivo,
        ensure_ascii=False,
        indent=2
    )


print("Trial elegido:", trial_final_elegido.number)
print("Parámetros finales:")

display(parametros_finales)

print("Iteraciones por semilla:")

display(
    pd.DataFrame({
        "semilla":
            iteraciones_finales_por_semilla.keys(),
        "iteraciones":
            iteraciones_finales_por_semilla.values()
    })
)


# ------------------------------------------------------------
# Gráficos de Optuna
# ------------------------------------------------------------

plot_optimization_history(
    study_candidato_final
).show()

plot_param_importances(
    study_candidato_final
).show()

plot_slice(
    study_candidato_final,
    params=[
        "learning_rate",
        "min_data_in_leaf",
        "feature_fraction",
        "bagging_fraction",
        "num_leaves",
        "max_bin"
    ]
).show()

plot_parallel_coordinate(
    study_candidato_final,
    params=[
        "learning_rate",
        "min_data_in_leaf",
        "feature_fraction",
        "bagging_fraction",
        "num_leaves",
        "max_bin"
    ]
).show()

print("CONFIGURACIÓN FINAL GUARDADA.")

Trial elegido: 25
Parámetros finales:


{'max_depth': 12,
 'num_leaves': 57,
 'learning_rate': 0.014449396240655157,
 'min_data_in_leaf': 1614,
 'feature_fraction': 0.2613620757159252,
 'bagging_fraction': 0.5276823217951291,
 'lambda_l1': 6.957248085997317e-05,
 'lambda_l2': 4.968141077185522e-06,
 'min_gain_to_split': 1.5074952598260436,
 'max_bin': 31}

Iteraciones por semilla:


,semilla,iteraciones
0,100129,129
1,100183,130
2,100151,150
3,500041,164
4,900007,143


CONFIGURACIÓN FINAL GUARDADA.


In [167]:
# ============================================================
# VALIDACIÓN DEL ENSAMBLE DE CINCO SEMILLAS EN JUNIO
# ============================================================

predicciones_validacion = {}
filas_modelos = []

for semilla_actual in semillas_auditoria:

    iteraciones_actuales = (
        iteraciones_finales_por_semilla[
            semilla_actual
        ]
    )

    params_semilla = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "bagging_freq": 1,

        **parametros_finales,

        "seed": semilla_actual,
        "feature_fraction_seed": semilla_actual,
        "bagging_seed": semilla_actual,
        "data_random_seed": semilla_actual,
    }

    dataset_train = lgb.Dataset(
        X_train_optuna,
        label=y_train_optuna
    )

    modelo = lgb.train(
        params=params_semilla,
        train_set=dataset_train,
        num_boost_round=iteraciones_actuales,
        callbacks=[
            lgb.log_evaluation(period=0)
        ]
    )

    probabilidades = modelo.predict(
        X_valid_optuna
    )

    predicciones_validacion[
        semilla_actual
    ] = probabilidades

    ganancia_individual = float(
        ganancia(
            probabilidades,
            ganancia_fila_valid_optuna
        )
    )

    auc_individual = float(
        roc_auc_score(
            y_valid_optuna,
            probabilidades
        )
    )

    filas_modelos.append({
        "semilla": semilla_actual,
        "iteraciones": iteraciones_actuales,
        "ganancia": ganancia_individual,
        "auc": auc_individual,
        "clientes_estimulados": int(
            (probabilidades >= BREAK_EVEN).sum()
        )
    })

    print(
        f"Semilla {semilla_actual} | "
        f"Ganancia {ganancia_individual:,.0f} | "
        f"AUC {auc_individual:.4f}"
    )

    del modelo
    del dataset_train


matriz_predicciones = np.column_stack([
    predicciones_validacion[semilla]
    for semilla in semillas_auditoria
])

probabilidad_promedio = (
    matriz_predicciones.mean(axis=1)
)

probabilidad_mediana = (
    np.median(
        matriz_predicciones,
        axis=1
    )
)


# ------------------------------------------------------------
# Métricas de los dos agregados
# El promedio queda predefinido como candidato principal.
# ------------------------------------------------------------

resumen_ensemble = pd.DataFrame([

    {
        "metodo": "PROMEDIO_5_SEMILLAS",
        "ganancia": float(
            ganancia(
                probabilidad_promedio,
                ganancia_fila_valid_optuna
            )
        ),
        "auc": float(
            roc_auc_score(
                y_valid_optuna,
                probabilidad_promedio
            )
        ),
        "clientes_estimulados": int(
            (
                probabilidad_promedio
                >= BREAK_EVEN
            ).sum()
        )
    },

    {
        "metodo": "MEDIANA_5_SEMILLAS",
        "ganancia": float(
            ganancia(
                probabilidad_mediana,
                ganancia_fila_valid_optuna
            )
        ),
        "auc": float(
            roc_auc_score(
                y_valid_optuna,
                probabilidad_mediana
            )
        ),
        "clientes_estimulados": int(
            (
                probabilidad_mediana
                >= BREAK_EVEN
            ).sum()
        )
    }
])


# ------------------------------------------------------------
# Guardar predicciones de validación
# ------------------------------------------------------------

validacion_ensemble = pd.DataFrame({
    "numero_de_cliente":
        base_candidato_6m.loc[
            mascara_valid_optuna,
            "numero_de_cliente"
        ].to_numpy(),

    "clase_ternaria":
        clases_valid_optuna,

    "target_baja2":
        y_valid_optuna,

    "probabilidad_promedio":
        probabilidad_promedio,

    "probabilidad_mediana":
        probabilidad_mediana,

    "desvio_entre_semillas":
        matriz_predicciones.std(axis=1)
})

for posicion, semilla in enumerate(
    semillas_auditoria
):
    validacion_ensemble[
        f"probabilidad_{semilla}"
    ] = matriz_predicciones[:, posicion]


display(
    pd.DataFrame(filas_modelos)
)

display(resumen_ensemble)

display(
    pd.DataFrame(
        matriz_predicciones,
        columns=[
            str(semilla)
            for semilla in semillas_auditoria
        ]
    ).corr().round(4)
)

validacion_ensemble.to_csv(
    "validacion_ensemble_trial25.csv",
    index=False
)

print("VALIDACIÓN DEL ENSAMBLE FINALIZADA.")

Semilla 100129 | Ganancia 409,612,500 | AUC 0.9010
Semilla 100183 | Ganancia 399,657,500 | AUC 0.9003
Semilla 100151 | Ganancia 387,172,500 | AUC 0.8999
Semilla 500041 | Ganancia 410,932,500 | AUC 0.9012
Semilla 900007 | Ganancia 411,345,000 | AUC 0.9010


,semilla,iteraciones,ganancia,auc,clientes_estimulados
0,100129,129,409612500.0,0.901029,8585
1,100183,130,399657500.0,0.900284,8427
2,100151,150,387172500.0,0.899939,8481
3,500041,164,410932500.0,0.901247,8937
4,900007,143,411345000.0,0.901040,8562


,metodo,ganancia,auc,clientes_estimulados
0,PROMEDIO_5_SEMILLAS,408732500.0,0.901335,8617
1,MEDIANA_5_SEMILLAS,415332500.0,0.901260,8537


,100129,100183,100151,500041,900007
100129,1.0000,0.9908,0.9899,0.9887,0.9907
100183,0.9908,1.0000,0.9918,0.9900,0.9920
100151,0.9899,0.9918,1.0000,0.9903,0.9920
500041,0.9887,0.9900,0.9903,1.0000,0.9900
900007,0.9907,0.9920,0.9920,0.9900,1.0000


VALIDACIÓN DEL ENSAMBLE FINALIZADA.


Ahora entrenamos los cinco modelos con marzo–junio y predecimos agosto:

In [168]:
# ============================================================
# ENTRENAMIENTO FINAL Y PREDICCIÓN DE AGOSTO
# ============================================================

import gc
from pathlib import Path


# ------------------------------------------------------------
# 1. Liberar objetos de validación que ya están guardados
# ------------------------------------------------------------

objetos_validacion = [
    "X_train_optuna",
    "X_valid_optuna",
    "y_train_optuna",
    "y_valid_optuna",
    "predicciones_validacion",
    "matriz_predicciones",
    "probabilidad_promedio",
    "probabilidad_mediana",
    "validacion_ensemble",
]

for nombre in objetos_validacion:
    globals().pop(nombre, None)

gc.collect()


# ------------------------------------------------------------
# 2. Preparar entrenamiento y scoring
# ------------------------------------------------------------

meses_entrenamiento_final = [
    202103,
    202104,
    202105,
    202106
]

mascara_entrenamiento_final = (
    base_candidato_6m["foto_mes"]
    .isin(meses_entrenamiento_final)
    & base_candidato_6m["target_baja2"].notna()
)

mascara_agosto = (
    base_candidato_6m["foto_mes"] == 202108
)

X_entrenamiento_final = (
    base_candidato_6m.loc[
        mascara_entrenamiento_final,
        features_candidato_final
    ]
    .copy()
)

y_entrenamiento_final = (
    base_candidato_6m.loc[
        mascara_entrenamiento_final,
        "target_baja2"
    ]
    .astype("int8")
    .to_numpy()
)

X_agosto = (
    base_candidato_6m.loc[
        mascara_agosto,
        features_candidato_final
    ]
    .copy()
)

clientes_agosto = (
    base_candidato_6m.loc[
        mascara_agosto,
        "numero_de_cliente"
    ]
    .to_numpy()
)


# La base completa ya puede salir de memoria.
del base_candidato_6m
gc.collect()


assert len(X_entrenamiento_final) == 654_066
assert len(y_entrenamiento_final) == 654_066
assert y_entrenamiento_final.sum() == 4_067

assert len(X_agosto) == 164_647
assert len(clientes_agosto) == 164_647
assert len(np.unique(clientes_agosto)) == 164_647

assert (
    X_entrenamiento_final
    .select_dtypes(exclude="number")
    .columns
    .tolist()
    == []
)

print(
    "Entrenamiento final:",
    len(X_entrenamiento_final),
    "filas"
)

print(
    "Casos BAJA+2:",
    int(y_entrenamiento_final.sum())
)

print(
    "Scoring agosto:",
    len(X_agosto),
    "clientes"
)

print(
    "Features:",
    len(features_candidato_final)
)


# ------------------------------------------------------------
# 3. Entrenar y guardar los cinco modelos
# ------------------------------------------------------------

directorio_modelos = Path(
    "modelos_finales_trial25"
)

directorio_modelos.mkdir(
    exist_ok=True
)

predicciones_agosto_semillas = {}

for semilla_actual in semillas_auditoria:

    iteraciones_actuales = int(
        iteraciones_finales_por_semilla[
            semilla_actual
        ]
    )

    params_semilla = {
        "objective": "binary",
        "metric": "None",
        "boosting_type": "gbdt",
        "boost_from_average": True,
        "feature_pre_filter": False,
        "first_metric_only": True,
        "verbosity": -1,
        "bagging_freq": 1,

        **parametros_finales,

        "seed": semilla_actual,
        "feature_fraction_seed": semilla_actual,
        "bagging_seed": semilla_actual,
        "data_random_seed": semilla_actual,
    }

    dataset_entrenamiento = lgb.Dataset(
        X_entrenamiento_final,
        label=y_entrenamiento_final
    )

    modelo_final = lgb.train(
        params=params_semilla,
        train_set=dataset_entrenamiento,
        num_boost_round=iteraciones_actuales,
        callbacks=[
            lgb.log_evaluation(period=0)
        ]
    )

    probabilidades_agosto = (
        modelo_final.predict(X_agosto)
    )

    predicciones_agosto_semillas[
        semilla_actual
    ] = probabilidades_agosto

    ruta_modelo = (
        directorio_modelos
        / f"modelo_trial25_semilla_{semilla_actual}.txt"
    )

    modelo_final.save_model(
        str(ruta_modelo)
    )

    print(
        f"Semilla {semilla_actual} | "
        f"Iteraciones {iteraciones_actuales} | "
        f"Modelo guardado"
    )

    del modelo_final
    del dataset_entrenamiento

    gc.collect()


# ------------------------------------------------------------
# 4. Construir los ensambles
# ------------------------------------------------------------

matriz_agosto = np.column_stack([
    predicciones_agosto_semillas[semilla]
    for semilla in semillas_auditoria
])

probabilidad_agosto_promedio = (
    matriz_agosto.mean(axis=1)
)

probabilidad_agosto_mediana = (
    np.median(
        matriz_agosto,
        axis=1
    )
)

desvio_agosto = (
    matriz_agosto.std(axis=1)
)


# ------------------------------------------------------------
# 5. Guardar probabilidades
# ------------------------------------------------------------

predicciones_agosto = pd.DataFrame({
    "numero_de_cliente":
        clientes_agosto,

    "probabilidad_promedio":
        probabilidad_agosto_promedio,

    "probabilidad_mediana":
        probabilidad_agosto_mediana,

    "desvio_entre_semillas":
        desvio_agosto,

    "estimular_promedio":
        (
            probabilidad_agosto_promedio
            >= BREAK_EVEN
        ).astype("int8"),

    "estimular_mediana":
        (
            probabilidad_agosto_mediana
            >= BREAK_EVEN
        ).astype("int8"),
})

for posicion, semilla in enumerate(
    semillas_auditoria
):
    predicciones_agosto[
        f"probabilidad_{semilla}"
    ] = matriz_agosto[:, posicion]


# ------------------------------------------------------------
# 6. Auditoría de las predicciones
# ------------------------------------------------------------

assert len(predicciones_agosto) == 164_647

assert (
    predicciones_agosto[
        "numero_de_cliente"
    ].duplicated().sum()
    == 0
)

assert np.isfinite(
    probabilidad_agosto_promedio
).all()

assert (
    (probabilidad_agosto_promedio >= 0)
    & (probabilidad_agosto_promedio <= 1)
).all()

resumen_prediccion_agosto = pd.DataFrame({

    "metodo": [
        "PROMEDIO",
        "MEDIANA"
    ],

    "clientes_estimulados": [
        int(
            predicciones_agosto[
                "estimular_promedio"
            ].sum()
        ),
        int(
            predicciones_agosto[
                "estimular_mediana"
            ].sum()
        )
    ],

    "pct_clientes_estimulados": [
        round(
            100.0
            * predicciones_agosto[
                "estimular_promedio"
            ].mean(),
            3
        ),
        round(
            100.0
            * predicciones_agosto[
                "estimular_mediana"
            ].mean(),
            3
        )
    ],

    "probabilidad_mediana": [
        float(
            np.median(
                probabilidad_agosto_promedio
            )
        ),
        float(
            np.median(
                probabilidad_agosto_mediana
            )
        )
    ],

    "probabilidad_p99": [
        float(
            np.quantile(
                probabilidad_agosto_promedio,
                0.99
            )
        ),
        float(
            np.quantile(
                probabilidad_agosto_mediana,
                0.99
            )
        )
    ]
})

display(resumen_prediccion_agosto)

predicciones_agosto.to_csv(
    "predicciones_agosto_trial25_5semillas.csv",
    index=False
)

print(
    "PREDICCIONES DE AGOSTO GUARDADAS CORRECTAMENTE."
)

Entrenamiento final: 654066 filas
Casos BAJA+2: 4067
Scoring agosto: 164647 clientes
Features: 238
Semilla 100129 | Iteraciones 129 | Modelo guardado
Semilla 100183 | Iteraciones 130 | Modelo guardado
Semilla 100151 | Iteraciones 150 | Modelo guardado
Semilla 500041 | Iteraciones 164 | Modelo guardado
Semilla 900007 | Iteraciones 143 | Modelo guardado


,metodo,clientes_estimulados,pct_clientes_estimulados,probabilidad_mediana,probabilidad_p99
0,PROMEDIO,8349,5.071,0.001679,0.079188
1,MEDIANA,8276,5.027,0.001687,0.079309


PREDICCIONES DE AGOSTO GUARDADAS CORRECTAMENTE.
